In [1]:
import re
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from Bio import SeqIO, Entrez
import py3Dmol

from dotenv import load_dotenv
env_file = '/home/yuan/bio/bio_omics/src/.env'
load_dotenv(dotenv_path = env_file)

%load_ext autoreload
%autoreload 2

src_dir = os.path.dirname(os.getcwd())
print('src direcotry is ', src_dir)
bioomics_dir = '/home/yuan/bio/bio_omics/src'
for _dir in (src_dir, bioomics_dir):
    if _dir not in sys.path:
        sys.path.append(_dir)
print(sys.path)

src direcotry is  /home/yuan/bio/predict_antibody
['/home/yuan/bio/anaconda3/envs/simulate2/lib/python312.zip', '/home/yuan/bio/anaconda3/envs/simulate2/lib/python3.12', '/home/yuan/bio/anaconda3/envs/simulate2/lib/python3.12/lib-dynload', '', '/home/yuan/bio/anaconda3/envs/simulate2/lib/python3.12/site-packages', '/home/yuan/bio/predict_antibody', '/home/yuan/bio/bio_omics/src']


In [2]:
from bioomics import QueryComplex, DistanceAnalyze, DistanceProcess, ProcessPickle
from layout import Layout
from plot_predict import PlotPredict
from plot_binding import PlotBinding
from load_data import LoadData

/home/yuan/bio/predict_antibody/src/plot_binding.py:53: SyntaxWarning: invalid escape sequence '\A'
  ax.set_xlabel('Minimum distance of Ca, -$\AA$')
/home/yuan/bio/predict_antibody/src/plot_binding.py:77: SyntaxWarning: invalid escape sequence '\A'
  ax.set_xlabel(f"Minimum distance of the top {col} ranked Ca, $\AA$")


In [6]:
infile = '/home/yuan/bio/ANARCI/Example_scripts_and_sequences/12e8_H.csv'
data = pd.read_csv(infile)
data.iloc[0].to_dict()

{'Id': '12E8:H|PDBID|CHAIN|SEQUENCE',
 'domain_no': 0,
 'hmm_species': 'mouse',
 'chain_type': 'H',
 'e-value': 3.3e-54,
 'score': 173.4,
 'seqstart_index': 0,
 'seqend_index': 119,
 'identity_species': nan,
 'v_gene': nan,
 'v_identity': 0.0,
 'j_gene': nan,
 'j_identity': 0.0,
 '1': 'E',
 '2': 'V',
 '3': 'Q',
 '4': 'L',
 '5': 'Q',
 '6': 'Q',
 '7': 'S',
 '8': 'G',
 '9': 'A',
 '10': '-',
 '11': 'E',
 '12': 'V',
 '13': 'V',
 '14': 'R',
 '15': 'S',
 '16': 'G',
 '17': 'A',
 '18': 'S',
 '19': 'V',
 '20': 'K',
 '21': 'L',
 '22': 'S',
 '23': 'C',
 '24': 'T',
 '25': 'A',
 '26': 'S',
 '27': 'G',
 '28': 'F',
 '29': 'N',
 '30': 'I',
 '31': '-',
 '32': '-',
 '33': '-',
 '34': '-',
 '35': 'K',
 '36': 'D',
 '37': 'Y',
 '38': 'Y',
 '39': 'I',
 '40': 'H',
 '41': 'W',
 '42': 'V',
 '43': 'K',
 '44': 'Q',
 '45': 'R',
 '46': 'P',
 '47': 'E',
 '48': 'K',
 '49': 'G',
 '50': 'L',
 '51': 'E',
 '52': 'W',
 '53': 'I',
 '54': 'G',
 '55': 'W',
 '56': 'I',
 '57': 'D',
 '58': 'P',
 '59': 'E',
 '60': '-',
 '61': '-

In [12]:
data.iloc[0][13:].str.cat(sep='')

'DIVMTQSQKFMSTSVGDRVSITCKASQNV------GTAVAWYQQKPGQSPKLMIYSA-------SNRYTGVP-DRFTGSG--SGTDFTLTISNMQSEDLADYFCQQYSS----YPLTFGAGTKLELK'

In [7]:
infile = '/home/yuan/bio/ANARCI/Example_scripts_and_sequences/12e8_KL.csv'
data = pd.read_csv(infile)
data.iloc[0].to_dict()

{'Id': '12E8:L|PDBID|CHAIN|SEQUENCE',
 'domain_no': 0,
 'hmm_species': 'mouse',
 'chain_type': 'K',
 'e-value': 3.3999999999999995e-53,
 'score': 170.2,
 'seqstart_index': 0,
 'seqend_index': 106,
 'identity_species': nan,
 'v_gene': nan,
 'v_identity': 0.0,
 'j_gene': nan,
 'j_identity': 0.0,
 '1': 'D',
 '2': 'I',
 '3': 'V',
 '4': 'M',
 '5': 'T',
 '6': 'Q',
 '7': 'S',
 '8': 'Q',
 '9': 'K',
 '10': 'F',
 '11': 'M',
 '12': 'S',
 '13': 'T',
 '14': 'S',
 '15': 'V',
 '16': 'G',
 '17': 'D',
 '18': 'R',
 '19': 'V',
 '20': 'S',
 '21': 'I',
 '22': 'T',
 '23': 'C',
 '24': 'K',
 '25': 'A',
 '26': 'S',
 '27': 'Q',
 '28': 'N',
 '29': 'V',
 '30': '-',
 '31': '-',
 '32': '-',
 '33': '-',
 '34': '-',
 '35': '-',
 '36': 'G',
 '37': 'T',
 '38': 'A',
 '39': 'V',
 '40': 'A',
 '41': 'W',
 '42': 'Y',
 '43': 'Q',
 '44': 'Q',
 '45': 'K',
 '46': 'P',
 '47': 'G',
 '48': 'Q',
 '49': 'S',
 '50': 'P',
 '51': 'K',
 '52': 'L',
 '53': 'M',
 '54': 'I',
 '55': 'Y',
 '56': 'S',
 '57': 'A',
 '58': '-',
 '59': '-',
 '60':

In [13]:
all_germlines = {
    'J': {
        'H': {
            'human': {
                'IGHJ2*01': '------------------------------------------------------------------------------------------------------------------FDLWGRGTLVTVSS',
                'IGHJ5*04': '------------------------------------------------------------------------------------------------------------------FDPWGQGTLVSVSS',
                'IGHJ5*01': '------------------------------------------------------------------------------------------------------------------FDSWGQGTLVTVSS',
                'IGHJ5*02': '------------------------------------------------------------------------------------------------------------------FDPWGQGTLVTVSS',
                'IGHJ1*01': '------------------------------------------------------------------------------------------------------------------FQHWGQGTLVTVSS',
                'IGHJ4*01': '------------------------------------------------------------------------------------------------------------------FDYWGQGTLVTVSS',
                'IGHJ4*02': '------------------------------------------------------------------------------------------------------------------FDYWGQGTLVTVSS',
                'IGHJ4*03': '------------------------------------------------------------------------------------------------------------------FDYWGQGTLVTVSS',
                'IGHJ6*01': '------------------------------------------------------------------------------------------------------------------MDVWGQGTTVTVSS',
                'IGHJ6*02': '------------------------------------------------------------------------------------------------------------------MDVWGQGTTVTVSS',
                'IGHJ6*03': '------------------------------------------------------------------------------------------------------------------MDVWGKGTTVTVSS',
                'IGHJ6*04': '------------------------------------------------------------------------------------------------------------------MDVWGKGTTVTVSS',
                'IGHJ3*01': '------------------------------------------------------------------------------------------------------------------FDVWGQGTMVTVSS',
                'IGHJ3*02': '------------------------------------------------------------------------------------------------------------------FDIWGQGTMVTVSS'
            },
            'mouse': {
                'IGHJ1*01': '------------------------------------------------------------------------------------------------------------------FDVWGAGTTVTVSS',
                'IGHJ1*02': '------------------------------------------------------------------------------------------------------------------FDVWGAGTTVTVSS',
                'IGHJ1*03': '------------------------------------------------------------------------------------------------------------------FDVWGTGTTVTVSS',
                'IGHJ2*02': '------------------------------------------------------------------------------------------------------------------FDYWGQGTSLTVSS',
                'IGHJ2*03': '------------------------------------------------------------------------------------------------------------------FDYWGQGTSLTVSS',
                'IGHJ2*01': '------------------------------------------------------------------------------------------------------------------FDYWGQGTTLTVSS',
                'IGHJ3*01': '------------------------------------------------------------------------------------------------------------------FAYWGQGTLVTVSA',
                'IGHJ4*01': '------------------------------------------------------------------------------------------------------------------MDYWGQGTSVTVSS'
            },
            'rabbit': {
                'IGHJ1*01': '------------------------------------------------------------------------------------------------------------------LDPWGTGTLVTISS',
                'IGHJ4*01': '------------------------------------------------------------------------------------------------------------------FNLWGPGTLVTVSS',
                'IGHJ4*02': '------------------------------------------------------------------------------------------------------------------FNIWGPGTLVTVSS',
                'IGHJ3*02': '------------------------------------------------------------------------------------------------------------------LDPWGQGTLVTVSS',
                'IGHJ3*01': '------------------------------------------------------------------------------------------------------------------LDLWGQGTLVTVSS',
                'IGHJ5*01': '------------------------------------------------------------------------------------------------------------------LDLWGQGTLVTVSS',
                'IGHJ5*02': '------------------------------------------------------------------------------------------------------------------LDLWGQGTLVTVSS',
                'IGHJ2*01': '------------------------------------------------------------------------------------------------------------------FDPWGPGTLVTVSS',
                'IGHJ2*02': '------------------------------------------------------------------------------------------------------------------FDPWGPGTLVTVSS',
                'IGHJ6*01': '------------------------------------------------------------------------------------------------------------------MDLWGPGTLVTVSS',
                'IGHJ6*02': '------------------------------------------------------------------------------------------------------------------MDPWGPGTLVTVSS'
            },
            'rhesus': {
                'IGHJ6*01': '------------------------------------------------------------------------------------------------------------------LDSWGQGVVVTVSS',
                'IGHJ2*01': '------------------------------------------------------------------------------------------------------------------FDLWGPGTPITISS',
                'IGHJ1*01': '------------------------------------------------------------------------------------------------------------------FEFWGQGALVTVSS',
                'IGHJ1*02': '------------------------------------------------------------------------------------------------------------------FEFWGQGALVTVSS',
                'IGHJ4*01': '------------------------------------------------------------------------------------------------------------------FDYWGQGVLVTVSS',
                'IGHJ5-1*02': '------------------------------------------------------------------------------------------------------------------FDVWGPGVLVTVSS',
                'IGHJ5-1*01': '------------------------------------------------------------------------------------------------------------------FDVWGPGVLVTVSS',
                'IGHJ5-1*03': '------------------------------------------------------------------------------------------------------------------FDVWGAGVLVTVSS',
                'IGHJ5-2*01': '------------------------------------------------------------------------------------------------------------------LDVWGQGVLVTVSS',
                'IGHJ5-2*02': '------------------------------------------------------------------------------------------------------------------LDVWGRGVLVTVSS',
                'IGHJ3*01': '------------------------------------------------------------------------------------------------------------------FDFWGQGLRVTVSS'
            },
            'pig': {
                'IGHJ4*01': '------------------------------------------------------------------------------------------------------------------LESWGQGTLVYDAS',
                'IGHJ3*01': '------------------------------------------------------------------------------------------------------------------LHSWGRGVEVTVSS',
                'IGHJ5*01': '------------------------------------------------------------------------------------------------------------------MDLWGPGVEVVVSS',
                'IGHJ1*01': '------------------------------------------------------------------------------------------------------------------LDSWGQGILVTVSS',
                'IGHJ2*01': '------------------------------------------------------------------------------------------------------------------LDHWGRGVLVTVSS'
            },
            'alpaca': {
                'IGHJ6*01': '------------------------------------------------------------------------------------------------------------------FGSWGQGTQVTVSS',
                'IGHJ4*01': '------------------------------------------------------------------------------------------------------------------YDYWGQGTQVTVSS',
                'IGHJ2*01': '------------------------------------------------------------------------------------------------------------------LEVWGQGTLVTVSS',
                'IGHJ5*01': '------------------------------------------------------------------------------------------------------------------FEYWGQGTLVTVS-',
                'IGHJ3*01': '------------------------------------------------------------------------------------------------------------------LDAWGQGTLVTVSS',
                'IGHJ7*01': '------------------------------------------------------------------------------------------------------------------MDYWGKGTLVTVSS'
            },
            'cow': {
                'IGHJ1-4*01': '------------------------------------------------------------------------------------------------------------------FDNWGPGIQNTVSS',
                'IGHJ1-6*01': '------------------------------------------------------------------------------------------------------------------IDAWGRGLRVTVSS',
                'IGHJ2-4*01': '------------------------------------------------------------------------------------------------------------------VDAWGQGLLVTVSS'
            }
        },
        'K': {
            'human': {
                'IGKJ3*01': '-------------------------------------------------------------------------------------------------------------------FTFGPGTKVDIK-',
                'IGKJ4*01': '-------------------------------------------------------------------------------------------------------------------LTFGGGTKVEIK-',
                'IGKJ1*01': '-------------------------------------------------------------------------------------------------------------------WTFGQGTKVEIK-',
                'IGKJ5*01': '-------------------------------------------------------------------------------------------------------------------ITFGQGTRLEIK-',
                'IGKJ2*04': '-------------------------------------------------------------------------------------------------------------------CSFGQGTKLEIK-',
                'IGKJ2*01': '-------------------------------------------------------------------------------------------------------------------YTFGQGTKLEIK-'
            },
            'mouse': {
                'IGKJ5*01': '-------------------------------------------------------------------------------------------------------------------LTFGAGTKLELK-',
                'IGKJ1*02': '-------------------------------------------------------------------------------------------------------------------PTFGGGTKLEIN-',
                'IGKJ4*02': '-------------------------------------------------------------------------------------------------------------------FTFGTGTKLEIK-',
                'IGKJ4*01': '-------------------------------------------------------------------------------------------------------------------FTFGSGTKLEIK-',
                'IGKJ2*02': '-------------------------------------------------------------------------------------------------------------------YTFGSGTKLEMK-',
                'IGKJ2*03': '-------------------------------------------------------------------------------------------------------------------YTFGSGTKLEIK-',
                'IGKJ1*01': '-------------------------------------------------------------------------------------------------------------------WTFGGGTKLEIK-',
                'IGKJ2*01': '-------------------------------------------------------------------------------------------------------------------YTFGGGTKLEIK-'
            },
            'rat': {
                'IGKJ2-2*01': '-------------------------------------------------------------------------------------------------------------------DTFGAGTKLELK-',
                'IGKJ2-1*01': '-------------------------------------------------------------------------------------------------------------------NTFGAGTKLELK-',
                'IGKJ2-3*01': '-------------------------------------------------------------------------------------------------------------------YTFGAGTKLELK-',
                'IGKJ5*01': '-------------------------------------------------------------------------------------------------------------------LTFGSGTKLEIK-',
                'IGKJ4*01': '-------------------------------------------------------------------------------------------------------------------FTFGSGTKLEIK-',
                'IGKJ1*01': '-------------------------------------------------------------------------------------------------------------------WTFGGGTKLELK-'
            },
            'rabbit': {
                'IGKJ1-2*01': '------------------------------------------------------------------------------------------------------------------YNAFGGGTEVVVK-',
                'IGKJ1-2*02': '------------------------------------------------------------------------------------------------------------------YNAFGGGTEVVVK-',
                'IGKJ1-2*03': '------------------------------------------------------------------------------------------------------------------YNTFGGGTKVVVE-',
                'IGKJ1-1*03': '-------------------------------------------------------------------------------------------------------------------WAFGAGTNVEIK-',
                'IGKJ2-3*01': '-------------------------------------------------------------------------------------------------------------------ITFGKGTKLEIK-',
                'IGKJ2-2*01': '------------------------------------------------------------------------------------------------------------------SNTFGAGTKVEIK-',
                'IGKJ2-1*01': '-------------------------------------------------------------------------------------------------------------------LTFGAGTKVEIK-'},
                'rhesus': {
                    'IGKJ1*01': '-------------------------------------------------------------------------------------------------------------------WTFGQGTKVEIK-'},
                'pig': {'IGKJ5*01': '-------------------------------------------------------------------------------------------------------------------ITFGEGTSVEIE-',
                'IGKJ5*02': '-------------------------------------------------------------------------------------------------------------------ITFGEGTSVEIE-',
                'IGKJ2*02': '-------------------------------------------------------------------------------------------------------------------NGFGAGTKLELK-',
                'IGKJ2*01': '-------------------------------------------------------------------------------------------------------------------YGFGAGTKLELK-',
                'IGKJ3*01': '-------------------------------------------------------------------------------------------------------------------FTFGSGTKVEPK-',
                'IGKJ4*01': '-------------------------------------------------------------------------------------------------------------------VVFGSGTKLEIK-',
                'IGKJ4*02': '-------------------------------------------------------------------------------------------------------------------VVFGSGTKLEIK-',
                'IGKJ1*01': '-------------------------------------------------------------------------------------------------------------------WTFGQGTKLELK-'},

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'cow': {'IGKJ2*01': '-------------------------------------------------------------------------------------------------------------------NTFGQGTKVEIK-'}},
                'L': {'human': {'IGLJ2A*01': '---------------------------------------------------------------------------------------------------------------------CGIRRRDQADR',
                'IGLJ1*01': '-------------------------------------------------------------------------------------------------------------------YVFGTGTKVTVL-',
                'IGLJ6*01': '-------------------------------------------------------------------------------------------------------------------NVFGSGTKVTVL-',
                'IGLJ7*02': '-------------------------------------------------------------------------------------------------------------------AVFGGGTQLTAL-',
                'IGLJ7*01': '-------------------------------------------------------------------------------------------------------------------AVFGGGTQLTVL-',
                'IGLJ2*01': '-------------------------------------------------------------------------------------------------------------------VVFGGGTKLTVL-',
                'IGLJ3*01': '-------------------------------------------------------------------------------------------------------------------VVFGGGTKLTVL-',
                'IGLJ3*02': '-------------------------------------------------------------------------------------------------------------------WVFGGGTKLTVL-'},
                'mouse': {'IGLJ3*01': '-------------------------------------------------------------------------------------------------------------------FIFGSGTKVTVL-',
                'IGLJ2*01': '-------------------------------------------------------------------------------------------------------------------YVFGGGTKVTVL-',
                'IGLJ1*01': '-------------------------------------------------------------------------------------------------------------------WVFGGGTKLTVL-'},
                'rat': {'IGLJ1*02': '-------------------------------------------------------------------------------------------------------------------YIFGGGTKLTVL-',
                'IGLJ1*01': '-------------------------------------------------------------------------------------------------------------------PVFGGGTKLTVL-',
                'IGLJ3*01': '-------------------------------------------------------------------------------------------------------------------PVFGGGTKLTVL-'},
                'rabbit': {'IGLJ5*01': '-------------------------------------------------------------------------------------------------------------------YVFGGGTQLTVT-',
                'IGLJ6*01': '-------------------------------------------------------------------------------------------------------------------VVFGGGTQLTVT-'},
                'rhesus': {'IGLJ6*01': '-------------------------------------------------------------------------------------------------------------------DVFGSGTKLTVL-',
                'IGLJ1*01': '-------------------------------------------------------------------------------------------------------------------YIFGAGTRLTVL-',
                'IGLJ3*01': '-------------------------------------------------------------------------------------------------------------------VLFGGGTRLTVL-',
                'IGLJ2*01': '-------------------------------------------------------------------------------------------------------------------GLFGGGTRLTVL-',
                'IGLJ5*01': '-------------------------------------------------------------------------------------------------------------------WVFGEGTKLTIL-',
                'IGLJ2A*01': '-------------------------------------------------------------------------------------------------------------------WVFGGGTRLTVL-',
                'IGLJ7*01': '-------------------------------------------------------------------------------------------------------------------VMFGRGTRLTDI-'},
                'pig': {'IGLJ4*01': '-------------------------------------------------------------------------------------------------------------------DRFGRGTRLSVL-',
                'IGLJ2*01': '-------------------------------------------------------------------------------------------------------------------NIFGGGTHLTVL-',
                'IGLJ2*02': '-------------------------------------------------------------------------------------------------------------------NIFGGGTHLTVL-',
                'IGLJ3*01': '-------------------------------------------------------------------------------------------------------------------VPFGGGTHLTVL-'},
                'cow': {'IGLJ4*01': '-------------------------------------------------------------------------------------------------------------------AVFGSGTTLTVL-',
                'IGLJ7*01': '-------------------------------------------------------------------------------------------------------------------AVFGSGTTLTVL-',
                'IGLJ8*01': '-------------------------------------------------------------------------------------------------------------------AVFGSGTTLTVL-',
                'IGLJ3*01': '-------------------------------------------------------------------------------------------------------------------DLFGGGTTVTVL-',
                'IGLJ2*01': '-------------------------------------------------------------------------------------------------------------------DLFGGGTRVTVL-'}},
                'A': {'human': {'TRAJ16*01': '------------------------------------------------------------------------------------------------------------------KLLFARGTMLKVDL',
                'TRAJ16*02': '------------------------------------------------------------------------------------------------------------------KLLFARGTMLKVDL',
                'TRAJ10*01': '------------------------------------------------------------------------------------------------------------------KLTFGTGTQLKVEL',
                'TRAJ44*01': '------------------------------------------------------------------------------------------------------------------KLTFGTGTRLQVTL',
                'TRAJ56*01': '------------------------------------------------------------------------------------------------------------------KLTFGKGITLSVRP',
                'TRAJ11*01': '------------------------------------------------------------------------------------------------------------------TLTFGKGTMLLVSP',
                'TRAJ41*01': '------------------------------------------------------------------------------------------------------------------ALNFGKGTSLLVTP',
                'TRAJ8*01': '------------------------------------------------------------------------------------------------------------------KLVFGTGTRLLVSP',
                'TRAJ49*01': '------------------------------------------------------------------------------------------------------------------QFYFGTGTSLTVIP',
                'TRAJ18*01': '------------------------------------------------------------------------------------------------------------------RLYFGRGTQLTVWP',
                'TRAJ46*01': '------------------------------------------------------------------------------------------------------------------KLTFGTGTRLAVRP',
                'TRAJ30*01': '------------------------------------------------------------------------------------------------------------------KIIFGKGTRLHILP',
                'TRAJ54*01': '------------------------------------------------------------------------------------------------------------------KLVFGQGTRLTINP',
                'TRAJ36*01': '------------------------------------------------------------------------------------------------------------------NLFFGTGTRLTVIP',
                'TRAJ12*01': '------------------------------------------------------------------------------------------------------------------KLIFGSGTRLLVRP',
                'TRAJ13*01': '------------------------------------------------------------------------------------------------------------------KVTFGIGTKLQVIP',
                'TRAJ13*02': '------------------------------------------------------------------------------------------------------------------KVTFGTGTKLQVIP',
                'TRAJ20*01': '------------------------------------------------------------------------------------------------------------------KLSFGAGTTVTVRA',
                'TRAJ4*01': '------------------------------------------------------------------------------------------------------------------KLIFGAGTRLAVHP',
                'TRAJ29*01': '------------------------------------------------------------------------------------------------------------------PLVFGKGTRLSVIA',
                'TRAJ43*01': '------------------------------------------------------------------------------------------------------------------DMRFGAGTRLTVKP',
                'TRAJ17*01': '------------------------------------------------------------------------------------------------------------------KLTFGGGTRVLVKP',
                'TRAJ39*01': '------------------------------------------------------------------------------------------------------------------MLTFGGGTRLMVKP',
                'TRAJ57*02': '------------------------------------------------------------------------------------------------------------------KLVFGKGMKLTVNP',
                'TRAJ57*01': '------------------------------------------------------------------------------------------------------------------KLVFGKGTKLTVNP',
                'TRAJ26*01': '------------------------------------------------------------------------------------------------------------------NFVFGPGTRLSVLP',
                'TRAJ9*01': '------------------------------------------------------------------------------------------------------------------KTIFGAGTRLFVKA',
                'TRAJ22*01': '------------------------------------------------------------------------------------------------------------------QLTFGSGTQLTVLP',
                'TRAJ40*01': '------------------------------------------------------------------------------------------------------------------KYIFGTGTRLKVLA',
                'TRAJ7*01': '------------------------------------------------------------------------------------------------------------------RLAFGKGNQVVVIP',
                'TRAJ50*01': '------------------------------------------------------------------------------------------------------------------KVIFGPGTSLSVIP',
                'TRAJ15*01': '------------------------------------------------------------------------------------------------------------------ALIFGKGTTLSVSS',
                'TRAJ15*02': '------------------------------------------------------------------------------------------------------------------ALIFGKGTHLSVSS',
                'TRAJ45*01': '------------------------------------------------------------------------------------------------------------------GLTFGKGTHLIIQP',
                'TRAJ31*01': '------------------------------------------------------------------------------------------------------------------RLMFGDGTQLVVKP',
                'TRAJ38*01': '------------------------------------------------------------------------------------------------------------------KLIWGLGTSLAVNP',
                'TRAJ24*02': '------------------------------------------------------------------------------------------------------------------KLQFGAGTQVVVTP',
                'TRAJ24*01': '------------------------------------------------------------------------------------------------------------------KFEFGAGTQVVVTP',
                'TRAJ24*03': '------------------------------------------------------------------------------------------------------------------KFQFGAGTQVVVTP',
                'TRAJ35*01': '------------------------------------------------------------------------------------------------------------------VLHCGSGTQVIVLP',
                'TRAJ6*01': '------------------------------------------------------------------------------------------------------------------IPTFGRGTSLIVHP',
                'TRAJ28*01': '------------------------------------------------------------------------------------------------------------------QLTFGKGTKLSVIP',
                'TRAJ47*01': '------------------------------------------------------------------------------------------------------------------KLVFGAGTILRVKS',
                'TRAJ47*02': '------------------------------------------------------------------------------------------------------------------KLVFGAGTILRVKS',
                'TRAJ14*01': '------------------------------------------------------------------------------------------------------------------TFIFGSGTRLSVKP',
                'TRAJ3*01': '------------------------------------------------------------------------------------------------------------------KIIFGSGTRLSIRP',
                'TRAJ5*01': '------------------------------------------------------------------------------------------------------------------ALTFGSGTRLQVQP',
                'TRAJ52*01': '------------------------------------------------------------------------------------------------------------------KLTFGQGTILTVHP',
                'TRAJ27*01': '------------------------------------------------------------------------------------------------------------------KSTFGDGTTLTVKP',
                'TRAJ37*01': '------------------------------------------------------------------------------------------------------------------KLIFGQGTTLQVKP',
                'TRAJ37*02': '------------------------------------------------------------------------------------------------------------------KLIFGQGTTLQVKP',
                'TRAJ34*01': '------------------------------------------------------------------------------------------------------------------KLIFGTGTRLQVFP',
                'TRAJ23*01': '------------------------------------------------------------------------------------------------------------------KLIFGQGTELSVKP',
                'TRAJ23*02': '------------------------------------------------------------------------------------------------------------------KLIFGQGTELSVKP',
                'TRAJ48*01': '------------------------------------------------------------------------------------------------------------------KLTFGTGTRLTIIP',
                'TRAJ53*01': '------------------------------------------------------------------------------------------------------------------KLTFGKGTLLTVNP',
                'TRAJ33*01': '------------------------------------------------------------------------------------------------------------------QLIWGAGTKLIIKP',
                'TRAJ21*01': '------------------------------------------------------------------------------------------------------------------KFYFGSGTKLNVKP',
                'TRAJ42*01': '------------------------------------------------------------------------------------------------------------------NLIFGKGTKLSVKP',
                'TRAJ32*01': '------------------------------------------------------------------------------------------------------------------KLIFGTGTLLAVQP',
                'TRAJ32*02': '------------------------------------------------------------------------------------------------------------------KLIFGTGTLLAVQP'},
                'mouse': {'TRAJ16*01': '------------------------------------------------------------------------------------------------------------------KLVFGQGTILKVYL',
                'TRAJ50*01': '------------------------------------------------------------------------------------------------------------------KLVFGQGTSLSVVP',
                'TRAJ56*01': '------------------------------------------------------------------------------------------------------------------KLTFGQGTVLSVIP',
                'TRAJ11*01': '------------------------------------------------------------------------------------------------------------------KLTFGKGTVLLVSP',
                'TRAJ9*01': '------------------------------------------------------------------------------------------------------------------KLTFGTGTSLLVDP',
                'TRAJ9*02': '------------------------------------------------------------------------------------------------------------------KLTFGTGTSLLVDP',
                'TRAJ12*01': '------------------------------------------------------------------------------------------------------------------KVVFGSGTRLLVSP',
                'TRAJ12*02': '------------------------------------------------------------------------------------------------------------------KVVFGSGTRLLVSP',
                'TRAJ49*01': '------------------------------------------------------------------------------------------------------------------NFYFGKGTSLTVIP',
                'TRAJ18*01': '------------------------------------------------------------------------------------------------------------------RLHFGAGTQLIVIP',
                'TRAJ2*01': '------------------------------------------------------------------------------------------------------------------KLTFGEGTQVTVIS',
                'TRAJ2*02': '------------------------------------------------------------------------------------------------------------------KLTFGEGTQVTVIS',
                'TRAJ58*01': '------------------------------------------------------------------------------------------------------------------KLSFGKGAKLTVSP',
                'TRAJ5*01': '------------------------------------------------------------------------------------------------------------------QLTFGRGTRLQVYA',
                'TRAJ4*02': '------------------------------------------------------------------------------------------------------------------KLTFGAGTRLAVCP',
                'TRAJ22*01': '------------------------------------------------------------------------------------------------------------------QLIFGSGTQLTVMP',
                'TRAJ43*01': '------------------------------------------------------------------------------------------------------------------APRFGAGTKLSVKP',
                'TRAJ17*01': '------------------------------------------------------------------------------------------------------------------KLTFGIGTRVLVRP',
                'TRAJ39*01': '------------------------------------------------------------------------------------------------------------------KLTFGGGTRLTVRP',
                'TRAJ30*01': '------------------------------------------------------------------------------------------------------------------KVIFGKGTHLHVLP',
                'TRAJ57*01': '------------------------------------------------------------------------------------------------------------------KLIFGEGTKLTVSS',
                'TRAJ21*01': '------------------------------------------------------------------------------------------------------------------VLYFGSGTKLTVEP',
                'TRAJ40*01': '------------------------------------------------------------------------------------------------------------------KYVFGAGTRLKVIA',
                'TRAJ13*01': '------------------------------------------------------------------------------------------------------------------YQRFGTGTKLQVVP',
                'TRAJ26*01': '------------------------------------------------------------------------------------------------------------------GLTFGLGTRVSVFP',
                'TRAJ15*01': '------------------------------------------------------------------------------------------------------------------ALIFGTGTTVSVSP',
                'TRAJ45*01': '------------------------------------------------------------------------------------------------------------------RLTFGKGTQLIIQP',
                'TRAJ31*01': '------------------------------------------------------------------------------------------------------------------RIFFGDGTQLVVKP',
                'TRAJ38*01': '------------------------------------------------------------------------------------------------------------------KLIWGLGTSLVVNP',
                'TRAJ24*01': '------------------------------------------------------------------------------------------------------------------KLQFGTGTQVVVTP',
                'TRAJ24*02': '------------------------------------------------------------------------------------------------------------------KLQFGTGTQVVVTP',
                'TRAJ6*01': '------------------------------------------------------------------------------------------------------------------KPTFGKGTSLVVHP',
                'TRAJ28*01': '------------------------------------------------------------------------------------------------------------------RLTFGKGTKFSLIP',
                'TRAJ47*02': '------------------------------------------------------------------------------------------------------------------KMIFGLGTILRVRP',
                'TRAJ52*01': '------------------------------------------------------------------------------------------------------------------KLTFGHGTILRVHP',
                'TRAJ27*01': '------------------------------------------------------------------------------------------------------------------KLTFGDGTVLTVKP',
                'TRAJ37*01': '------------------------------------------------------------------------------------------------------------------KLIFGLGTTLQVQP',
                'TRAJ34*02': '------------------------------------------------------------------------------------------------------------------KVVFGTGTRLQVLP',
                'TRAJ34*01': '------------------------------------------------------------------------------------------------------------------KVVFGTGTRLQVSP',
                'TRAJ53*01': '------------------------------------------------------------------------------------------------------------------KLTFGKGTLLTVTP',
                'TRAJ42*01': '------------------------------------------------------------------------------------------------------------------KLTFGKGTKLSVKS',
                'TRAJ33*01': '------------------------------------------------------------------------------------------------------------------QLIWGSGTKLIIKP',
                'TRAJ23*01': '------------------------------------------------------------------------------------------------------------------KLIFGQGTKLSIKP',
                'TRAJ35*02': '------------------------------------------------------------------------------------------------------------------ALTFGSGTKVIVLP',
                'TRAJ35*01': '------------------------------------------------------------------------------------------------------------------ALTFGSGTKVIPCL',
                'TRAJ48*01': '------------------------------------------------------------------------------------------------------------------KITFGAGTKLTIKP',
                'TRAJ32*01': '------------------------------------------------------------------------------------------------------------------KLIFGIGTLLSVKP'}},
                'B': {'human': {'TRBJ1-6*01': '------------------------------------------------------------------------------------------------------------------PLHFGNGTRLTVT-',
                'TRBJ1-6*02': '------------------------------------------------------------------------------------------------------------------PLHFGNGTRLTVT-',
                'TRBJ2-2*01': '------------------------------------------------------------------------------------------------------------------ELFFGEGSRLTVL-',
                'TRBJ1-3*01': '------------------------------------------------------------------------------------------------------------------TIYFGEGSWLTVV-',
                'TRBJ1-1*01': '------------------------------------------------------------------------------------------------------------------EAFFGQGTRLTVV-',
                'TRBJ2-3*01': '------------------------------------------------------------------------------------------------------------------TQYFGPGTRLTVL-',
                'TRBJ2-5*01': '------------------------------------------------------------------------------------------------------------------TQYFGPGTRLLVL-',
                'TRBJ2-1*01': '------------------------------------------------------------------------------------------------------------------EQFFGPGTRLTVL-',
                'TRBJ2-7*01': '------------------------------------------------------------------------------------------------------------------EQYFGPGTRLTVT-',
                'TRBJ1-5*01': '------------------------------------------------------------------------------------------------------------------PQHFGDGTRLSIL-',
                'TRBJ2-4*01': '------------------------------------------------------------------------------------------------------------------IQYFGAGTRLSVL-',
                'TRBJ2-6*01': '------------------------------------------------------------------------------------------------------------------VLTFGAGSRLTVL-',
                'TRBJ1-2*01': '------------------------------------------------------------------------------------------------------------------GYTFGSGTRLTVV-',
                'TRBJ1-4*01': '------------------------------------------------------------------------------------------------------------------KLFFGSGTQLSVL-'},
                'mouse': {'TRBJ2-2*01': '------------------------------------------------------------------------------------------------------------------QLYFGEGSKLTVL-',
                'TRBJ1-3*01': '------------------------------------------------------------------------------------------------------------------TLYFGEGSRLIVV-',
                'TRBJ1-2*01': '------------------------------------------------------------------------------------------------------------------DYTFGSGTRLLVI-',
                'TRBJ1-1*01': '------------------------------------------------------------------------------------------------------------------EVFFGKGTRLTVV-',
                'TRBJ2-5*01': '------------------------------------------------------------------------------------------------------------------TQYFGPGTRLLVL-',
                'TRBJ2-1*01': '------------------------------------------------------------------------------------------------------------------EQFFGPGTRLTVL-',
                'TRBJ2-7*01': '------------------------------------------------------------------------------------------------------------------EQYFGPGTRLTVL-',
                'TRBJ1-5*03': '------------------------------------------------------------------------------------------------------------------AQHFGEGTRLSVL-',
                'TRBJ1-5*01': '------------------------------------------------------------------------------------------------------------------APLFGEGTRLSVL-',
                'TRBJ2-4*01': '------------------------------------------------------------------------------------------------------------------TLYFGAGTRLSVL-',
                'TRBJ2-3*01': '------------------------------------------------------------------------------------------------------------------TLYFGSGTRLTVL-',
                'TRBJ1-4*01': '------------------------------------------------------------------------------------------------------------------RLFFGHGTKLSVL-',
                'TRBJ1-4*02': '------------------------------------------------------------------------------------------------------------------RLFFGHGTKLSVL-'}},
                'G': {'human': {'TRGJP2*01': '------------------------------------------------------------------------------------------------------------------IKTFAKGTRLIVTS',
                'TRGJP1*01': '------------------------------------------------------------------------------------------------------------------FKIFAEGTKLIVTS',
                'TRGJ1*01': '------------------------------------------------------------------------------------------------------------------KKLFGSGTTLVVT-',
                'TRGJ1*02': '------------------------------------------------------------------------------------------------------------------KKLFGSGTTLVVT-',
                'TRGJ2*01': '------------------------------------------------------------------------------------------------------------------KKLFGSGTTLVVT-',
                'TRGJP*01': '------------------------------------------------------------------------------------------------------------------IKVFGPGTKLIIT-'},
                'mouse': {'TRGJ4*01': '------------------------------------------------------------------------------------------------------------------VKIFAKGTKLVVIP',
                'TRGJ1*01': '------------------------------------------------------------------------------------------------------------------HKVFAEGTKLIVIP',
                'TRGJ2*01': '------------------------------------------------------------------------------------------------------------------HKVFAEGTKLIVIP',
                'TRGJ3*01': '------------------------------------------------------------------------------------------------------------------HKVFAEGTKLIVIP'}},
                'D': {'human': {'TRDJ3*01': '------------------------------------------------------------------------------------------------------------------QMFFGTGIKLFVEP',
                'TRDJ4*01': '------------------------------------------------------------------------------------------------------------------PLIFGKGTYLEVQQ',
                'TRDJ4*02': '------------------------------------------------------------------------------------------------------------------PLIFGKGTYLEVQQ',
                'TRDJ2*01': '------------------------------------------------------------------------------------------------------------------QLFFGKGTQLIVEP',
                'TRDJ1*01': '------------------------------------------------------------------------------------------------------------------KLIFGKGTRVTVEP'},
                'mouse': {'TRDJ2*02': '------------------------------------------------------------------------------------------------------------------TDVFGTGIELFVEP',
                'TRDJ2*01': '------------------------------------------------------------------------------------------------------------------QMFFGTGIELFVEP',
                'TRDJ1*01': '------------------------------------------------------------------------------------------------------------------KLVFGQGTQVTVEP'}}},
                'V': {'H': {'human': {'IGHV1-18*01': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYGISWVRQAPGQGLEWMGWISAY--NGNTNYAQKLQ-GRVTMTTDTSTSTAYMELRSLRSDDTAVYYCAR----------------------',
                'IGHV1-18*03': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYGISWVRQAPGQGLEWMGWISAY--NGNTNYAQKLQ-GRVTMTTDTSTSTAYMELRSLRSDDMAVYYCAR----------------------',
                'IGHV1-18*04': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYGISWVRQAPGQGLEWMGWISAY--NGNTNYAQKLQ-GRVTMTTDTSTSTAYMELRSLRSDDTAVYYCAR----------------------',
                'IGHV1-2*01': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TGYYMHWVRQAPGQGLEWMGRINPN--SGGTNYAQKFQ-GRVTSTRDTSISTAYMELSRLRSDDTVVYYCAR----------------------',
                'IGHV1-2*02': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TGYYMHWVRQAPGQGLEWMGWINPN--SGGTNYAQKFQ-GRVTMTRDTSISTAYMELSRLRSDDTAVYYCAR----------------------',
                'IGHV1-2*04': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TGYYMHWVRQAPGQGLEWMGWINPN--SGGTNYAQKFQ-GWVTMTRDTSISTAYMELSRLRSDDTAVYYCAR----------------------',
                'IGHV1-2*06': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TGYYMHWVRQAPGQGLEWMGRINPN--SGGTNYAQKFQ-GRVTMTRDTSISTAYMELSRLRSDDTAVYYCAR----------------------',
                'IGHV1-2*07': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TGYYMHWVRQAPGQGLEWMGWINPN--SGGTNYAHKFQ-GRVTMTRDTSISTAYMELSRLRSDDTAVYYCAR----------------------',
                'IGHV1-24*01': 'QVQLVQSGA-EVKKPGASVKVSCKVSGYTL----TELSMHWVRQAPGKGLEWMGGFDPE--DGETIYAQKFQ-GRVTMTEDTSTDTAYMELSSLRSEDTAVYYCAT----------------------',
                'IGHV1-3*01': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYAMHWVRQAPGQRLEWMGWINAG--NGNTKYSQKFQ-GRVTITRDTSASTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-3*02': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYAMHWVRQAPGQRLEWMGWSNAG--NGNTKYSQEFQ-GRVTITRDTSASTAYMELSSLRSEDMAVYYCAR----------------------',
                'IGHV1-3*03': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYAMHWVRQAPGQRLEWMGWINAG--NGNTKYSQEFQ-GRVTITRDTSASTAYMELSSLRSEDMAVYYCAR----------------------',
                'IGHV1-3*05': 'QVQLVQSGA-EEKKPGASVKVSCKASGYTF----TSYAMHWVRQAPGQRLEWMGWINAG--NGNTKYSQKFQ-GRVTITRDTSASTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-45*01': 'QMQLVQSGA-EVKKTGSSVKVSCKASGYTF----TYRYLHWVRQAPGQALEWMGWITPF--NGNTNYAQKFQ-DRVTITRDRSMSTAYMELSSLRSEDTAMYYCAR----------------------',
                'IGHV1-45*02': 'QMQLVQSGA-EVKKTGSSVKVSCKASGYTF----TYRYLHWVRQAPGQALEWMGWITPF--NGNTNYAQKFQ-DRVTITRDRSMSTAYMELSSLRSEDTAMYYCAR----------------------',
                'IGHV1-45*03': 'QMQLVQSGA-EVKKTGSSVKVSCKASGYTF----TYRYLHWVRQAPRQALEWMGWITPF--NGNTNYAQKFQ-DRVTITRDRSMSTAYMELSSLRSEDTAMYYCAR----------------------',
                'IGHV1-46*01': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYYMHWVRQAPGQGLEWMGIINPS--GGSTSYAQKFQ-GRVTMTRDTSTSTVYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-46*02': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----NSYYMHWVRQAPGQGLEWMGIINPS--GGSTSYAQKFQ-GRVTMTRDTSTSTVYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-46*03': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYYMHWVRQAPGQGLEWMGIINPS--GGSTSYAQKFQ-GRVTMTRDTSTSTVYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-46*04': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYYMHWVRQAPGQGLEWMGIINPS--GGSTSYAQKLQ-GRVTMTRDTSTSTVYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-58*01': 'QMQLVQSGP-EVKKPGTSVKVSCKASGFTF----TSSAVQWVRQARGQRLEWIGWIVVG--SGNTNYAQKFQ-ERVTITRDMSTSTAYMELSSLRSEDTAVYYCAA----------------------',
                'IGHV1-58*02': 'QMQLVQSGP-EVKKPGTSVKVSCKASGFTF----TSSAMQWVRQARGQRLEWIGWIVVG--SGNTNYAQKFQ-ERVTITRDMSTSTAYMELSSLRSEDTAVYYCAA----------------------',
                'IGHV1-69*01': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*02': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYTISWVRQAPGQGLEWMGRIIPI--LGIANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*04': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGRIIPI--LGIANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*05': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITTDESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*06': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*08': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYTISWVRQAPGQGLEWMGRIIPI--LGTANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*09': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGRIIPI--LGIANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*10': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--LGIANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*11': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGRIIPI--LGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*12': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*13': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*14': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*15': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGRIIPI--FGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*16': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYTISWVRQAPGQGLEWMGGIIPI--LGTANYAQKFQ-GRVTITTDESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*17': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGIANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*19': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69*21': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYTISWVRQAPGQGLEWMGRIIPI--LGIANYAQKFQ-GRVTITADKSTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69-2*01': 'EVQLVQSGA-EVKKPGATVKISCKVSGYTF----TDYYMHWVQQAPGKGLEWMGLVDPE--DGETIYAEKFQ-GRVTITADTSTDTAYMELSSLRSEDTAVYYCAT----------------------',
                'IGHV1-69D*01': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITADESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-69D*05': 'QVQLVQSGA-EVKKPGSSVKVSCKASGGTF----SSYAISWVRQAPGQGLEWMGGIIPI--FGTANYAQKFQ-GRVTITTDESTSTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-8*01': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYDINWVRQATGQGLEWMGWMNPN--SGNTGYAQKFQ-GRVTMTRNTSISTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV1-8*03': 'QVQLVQSGA-EVKKPGASVKVSCKASGYTF----TSYDINWVRQATGQGLEWMGWMNPN--SGNTGYAQKFQ-GRVTITRNTSISTAYMELSSLRSEDTAVYYCAR----------------------',
                'IGHV2-26*01': 'QVTLKESGP-VLVKPTETLTLTCTVSGFSLS--NARMGVSWIRQPPGKALEWLAHIFSN---DEKSYSTSLK-SRLTISKDTSKSQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-26*02': 'QVTLKESGP-VLVKPTETLTLTCTVSGFSLS--NARMGVSWIRQPPGKALEWLAHIFSN---DEKSYSTSLK-SRLTISKDTSKSQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-26*03': 'QVTLKESGP-VLVKPTETLTLTCTISGFSLS--NARMGVSWIRQPPGKALEWLAHIFSN---DEKSYSTSLK-SRLTISKDTSKSQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-26*04': 'QVTLKESGP-VLVKPTETLTLTCTVSGFSLS--NARMGVSWIRQPPGKALEWLAHIFSN---DEKSYSTSLK-SRLTISKDTSKSQVVLTMTNMDPVDTATYYCAWI---------------------',
                'IGHV2-26*05': 'QVTLKESGP-VLVKPTETLTLTCTVSGFSLS--NARMGVSWIRQPPGKALEWLAHIFSN---DEKSYSTSLK-SRLTISKDTSKSQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-5*01': 'QITLKESGP-TLVKPTQTLTLTCTFSGFSLS--TSGVGVGWIRQPPGKALEWLALIYWN---DDKRYSPSLK-SRLTITKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-5*02': 'QITLKESGP-TLVKPTQTLTLTCTFSGFSLS--TSGVGVGWIRQPPGKALEWLALIYWD---DDKRYSPSLK-SRLTITKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-5*05': 'QITLKESGP-TLVKPTQTLTLTCTFSGFSLS--TSGVGVGWIRQPPGKALEWLALIYWD---DDKRYGPSLK-SRLTITKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-5*06': 'QITLKESGP-TLVKPTQTLTLTCTFSGFSLS--TSGVGVGWIRQPPGKALEWLALIYWD---DDKRYGPSLK-SRLTITKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-5*08': 'QVTLKESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMRVSWIRQPPGKALEWLALIYWD---DDKRYSPSLK-SRLTITKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-5*09': 'QVTLKESGP-TLVKPTQTLTLTCTFSGFSLS--TSGVGVGWIRQPPGKALEWLALIYWD---DDKRYGPSLK-SRLTITKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-70*01': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLALIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*04': 'QVTLKESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMRVSWIRQPPGKALEWLARIDWD---DDKFYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*05': 'QVTLKESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMRASWIRQPPGKALEWLARIDWD---DDKFYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*10': 'QVTLKESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMRVSWIRQPPGKALEWIARIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*11': 'RVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLARIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*12': 'QITLKESGP-TLVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLALIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCAHR---------------------',
                'IGHV2-70*13': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLALIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*15': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLARIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*16': 'QVTLKESGP-VLVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLARIDWD---DDKFYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*17': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLARIDWD---DDKFYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*18': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSEMCVSWVRQPPGKALEWLALIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*19': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWVRQPPGKALEWLALIDWD---DDKHYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*20': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWVRQPPGKALEWLALIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*21': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSEMCVSWSRQPPGKALEWLARIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70*23': 'QVTLRESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMCVSWIRQPPGKALEWLARIDWD---DDKYYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARM---------------------',
                'IGHV2-70D*04': 'QVTLKESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMRVSWIRQPPGKALEWLARIDWD---DDKFYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV2-70D*14': 'QVTLKESGP-ALVKPTQTLTLTCTFSGFSLS--TSGMRVSWIRQPPGKALEWLARIDWD---DDKFYSTSLK-TRLTISKDTSKNQVVLTMTNMDPVDTATYYCARI---------------------',
                'IGHV3-11*01': 'QVQLVESGG-GLVKPGGSLRLSCAASGFTF----SDYYMSWIRQAPGKGLEWVSYISSS--GSTIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-11*03': 'QVQLLESGG-GLVKPGGSLRLSCAASGFTF----SDYYMSWIRQAPGKGLEWVSYISSS--SSYTNYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-11*04': 'QVQLVESGG-GLVKPGGSLRLSCAASGFTF----SDYYMSWIRQAPGKGLEWVSYISSS--GSTIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-11*05': 'QVQLVESGG-GLVKPGGSLRLSCAASGFTF----SDYYMSWIRQAPGKGLEWVSYISSS--SSYTNYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-11*06': 'QVQLVESGG-GLVKPGGSLRLSCAASGFTF----SDYYMSWIRQAPGKGLEWVSYISSS--SSYTNYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-13*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYDMHWVRQATGKGLEWVSAIGTA---GDTYYPGSVK-GRFTISRENAKNSLYLQMNSLRAGDTAVYYCAR----------------------',
                'IGHV3-13*02': 'EVHLVESGG-GLVQPGGALRLSCAASGFTF----SNYDMHWVRQATGKGLEWVSANGTA---GDTYYPGSVK-GRFTISRENAKNSLYLQMNSLRAGDTAVYYCAR----------------------',
                'IGHV3-13*03': 'EVQLVESGG-GLVQPGGSLRLSCAACGFTF----SSYDMHWVRQATGKGLEWVSAIGTA---GDTYYPGSVK-GQFTISRENAKNSLYLQMNSLRAGDTAVYYCAR----------------------',
                'IGHV3-13*04': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYDMHWVRQATGKGLEWVSAIGTA---GDTYYPGSVK-GRFTISRENAKNSLYLQMNSLRAGDTAVYYCAR----------------------',
                'IGHV3-13*05': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYDMHWVRQATGKGLEWVSAIGTA---GDPYYPGSVK-GRFTISRENAKNSLYLQMNSLRAGDTAVYYCAR----------------------',
                'IGHV3-13*06': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYDMHWVRQATGKGLEWVSAIGTA---GDTYYPGSVK-GRFTISRENAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-15*01': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SNAWMSWVRQAPGKGLEWVGRIKSKTDGGTTDYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*02': 'EVQLVESGG-ALVKPGGSLRLSCAASGFTF----SNAWMSWVRQAPGKGLEWVGRIKSKTDGGTTDYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*03': 'EVQLVESAG-ALVQPGGSLRLSCAASGFTC----SNAWMSWVRQAPGKGLEWVGRIKSKANGGTTDYAAPVK-GRFTISRVDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*04': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SNAWMSWVRQAPGKGLEWVGRIESKTDGGTTDYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*05': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SNAWMSWVRQAPGKGLEWVGRIKSKTDGGTTDYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*06': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SNAWMSWVRQAPGKGLEWVGRIKSKTDGGTTNYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*07': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SNAWMNWVRQAPGKGLEWVGRIKSKTDGGTTDYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-15*08': 'EVQLVESAG-GLVQPGGSLRLSCAASGFTC----SNAWMSWVRQAPGKGLEWVGCIKSKANGGTTDYAAPVK-GRFTISRDDSKNTLYLQMISLKTEDTAVYYCTT----------------------',
                'IGHV3-15*09': 'EVQLVESGG-GLVKPRGSLRLSCAASGFTF----SNAWMSWVRQAPGKGLEWVGRIKSKTDGGTTDYAAPVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT----------------------',
                'IGHV3-20*01': 'EVQLVESGG-GVVRPGGSLRLSCAASGFTF----DDYGMSWVRQAPGKGLEWVSGINWN--GGSTGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTALYHCAR----------------------',
                'IGHV3-20*04': 'EVQLVESGG-GVVRPGGSLRLSCAASGFTF----DDYGMSWVRQAPGKGLEWVSGINWN--GGSTGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTALYYCAR----------------------',
                'IGHV3-20*05': 'EVQLVESGG-GVVRPGGSLRLSCAASGFTF----GDYGMSWVRQAPGKGLEWVSGINWN--GGSTGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTALYYCAR----------------------',
                'IGHV3-21*01': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSSISSS--SSYIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-21*02': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSSISSS--SSYIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-21*03': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSSISSS--SSYIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-21*06': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSSISSS--SSYIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-21*08': 'EVQLVESGG-GLVKPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSSISSS---SYIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-23*01': 'EVQLLESGG-GLVQPGGSLRLSCAASGFTF----SSYAMSWVRQAPGKGLEWVSAISGS--GGSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-23*02': 'EVQLLESGG-GLVQPGGSLRLSCAASGFTF----SSYAMSWVRQAPGKGLEWVSAISGS--GGSTYYGDSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-23*03': 'EVQLLESGG-GLVQPGGSLRLSCAASGFTF----SSYAMSWVRQAPGKGLEWVSVIYSG--GSSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-23*04': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYAMSWVRQAPGKGLEWVSAISGS--GGSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-23D*01': 'EVQLLESGG-GLVQPGGSLRLSCAASGFTF----SSYAMSWVRQAPGKGLEWVSAISGS--GGSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-30*01': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*02': 'QVQLVESGG-GVVQPGGSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAFIRYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-30*03': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*04': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*05': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEGTAVYYCAR----------------------',
                'IGHV3-30*06': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*07': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*08': 'QVQLVDSGG-GVVQPGRSLRLSCAASAFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*09': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFAISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*10': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYTDSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*11': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*12': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*13': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNRLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*14': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*15': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMSSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*16': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*17': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30*18': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-30*19': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-30-3*02': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-30-3*03': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-33*01': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-33*02': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSNKYYADSAK-GRFTISRDNSTNTLFLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-33*03': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-33*04': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-33*05': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-33*06': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV3-33*07': 'QVQLVESGG-RVVQPGRSLRLSCAASGFTF----SRYGMYWVRQAPGKGLEWVAVIWYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-33*08': 'QVQLVESGG-GVVQPGRSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSNKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-35*02': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SNSDMNWVHQAPGKGLEWVSGVSWN--GSRTHYADSVK-GQFIISRDNSRNTLYLQTNSLRAEDTAVYYCVR----------------------',
                'IGHV3-43*01': 'EVQLVESGG-VVVQPGGSLRLSCAASGFTF----DDYTMHWVRQAPGKGLEWVSLISWD--GGSTYYADSVK-GRFTISRDNSKNSLYLQMNSLRTEDTALYYCAKD---------------------',
                'IGHV3-43*02': 'EVQLVESGG-GVVQPGGSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSLISGD--GGSTYYADSVK-GRFTISRDNSKNSLYLQMNSLRTEDTALYYCAKD---------------------',
                'IGHV3-43D*03': 'EVQLVESGG-VVVQPGGSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSLISWD--GGSTYYADSVK-GRFTISRDNSKNSLYLQMNSLRAEDTALYYCAKD---------------------',
                'IGHV3-43D*04': 'EVQLVESGG-VVVQPGGSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSLISWD--GGSTYYADSVK-GRFTISRDNSKNSLYLQMNSLRAEDTALYYCAKD---------------------',
                'IGHV3-43D*05': 'EMQLVESGG-VVVQPGGSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSLISWD--GGSTYYADSVK-GRFTISRDNSKNSLYLQMNSLRAEDTALYYCAKD---------------------',
                'IGHV3-43D*06': 'EVQLVESGG-VVVQPGGFLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSLISWD--GGSTYYADSVK-GRFTISRDNSKNSLYLQMNSLRAEDTALYYCAKD---------------------',
                'IGHV3-48*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSYISSS--SSTIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-48*02': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYSMNWVRQAPGKGLEWVSYISSS--SSTIYYADSVK-GRFTISRDNAKNSLYLQMNSLRDEDTAVYYCAR----------------------',
                'IGHV3-48*03': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYEMNWVRQAPGKGLEWVSYISSS--GSTIYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-49*01': 'EVQLVESGG-GLVQPGRSLRLSCTASGFTF----GDYAMSWFRQAPGKGLEWVGFIRSKAYGGTTEYTASVK-GRFTISRDGSKSIAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-49*02': 'EVQLVESGG-GLVQPGPSLRLSCTASGFTF----GYYPMSWVRQAPGKGLEWVGFIRSKAYGGTTEYAASVK-GRFTISRDDSKSIAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-49*03': 'EVQLVESGG-GLVQPGRSLRLSCTASGFTF----GDYAMSWFRQAPGKGLEWVGFIRSKAYGGTTEYAASVK-GRFTISRDDSKSIAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-49*04': 'EVQLVESGG-GLVQPGRSLRLSCTASGFTF----GDYAMSWVRQAPGKGLEWVGFIRSKAYGGTTEYAASVK-GRFTISRDDSKSIAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-49*05': 'EVQLVESGG-GLVKPGRSLRLSCTASGFTF----GDYAMSWFRQAPGKGLEWVGFIRSKAYGGTTEYAASVK-GRFTISRDDSKSIAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-53*01': 'EVQLVESGG-GLIQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-53*02': 'EVQLVETGG-GLIQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-53*03': 'EVQLVESGG-GLIQPGGSLRLSCAASGFTV----SSNYMSWVRQPPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-53*04': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRHNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-53*06': 'EVQLVESGG-GLIQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRHNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-62*04': 'EVQLVKSGG-GLVQPGGSLRLSCAASGFTF----SSSAMHWVRQAPRKGLEWVSVISTS--GDTVLYTDSVK-GRFTISRDNAQNSLSLQMNSLRAEDMAVYYCVK----------------------',
                'IGHV3-64*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYANSVK-GRFTISRDNSKNTLYLQMGSLRAEDMAVYYCAR----------------------',
                'IGHV3-64*02': 'EVQLVESGE-GLVQPGGSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYLQMGSLRAEDMAVYYCAR----------------------',
                'IGHV3-64*03': 'EVQLVESGG-GLVQPGGSLRLSCSASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYVQMSSLRAEDTAVYYCVK----------------------',
                'IGHV3-64*04': 'QVQLVESGG-GLVQPGGSLRLSCSASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-64*05': 'EVQLVESGG-GLVQPGGSLRLSCSASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYVQMSSLRAEDTAVYYCVK----------------------',
                'IGHV3-64*07': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYLQMGSLRAEDMAVYYCAR----------------------',
                'IGHV3-64D*06': 'EVQLVESGG-GLVQPGGSLRLSCSASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYLQMSSLRAEDTAVYYCVK----------------------',
                'IGHV3-64D*08': 'EVQLVESGG-GLVQPGGSLRLSCSASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYLQMSSLRAEDTAVYYCVK----------------------',
                'IGHV3-64D*09': 'EVQLVESGG-GLVQPGGSLRLSCSASGFTF----SSYAMHWVRQAPGKGLEYVSAISSN--GGSTYYADSVK-GRFTISRDNSKNTLYLQMSSLRAEDTAVYYCVK----------------------',
                'IGHV3-66*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-66*02': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-66*03': 'EVQLVESGG-GLIQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSC---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-66*04': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTV----SSNYMSWVRQAPGKGLEWVSVIYSG---GSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-7*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMSWVRQAPGKGLEWVANIKQD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-7*02': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMSWVRQAPGKGLEWVANIKQD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-7*04': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMSWVRQAPGKGLEWVANIKQD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-7*05': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMSWVRQAPGKGLEWVANIKQD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-72*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SDHYMDWVRQAPGKGLEWVGRTRNKANSYTTEYAASVK-GRFTISRDDSKNSLYLQMNSLKTEDTAVYYCAR----------------------',
                'IGHV3-73*01': 'EVQLVESGG-GLVQPGGSLKLSCAASGFTF----SGSAMHWVRQASGKGLEWVGRIRSKANSYATAYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-73*02': 'EVQLVESGG-GLVQPGGSLKLSCAASGFTF----SGSAMHWVRQASGKGLEWVGRIRSKANSYATAYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCTR----------------------',
                'IGHV3-74*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMHWVRQAPGKGLVWVSRINSD--GSSTSYADSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-74*02': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMHWVRQAPGKGLVWVSRINSD--GSSTSYADSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-74*03': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMHWVRQAPGKGLVWVSRINSD--GSSTTYADSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR----------------------',
                'IGHV3-9*01': 'EVQLVESGG-GLVQPGRSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSGISWN--SGSIGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTALYYCAKD---------------------',
                'IGHV3-9*02': 'EVQLVESGG-GLVQPGRSLRLSCAASGFTS----DDYAMHWVRQAPGKGLEWVSGISWN--SGSIGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTALYYCAKD---------------------',
                'IGHV3-9*03': 'EVQLVESGG-GLVQPGRSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSGISWN--SGSIGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDMALYYCAKD---------------------',
                'IGHV3-9*04': 'EVQLVESGG-GLVQPGRSLRLSCAASGFTF----DDYAMHWVRQAPGKGLEWVSGISWN--SGSIGYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTALYHCAKD---------------------',
                'IGHV3-NL1*01': 'QVQLVESGG-GVVQPGGSLRLSCAASGFTF----SSYGMHWVRQAPGKGLEWVSVIYSG--GSSTYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK----------------------',
                'IGHV4-28*01': 'QVQLQESGP-GLVKPSDTLSLTCAVSGYSIS---SSNWWGWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAVDTAVYYCAR----------------------',
                'IGHV4-28*02': 'QVQLQESGP-GLVKPSQTLSLTCAVSGYSIS---SSNWWGWIRQPPGKGLEWIGYIYYS---GSIYYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAVDTAVYYCAR----------------------',
                'IGHV4-28*03': 'QVQLQESGP-GLVKPSDTLSLTCAVSGYSIS---SSNWWGWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAVDTAVYYCAR----------------------',
                'IGHV4-28*04': 'QVQLQESGP-GLVKPSDTLSLTCAVSGYSIS---SSNWWGWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAVDTGVYYCAR----------------------',
                'IGHV4-28*05': 'QVQLQESGP-GLVKPSDTLSLTCAVSGYSIS---SSNWWGWIRQPPGKGLEWIGYIYYS---GSIYYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAVDTAVYYCAR----------------------',
                'IGHV4-28*07': 'QVQLQESGP-GLVKPSDTLSLTCAVSGYSIS---SSNWWGWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAVDTAVYYCAR----------------------',
                'IGHV4-30-2*01': 'QLQLQESGS-GLVKPSQTLSLTCAVSGGSIS--SGGYSWSWIRQPPGKGLEWIGYIYHS---GSTYYNPSLK-SRVTISVDRSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-2*03': 'QLQLQESGS-GLVKPSQTLSLTCAVSGGSIS--SGGYSWSWIRQPPGKGLEWIGSIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-2*05': 'QLQLQESGS-GLVKPSQTLSLTCAVSGGSIS--SGGYSWSWIRQPPGKGLEWIGYIYHS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-2*06': 'QLQLQESGS-GLVKPSQTLSLTCAVSGGSIS--SGGYSWSWIRQSPGKGLEWIGYIYHS---GSTYYNPSLK-SRVTISVDRSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-2*07': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGDYYWSWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-4*01': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGDYYWSWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-4*02': 'QVQLQESGP-GLVKPSDTLSLTCTVSGGSIS--SGDYYWSWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-4*07': 'QVQLQESGP-GLVKPSQTLSLTCAVSGGSIS--SGGYSWSWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-30-4*10': 'QLQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGDYYWSWIRQPPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-31*01': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGGYYWSWIRQHPGKGLEWIGYIYYS---GSTYYNPSLK-SLVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-31*02': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGGYYWSWIRQHPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-31*03': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGGYYWSWIRQHPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-31*10': 'QVQLQESGP-GLLKPSQTLSLTCTVSGGSIS--SGGYYWSWIRQHPGKGLEWIGCIYYS---GSTYYNPSLK-SRVTISVDPSKNQFSLKPSSVTAADTAVDYCAR----------------------',
                'IGHV4-31*13': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGGYYWSWIRQHPGKGLEWIGYIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*01': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSF----SGYYWSWIRQPPGKGLEWIGEINHS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*02': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSF----SGYYWSWIRQPPGKGLEWIGEINHS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*04': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSF----SGYYWSWIRQPPGKGLEWIGEINHS---GSTNNNPSLK-SRATISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*05': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSF----SGYYWCWIRQPLGKGLEWIGEINHS---GSTNNNPSLK-SRATISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*09': 'QVQLQESGP-GLVKPSQTLSLTCAVYGGSF----SGYYWSWIRQPPGKGLEWIGEINHS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*10': 'QVQLQESGP-GLVKPSETLSLTCAVYGGSF----SGYYWSWIRQPPGKGLEWIGEINHS---GSTNYNPSLK-SRITMSVDTSKNQFYLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-34*11': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSV----SGYYWSWIRQPPGKGLEWIGYIYYS---GSTNNNPSLK-SRATISVDTSKNQFSLNLSSVTAADTAVYCCAR----------------------',
                'IGHV4-34*12': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSF----SGYYWSWIRQPPGKGLEWIGEIIHS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-38-2*01': 'QVQLQESGP-GLVKPSETLSLTCAVSGYSIS---SGYYWGWIRQPPGKGLEWIGSIYHS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-38-2*02': 'QVQLQESGP-GLVKPSETLSLTCTVSGYSIS---SGYYWGWIRQPPGKGLEWIGSIYHS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-38-2*03': 'QVQLQESGP-GLVKPSETLSLTCAVSGYSIS---SGYYWGWIRQPPGKGLEWIGSIYHS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-39*01': 'QLQLQESGP-GLVKPSETLSLTCTVSGGSIS--SSSYYWGWIRQPPGKGLEWIGSIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-39*02': 'QLQLQESGP-GLVKPSETLSLTCTVSGGSIS--SSSYYWGWIRQPPGKGLEWIGSIYYS---GSTYYNPSLK-SRVTISVDTSKNHFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-39*06': 'RLQLQESGP-GLVKPSETLSLTCTVSGGSIS--SSSYYWGWIRQPPGKGLEWIGSIYYS---GSTYYNPSLK-SRVTISVDTSKNQFPLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-39*07': 'QLQLQESGP-GLVKPSETLSLTCTVSGGSIS--SSSYYWGWIRQPPGKGLEWIGSIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-39*10': 'QLQLQESGP-GLVKPSETLSLTCTVSGGSIS--SSSYYWGWIRQPPGKGLEWIGSIYYS---GSTYYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-4*01': 'QVQLQESGP-GLVKPPGTLSLTCAVSGGSIS---SSNWWSWVRQPPGKGLEWIGEIYHS---GSTNYNPSLK-SRVTISVDKSKNQFSLKLSSVTAADTAVYCCAR----------------------',
                'IGHV4-4*02': 'QVQLQESGP-GLVKPSGTLSLTCAVSGGSIS---SSNWWSWVRQPPGKGLEWIGEIYHS---GSTNYNPSLK-SRVTISVDKSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-4*07': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSI----SSYYWSWIRQPAGKGLEWIGRIYTS---GSTNYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-4*08': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSI----SSYYWSWIRQPPGKGLEWIGYIYTS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-59*01': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSI----SSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-59*02': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSV----SSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-59*07': 'QVQLQESGP-GLVKPSDTLSLTCTVSGGSI----SSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-59*10': 'QVQLQQWGA-GLLKPSETLSLTCAVYGGSI----SSYYWSWIRQPAGKGLEWIGRIYTS---GSTNYNPSLK-SRVTMSVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-59*11': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSI----SSHYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-59*13': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSI----SSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*01': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSVS--SGSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*02': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGSYYWSWIRQPAGKGLEWIGRIYTS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*03': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSVS--SGSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNHFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*05': 'QLQLQESGP-GLVKPSETLSLTCTVSGGSIS--SSSYYWGWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDKSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*08': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSVS--SGGYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*09': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIS--SGSYYWSWIRQPAGKGLEWIGHIYTS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*12': 'QVQLQESGP-GLVKPSETLSLTCTVSGGSI----SSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV4-61*13': 'QVQLQESGP-GLVRPSETLSLTCTVSGGSVS--SGSYYWSWIRQPPGKGLEWIGYIYYS---GSTNYNPSLK-SRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR----------------------',
                'IGHV5-10-1*01': 'EVQLVQSGA-EVKKPGESLRISCKGSGYSF----TSYWISWVRQMPGKGLEWMGRIDPS--DSYTNYSPSFQ-GHVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-10-1*02': 'EVQLVQSGA-EVKKPGESLRISCKGSGYSF----TSYWISWVRQMPGKGLEWMGRIDPS--DSYTNYSPSFQ-GHVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-10-1*03': 'EVQLVQSGA-EVKKPGESLRISCKGSGYSF----TSYWISWVRQMPGKGLEWMGRIDPS--DSYTNYSPSFQ-GHVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-10-1*04': 'EVQLVQSGA-EVKKPGESLRISCKGSGYSF----TSYWISWVRQMPGKGLEWMGRIDPS--DSYTNYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-51*01': 'EVQLVQSGA-EVKKPGESLKISCKGSGYSF----TSYWIGWVRQMPGKGLEWMGIIYPG--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-51*02': 'EVQLVQSGA-EVKKPGESLKISCKGSGYSF----TSYWTGWVRQMPGKGLEWMGIIYPG--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-51*04': 'EVQLVQSGA-EVKKPGESLKISCKGSGYSF----TSYWIGWVRQMPGKGLEWMGIIYPG--DSDTRYSPSFQ-GQVTISADKPISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV5-51*07': 'EVQLVQSGA-EVKKPGESLKISCKGSGYSF----TSYWIGWVHQMPGKGLEWMGIIYPG--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTAMYYCAR----------------------',
                'IGHV6-1*01': 'QVQLQQSGP-GLVKPSQTLSLTCAISGDSVS--SNSAAWNWIRQSPSRGLEWLGRTYYRS-KWYNDYAVSVK-SRITINPDTSKNQFSLQLNSVTPEDTAVYYCAR----------------------',
                'IGHV6-1*02': 'QVQLQQSGP-GLVKPSQTLSLTCAISGDSVS--SNSAAWNWIRQSPSRGLEWLGRTYYRS-KWYNDYAVSVK-SRITINPDTSKNQFSLQLNSVTPEDTAVYYCAR----------------------',
                'IGHV7-4-1*01': 'QVQLVQSGS-ELKKPGASVKVSCKASGYTF----TSYAMNWVRQAPGQGLEWMGWINTN--TGNPTYAQGFT-GRFVFSLDTSVSTAYLQICSLKAEDTAVYYCAR----------------------',
                'IGHV7-4-1*02': 'QVQLVQSGS-ELKKPGASVKVSCKASGYTF----TSYAMNWVRQAPGQGLEWMGWINTN--TGNPTYAQGFT-GRFVFSLDTSVSTAYLQISSLKAEDTAVYYCAR----------------------'},
                'mouse': {'IGHV1-11*01': 'QIQLQQSGA-ELASPGASVTLSCKASGYTF----TDHIMNWVKKRPGQGLEWIGRIYPV--SGETNYNQKFM-GKATFSVDRSSSTVYMVLNSLTSEDPAVYYCGR----------------------',
                'IGHV1-11*02': 'QIQLQQSGA-ELASPGASVTLSCKASGYTF----TDHIMNWVKKRPGQGLEWIGRIYPV--SGETNYNQKFM-GKATFSVDRSSSTVYMVLNSLTSEDPAVYYCGR----------------------',
                'IGHV1-12*01': 'QAYLQQSGA-ELVRPGASVKMSCKASGYTF----TSYNMHWVKQTPRQGLEWIGAIYPG--NGDTSYNQKFK-GKATLTVDKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-12*02': 'QAYLQQSGA-ELVRSGASVKMSCKASGYTF----TSYNMHWVKQTPGQGLEWIGYIYPG--NGGTNYNQKFK-GKATLTADTSSSTAYMQISSLTSEDSAVYFCAR----------------------',
                'IGHV1-12*03': 'QAYLPQSGA-ELVRPGASVKMSCKASGYTF----TSYNVHWVKQTPGQGLEWIGYIYPG--NGGTNYNQKF---KATLTADTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-14*02': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TSYVMHWVKQKPGQGLEWIGYINPY--NDGTKYNEKFK-GKATLTSDKSSSTAYMELSSLTSEDSAVYYCAR----------------------',
                'IGHV1-14*03': 'QVQLQQSGP-ELVKPGASVKMSCKASGYTF----ANHVMHWVKQKPGQGLEWIGYIYPY--NDGTKYNEKFK-GKATLTSDKSSSTAYMELSSLASEDSAVYYCAR----------------------',
                'IGHV1-15*01': 'QVQLQQSGA-ELVRPGASVTLSCKASGYTF----TDYEMHWVKQTPVHGLEWIGAIDPE--TGGTAYNQKFK-GKAILTADKSSSTAYMELRSLTSEDSAVYYCTR----------------------',
                'IGHV1-15*02': 'QVQLQQSGA-ELVRPGASVKLSCKALGYTF----TDYEMHWVKQTPVHGLEWIGAIHPG--SGGTAYNQKFK-GKATLTADKSSSTAYMELSSLTSEDSAVYYCTR----------------------',
                'IGHV1-18*01': 'EVQLQQSGP-ELVKPGASVKIPCKASGYTF----TDYNMDWVKQSHGKSLEWIGDINPN--NGGTIYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDTAVYYCAR----------------------',
                'IGHV1-18*04': 'EVQLQQSGP-ELVKPGASVKISCKTSGYTF----TEYTMHWVKQSHGKSLEWIGGINPN--NGGTSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18*05': 'EVQLQQSGP-ELVKPGSSVKISCKASGYTF----TDYSMDWVKQSHGKSLEWIGAINPN--NGGTSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-11*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSHVKSLEWIGRINPY--NGATSYNQNFK-DKASLTVDKSSSTAYMELHSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-13*01': 'QVQLQQSGA-ELARPGASVKMSCKASGYTF----TSYTMHWVKQRPGQGLEWIGYINPS--SGYTNYNQKFK-DKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-14*01': 'EVQLQQSGT-VLARPGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLEWIGAIYPG--NSDTSYNQKFK-GKAKLTAVTSTSTAYMELSSLTNEDSAVYYCTR----------------------',
                'IGHV1-18-14*02': 'EVQLQQSGA-ELARPGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLECIGAIYPG--NSDTSYNQKFK-GKAKLTAVTSASTAYMELSSLTNEDSAVYYCTR----------------------',
                'IGHV1-18-17*01': 'QGQMQQSGA-ELVKPGASVKLSCKTSGFTF----SSSYISWLKQKPGQSLEWIAWIYAG--TGGTSYNQKFT-GKAQLTVDTSSSTAYMQFSSLTTEDSAIYYCARH---------------------',
                'IGHV1-18-17*02': 'QGQMQQSGA-ELVKPGASVKLSCKTSGFTF----SSSYISWLKQKPGQSLEWIAWIYPG--SGSTSYNQKFT-GKAQLTADTSSSTAYMQLSSLTSEDSAIYYCAR----------------------',
                'IGHV1-18-19*01': 'QIQLQQSGA-ELASPGASVTLSCKASGYTF----TDHIMNWVKKRPGQGLEWIGRIYPV--SCETNYNQKFM-GKATFSVDRSSSTVYMVLNSLTSEDPAVYYCGR----------------------',
                'IGHV1-18-2*01': 'EIQLQQSGP-ELMKPGASVKISCKASGYSF----TSYYMHWVKQSHGKSLEWIGYIDPF--NGGTSYNQKFK-GKATLTVDKSSSTAYMHLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-22*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYVMHWVKQKPGQGLEWIGYIYPY--NDGTEYTEKFK-GKATLTLDKSSSTAYMDLSSLTSEDSTVYYCAR----------------------',
                'IGHV1-18-23*01': 'QVQLQQSGA-ELVRPGASVTLSCKASGYTF----TDYEMHWVKQTPVHGLEWIGAIDPE--TGGTAYNQKFK-GKATLTADKSSSTAYMELRSLTSEDSAVYYCTR----------------------',
                'IGHV1-18-23*02': 'QVQLQQSGA-ELVRPGASVTLSCKASGYTF----TDYEMHWVKQTPVHGLEWIGAIDPE--TGDTAYNQKFK-GKATLTADKSSSTAYMELSSLTSEDSAVYYCTR----------------------',
                'IGHV1-18-26*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYTF----TDYNMHWVKQSHGKSLEWIGYIYPY--NGGTGYNQKFK-SKATLTVDNSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-28*01': 'EIQLQQSGP-ELVKPGASVKVSCKASGYAF----TSYNMYWVKQSHGKSLEWIGYIDPY--NGGTSYNQKFK-GKATLTVDKSSSTAYMHLNSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-32*01': 'EVQLQQSGP-ELGKPGASVKISCKASGYSF----TGYNMYWVKQSHRKSLEWIGYIDPY--NGGTSYNQKSK-GKATLTVDKSSSTAYMHLNSLTSEDSAIYYCAR----------------------',
                'IGHV1-18-36*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYYMKWVKQSHGKSLEWIGDINPN--NGDTFYNQKFK-GKATLTVDKSSSTAYMQLNSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-37*01': 'QVQLQQSGA-ELAKPGASVKMSCKASGYPF----TSYWMHWVKQRPGQGLEWIGAINPS--SDYTEYNQKFK-DKATLTADKSSSTAYMQLSILASEDSAVYYCAR----------------------',
                'IGHV1-18-39*01': 'QVQLQQSGA-ELAKPGTSVKMSCKASGYTF----TSYWMNWVKQRPGQGLEWIGAINPS--NGYTEYNQKFK-DKAILTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-4*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSPENSLEWIGEINPS--TGGTSYNQKFK-GKATLTVDKSSSTAYMQLKSLTSEESAVYYCTR----------------------',
                'IGHV1-18-40*01': 'EVQLQQSGT-VLARHGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLEWIGAIYPG--NSDTSYNQKFK-GKAKLTAVTSASTAYMELSSLTNEDSAVYYCTR----------------------',
                'IGHV1-18-41*01': 'QVQLQQSGA-ELAKPGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLEWIGYINPS--SGYTKYNQKFK-DKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-45*01': 'EVQLQQSEP-ELMKPGASVKISCKTSGYSF----TDYYMHWVKHGPRNSLEWIGYIYTY--NGVSSYNQKFK-GKATLTVDKSSSTAYMELHSLTSEDSAVHYCAR----------------------',
                'IGHV1-18-50*01': 'EVHLQQSGP-ELVNPRDSVKISCKASGYSF----TGYYMNWVKQGPGKSLEWIGYISWY--NGATSYNQKFK-GKAIFTVDKSSSTAYMQFNSLTSEDSVVYYCAR----------------------',
                'IGHV1-18-53*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TNYYMHWVKQSHGKSLEWIGYIYPN--NGDTSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-58*01': 'EVQLQQSGP-ELVKPGSSVKISCKASGYTF----TDYYMNWVKQSHGKSLEWIGDINPI--NGGTSYNQKFK-GKATLTVDKSSSTDYMELRGLTSEDSAVYYCAR----------------------',
                'IGHV1-18-58D*01': 'EVQLQQSGP-ELVKPGSSVKISCKASGYTF----TDYYMNWVKQSHGKSLEWIGDINPI--NGGTSYNQKFK-GKATLTVDKSSSTDYMELRGLTSEDSAVYYCAR----------------------',
                'IGHV1-18-60*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSHGKSLEWIGYIYPY--NGVSSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-60D*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSHGKSLEWIGYIYPY--NGVSSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-62*01': 'EIQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYYMHWVKQSNGKSLEWIGYINPY--NDYTSYNQKFN-GKATLTVDKSSSTAYMQLNSLTSEDSAFYYCAR----------------------',
                'IGHV1-18-65*01': 'EVQLQQSGP-ELVKTGASVKISCKASGYSF----TGYYMHWVKQSHGKSLEWIGYISCY--NGATSYNREIK-GKAIYTIDTSSSTAYMQFNSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-66*01': 'EVHLQQSLP-KVVKAGPSVKISCKASGYSF----TGYYMHWVKQSHGKILQRVEYINPY--NGGTGYIEKFK-DKATLTADKSFSTAYMHLSSLTSEDSEVYSCAR----------------------',
                'IGHV1-18-68*01': 'EVQLQQSGP-ELLKPGASVKISCKASGYTF----TDYTMHWVKQSHGKSLEWIGGINPN--NGGTSYNEKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-7*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYYMDWVKQSHGESFEWIGRVNPY--NGGTSYNQKFK-GKATLTVDKSSSTAYMELNSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-73*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYTF----TDYTMHWVKQSHGKSLEWIGLVNPN--NGGTNYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSSVYYCAR----------------------',
                'IGHV1-18-74*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYTFT---DYYYMNWVKQSHGKSLEWIGYIYPN--NGGTSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-75*01': 'EIQLQQSGP-ELVKPGASVKMSCKASGYTF----TNYYMHWVKQSHGKSLEWIRRVNPN--NGGTSYNQKFK-DKATLTVEKSSITAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-80*01': 'EVQLQQSGP-DLVKPGASVKISCKASGYTF----TDYDMNWVKQSHGKSLEWIGYIYPN--NGGTSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-86*01': 'EIQLQQSGP-ELVKPGASVKVSCKASGYAF----TSYNMYWVKQSHGKSLEWIGYIDPY--NGGTSYNQKFK-GKATLTIDKSSSTAYMHLNSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-89*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYVMHWVKQSNGKSLEWIGYINPY--NDYTSYNQKFK-GKATLTVDKSSSTAYMQLNSLTSEDSAVYYCAR----------------------',
                'IGHV1-18-92*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYTF----TDYNMHWVK--HGKSLEWIGYIYPN--NGGTGYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-19*01': 'EVQLQQSGP-VLVKPGASVKMSCKASGYTF----TDYYMNWVKQSHGKSLEWIGVINPY--NGGTSYNQKFK-GKATLTVDKSSSTAYMELNSLTSEDSAVYYCAR----------------------',
                'IGHV1-20*01': 'EVQLQQSGP-ELVKPGDSVKISCKASGYSF----TGYFMNWVMQSHGKSLEWIGRINPY--NGDTFYNQKFK-GKATLTVDKSSSTAHMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-20*02': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYFMNWVMQSHGKSLEWIGRINPY--NGDTFYNQKFK-GKATLTVDKSSSTAHMELRSLASEDSAVYYCAR----------------------',
                'IGHV1-22*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYNMHWVKQSHGKSLEWIGYINPN--NGGTSYNQKFK-GKATLTVNKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-26*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYTF----TDYYMNWVKQSHGKSLEWIGDINPN--NGGTSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-31*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSHGNILDWIGYIYPY--NGVSSYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-34*01': 'EVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYYMHWVKQSHGKSLEWIGYIYPN--NGGNGYNQKFK-GKATLTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-34*02': 'EVQLQQSGP-ELVKPGDSVKMSCKASGYTF----TDYYMDWVKQSHGKSLEWIGYIYPN--NGGTSYNQKFK-GKATLTVDKSSSTAYMELHSLTSEDSAVYYCAR----------------------',
                'IGHV1-36*01': 'EVQLQQSGP-VLVKPGPSVKISCKASGFTF----TDYYMHWVKQSHGKSLEWIGLVYPY--NGGTSYNQKFK-GKATLTVDTSSSTAYMELNSLTSEDSAVYYCAR----------------------',
                'IGHV1-36*03': 'EVQLQQSGP-ELVKPGPSVKISCKASGYSF----TGYYMHWVKQSHGKSLEWIGLIIPY--NGDTFYNQKFK-GKATLTVDTSSSTAYMELGSLTSEDSAVYYCAR----------------------',
                'IGHV1-37*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYFMNWVKQSHGKSLEWIGRINPY--NGDTFYNQKFK-GKATLTVDKSSSTAHMELLSLTSEDFAVYYCAR----------------------',
                'IGHV1-37*02': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYFMNWVKQSHGKSLEWIGRINPY--NGDTFYNQKFK-GKATLTVDKSSSTAHMELLSLTSEDSAVYYCGR----------------------',
                'IGHV1-37*03': 'EVQLQQSGP-DLVKPGDSVKISCKASGYSF----TGYFMNWVKPSHGKSLEWIGRINPY--NGVTFYNQKFK-GKATLTVDKSSSTAHMELLSLTSEDFAVYYCAR----------------------',
                'IGHV1-39*01': 'EFQLQQSGP-ELVKPGASVKISCKASGYSF----TDYNMNWVKQSNGKSLEWIGVINPN--YGTTSYNQKFK-GKATLTVDQSSSTAYMQLNSLTSEDSAVYYCAR----------------------',
                'IGHV1-39*02': 'EIQLQQTGP-ELVKPGASVKISCKASGYSF----TDYIMLWVKQSHGKSLEWIGNINPY--YGSTSYNLKFK-GKATLTVDKSSSTAYMQLNSLTSEDSAVYYCAR----------------------',
                'IGHV1-39*03': 'EIQLQQTGP-ELVKPGASVKISCKASGYSF----TDYIMLWVKQSHGKSLEWIGNINPY--YGSTSYNQKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-4*01': 'QVQLQQSGA-ELARPGASVKMSCKASGYTF----TSYTMHWVKQRPGQGLEWIGYINPS--SGYTKYNQKFK-DKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-4*02': 'QVQLQQSAA-ELARPGASVKMSCKASGYTF----TSYTMHWVKQRPGQGLEWIGYINPS--SGYTEYNQKFK-DKTTLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-42*01': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMNWVKQSPEKSLEWIGEINPS--TGGTTYNQKFK-AKATLTVDKSSSTAYMQLKSLTSEDSAVYYCAR----------------------',
                'IGHV1-42*04': 'EVQLQQSGP-ELEKPGASVKISCKASGYSF----TGYNMNWVKQSNGKSLEWIGNIDPY--YGGTSYNQKFK-GKATLTVDKSSSTAYMQLKSLTSEDSAVYYCAR----------------------',
                'IGHV1-42*05': 'EVQLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSPEKSLEWIGEINPS--TGGTTYNQKFT-GKATLTVDKSSSTAYMQLKSLTSEDSAVYYCAR----------------------',
                'IGHV1-42-1*01': 'EVQLQQSGP-ELVKPGASMKISCKASGYSF----TGYTMNWVKQSHGKNLEWIGLINPY--NGGTSYNQKFK-GKATLTVDKSSSTAYMELLSLTSEDSAVYYCAR----------------------',
                'IGHV1-43*01': 'EVKLQQSGP-ELVKPGASVKISCKASGYSF----TGYYMHWVKQSSEKSLEWIGEINPS--TGGTSYNQKFK-GKATLTVDKSSSTAYMQLKSLTSEDSAVYYCAR----------------------',
                'IGHV1-43*02': 'EVQLQQSGP-DLVKPGASVKISCKASGYSF----TGYYMHWVKQSHGKSLEWIGRVNPN--NGGTSYNQKFK-GKAILTVDKSSSTAYMELRSLTSEDSAVYYCAR----------------------',
                'IGHV1-47*01': 'QVQLQQSGA-ELVKPGASVKMSCKASGYTF----TTYPIEWMKQNHGKSLEWIGNFHPY--NDDTKYNEKFK-GKATLTVEKSSSTVYLELSRLTSDDSAVYYCAR----------------------',
                'IGHV1-47*02': 'QVQLQQSGA-ELVKPGASVKMSCKAFGYTF----TTYPIEWMKQNHGKSLEWIGNFHPY--NDDTKYNEKFK-GKAKLTVEKSSSTVYLELSRLTSDDSAVYYCAR----------------------',
                'IGHV1-47*03': 'QVQLQQSGA-ELVKPGASVKMSCKAFGYTF----TTYPIEWMKQNHGKSLEWIGNFHPY--NDDTKYNEKFK-GKAKLTVEKSSSTVYLELSRLTSDDSAVYYCAR----------------------',
                'IGHV1-47-1*01': 'QVQLQQSGA-ELVRPGTSVKISCKASGYTF----TNYWLGWVKQRPGHGLEWIGDIYPG--GGYTNYNEKFK-GKATLTADTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-11*01': 'QVQLQQPGA-ELVRPGASVKLSCKASGYSF----TSYWMNWVKQRPGQGLEWIGMIHPS--DSETRLNQKFK-DKATLTVDKSSSTAYMQLSSPTSEDSAVYYCAR----------------------',
                'IGHV1-47-13*01': 'QVQLQQPGA-ELVKPGASVKISCKASGYTF----TSYWMNWVKQRPGQGLEWIGEIDPS--DSYTNNNQKFK-DKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-47-14*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYYMYWVKQRPGQGLEWIGGINPS--NGGTNFNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCTR----------------------',
                'IGHV1-47-17*01': 'QVQLQQSGP-ELVRPGVSVKISCKGSGYTF----TDYAMHWVKQSHAKSLEWIGVISTY--YGNTNYNQKFK-GKATMTVDKSSSTAYMELARLTSEDSAIYYCAR----------------------',
                'IGHV1-47-17D*01': 'QVQLQQSGP-ELVRPGVSVKISCKGSGYTF----TDYAMHWVKQSHAKSLEWIGVISTY--YGNTNYNQKFK-GKATMTVDKSSSTAYMELARLTSEDSAIYYCAR----------------------',
                'IGHV1-47-18*01': 'QVQLQQPGA-ELVKPGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLEWIGTIDPS--DSYTSYNQKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYYCTR----------------------',
                'IGHV1-47-2*01': 'QVQLQQPGA-ELVRPGASVKLSCKASGYTF----TSYWINWVKQRPGQGLEWIGNIYPS--DSYTNYNQKFK-DKATLTVDKSSSTAYMQLSSPTSEDSAVYYCTR----------------------',
                'IGHV1-47-22*01': 'QVQLQQSDT-ELVKPGASVKISCKASGYTF----TDHAIHWVKQRPEQGLEWIGYISPG--NGDIKYNEKFK-GKATLTADKSSSTAYMQLNSLTSEDSAVYFCKR----------------------',
                'IGHV1-47-23*01': 'QVQLQQSGA-ELVRPGSSVKISCKASGYAF----SSYWMNWVKQRPGQGLEWIGQIYPG--DGDTNYNGKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-24*01': 'QVQLQQSGD-DLVKPGASVKLSCKASGYTF----TSYWINWIKQRPGQGLEWIGRIAPG--SGSTYYNEMFK-GKATLTVDTSSSTAYIQLSSLSSEDSAVYFCAR----------------------',
                'IGHV1-47-27*01': 'QVQLQQPGA-ELVKPGASVKMSCKASGYTF----TSYNMHWVKQTPGQGLEWIGAIYPG--NGDTSYNQKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-47-28*01': 'QVQLQQPGT-ELVKPGASVKLSCKASGYTF----TSYWMYWVKQRPGQGLEWIGEINPS--NGGTNYNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCTR----------------------',
                'IGHV1-47-29*01': 'QVQLQQPGA-ELVKPGSSVKMSCKASGYTF----TRYWMHWVKQRPGQGLEWIGNIYPG--RGSTNYNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-47-3*01': 'QVQLQQPGS-ELVRPGASVKLSCKASGYTF----TSYWMHWVKQRHGQGLEWIGNIYPG--SGSTNYDEKFK-SKGTLTVDTSSSTAYMHLSSLTSEDSAVYYCTR----------------------',
                'IGHV1-47-30*01': 'KVQLQQSGA-ELVKPGASVKLSCKASGYTF----TEYIIHWVKQRSGQGLEWIGWFYPG--SGSIKYNEKFK-DKATLTADKSSSTVYMELSRLTSEDSAVYFCARHE--------------------',
                'IGHV1-47-30D*01': 'KVQLQQSGA-ELVKPGASVKLSCKASGYTF----TEYIIHWVKQRSGQGLEWIGWFYPG--SGSIKYNEKFK-DKATLTADKSSSTVYMELSRLTSEDSAVYFCARHE--------------------',
                'IGHV1-47-31*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYAF----SSSWMNWVKQRPGQGLEWIGRIYPG--DGDTNYNGKFK-GKATLTADKSSSTAYMQLSSLTSVDSAVYFC------------------------',
                'IGHV1-47-32*01': 'QVQLQQPGA-ELVRPGASVKLSCKASGYTF----TSYWMNWVKQRPGQGLEWIGMIHPS--DSETRLNQKFK-DKATLTVDKSSSTAYMQLSSPTSEDSAVYYCAR----------------------',
                'IGHV1-47-33*01': 'EVQLQQSGA-ELGRPGSSVKLSCKTSGYTF----TSYGINWVKQRPGQGLEWIGYIYPG--NGYTAYNEKFQ-GEATLTSDTSSSTAYMQLRSLTSEDSAIYFCAR----------------------',
                'IGHV1-47-34*01': 'QVQLQQPGS-VLVRPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGEIYPN--SGRTNYNEKFK-GKATLTVDTSSSTAYMDLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-47-36*01': 'QVQLQQSGP-ELVKPGASVRISCKASGYTF----TSYYIHWVKQRPGQGLEWIGWIYPG--NGNTKYNEKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-37*01': 'QVQLQQPGA-ELVRPGASVKLSCKASGYTF----TSYWMNWVKQRPGQGLEWIGMIHPS--DSETRLNQKFK-DKATLTVDKSSSTAYMQLSSPTSEDSAVYYCAR----------------------',
                'IGHV1-47-39*01': 'QVQLQQPGA-EIVRPGASVKLSCKASGYTF----TDYWMNWVKQRPGQGLEWIGAIDPS--DSYTSYNQKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-40*01': 'EVQLQQSGA-ELVTPGYSVKLSCKTSGYTF----TSYDINWVKQRPGQGLEWIGYIYPG--NGYTEYNEMFK-GKATLTSDTSSSTAYMQLRSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-43*01': 'QVQLQQPGA-ELVRPGASVKLSCKASSYTF----TRYWMNWVKQRPEQGLEWIGRIDPY--DSETHYNQKFK-DKAILTVDKSSSTAYMQLSTLTSEDSAVYYCAR----------------------',
                'IGHV1-47-44*01': 'QVQLQQPGA-ELVKPGASVKMSCKASGYTF----TSYWINWVKLRPGQGLEWIGDIYPG--SGSTNYNEKFK-SKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-47-45*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYIF----TSYWMYWVKQRPGQGLERIGEINPS--NGGTNYNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCTR----------------------',
                'IGHV1-47-47*01': 'EVQLQQSGA-ELVRPGSSVKLSCKTSGYTF----TSYGINWVKQRPGQGLEWIGYIYPG--NGYTAYNEKFQ-GKATLTSDTSSSTAYMQLRSLTSEDSAIYFCAR----------------------',
                'IGHV1-47-48*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSDDSAVYFCAR----------------------',
                'IGHV1-47-48D*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSDDSAVYFCAR----------------------',
                'IGHV1-47-5*01': 'KVQLQQSGA-GLVKPGASVKLSCKASGYTF----TEYIIHWVKQRSGQGLEWIGWFYPG--SGSIKYNEKFK-DKATLTADKSSSTVYMELSRLTSEDSAVYFCARHE--------------------',
                'IGHV1-47-52*01': 'EVQLQQSGA-ELVRPGSSVKLSCKTSGYTF----TSYGINWVKQRPGQGLECIGYIYIG--NGNTEYNEKFK-SKATLTSDTSSSTAYMELSSLTSEDSAIYFCAR----------------------',
                'IGHV1-47-53*01': 'QVQLQQPGA-EIVRPGASVKLSCKASGYTF----TDYWMNWVKQRPGQGLEWIRAIDPS--DSYTSYNQKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-56*01': 'QVQLQQSGA-ELVKPGASVKISCKASGYAF----SSSWMNWVKQRPGKGLEWIGQIYPG--DGDTNYNGKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-58*01': 'QVQLQQSGA-ELARPGASVKLSCKASGYTF----TSYGISWVKQRTGQGLEWIGEIFPG--SGNTYYNEKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-47-6*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYAF----SSSWMNWVKQRPGQGLEWIGRIYPG--DGDTNYNGKFK-GKATLTADKSSSTAYMQLSSLTSVDSAVYFCAR----------------------',
                'IGHV1-47-7*01': 'QVQLQQSGP-ELVRPGVSVKISCKGSGYTF----TDYAMHWVKQSHAKSLEWIGVISTY--SGNTNYNQKFK-GKATMTVDKSSSTAYMELARLTSEDSAIYYCAR----------------------',
                'IGHV1-47-8*01': 'QVQLQQPGA-ELVRPGASVKLSCKASGYTF----TSYWMNWVKQRPEQGLEWIGRIDPY--DSETHYNQKFK-DKAILTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-47-9*01': 'QVQLQQPGA-ELVKPGTSVKLSCKASGYNF----TSYWINWVKLRPGQGLEWIGDIYPG--SGSTNYNEKFK-SKATLTVDTSSSTAYMQLSSLASEDSALYYCAR----------------------',
                'IGHV1-49*01': 'QRELQQSGA-ELVRPGSSVKLSCKDSYFAF----MASAMHWVKQRPGHGLEWIGSFTMY--SDATEYSENFK-GKATLTANTSSSTAYMELSSLTSEDSAVYYCAR----------------------',
                'IGHV1-5*01': 'EVQLQQSGT-VLARPGASVKMSCKTSGYTF----TSYWMHWVKQRPGQGLEWIGAIYPG--NSDTSYNQKFK-GKAKLTAVTSASTAYMELSSLTNEDSAVYYCTR----------------------',
                'IGHV1-5*02': 'EVQLQQSGT-VLARPGASVKMSCKASGYSF----TSYWMHWVKQRPGQGLEWIGAIYPG--NSDTSYNQKFK-GKAKLTAVTSASTAYMELSSLTNEDSAVYYCTR----------------------',
                'IGHV1-50*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMQWVKQRPGQGLEWIGEIDPS--DSYTNYNQKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-52*01': 'QVQLQQPGA-ELVRPGSSVKLSCKASGYTF----TSYWMHWVKQRPIQGLEWIGNIDPS--DSETHYNQKFK-DKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-53*01': 'QVQLQQPGT-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGNINPS--NGGTNYNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-54*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-54*02': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSNTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-55*01': 'QVQLQQPGA-ELVKPGASVKMSCKASGYTF----TSYWITWVKQRPGQGLEWIGDIYPG--SGSTNYNEKFK-SKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-56*01': 'QVQLQQSGP-ELVRPGASVKISCKAPGYTF----TSHWMQWVRQRPGQGLEWIGEIFPG--SGSTYYNEKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-58*01': 'EVQLQQSGA-ELVRPGSSVKMSCKTSGYTF----TSYGINWVKQRPGQGLEWIGYIYIG--NGYTEYNEKFK-GKATLTSDTSSSTAYMQLSSLTSEDSAIYFCAR----------------------',
                'IGHV1-59*01': 'QVQLQQPGA-ELVRPGTSVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGVIDPS--DSYTNYNQKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-61*01': 'QVQLQQPGA-ELVRPGSSVKLSCKASGYTF----TSYWMDWVKQRPGQGLEWIGNIYPS--DSETHYNQKFK-DKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-62-1*01': 'QVQLQQSGA-ELVRPGASVKLSCKASGYTF----TSYWMQWVKQRPGQGLEWIGEIFPG--SGSTYYNEKFK-GKATLTVDTSSSTAYMQLSSLTAENSAIYLCK-----------------------',
                'IGHV1-62-2*01': 'QVQLQQSGA-ELVKPGASVKLSCKASGYTF----TEYTIHWVKQRSGQGLEWIGWFYPG--SGSIKYNEKFK-DKATLTADKSSSTVYMELSRLTSEDSAVYFCARHE--------------------',
                'IGHV1-63*01': 'QVQLQQSGA-ELVRPGTSVKMSCKASGYTF----TNYWIGWAKQRPGHGLEWIGDIYPG--GGYTNYNEKFK-GKATLTADKSSSTAYMQFSSLTSEDSAIYYCAR----------------------',
                'IGHV1-63*02': 'QVQLQQSGA-ELVRPGTSVKMSCKAAGYTF----TNYWIGWVKQRPGHGLEWIGDIYPG--GGYTNYNEKFK-GKATLTADTSSSTAYMQLSSLTSEDSAIYYCAR----------------------',
                'IGHV1-64*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGMIHPN--SGSTNYNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-66*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYSF----TSYYIHWVKQRPGQGLEWIGWIYPG--SGNTKYNEKFK-GKATLTADTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69*01': 'QVQLQQPGA-ELVMPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGEIDPS--DSYTNYNQKFK-GKSTLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69*02': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGEIDPS--DSYTNYNQKFK-GKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69*03': 'QVQLQQPGA-ELVMPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGEIDPS--DSYTNYNQKFK-GKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69-1*01': 'HVQLQQSGP-ELVRPGASVKLSCKASGYIF----ITYWMNWVKQRPGQGLEWIGQIFPA--SGSTNYNEMFE-GKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69-2*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSDDSAVYFCAR----------------------',
                'IGHV1-69-2D*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSDDSAVYFCAR----------------------',
                'IGHV1-69-3*01': 'QVQLQQPGA-ELVKPGASVKMSCKASGYTF----TSYWINWVKQRPGQGLEWIGDIYPG--RGITNYNEKFK-SKATLTLDTSSSTAYMQLSSLTSEDSAVYYCSR----------------------',
                'IGHV1-69-4*01': 'PVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGRGLEWIGRIDPN--SGGTKYNEKFK-SKATLTVDKPSSTAYMQLSSLTSEDSAVYYCTR----------------------',
                'IGHV1-69-5*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGEINPS--NGRTNYNEKFK-SKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69-6*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYTF----TSYYIHWVKQRPGQGLEWIGYIYPR--DGSTNYNEKFK-GKATLTADTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-69-7*01': 'QVQLQQSGA-ELVRPGVSVKISCKGSGYTF----TDYAMHWVKQSHAKSLEWIGVISTY--YGDASYNQKFK-GKATMTVDKSSSTAYMELARLTSEDSAVYYCAR----------------------',
                'IGHV1-69-8*01': 'QVQLQQSGP-ELVRPGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLEWIGMIDPS--NSETRLNQKFK-DKATLNVDKSSNTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-69-9*01': 'QVQLQQSGA-ELVKPGASVKLSCKTSGYTF----TSYWIQWVKQRPGQGLGWIGEIFPG--TGTTYYNEKFK-GKATLTIDTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-7*01': 'QVQLQQSGA-ELAKPGASVKLSCKASGYTF----TSYWMHWVKQRPGQGLEWIGYINPS--SGYTKYNQKFK-DKATLTADKSSSTAYMQLSSLTYEDSAVYYCAR----------------------',
                'IGHV1-7*02': 'QVQLQQSGA-ELAKPGASVKMSCKASGYTF----TSYWMHWVKQRPGQGLEWIGYINPS--TGYTEYNQKFK-DKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-7*03': 'QVQLQQSGA-ELAKPGASVKMSCKASGYTF----TSHWMHWVKQRPGQGLEWIGYINPS--SGHTGYNQKFK-DKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-71*01': 'QVQLQQSGA-ELVKPGASVKLSCKASGYTF----TEYTIHWVKQRSGQGLEWIGWFYPG--SGSIKYNEKFK-DKATLTADKSSSTVYMELSRLTSEDSAVYFCARHE--------------------',
                'IGHV1-71-1*01': 'QVQLQQSGP-ELVKPGASVKMSCKASGYTF----TSYYIHWVKQRPGQGLEWIGWIYPG--DGSTKYNEKFK-GKTTLTADKSSSTAYMLLSSLTSEDSAIYFCAR----------------------',
                'IGHV1-71-10*01': 'QVQLKQSGA-ELVRPGASVKLSCKTSGYIF----TSYWIHWVKQRSGQGLEWIARIYPG--TGSTYYNEKFK-GKATLTADKSSSTAYMQLSSLKSEDSAVYFCAR----------------------',
                'IGHV1-71-11*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYAF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSDDSAVYFCAR----------------------',
                'IGHV1-71-12*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKLRPGQGFEWIGEINPS--NGGTNYNEKFK-RKATLTVDKSSSTAYMQLSSLTSEDSAVYYCTI----------------------',
                'IGHV1-71-15*01': 'QVQLQQSDA-ELVKPGASVKISCKASGYTF----TDHAIHWVKQKPEQGLEWIGYISPG--NGDIKYNEKFK-GKATLTADKSSSTAYMQLNSLTSEDSAVYFCKR----------------------',
                'IGHV1-71-16*01': 'QVQLQQSGP-ELVKPGASVKMSCKASGYTF----TDYVISWVKQRTGQGLEWIGEIYPG--SGSTYYNEKFK-GKATLTADKSSNTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-71-2*01': 'QVQLQQSGP-ELVRPGESVKISCKGSGYTF----TDYAMHWVKQSHAKSLEWIGVISIY--YDNTNYNQKFK-GKATMTVDKSSSTAYMELARLTSEDSAIYYCAR----------------------',
                'IGHV1-71-3*01': 'QVQLQQPGA-ELVKPGAPVKLSCKASGYTF----TSYWMNWVKQRPGRGLEWIGRIDPS--DSETHYNQKFK-DKATLTVDKSSSTAYIQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-71-4*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYSF----TSYYIHWVKQRPGQGLEWIGWIFPG--SGNTKYNEKFK-GKATLTADTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-71-5*01': 'QVQLQQSGP-ELVRPGVSVKISCKGSSYTF----TDYAMHWVKQSHAKSLEWIGVISTY--YGNTNYNQKFK-GKATMTVDKSSSTAYMELARLTSEDSAVYYCAR----------------------',
                'IGHV1-71-6*01': 'QVQLQQPGA-ELVMPGASVKMSCKASGYTF----TDYWMHWVKQRPGQGLEWIGAIDTS--DSYTSYNQKFK-GKATLTVDESSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-71-8*01': 'QVQLQQSGA-ELARPGASVKLSCKASGYTF----TDYYINWVKQRTGQGLEWIGEIYPG--SGNTYYNEKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-71-9*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYTF----TSYGINWVKQRPGQGLEWIGYIYPG--SGGTAYNQKFK-GKATLTADKSSSTVYMQLSSLTSEDSAIYFCAR----------------------',
                'IGHV1-72*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGRGLEWIGRIDPN--SGGTKYNEKFK-SKATLTVDKPSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1-72*04': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGRGLEWIGRIDPN--SGGTKYNEKFK-SKATLTVDTSSSTAYMQLSSLTSEDSAVHYCAR----------------------',
                'IGHV1-74*01': 'QVQLQQPGA-ELVKPGASVKVSCKASGYTF----TSYWMHWVKQRPGQGLEWIGRIHPS--DSDTNYNQKFK-GKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAI----------------------',
                'IGHV1-74*04': 'HVQLQQPGA-ELVKPGASVKVSCKASGYTF----TSYWMHWVKQRPGQGLEWIGRIHPS--DSDTNYNQKFK-GKATLTVDKSSSTAYMQLSSLTSEDSAVYYCAI----------------------',
                'IGHV1-75*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYTF----TDYYINWVKQRPGQGLEWIGWIFPG--SGSTYYNEKFK-GKATLTVDKSSSTAYMLLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-76*01': 'QVQLKQSGA-ELVRPGASVKLSCKASGYTF----TDYYINWVKQRPGQGLEWIARIYPG--SGNTYYNEKFK-GKATLTAEKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-77*01': 'QVQLKQSGA-ELVKPGASVKISCKASGYTF----TDYYINWVKQRPGQGLEWIGKIGPG--SGSTYYNEKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-78*01': 'QVQLQQSDA-ELVKPGASVKISCKVSGYTF----TDHTIHWMKQRPEQGLEWIGYIYPR--DGSTKYNEKFK-GKATLTADKSSSTAYMQLNSLTSEDSAVYFCAR----------------------',
                'IGHV1-80*01': 'QVQLQQSGA-ELVKPGASVKISCKASGYAF----SSYWMNWVKQRPGKGLEWIGQIYPG--DGDTNYNGKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-81*01': 'QVQLQQSGA-ELARPGASVKLSCKASGYTF----TSYGISWVKQRTGQGLEWIGEIYPR--SGNTYYNEKFK-GKATLTADKSSSTAYMELRSLTSEDSAVYFCAR----------------------',
                'IGHV1-82*01': 'QVQLQQSGP-ELVKPGASVKISCKASGYAF----SSSWMNWVKQRPGKGLEWIGRIYPG--DGDTNYNGKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-84*01': 'QIQLQQSGP-ELVKPGASVKISCKASGYTF----TDYYINWVKQRPGQGLEWIGWIYPG--SGNTKYNEKFK-GKATLTVDTSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-84*02': 'QIQLQQSGP-ELVKPGASVKISCKASGYTF----TDYYINWVKQKPGQGLEWIGWIYPG--SGNTKYNEKFK-GKATLTVDTSSSTAYMQLSSLTSEDTAVYFCAR----------------------',
                'IGHV1-85*01': 'QVQLQQSGP-ELVKPGASVKLSCKASGYTF----TSYDINWVKQRPGQGLEWIGWIYPR--DGSTKYNEKFK-GKATLTVDTSSSTAYMELHSLTSEDSAVYFCAR----------------------',
                'IGHV1-85*02': 'QVQLQQSGA-ELVKPGASVKLSCKASGYTF----TSYDINWVRQRPEQGLEWIGWIFPG--DGSTKYNEKFK-GKATLTTDKSSSTAYMQLSRLTSEDSAVYFCAR----------------------',
                'IGHV1-85*03': 'QVQLQQSGA-ELVKPGASVKLSCKASDYTF----TSYDINWVKQRPGQGLEWIGWIYPG--SGNTKYNEKFK-GKATLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1-86*02': 'QVQLQQSGA-ELVRPGASVKISCKAFGYTF----TNHHINWVKQRPGQGLDWIGYINPY--NDYTSYNQKFK-GKATLTVDKSSSTAYMELSSLTSEDSAVYYCAR----------------------',
                'IGHV1-87*01': 'QVQLQQSGA-ELARPGASVKMSCKASGYTF----TSYWMQWVKQRPGQGLEWIGAIYPG--DGDTRYTQKFK-GRATLTADKSSSTAYMQLSSLTSEDSAVYYCAT----------------------',
                'IGHV1-87*02': 'QVQLQQSGA-ELARPGASVKLSCKASGYTF----TSYWMQWVKQRPGQGLEWIGAIYPG--DGDTRYTQKFK-GKATLTADKSSSTAYMQLSSLASEDSAVYYCAR----------------------',
                'IGHV1-9*01': 'QVQLQQSGA-ELMKPGASVKLSCKATGYTF----TGYWIEWVKQRPGHGLEWIGEILPG--SGSTNYNEKFK-GKATFTADTSSNTAYMQLSSLTTEDSAIYYCAR----------------------',
                'IGHV1-9*02': 'QVQLQQSGA-ELMKPGASVKISCKATGYTF----SSYWIEWVKQRPGHGLEWIGEILPG--SGSTNYNEKFK-GKATFTADTSSNTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV10-1*01': 'EVQLVESGG-GLVQPKGSLKLSCAASGFSF----NTYAMNWVRQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSESMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-1*02': 'EVQLVESGG-GLVQPKGSLKLSCAASGFTF----NTYAMNWVRQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVS----------------------',
                'IGHV10-1*03': 'EVQLVESGG-GLVQPKGTLKLSCAASGFSF----NTYAMNWVRQAPGKSLEWVARTMSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-1*04': 'EVQLVESGG-GLVQPKGSLKLSCAASGFTF----NTYAMNWVRQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-1-2*01': 'EVQLVETGG-GLVQPKGSLKLSCAASGFTF----NTNAMNWVRQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-1-4*01': 'EVQLVESGG-GLVQPKGSLKLSCATSGFTF----NTYAMHWVRQAPGKGLEWVARIRSKSYNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-1-6*01': 'EVQLVESGG-RLVQPKGSLKLSCAASGFTF----NTYAMYWIRQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-1-8*01': 'EVQLVESAG-GLVQPKGSLKLSCAASGFSF----NTYAMNWVRKAPGKGLEWVARTMSKNNNYATYYADSVK-DRFTITRDDSQSMLYLQMNNLKTEDTAMYYCVK----------------------',
                'IGHV10-3*01': 'EVQLVESGG-GLVQPKGSLKLSCAASGFTF----NTYAMHWVRQAPGKGLEWVARIRSKSSNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-3*02': 'EVQLVESGG-GLVQPKGSLKLSCAASVFTF----NTYAMHWVRQAPGKGLEWVARIRSKSSNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-3*03': 'EVQLVESGG-GLVQPKGSLKLSCAASVFTF----NTYAMHWVCQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV10-3*04': 'EVQLVESGG-GLVQPKGSLKLSCAASGFTF----NTYAMHWVCQAPGKGLEWVARIRSKSNNYATYYADSVK-DRFTISRDDSQSMLYLQMNNLKTEDTAMYYCVR----------------------',
                'IGHV11-1*01': 'EVQLLETGE-GLVPPGGSRGLSCEGSGFTF----SGFWMSWVRQTPGKTLEWIGDINSD--GSAINYAPSIK-DRFTIFRDNDKSTLYLQMSNVRSEDTATYFCMR----------------------',
                'IGHV11-1*02': 'EVQLLETGG-GLVQPGGSRGLSCEGSGFTF----SGFWMSWVRQTPGKTVEWIGDINSD--GSAINYAPSIK-DRFTIFRDNDKSTLYLQMSNVRSEDPATYFCMR----------------------',
                'IGHV11-1*03': 'EVQLLETGG-GLVQPGGSRGLSCEGSGFTF----SGFWMSWVRQTPGKTLEWIGDINSD--GSAINYAPSIK-DRFTIFRDNDKSTLYLQMSNVRSEDPATYFCMR----------------------',
                'IGHV11-2*01': 'EVQLLETGG-GLVQPGGSRGLSCEGSGFTF----SGFWMSWVRQTPGKTLEWIGDINSD--GSAINYAPSIK-DRFTIFRDNDKSTLYLQMSNVRSEDTATYFCMR----------------------',
                'IGHV11-2*02': 'EVQLLETGG-GLVQPGGSRGLSCEGSGFTF----SGFWMSWVRQTPGKTLEWIGDINSD--GSAINYAPSIK-DRFTIFRDNDKSTLYLQMSNVRSEDTATYFCMR----------------------',
                'IGHV11-2*03': 'EVQLLETGG-GLVQPGGSRGLSCEGSGFTF----SGFWMSWVRQTPGKTLEWIGDINSD--GSAINYAPSIK-DRFTIFRDNDKSTLYLQMSNVRSEDPATYFCMR----------------------',
                'IGHV12-1-1*01': 'QIQLKESGP-AVIKPSQSLSLTCIVSGFSIT--SSSYCWHWIRQPPGKGLEWMGRICYE---GSIYYSPSIK-SRSTISRDTSLNKFFIQLSSVTNEDTAMYYCSREN--------------------',
                'IGHV12-1-1*02': 'QIQLKESGP-AVIKPSQSLSLTCTVSGFSIT--SSSYCWHWIRQPPGKGLEWMGRICYE---GSIYYSPSIK-SRSTISRDTSLNKFFIQLSSVTNEDTAMYYCSREN--------------------',
                'IGHV12-1-4*01': 'QIQLKESGP-AVIKPSQSLSLTCTVSGFSIT--SSGFCWHWIRQAPGKGLEWMGSICYE---GSVYYSPSFK-SRSTISRDTSLNKFFIQLSSVTDEDTAMYYCSREN--------------------',
                'IGHV12-1-5*01': 'QIQLKESGP-AVIKPSQSLSLTCTVSGFSIT--SSSYCWHWIRQPPGKGLEWMGRICYE---GSLNNSPSLK-SRSTISRDTSLNKFFIQLSSVTNEDTAMYYCSREN--------------------',
                'IGHV12-3*01': 'QMQLQESGP-GLVKPSQSLFLTCSITGFPIT---SGYYWIWIRQSPGKPLEWMGYITHS---GETFYNPSLQ-SPISITRETSKNQFFLQLNSVTTEDTAMYYCAGDR--------------------',
                'IGHV12-3*02': 'QMQLQESGP-GLVKPSQSLFLACSITGFPIT---SGYYWIWIRQSPGKPLEWMGYITHS---GETFYNPSLQ-SPISITRETSKNQFFLQLNSVTTEDTAMYYCAGDR--------------------',
                'IGHV12-3*03': 'QMQLQESGP-GLVKPSQSLFLACSITGFPIT---SGYYWIWIRQSPGKPLEWMGYITHS---GETFYNPSLQ-SPISITRETSKNQFFLQLNSVTTEDTAMYYCAGDR--------------------',
                'IGHV13-2*01': 'QVQLVETGG-GLVRPGNSLKLSCVTSGFTF----SNYRMHWLRQPPGKRLEWIAVITVKSDNYGANYAESVK-GRFAISRDDSKSSVYLEMNRLREEDTATYFCSR----------------------',
                'IGHV13-2*02': 'QVQLVETGG-GLVRPGNSLKLSCVTSGFTF----SNYRMHWLRQPPGKRLEWIAVITVKSDNYGANYAESVK-GRFTISRDDSKSSVYLQMNRLREEDTATYYCSR----------------------',
                'IGHV13-2*04': 'QVQLVETGG-GLARPGNSLKLSCVTSGFTF----SNYRMHWLRQPPGKRLEWIAVITVKSDNYGANYAESVK-GRFTISRDDSKSSVYLQMNRLREEDTATYYCSR----------------------',
                'IGHV14-1*01': 'EVQLQQSGA-ELVRPGASVKLSCTASGFNI----KDYYMHWVKQRPEQGLEWIGRIDPE--DGDTEYAPKFQ-GKATMTADTSSNTAYLQLSSLTSEDTAVYYCTT----------------------',
                'IGHV14-1*02': 'EVQLQQSGA-ELVRPGALVKLSCKASGFNI----KDYYMHWVKQRPEQGLEWIGWIDPE--NGNTIYDPKFQ-GKASITADTSSNTAYLQLSSLTSEDTAVYYCAR----------------------',
                'IGHV14-2*01': 'EVQLQQSGA-ELVKPGASVKLSCTASGFNI----KDYYMHWVKQRTEQGLEWIGRIDPE--DGETKYAPKFQ-GKATITADTSSNTAYLQLSSLTSEDTAVYYCAR----------------------',
                'IGHV14-3*01': 'EVQLQQSVA-ELVRPGASVKLSCTASGFNI----KNTYMHWVKQRPEQGLEWIGRIDPA--NGNTKYAPKFQ-GKATITADTSSNTAYLQLSSLTSEDTAIYYCAR----------------------',
                'IGHV14-3*02': 'EVQLQQSGA-ELVKPGASVKLSCTASGFNI----KDTYMHWVKQRPEQGLEWIGRIDPA--NGNTKYDPKFQ-GKATITADTSSNTAYLQLSSLTSEDTAVYYCAR----------------------',
                'IGHV14-3-1*01': 'EVQLQQSGA-ELVKPGASVKLSCTASGFNI----KDTYMHWVKQRPEQGLEWIGRIDPA--NGNTKYASKFQ-GKATITADTSSNTVYMQLSSLTSEDTAVYYCAR----------------------',
                'IGHV14-4*01': 'EVQLQQSGA-ELVRPGASVKLSCTASGFNI----KDDYMHWVKQRPEQGLEWIGWIDPE--NGDTEYASKFQ-GKATITADTSSNTAYLQLSSLTSEDTAVYYCTT----------------------',
                'IGHV14-4*02': 'EVQLQQSGA-ELVRSGASVKLSCTASGFNI----KDYYMHWVKQRPEQGLEWIGWIDPE--NGDTEYAPKFQ-GKATMTADTSSNTAYLQLSSLTSEDTAVYYCNA----------------------',
                'IGHV15-2*01': 'QVHLQQSGS-ELRSPGSSVKLSCKDFDSEVF---PIAYMSWVRQKPGHGFEWIGGILPS--IGRTIYGEKFE-DKATLDADTLSNTAYLELNSLTSEDSAIYYCAR----------------------',
                'IGHV15-2*02': 'QVHLQQSGS-ELRSPGSSVKLSCKDFDSEVF---PIAYMSWVRQKPGHGFEWIGDILPS--IGRTIYGEKFE-DKATLDADTVSNTAYLELNSLTSEDSAIYYCAR----------------------',
                'IGHV15-2*03': 'QVHLQQSGS-ELRSPGSSVKLSCKDFDSEVF---PIAYMSWVRQKPGHGFEWIGGILPS--IGRTIYGEKFE-DKATLDADTMSNTAYLELNSLTSEDSAIYYCAR----------------------',
                'IGHV1S14*01': 'QVQLQQPGS-VLVRPGTSVKLSCKASGYTF----TSYWMHWAKQRPGQGLEWIGEIHPN--CGNINYNEKFK-GKATLTVDTSSSTAYVDLSSLTSEDSAVYYCAR----------------------',
                'IGHV1S26*01': 'QVQLQQSGA-ELVKTGASVKMSCKASGYTF----TSYTMHWVKQRPGQGLEWIGYINPS--SGYTNYNQKFK-DKATLTADKSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1S29*02': 'EVQLQQSGP-ELVKPGASVKISCKASGYTF----TDYNMHWVKQSHGKSLEWIGYIYPY--NGGTGYNQKFK-SKATLTVDNSSSTAYMELSSLTSEDSAVYYCAR----------------------',
                'IGHV1S35*01': 'QVQLQQPGA-VLVRHGASVKLSCKASGYTF----TSSWMHWAKQRHGQGLEWIGEIHPN--SGNTNYNEKFK-GKATLTVDKSSSTAYVDLSSLTSEDSAVYYCAR----------------------',
                'IGHV1S36*01': 'PVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGRGLEWIGNIDPN--SGGTKYNEKFK-SKATLTVDKPSSTAYMQLSSLTSEDSAVYYCTR----------------------',
                'IGHV1S40*01': 'QVQLQQSGP-ELVRPGLSVKLSCKASGYIF----ITYWMNWVKQRPGQGLEWIGQIFPA--SGSTNYNEMFE-GKATLTVDTSSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1S49*01': 'EVQLQQSGA-ELVRPGSSVKLSCKTSGYTF----TSYGINWVKQRPGQGLEWIGYIYLG--NGYTAYNEKFK-GKATLTSDTSSSTAYMQLRSLTSEDSVIKFCAR----------------------',
                'IGHV1S5*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWMHWVKQRPGEGLEWIGNIYPG--SSSTNYNEKFK-SKATLTVDTPSSTAYMQLSSLTSEDSAVYYCAR----------------------',
                'IGHV1S50*01': 'QVQLQQSGA-ELVKPGASVRISCKTSGYTF----TSYNIHWVKERPGQGLEWIGWIYPG--DGNTKYNEKFK-GKTTLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1S52*01': 'QVQLQQSGA-ELVRPGTSVKVSCKASGYVF----TNYLIEWVKQRPGQGLEWIGVINPG--SGGTNYNEKFK-GKATLTADKSSSTAYMQLSSLTSDDSAVYFCAR----------------------',
                'IGHV1S53*01': 'QVQLQQSDA-ELVKPGASVKISCKASGYTF----TDHAIHWVKQKPEQGLEWIGYISPG--NGDIKYNEKFK-GKATLTADKSSSTAYMQLNSLTSEDSAVYFCKR----------------------',
                'IGHV1S55*01': 'QVQLQQPGS-VLVRPGASVKLSCKASGYTF----TSYWMNWVKQRPGQGLEWIGGIYPN--SGSTDYNEKFK-GKATLTVDTSSSTTYMDLSSLTSKDSAVYYCAR----------------------',
                'IGHV1S56*01': 'QVQLQQSGP-ELVKPGASVRISCKASGYTF----TSYNIHWVKQRPGQGLEWIGWIYPG--DGNTKYNEKFK-GKTTLTADKSSSTAYMQLSSLTSEDSAVYFCAR----------------------',
                'IGHV1S61*01': 'QVQLQQPGA-ELVKPGASVKLSCKASGYTF----TSYWINWVKQRPGQGLEWIGNIYPG--SSSTNYNEKFK-SKATLTVDTSSSTAYMQLSSLTSDDSAVYYCAR----------------------',
                'IGHV2-2*01': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQADDTAIYYCAR----------------------',
                'IGHV2-2*02': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQANDTAIYYCAR----------------------',
                'IGHV2-2*03': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQSNDTAIYYCAR----------------------',
                'IGHV2-2-2*01': 'QVQMKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQADDTAIYYCVR----------------------',
                'IGHV2-2-2*02': 'QVQMKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQADDTAIYYCVR----------------------',
                'IGHV2-3*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVSWVRQPPGKGLEWLGVIWGD---GSTNYHSALI-SRLSISKDNSKSQVFLKLNSLQTDDTATYYCAK----------------------',
                'IGHV2-3-1*01': 'QVQLKESGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAPFI-SRLSISKDNSKSQIFFKMNSLQADDTAIYYCAR----------------------',
                'IGHV2-4*01': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQADDTAIYYCAK----------------------',
                'IGHV2-4*02': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQADDTAIYYCAR----------------------',
                'IGHV2-4-1*01': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWSG---GSTDYNAAFI-SRLSISKDNSKSQVFFKMNSLQADDTAIYYCAR----------------------',
                'IGHV2-5*01': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWRG---GSTDYNAAFM-SRLSITKDNSKSQVFFKMNSLQADDTAIYYCAK----------------------',
                'IGHV2-5-1*01': 'QVQLKQSGP-SLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWRG---GSTDYNAAFM-SRLSITKDNSKSQVFFKMNSLQADDTAIYYCAK----------------------',
                'IGHV2-5D*01': 'QVQLKQSGP-GLVQPSQSLSITCTVSGFSL----TSYGVHWVRQSPGKGLEWLGVIWRG---GSTDYNAAFM-SRLSITKDNSKSQVFFKMNSLQADDTAIYYCAK----------------------',
                'IGHV2-6*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVDWVRQSPGKGLEWLGVIWGV---GSTNYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAS----------------------',
                'IGHV2-6*02': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLVVIWSD---GSTTYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-6*03': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLVVIWSD---GSTTYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-6-1*01': 'QVQLKESGP-GLVAPSQSLSITCTISGFSL----TSYGVHWVRQPPGKGLEWLVVIWSD---GSTTYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-6-2*01': 'QVQLKESGP-DLVAPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLVVIWSD---GSTTYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-6-3*01': 'QVQ-NKSGP-GLVEPSQSLSITCTVYWFSL----TSYGVSWVRQPPGKGLKWLGVIWAG---GSTNYNSALI-SRLSISKDNSKSQVFLKMNSLQTDDTAIYYCVR----------------------',
                'IGHV2-6-4*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----SRYSVHWVRQPPGKGLEWLGMIWGG---GSTDYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-6-5*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TDYGVSWIRQPPGKGLEWLGVIWGG---GSTYYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAK----------------------',
                'IGHV2-6-6*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TNSGVHWVRQSPGKGLEWLGVIWGD---GSTNYNSAFK-SRLSISKDNSKSQVFLKMNSLQTDDTARYYCAK----------------------',
                'IGHV2-6-7*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TGYGVNWVRQPPGKGLEWLGMIWGD---GSTDYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTARYYCAR----------------------',
                'IGHV2-6-7*02': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TGYGVNWVRQPPGKGLEWLGMIWGD---GSTDYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTARYYCAR----------------------',
                'IGHV2-6-8*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVDWVRQSPGKGLEWLGVIWGV---GSTNYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAS----------------------',
                'IGHV2-7*01': 'QVQMKESGP-DLVQPSQTLSLTCTVSGFSL----SSYGVHWFRKPPRKGLEWLGGIWSG---GSIYYTPALS-SRLSVSRDTSKSQVFFKMSSLQSEDTAVYHCAR----------------------',
                'IGHV2-7*03': 'QVQMKESGP-DLVQPSQTLSLTCTVSGFSL----SSYGVHWFRKPPRKGLEWLGGIWSG---GSIYYNPALS-SRLSVSRDISKSQVFFKMSSLQSEDTAVYHCAR----------------------',
                'IGHV2-9*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVDWVRQPPGKGLEWLGVIWGG---GSTNYNSALM-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAK----------------------',
                'IGHV2-9*02': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLGVIWAG---GSTNYNSALM-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-9*03': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYGVHWVRQPPGKGLEWLGVIWAG---GSTNYNSALI-SRLSISKDNSKSQVFLKMNSLQTDDTAMYYCAR----------------------',
                'IGHV2-9-1*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYAISWVRQPPGKGLEWLGVIWTG---GGTNYNSALK-SRLSISKDNSKSQVFLKMNSLQTDDTARYYCAR----------------------',
                'IGHV2-9-2*01': 'QVQLKESGP-GLVAPSQSLSITCTVSGFSL----TSYDISWIRQPPGKGLEWLGVIWTG---GGTNYNSAFM-SRLSISKDNSKSQVFLKMNSLQTDDTAIYYCVR----------------------',
                'IGHV2S3*01': 'QVQLKQSGP-GLVAPSQSLFITCTVYGFSL----TSYEINWVRQPPGKGLEWLGVIWTG---GSTNYNSALI-SRLSISKDNSKSLVFLKMNSLQTDDTAIYYCVR----------------------',
                'IGHV3-1*01': 'DVQLQESGP-GMVKPSQSLSLTCTVTGYSIT---SGYDWHWIRHFPGNKLEWMGYISYS---GSTNYNPSLK-SRISITHDTSKNHFFLKLNSVTTEDTATYYCAR----------------------',
                'IGHV3-1*02': 'DVQLQESGP-DLVKPSQSLSLTCTVTGYSIT---SGYSWHWIRQFPGNKLEWMGYIHYS---GSTNYNPSLK-SRISITRDTSKNQFFLQLNSVTTEDTATYYCAR----------------------',
                'IGHV3-2*02': 'DVQLQESGP-GLVKPSQSLSLTCTVTGYSIT---SDYAWNWIRQFPGNKLEWMGYISYS---GSTSYNPSLK-SRISITRDTSKNQFFLQLNSVTTEDTATYYCAR----------------------',
                'IGHV3-2-1*01': 'DVQLQESGP-GLVKPSQSLSLTCTVTGYSIT---SGYAWNWIRQFPGNKLEWMGYIIYS---GSTNYNPSLK-SRISITRDTSKNQFFLQLNSVTTEDTATYYCAR----------------------',
                'IGHV3-3*01': 'DVQLQESGP-SLVRPSQTLSLTCTVTGFSIN---SDCYWIWIRQFPGNKLEYIGYTFYS---GITYYNPSLE-SRTYITRDTSKNQFSLKLSSVTTEDTATYYCAR----------------------',
                'IGHV3-4*01': 'DVQLQESGP-ALVKPSQTVSLTCTVTGYSIT--NGNHWWNWIRQVSGSKLEWIGYISSS---GSTDSNPSLK-SRISITRDTSKNQLFLQLNSVTTEDIATYYCAR----------------------',
                'IGHV3-4*02': 'DVQLQESGP-GLVKPSQTVSLTCTVTGYSIT--NGNHWWNWIRQVSGNKLEWMGYISSS---GSTDSNPSLK-SQISITRDTSKNQLFLQLNSVTIEDIATYYCAR----------------------',
                'IGHV3-5*01': 'DVQLQESGP-GLVKPSQTVFLTCTVTGISIT--TGNYRWSWIRQFPGNKLEWIGYIYYS---GTITYNPSLT-SRTTITRDTPKNQFFLEMNSLTAEDTATYYCAR----------------------',
                'IGHV3-5*02': 'DVQLQESGP-GLVKPSQTVSLTCTVTGISIT--TGNYRWSWIRQFPGNKLEWIGYIYYS---GTITYNPSLT-SRTTITRDTSKNQFFLEMNSLTAEDTATYYCAR----------------------',
                'IGHV3-6*01': 'DVQLQESGP-GLVKPSQSLSLTCSVTGYSIT---SGYYWNWIRQFPGNKLEWMGYISYD---GSNNYNPSLK-NRISITRDTSKNQFFLKLNSVTTEDTATYYCAR----------------------',
                'IGHV3-6*02': 'DVQLQESGP-GLVKPSQSLSLTCSVTGYSIT---SGYYWNWIRQFPGNKLEWMGYISYD---GSNNYNPSLK-NRISITRDTSKNQFFLKLNSVTTEDTATYYCAR----------------------',
                'IGHV3-6*04': 'DVQLQESGP-GLVKPSQSLSLTCSVTGYSIT---SDYYWHWIRQFPGNKLEWMGYISYD---GSNNYNPSLK-NRISITRDTSKNQFFLKLNSVTTEDTATYYCAR----------------------',
                'IGHV3-8*01': 'EVQLQESGP-GLAKPSQTLSLTCSVTGYSI----TSDYWNWIRKFPGNKLEYMGYISYS---GSTYYNPSLK-SRISITRDTSKNQYYLQLNSVTTEDTATYYCAR----------------------',
                'IGHV3-8*02': 'EVQLQESGP-SLVKPSQTLSLTCSVTGDSI----TSGYWNWIRKFPGNKLEYMGYISYS---GSTYYNPSLK-SRISITRDTSKNQYYLQLNSVTTEDTATYYCAR----------------------',
                'IGHV3-8*03': 'EVQLQESGP-GLVKPSQTLSLTCSVTGYSI----TSDYWNWLRKFPGNKLEYMGYISYS---GSTYYNPSLK-SRISITRDTSKNQYYLQLNSVTSEDTATYYCAR----------------------',
                'IGHV3S1*01': 'EVQLQESGP-SLVKPSQTLSLTCSVTGDSI----TSDYWNWIRKFPGNKLEYMGYISYS---GSTYYNPSLK-SRISITRDTSKNQYYLQLNSVTSEDTATYYCAR----------------------',
                'IGHV3S1*02': 'EVQLQESGP-SLVKPSQTLSLTCSVTGDSI----TSGYWNWIRKFPGNKLEYMGYISYS---GSTYYNPSLK-SRISITRDTSKNQYYLQLNSVTTEDTPTYYCAR----------------------',
                'IGHV4-1*01': 'EVKLLQSGG-GLVQPGGSLKLSCAASGIDF----SRYWMSWVRRAPGKGLEWIGEINPD--SSTINYAPSLK-DKFIISRDNAKNTLYLQMSKVRSEDTALYYCAR----------------------',
                'IGHV4-1*02': 'EVKLLESGG-GLVQPGGSLKLSCAASGFDF----SRYWMSWVRQAPGKGLEWIGEINPD--SSTINYTPSLK-DKFIISRDNAKNTLYLQMSKVRSEDTALYYCAR----------------------',
                'IGHV4-2*02': 'EVKLLESGG-GLVQPGGSLNLSCAASGFDF----SRYWMSWARQAPGKGQEWIGEINPG--SSTINYTPSLK-DKFIISRDNAKNTLYLQMSKVRSEDTALYYCAR----------------------',
                'IGHV4-2-1*01': 'EVKLLESGG-GLVQPGGSLNLSCAASGFDF----SKDWMSWVRQAPGKGLEWIGEINPG--SSTINYAPSLK-DKFIISRDNAKNTLYLQMSKVRSEDTALYYCAR----------------------',
                'IGHV5-1-1*01': 'ELQLVESGG-GLVQPGGSLKLSCAASGFTF----SSHFMAWVRQTPEKRLEWAANIYPD--GGTAYYPNTVK-GRFTISRDNARNIMYLQMSSLRSEDTAMYYCVRQD--------------------',
                'IGHV5-12*01': 'EVKLVESGG-GLVQPGGSLKLSCAASGFTF----SDYYMYWVRQTPEKRLEWVAYISNG--GGSTYYPDTVK-GRFTISRDNAKNTLYLQMSRLKSEDTAMYYCAR----------------------',
                'IGHV5-12*02': 'EVKLVESGG-GLVQPGGSLKLSCATSGFTF----SDYYMYWVRQTPEKRLEWVAYISNG--GGSTYYPDTVK-GRFTISRDNAKNTLYLQMSRLKSEDTAMYYCAR----------------------',
                'IGHV5-12-1*01': 'EVQLVESGG-GLVKPGGSLKLSCAASGFAF----SSYDMSWVRQTPEKRLEWVAYISSG--GGSTYYPDTVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCAR----------------------',
                'IGHV5-12-2*01': 'EVKLVESGG-GLVQPGGSLKLSCAASGFTF----SSYTMSWVRQTPEKRLEWVAYISNG--GGSTYYPDTVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCAR----------------------',
                'IGHV5-12-4*01': 'EVKLVESGG-GLVKPGQSLKLSCAASGFTF----SNYYMSWVHQTPEKRLEWVAYISSS--GVSTYYPDNVK-GRFAISRDNAKNTLYLQMTSLKSEDTALYYCAR----------------------',
                'IGHV5-15*01': 'EVKLVESGG-GLVQPGGSLKLSCAASGFTF----SDYGMAWVRQAPRKGPEWVAFISNL--AYSIYYADTVT-GRFTISRENAKNTLYLEMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-15*02': 'EVKLVESGG-GLVQPGGSRKLSCAASGFTF----SDYGMAWVRQAPGKGPEWVAFISNL--AYSIYYADTVT-GRFTISRENAKNTLYLEMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-15*05': 'EVKLVESGG-ALVQPGGSLKLSCAASGFTF----SDYGMAWVRQAPRKGPEWVAFISNL--AYSIYYADTVT-GRFTISRENAKNTLYLEMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-16*01': 'EVKLVESEG-GLVQPGSSMKLSCTASGFTF----SDYYMAWVRQVPEKGLEWVANINYD--GSSTYYLDSLK-SRFIISRDNAKNILYLQMSSLKSEDTATYYCAR----------------------',
                'IGHV5-17*01': 'EVQLVESGG-GLVKPGGSLKLSCAASGFTF----SDYGMHWVRQAPEKGLEWVAYISSG--SSTIYYADTVK-GRFTISRDNAKNTLFLQMTSLRSEDTAMYYCAR----------------------',
                'IGHV5-17*02': 'DVQLVESGG-GLVQPGGSRKLSCAASGFTF----SSFGMHWVRQAPEKGLEWVAYISSG--SSTIYYADTVK-GRFTISRDNPKNTLFLQMTSLRSEDTAMYYCAR----------------------',
                'IGHV5-17*04': 'EVKLVESGG-GLVQPGGSRKLSCAASGFTF----SDYGMVWVRQAPGKGLEWVAYISSG--SSTIYYADTVK-GRFTISRDNPKNTLFLQMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-2*01': 'EVQLVESGG-GLVQPGESLKLSCESNEYEF----PSHDMSWVRKTPEKRLELVAAINSD--GGSTYYPDTME-RRFIISRDNTKKTLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5-2*02': 'EVQLVESGG-GLVQPRESLKLSCESNEYEF----PSHDMSWVRKTPEKRLELVAAINSD--GGSTYYPDTME-RRFIISRDNTKKTLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5-4*01': 'EVQLVESGG-GLVKPGGSLKLSCAASGFTF----SSYAMSWVRQTPEKRLEWVATISDG--GSYTYYPDNVK-GRFTISRDNAKNNLYLQMSHLKSEDTAMYYCAR----------------------',
                'IGHV5-4*02': 'EVQLVESGG-GLVKPGGSLKLSCAASGFTF----SDYYMYWVRQTPEKRLEWVATISDG--GSYTYYPDSVK-GRFTISRDNAKNNLYLQMSSLKSEDTAMYYCAR----------------------',
                'IGHV5-4*04': 'EVQLVESGG-GLVKPGGSLKLSCAASGFTF----SDYYMYWVRQTPEKRLEWVATISDG--GSYTYYPDNVK-GRFTISRDNAKNTLYLQMSHLKSEDTAMYYCAR----------------------',
                'IGHV5-5-2*01': 'DVKLVESGG-GLVKPEGSLKLSCAASGFAF----SSYGMAWVRQTPEKRLEWVATISNG--GVSTYYPDNVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCTR----------------------',
                'IGHV5-6*01': 'EVQLVESGG-DLVKPGGSLKLSCAASGFTF----SSYGMSWVRQTPDKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCAR----------------------',
                'IGHV5-6-1*01': 'EVQLVESGG-DLVKPGGSLKLSCAASGFTF----SSYGMSWVRQTPDKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCAR----------------------',
                'IGHV5-6-2*01': 'DVKLVESGG-GLVKLGGSLKLSCAASGFTF----SSYYMSWVRQTPEKRLELVAAINSN--GGSTYYPDTVK-GRFTISRDNAKNTLYLQMSSLKSEDTALYYCAR----------------------',
                'IGHV5-6-3*01': 'EVQLVESGG-GLVQPGGSLKLSCAASGFTF----SSYGMSWVRQTPDKRLELVATINSN--GGSTYYPDSVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCAR----------------------',
                'IGHV5-6-4*01': 'DVKLVESGG-GLVKPGGSLKLSCAASGFTF----SSYTMSWVRQTPEKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCTR----------------------',
                'IGHV5-6-4*02': 'EVKLVESGG-GLVKPGGSLKLSCAASGFTF----SSYTMSWVRQSPEKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNAKNTLYLQMSSLKSEDTAMYYCTR----------------------',
                'IGHV5-6-5*01': 'EVKLVESGG-GLVKPGGSLKLSCAASGFTF----SSYAMSWVRQTPEKRLEWVASISSG---GSTYYPDSVK-GRFTISRDNARNILYLQMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-6-6*01': 'EVKLVESGG-GLVQPGGSLKLSCAASGFTF----SSYAMSWIRQTPDKRLEWVASISSG--GSYTYYPDSVK-GRFTIPRDNTKNTLYLQMSSLSSKDTALYYCAR----------------------',
                'IGHV5-9*01': 'EVMLVESGG-GLVKPGGSLKLSCAASGFTF----SSYTMSWVRQTPEKRLEWVATISGG--GGNTYYPDSVK-GRFTISRDNAKNTLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5-9*02': 'EVKLVESGG-GLVKPGGSLKLSCAASGFAF----SSYDMSWVRQTPEKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNARNTLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5-9*03': 'EVMLVESGG-GLVKPGGSLKLSCAASGFTF----SSYTMSWVRQTPEKRLEWVATISSG--GGNTYYPDSVK-GRFTISRDNAKNNLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5-9-1*01': 'EVMLVESGG-GLVKPGGSLKLSCAASGFTF----SSYAMSWVRQTPEKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNAKNTLYLQMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-9-1*02': 'DVKLVESGE-GLVKPGGSLKLSCAASGFTF----SSYAMSWVRQTPEKRLEWVAYISSG--GDYIYYADTVK-GRFTISRDNARNTLYLQMSSLKSEDTAMYYCTR----------------------',
                'IGHV5-9-2*01': 'EVKLVESGG-GLVKPGGSLKLSCAASGFTF----SSYGMSWVRQTPEKRLEWVATISGG--GSYTYYPDSVK-GRFTISRDNAKNNLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5-9-3*01': 'EVQLVESGG-GLVKPGGSLKLSCAASGFTF----SSYAMSWVRQTPEKRLEWVATISSG--GSYTYYPDSVK-GRFTISRDNAKNTLYLQMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-9-4*01': 'EVQLVESGG-GLVKPGGSLKLSCAASGFTF----SSYAMSWVRQSPEKRLEWVAEISSG--GSYTYYPDTVT-GRFTISRDNAKNTLYLEMSSLRSEDTAMYYCAR----------------------',
                'IGHV5-9-5*01': 'EVMLVESGG-GLVKPGGSLKLSCAASGFTF----SSYTMSWVRQTPEKRLEWVATISSG--GGNTYYPDSVK-GRFTISRDNAKNNLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV5S4*01': 'EVKLVESGG-GLVKPGGSLKLSCATSGFTF----SSYGMSWVRQTPEKRLEWVATISGG--GSYTYYPDSVK-GRFTISRDNAKNNLYLQMSSLRSEDTALYYCAR----------------------',
                'IGHV6-3*01': 'EVKLEESGG-GLVQPGGSMKLSCVASGFTF----SNYWMNWVRQSPEKGLEWVAQIRLKSDNYATHYAESVK-GRFTISRDDSKSSVYLQMNNLRAEDTGIYYCT-----------------------',
                'IGHV6-3*02': 'EVKLEESGG-GLVQPGGSMKLSCVASGFTF----SNYWMSWVRQSPEKGLEWVAQIRLKSDNYATHYAESVK-GRFTISRDDSKSSVYLQMNNLRAEDTGIYYCT-----------------------',
                'IGHV6-3*03': 'EVKLEESGG-GLVQPGGSMKLSCVASGFTF----SNYWMSWVRQSPEKGLEWVAQIRLKSDNYATHYAESVK-GRFTISRDDSKSSVYLQMNNLRAEDTGIYYCTG----------------------',
                'IGHV6-3*04': 'EVKLEESGG-GLVQPGGSMKLSCVASGFTF----SNYWMYWVRQSPEKGLEWVAEIRLKSDNYATHYAESVK-GRFTISRDDSKSSVYLQMNSLRAEDTGIYYCT-----------------------',
                'IGHV6-6*01': 'EVKLEESGG-GLVQPGGSMKLSCAASGFTF----SDAWMDWVRQSPEKGLEWVAEIRNKANNHATYYAESVK-GRFTISRDDSKSSVYLQMNSLRAEDTGIYYCTR----------------------',
                'IGHV6-6*02': 'EVKLEESGG-GLVQPGGSMKLSCVASGFTF----SNYWMNWVRQSPEKGLEWVAEIRLKSNNYATHYAESVK-GRFTISRDDSKSSVYLQMNNLRAEDTGIYYCTR----------------------',
                'IGHV6-6*03': 'EGKLEESGG-GLVQPGGSIKLSCAASGFTF----SDAWMDWVRQSPEKGLEWVAEIRNKASNYATYYAEFVK-GRFTISRDDSKSSVYLQMNTLRAEDTGIYYCTR----------------------',
                'IGHV6-7*01': 'EEKLDESGG-GLVQPGRSMKLSCVASGFTF----TNSWMNWFCQSPEKGLEWVAQIKSKPYNYETYYSDSVK-GRFTISRDDSKSSVYLQMNNLRAEDTGIYYCTW----------------------',
                'IGHV6-7*02': 'EVKLDETGG-GLVQPGRPMKLSCVASGFTF----SDYWMNWVRQSPEKGLEWVAQIRNKPYNYETYYSDSVK-GRFTISRDDSKSSVYLQMNNLRAEDMGIYYCTW----------------------',
                'IGHV6-7*03': 'EVKLDETGG-GLVQPGRSMKLSCVASGFTF----SDYWMNWVRQSPEKGLEWVAQIRNKPYNYETYYSDSVK-GRFTISRDDSKSSVYLQMNNLRAEDMGIYYCTW----------------------',
                'IGHV6-7-1*01': 'EVKLEESGG-GLVQPGGSMKLSCVASGFTF----SSYWMSWVRQSPEKGLEWVAEIRLKSDNYATHYAESVK-GKFTISRDDSKSRLYLQMNSLRAEDTGIYYCT-----------------------',
                'IGHV6-7-4*01': 'EVKLEESGG-GLVQPGGSMKLSCAASGFTF----SDAWMDWVRQSPEKGLEWVAEIRSKANNHATYYAESVK-GRFTISRDDSKSSVYLQMNSLRAEDTGIYYCTR----------------------',
                'IGHV6-7-5*01': 'EEKLDESGG-GLVQPGRSMKLSCVAFGFTF----TNSWMNWFCQSPEKGLEWVAQIKSKPYNYETYYSDSVK-GRLTISRDDSKSSVYLQMNNLRAEDTGIYYCTW----------------------',
                'IGHV7-1*01': 'EVKLVESGG-GLVQSGRSLRLSCATSGFTF----SDFYMEWVRQAPGKGLEWIAASRNKANDYTTEYSASVK-GRFIVSRDTSQSILYLQMNALRAEDTAIYYCARDA--------------------',
                'IGHV7-1*02': 'EVKLVESGG-GLVQPGGSLRLSCATSGFTF----SDFYMEWVRQPPGKRLEWIAASRNKANDYTTEYSASVK-GRFIVSRDTSQSILYLQMNALRAEDTAIYYCARDA--------------------',
                'IGHV7-1-1*01': 'EVKLVESGG-GLVQPGASLRLSCATSGFTF----TDYYMNWVRQPPGKALEWLGFIRNKANGYTTEYSASVK-GRFTISRDNSQSILYLQMNTLRAEDSATYYCARD---------------------',
                'IGHV7-3*01': 'EVKLVESGG-GLVQPGGSLSLSCAASGFTF----TDYYMSWVRQPPGKALEWLGFIRNKANGYTTEYSASVK-GRFTISRDNSQSILYLQMNALRAEDSATYYCARY---------------------',
                'IGHV7-3*02': 'EVKLVESGG-GLVQPGGSLRLSCATSGFTF----TDYYMSWVRQPPGKALEWLGFIRNKANGYTTEYSASVK-GRFTISRDNSQSILYLQMNTLRAEDSATYYCARD---------------------',
                'IGHV7-3*03': 'EVKLVESGG-GLVQPGGSLSLSCAASGFTF----TDYYMSWVRQLPGKALEWLGFIRNKANGYTTEYSASVK-GRFTISRDNSQSILYLQMNALRAEDSATYYCAKD---------------------',
                'IGHV7-3*04': 'EVKLVESGG-GLVQPGGSLSLSCAASGFTF----TDYYMSWVRQPPGKALEWLALIRNKANGYTTEYSASVK-GRFTISRDNSQSILYLQMNALRAEDSATYYCARD---------------------',
                'IGHV7-4*01': 'EVKLMESGG-GLVQPGASLRLSCAASGFTF----TDYYMSWVRQPPGKAPEWLALIRNKANGYTTEYTASVK-GRFTISRDNSQNILYLQMNTLRAEDSATYYCVKAV--------------------',
                'IGHV7-4*02': 'EVKLMESGG-GLVQPGASLRLSCEASGFTF----TDYYMSWVRQPPGKSPEWLALIRNKANGYTTEYSASVK-GRFTISRDNSQNILYLQMNTLRAEASATYYCAKDV--------------------',
                'IGHV7-4*04': 'EVKLVESGG-GLVQPGGSLRLSCAASGFTF----TDYYMSWVRQPPGKAPEWLALIRNKANGYTTEYTASVK-GRFTISRDNSQNILYLQMNTLRAEDSATYYCVKAV--------------------',
                'IGHV8-11*01': 'QITQKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVGWIHQPSGNGLEWLAHIWWN---DNKYYNTALK-SRLTISKDTSNNQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-12*01': 'QVTLKESGP-GILQSSQTLSLTCSFSGFSLS--TSGMGVSWIRQPSGKGLEWLAHIYWD---DDKRYNPSLK-SRLTISKDTSRNQVFLKITSVDTADTATYYCARR---------------------',
                'IGHV8-12*02': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVSWIRQPSGKGLEWLAHIYWD---DDKRYNPSLK-SRLTISKDTSRNQVFLKITSVDTADTATYYCARR---------------------',
                'IGHV8-12-1*01': 'QVILKESGP-GILQPSQTLSLTCSFSGFSLS--TYGTAVNWIRQPSGKGLEWLAQIGSD---DSKLYNPFLK-SRITISKDTSNSQVFLKITSVDTEDSATYYCANR---------------------',
                'IGHV8-13-3*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVGWIRQPSGKGLEWLAHIWWD---DDKRYNPALK-SRLTISKDTSSNQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-13-4*01': 'QVTLKESGP-GMLQPSQTLSLTCSFSGFSMS--TFGMGVGWIHQPSGKGLEWLANIWWH---DNKYYNPALK-SRLTISKDTSKNQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-13-8*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVGWIRQPSGEGLEWLADIWWD---DNKYYNPSLK-SRLTISKDTSSNQVFLKITSVDTADTATYYCARR---------------------',
                'IGHV8-4*02': 'QITLKESGP-GIVQPSQPFRLTCTFSGFSLS--TSGIGVTWIRQPSGKGLEWLATIWWD---DDNRYNPSLK-SRLTVSKDTSNNQAFLNIITVETADTAIYYCAQS---------------------',
                'IGHV8-4-13*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVGWIRQPSGKGLEWLAHIWWN---DVKPYNPALK-SRLTISKDTSSSQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-4-14*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TFDMGVGWIRQPSGKGLEWLANIWWN---DNKYYNSALK-SRLTISKDTSNNQVFLKISSVDTADTATYYCAQI---------------------',
                'IGHV8-4-14D*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TFDMGVGWIRQPSGKGLEWLANIWWN---DNKYYNSALK-SRLTISKDTSNNQVFLKISSVDTADTATYYCAQI---------------------',
                'IGHV8-4-17*01': 'QVTLKESGP-GILQPSQTLSLTCSFSAFSLS--TFDMGVGWIRQPSGKGLEWLANIWWN---DNKYYNSALK-SRLTISKDTSNNQVFLKISSVDTADTATYYCAQI---------------------',
                'IGHV8-4-18*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVSWIRQPSGKGLEWLAHIYWD---DDKRYNPSLK-SRLTISKDTSSNQVFLKITSVDTADTATYYCARR---------------------',
                'IGHV8-4-18D*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVSWIRQPSGKGLEWLAHIYWD---DDKRYNPSLK-SRLTISKDTSSNQVFLKITSVDTADTATYYCARR---------------------',
                'IGHV8-4-19*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TYGMGVGWIRQPSGKGLEWLAHIWWD---DDKYYNPALK-SRLTISKDTSKNQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-4-21*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TYCMGVGWIRQPSGKGLEWLAHIWWN---DDKYYNPALK-SRLTISKDTSNNQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-4-4*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMSVGWIRQPSGKGLEWLAHIWWN---DDKYYNPALK-SRLTISKDTSNNQVFLKIASVVTADTATYYCARI---------------------',
                'IGHV8-4-6*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TYGIGVGWIRQPSGKGLEWLAHIWWN---DNKYYNTALK-SRLTISKDTSNNQVFLKIASVDTADTATYYCARI---------------------',
                'IGHV8-4-8*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVGWIRQPSGKGLEWLADIWWD---DDKYYNPSLK-SRLTISKDTSKNQVFLKIASVDTADTATYYCARR---------------------',
                'IGHV8-5*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSNMGIGWIRQPSGKGLEWLAHIWWN---DDKYYNPSLK-SRLTISKDTSNNQVFLKITSVDTADTATYYCAQI---------------------',
                'IGHV8-6*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TFGMGVSWIRQPSGKDLEWLAHIYWD---DDKHYNPSLK-SQLRISKDTSNNQVFLKITTVDTVDTATYYCARR---------------------',
                'IGHV8-8*01': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TFGMGVGWIRQPSGKGLEWLAHIWWD---DDKYYNPALK-SRLTISKDTSKNQVFLKIANVDTADTATYYCARI---------------------',
                'IGHV8-8*03': 'QVTLKESGP-GILQPSQTLSLTCSFSGFSLS--TSGMGVGWIRQPSGKGLECLANIWWD---DDKYYNPALK-SRLTISKDTSNNQVFLKIARVDTADTATCYCARI---------------------',
                'IGHV9-1*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TEYPMHWVKQAPGKGFKWMGMIYTD--TGEPTYAEEFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCVR----------------------',
                'IGHV9-1*02': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYGMNWVKQAPGKGLKWMGWINTY--TGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDMATYFCAR----------------------',
                'IGHV9-2*01': 'QIQFVQSGP-ELKKPGETVKISCKASVYTF----TEYPMHWVKQAPGKGFKWMGWINTY--SGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-2*02': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYAMHWVKQAPGKGLKWMGWKYTN--TGEPTYGDDFK-GRFAFSLETSASTAYLQINNLKNEDMATYFCAR----------------------',
                'IGHV9-2*03': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYAMHWVKQAPGKGLKWMGWIYTN--TGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDMATYFCAR----------------------',
                'IGHV9-2-1*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TDYSMHWVKQAPGKGLKWMGWINTE--TGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-2-2*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYGMNWVKQAPGKGLKRMGWINTE--TGVPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-2-4*01': 'QIHLVQSGP-ELKKPGETVKISCQASGYTF----TGYSMHWVKQAPRKGLKWMGLIYTN--TGEPTYDEEFK-GRFAFSLETSASTAYLQINNLKNEDMATYFCAR----------------------',
                'IGHV9-2-5*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TGYSMQWVKQAPGKGLKWMGWINTE--TGVPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-2-6*01': 'QIQFVQSGS-ELKKPGETVKISCKASGYTF----TNYPMHWVKQAPGKAFKWMGWINTN--TGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-2D*03': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYAMHWVKQAPGKGLKWMGWIYTN--TGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDMATYFCAR----------------------',
                'IGHV9-3*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TTYGMSWVKQAPGKGLKWMGWINTY--SGVPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-3*02': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYGMNWVKQAPGKGLKWMGWINTN--TGEPTYAEEFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-3-1*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TNYGMNWVKQAPGKGLKWMGWINTY--TGEPTYADDFK-GRFAFSLETSASTAYLQINNLKNEDTATYFCAR----------------------',
                'IGHV9-4*01': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TTAGMQWVQKMPGKGFKWIGWINTH--SGEPKYAEDFK-GRFAFSLETSASTAYLQISNLKNEDTATYFCAR----------------------',
                'IGHV9-4*02': 'QIQLVQSGP-ELKKPGETVRISCKASGYTF----TTAGMQWVQKMPGKGLKWIGWINTH--SGVPKYAEDFK-GRFAFSLETSASTAYLQISNLKNEDTATYFCAR----------------------',
                'IGHV9-4*04': 'QIQLVQSGP-ELKKPGETVKISCKASGYTF----TTAGMQWVQKMPGKGFKWIGWINTH--SGDPKYAEDFK-GRFAFSLETYASTAYLQISNLKNEDTATYFCAR----------------------'},
                'rat': {'IGHV2S13*01': 'QVQLKESGP-GLVQPSQTLSLTCTVSGFSL-----TSYNVHWVRQPPGKGLEWMGVIWSG---GNTDYNSALK-PRLSISRDTSKSQVFLTMNSLQTEDTGIY-YCNR--------------------',
                'IGHV2S18*01': 'QVQLKESGP-GLVQPSETLSLTCTVSGFSL-----TSYSVHWVRQHSGKSLEWMGRMWSD---GDTSYNSAFT-SRLSISRDTSKSQVFLKMNSLQTEDTGTY-YCAR--------------------',
                'IGHV2S61*01': 'QVQLKESGP-GLVQPSQTLSLTCTVSGFSL-----SSYGVIWVRQPPGKGLEWMGVIWGN---GNTNYNSALK-SRLSISRDTSKSQVFLKMNNLQTEDTAMY-FCA---------------------',
                'IGHV2S63*01': 'EVQLKESGP-GLVQPSQTLSLTCTVSGFSL-----TDYSVHWVRQPPGKGLEWMGVMWSG---GSTAYNSALK-SRLSISRDTSKSQVFLKMNSLQTEDTAIY-YCTR--------------------',
                'IGHV5-43*01': 'EVQLVESGG-GLVQPGSSLKVSCVASGFTF-----SSYVMHWFRQAPENGIEWLAYINTD--SSSTHYAETVK-GRFTISRDNAKNTVDMQLSSLRSEDTAMY-FCAR--------------------',
                'IGHV5S10*01': 'EVQLVESGG-GLVQPGRSLKLSCAASGFTF-----SDYNMAWVRQAPKKGLEWVATIIYD--GSRTYYRDSVK-GRFTISRDNAKSTLYLQMDSLRSEDTATY-YCAT--------------------',
                'IGHV5S11*01': 'EVQLVESGG-GLVQPGRSMKLSCAASGFTF-----SNYYMAWVRQAPTKGLEWVASISTG--GGNTYYRDSVK-GRFTISRDNAKSTLYLQMDSLRSEETATY-YCAR--------------------',
                'IGHV5S13*01': 'EVQLVESGG-GLVQPGRSLKLSCAASGFTF-----SNYGMAWVRQAPTKGLEWVASISTG--GGNTYYRDSVK-GRFTISRDNAKNTQYLQMDSLRSEDTATY-YCAR--------------------',
                'IGHV5S8*01': 'EVKLVESGG-GLVQPGRPLKLSCAASGFTF-----SSNWLNWIRQAPGKGLEWVASINPD--GSSTLYPDTVK-GRFVVSKDNAKNTRYLQMNNLRSEDTAMY-YCAR--------------------'},
                'rabbit': {'IGHV1S1*01': 'QEQLKESGG-RLVMPGGILTLTCTASGSNI----SSYGVSWFRQAPGKGLEWIRYISYG---GSAYYKSWVK-GRFTISKTSS--TVDLKMTSLTASDTATYFCAR----------------------',
                'IGHV1S13*01': 'QEQLEESGG-GLVTPGGTLTLTCTVSGFSL----SSYGVSWVRQAAGKGLEWIGYISSS---GSAYYASWVN-GRFTISKTSS--TVDLKMTSLRAADTATYFCAR----------------------',
                'IGHV1S17*01': 'QEQQKESGG-RLVMPGGSLTLTCTVSGFSL----SSYNMGWVRQAPGEGLEYIGWISTG---GSAYYASWVN-GRFTISKTST--TMDLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S24*01': 'QEQLKESGG-GLVTPGGILSLTCTASGFSI----SSYRMGWVRQAPGKGLEYIGYISYG---GSAYYKSWVK-GRFTISKTSS--TVDLKMTSLTASDKATYFCAR----------------------',
                'IGHV1S25*01': 'QEQLKESGG-VLVTPGGILSLTCTASGFSI----SSYRMGWVRQAPGKGLEYIGIIYTG---GSAYYASWVN-GRFTISKTSS--TVDLKMTSLTAADMATYFCAR----------------------',
                'IGHV1S26*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----SSYAISWVRQAPGNGLEWIGIINSY---GSTYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S28*01': 'Q-SLEESRG-GLIKPGGTLTLTCTASGFTI----SSYDMSWVRQAPGKELEWIGYISYG---GSAYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S31*01': 'Q-SVEESRG-GLIKPTDTLTLTCTVSGFSL----SSYGVIWVRQAPGNGLEYIGTIGSS---GSAYYASWAK-SRSTITRNTNLNTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S33*01': 'QEQLEESGG-GLVKPGDTLTLTCKASGFSL----SSYDMSWVRQAPGKGLEWIGFIWSG---GSTDYASWVN-GRIIISSDNTQNTVSLLMNSLSARDTATYFCAG----------------------',
                'IGHV1S34*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----SSNAISWVRQAPGNGLEWIGAIGSS---GSAYYASWAK-SRSTITRNTNLNTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S36*01': 'QEQLEESGG-GLVTPGGTLTLTCTASGFSL----SSYGVSWVRQAAGKGLEWIGYISSS---GSAYYASWVN-GRFTISKTSS--TVDLKMTSLTASDTATYFCAR----------------------',
                'IGHV1S40*01': 'Q-SLEESGG-DLVKPGASLTLTCTASGFSFS---SSYYMCWVRQAPGKGLEWIACIYAGS-SGSTYYASWAK-GRFTISKTSS-TTVTLQMTSLTAADTATYFCAR----------------------',
                'IGHV1S43*01': 'QQQLEESGG-GLVKPGGTLTLTCKASGIDFS---SYYYICWVRQAPGKGLELIACIYTS--SGSTWYASWVN-GRFTISRSTSLNTVDLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S44*01': 'Q-SLEESGG-RLVTPGGSLTLTCTVSGIDL----TSYAMGWVRQAPGKGLEYIGIISSS---GSTYYASWAK-GRFTISKTSST-TVDLKMTSLTTEDTATYFCAG----------------------',
                'IGHV1S45*01': 'QEQLEESGG-DLVKPEGSLTLTCTASGFSFS---SSYWICWVRQAPGKGLEWIACIYAGS-SGSTYYASWAK-GRFTISKTSST-TVTLQMTSLTAADTATYFCAR----------------------',
                'IGHV1S47*01': 'QEQLVESGG-GLVQPEGSLTLTCKASGFDF----SSNAMCWVRQAPGKGPEWIACIYNG--DGSTYYASWVN-GRFTISRSTSLNTVTLQMTSLTAVDTATYFCAR----------------------',
                'IGHV1S49*01': 'Q-SVKESEG-GLFKPADTLTLTCTASGFTI----SSYGVSWVRQAPGKGPEWIGAIDIN---GRTYYATWAK-SRATITRNVNENTVTLRVTSLTAADTATYFCAR----------------------',
                'IGHV1S50*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----SSYAISWVRQAPGNGLEYIGYISFT---NTAYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S51*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----SSYNIGWVRQAPGSGLEWIGIISYG---GSAYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S52*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----SSYNMGWVRQAPGNELEWIGIITSY---GSTYYASWAK-SRSTITRNTNENPVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S53*01': 'QEQLEESGG-GLVNPGGTLTLTCTVSGFTI----STYGVSWVRQAPGNGLEWIGTVNYD---GSTHYASWAK-SRSTITRNTNENTATLKMTSLTGADTATYFCAR----------------------',
                'IGHV1S54*01': 'Q-SLGESRG-GLIKPGGTLTLTCTASGFTI----SSYDMSWVRQAPGEGLEYIGCINSY---GTTYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCGRE---------------------',
                'IGHV1S55*01': 'Q-SVKESEG-GLFKPTDTLTLSCTVSGFSL----SSYGVSWVRQAPGEGLECIGWISTD---GSTYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCARA---------------------',
                'IGHV1S56*01': 'QEQLKESGG-GLVTPGGILSLTCTASGFSL----STYNMGWVPPAPGKGLEYIGWINTG---GSPYCTSWAG-KRSTITRNTSENTVTLEMTSLTAADTATYLCAK----------------------',
                'IGHV1S57*01': 'Q-TVKESEG-GLFKPTHTLTLTCTASGFSL----SSYPIIWVRQAPGNGLEWIGITNTY---GSPYYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S58*01': 'QEQLKESGG-RLVMPGGSLTITCTVSGFSL----SSNAISWVRQAPGNGLEWIGVINSG---GTAYYASWAK-GRSTISRNTKENTVTLQMTSLTAADTATYFCAR----------------------',
                'IGHV1S59*01': 'Q-SVKESEG-GLFKPTETLTLTCTVSGFSL----RDYRTGWVRQAPGKELEVVAYIRGD---GVIYYASWAK-KRSTITRNTNENTVTLKMTSLTAADTATYFCGR----------------------',
                'IGHV1S60*01': 'Q-SVKESEG-GLFKATETLTLTCTLSGFSL----NNNAIHWVRQAPGKGLEWIGMIYGS---GATYYASWVS-GRATITRDTNENTVTLKMTSLTDADTATYFCAR----------------------',
                'IGHV1S61*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----TSMSISWVRQAPGNGLEWIGAINRV---STTYYTTWAK-SRSTITRNTNENTVTLKMTSLTVADTATYFCTR----------------------',
                'IGHV1S62*01': 'Q-SVKESEG-GLIKPTDTLTLTCTVSGFSL----TNYIIFWVRQAPGKELEWIGYIHGG---GNTYYASWAK-SRSTITRDTKENTVTLKMASLTASDTATYFCAR----------------------',
                'IGHV1S63*01': 'Q-SVKESEG-GLFKPTDTLTLTCTASGFSI----SSYRMGWVRQAPGNGLEVIGYIRGD---GVTYCASWAK-SRSTITRNTNENTATLKMTSLTAADTATYFCGR----------------------',
                'IGHV1S65*01': 'Q-SLEEFGG-GLIRPASTLTLTCTVSGFSL----NEVGVIWVRQAPGKELEWIGYISYR---GNAYYASWAK-SRSTITRNTKENTVTLKVTGLTAADTATYFCAR----------------------',
                'IGHV1S66*01': 'Q-SVKESEG-GLFEPTDTLTLTCTVSGFSL----TKYGVMWVRQAPGNGLEWIGFIAYS---GNTYYTTWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAR----------------------',
                'IGHV1S67*01': 'Q-SVKESEG-GLFKPTDTLTLTCTVSGFSL----STYGVIWVRQAPGNGLEYIGSIASG---GSAGYASWAK-SRSTITRNTNENTVTLKMTSLTAADTATYFCAA----------------------',
                'IGHV1S68*01': 'Q-SVEESRG-GLIKPADTLTLTCTVSGFSL----NNYGVIWVRQAPGSGLEYIGTIDIG---VTAFYASWAK-SRSTITKNTNENTVTLRMTSLTAADTATYFCAS----------------------',
                'IGHV1S69*01': 'Q-SVEESGG-RLVTPGTPLTLTCTVSGFSL----SSYAMSWVRQAPGKGLEWIGIISSS---GSTYYASWAK-GRFTISKTST--TVDLKITSPTTEDTATYFCAR----------------------',
                'IGHV1S7*01': 'Q-QLKESGG-GLVKPGGSLKLCCKASGFTF----SSYYMCWVRQAPGKGLEWIGCIYAG--SGSTHYASWVN-GRFTLSRDNAQSTVCLQLNSLTAADTATYFCAR----------------------',
                'IGHV1S8*01': 'QKQLVESGG-GLDQPAGSLKLSCKDSGFTL----SSNAMCWVHQAPGKGLEWIACIDSY---GSTNYVSRVN-GRFTISSDNTQNMVDLEMNSLTAADMAIYFCAR----------------------'},
                'rhesus': {'IGHV1-111*01': 'EVQLVQSGA-EVKKP-GASVKISCKAS-GYTF----TDYYLHWVRQAPGKGLEWMGRVDPE--DGEAIHAQKFQ-DRVTITRDTSTDTAYMELSSLRSEDTAVYYCAT--------------------',
                'IGHV1-111*02': 'EVQLVQSGA-EVKKP-GASVKISCKAS-GYTF----TDYYLHWVRQAPGKGLEWMGRVDPE--DGEAIHAQKFQ-DRVTITADTSTDTAYMELSSLRSEDTAVYYCAT--------------------',
                'IGHV1-111*03': 'EVQLVQSGA-EVKKP-GASVKISCKAS-GYTF----TDYYLHWVRQAPGKGLEWMGRVDPE--DGEADYAQKFQ-DRVTITRDTSTDTAYMELSSLRSEDTAVYYCAT--------------------',
                'IGHV1-111*04': 'EVQLVQSGA-EVKKP-GASVKISCKAS-GYTF----TDYYLHWVRQAPGKGLEWMGRVDPE--DGEADYAQKFQ-DRVTITADTSTDTAYMELSSLRSEDTAMYYCAT--------------------',
                'IGHV1-138*01': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYIF----TDYYMHWVRQAPGQGLEWMGEINPK--TGGTNYAQKFQ-GRVTTTRDTSTSTAYMELSSLRSEDTAVYYCER--------------------',
                'IGHV1-138*02': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYTF----TDYYMHWVRQAPGQGLEWMGEINPK--TGGTNYAQKFQ-GRVTTTRDTSTSTAYMELSSLRSEDTAVYYCER--------------------',
                'IGHV1-138*03': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYTF----TDYYMQWVRQAPGQGLEWMGRINPK--TGGTDYAQKFQ-GRVTMTRDTSTSTAYMELSSLRSEDTAVYYCAT--------------------',
                'IGHV1-138*04': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYTF----TDYYMHWVRQAPGQGLEWMGEINPK--TGGTNYAQKFQ-GRVTMTRDTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-151*01': 'QVQLVQSGA-EVKKP-GASVKLSCKAS-GYTF----SIYAISWVRQAPGQGLEWMGGIIPL--VGITNYAQKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-151*02': 'QVQLVQSGA-EVKKP-GASVKLSCKAS-GYTF----SIYAISWVRQAPGQGLEWMGGIIPL--VGITNYAQKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-151*03': 'QVQLVQSGA-EVKKP-GASVKLSCKAS-GFTF----SIYAISWVRQAPGQGLEWMGEIIPL--VGITNYAQKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-156*01': 'EVQLVQSGA-EVKKP-GASVKVSCKVS-GYTF----TELSMHWVRQAPGKGLEWMGGVDPV--YGEIIHAEKFQ-GRVTMTEDTSTDTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-156D*01': 'EVQLVQSGA-EVKKP-GASVKVSCKVS-GYTF----TELSMHWVRQAPGKGLEWMGGVDPV--YGEIIHAEKFQ-GRVTMTEDTSTDTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-156D*02': 'EVQLVQSGA-EVKKP-GASVKVSCKVS-GYTF----TELSMHWVRQAPGKGLEWMGGVDPV--YGEIIHAEKFQ-GRVTMTEDTSTDTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-180*01': 'QVQLVQSGA-EIKQP-GASVKLSCKAS-GYTF----TSYYMHWVRQAPGQGLEWIGLISPY--NGNKGYAQNFQ-GRVTITTDTSTSTGYMELSSLRSEDTAVYYCTR--------------------',
                'IGHV1-198*01': 'QVQLVQSGA-EVKKP-GASVKVSCKAS-GFTF----GSYAISWVRQAPGQGLEWMGVIIPL--VGVTNYAEKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-198*02': 'QVQLVQSGA-EVKKP-GASVKVSCKAS-GFTF----GSYAISWVRQAPGQGLEWMGVIIPL--VGITNYAEKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-198*03': 'QVQLVQSGA-EVKKP-GASVKVSCKAS-GFTF----GSYAINWVRQAPGQGLEWMGVIIPL--VGITNYAEKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-198*04': 'QVQLVQSGA-EVKKP-GASVKVSCKAS-GFTF----GSYAISWVRQAPGQGLEWMGGIVPL--VGVTNYAQKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-198*05': 'QVQLVQSGA-EVKKP-GASVKVSCKAS-GFTF----GSYAISWVRQAPGQGLEWMGVIIPL--VGVTNYAEKFQ-GRVTITADTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-200*01': 'QVQLVQSGA-EVKKP-GASVKLSCKAS-GYTF----TSYYINWVRQAPGQGLDWMGWINPS--NGNTGYAQKFQ-GRVTMTRDTSTSTAYMELNSLRSEDTAVYYCAR--------------------',
                'IGHV1-200*02': 'QVQLVQSGA-EVKKP-GASVKLSCKAS-GYTF----TSYSINWVRQAPGQGLEWMGWINPS--NGNTGYAQKFQ-GRVTMTRDTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-69*01': 'EMQLVQSEA-EVKKP-GASVKISCKAS-GYTF----TYRYLHWLRQTPGQGLEWMGWITPY--NGNTNYAQKFQ-DRATITRDRSMSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1-70*01': 'QEQLVQSGA-EVKKP-GASVKVSCKAS-GYIF----TSYVISWLRQAPGQGFEWMGGIHPG--YGSTSYAQKFQ-GRVTITADMSTSTVYMELSSLRSEDMAVYYCAA--------------------',
                'IGHV1-70*02': 'HEQLVQSGA-EVKKP-GASVKVSCKAS-GYIF----TSYVISWLRQAPGQGFEWMGGIHPG--YGSTSYAQKFQ-GRVTITADMSTSTVYMELSSLRSEDMAVYYCAA--------------------',
                'IGHV1-70*03': 'QEQLVQSGA-EVKKP-GASVKVSCKAS-GYIF----TSYVISWLQQAPGQGFEWMGGIHPG--YGSTSYAQKFQ-GRVTITADMSTSTVYMELSSLRSEDMAVYYCAA--------------------',
                'IGHV1S18*01': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYTF----TDYYMHWVQQAPGQGLEWMGWINPY--NGNTKYAQKFQ-GRVTMTRDTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1S2*01': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYTF----TDYYMHWVRQAPRQGLEWMGWINPY--NGNTKYAQKFQ-GRVTMTRDTSTSTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV1S20*01': 'QVQLVQSGA-EVKKP-GSSVKVSCKAS-GYTF----TSSAMQWVRQAPGRGLEWIGVIIIG--NGNTNYAQKFQ-GRVTITRDTSTSTGYMELSSLRSEDMAVYYCAA--------------------',
                'IGHV1S27*01': 'EVQLVQSGA-EVKKP-GATVKISCKAS-GYTF----TDHYLNWVRQAPGKGLEWMGGVDPE--DGEADYAQKFQ-DRVTITADMSTDTAYMELSSLRSEDTAVYYCAR--------------------',
                'IGHV2-10*02': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLS--TSGMRVSWIRQPPGKALEWLARIDWD---DDKYYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-152*01': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLT--TSGMGVGWIRQPPGKALEWLALIYWD---DDKRYSTSLK-SRLTISKDTSKNQVVLTMTNMDPMDTATYYCAR--------------------',
                'IGHV2-161*01': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSIS--TTGTGVSWIRQPPGKALEWLASIYWD---DDKYYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-161*02': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLS--TSGMGVGWIRQPPGKALEWLASIYWD---DDKYYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-161*03': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLN--TSGMGVGWIRQPPGKALEWLASIYWD---DDKYYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-174*01': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLT--TSGMGVGWIRQPPGKALEWLALIYWD---DDKRYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-174*02': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLS--TSGMGVGWIRQPSRKTLEWLAHIYWD---DDKRYSTSLK-SRLTISKDTSKNQVVLTMTNMDPMDTATYYCAR--------------------',
                'IGHV2-174*03': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLS--TSGMGVGWIRQPSRKTLEWLAHIYWD---DDKRYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-5-1*01': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLS--TSGTGVSWIRQPPGKALEWLARIDWD---NDKYYSTSLK-SRLTISKDTSKNQVVLTITNMDPVDTATYYCAR--------------------',
                'IGHV2-95*01': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSIS--TTGTGVGWIRQPPGKALEWLASIYWN---DSKYYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-95*03': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSIS--TTGTGVGWIRQPPGKALEWLASIYWN---DSKYYSTSLK-SRLTISTDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV2-95*04': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSIS--TSGTGVGWIRQPPGKALEWLASIYWN---DSKYYSTSLK-SRLTISKDTSKNQVVLTITNMDPVDTATYYCAR--------------------',
                'IGHV2S1*01': 'QVTLKESGP-ALVKP-TQTLTLTCTFS-GFSLS--TSGMGVGWIRQPPGKALEWLASIYWD---DDKYYSTSLK-SRLTISKDTSKNQVVLTMTNMDPVDTATYYCAR--------------------',
                'IGHV3-100*01': 'EVQLVESGG-GLVKP-GGSLRLSCVAS-GFTF----SSYVMHWVRQAPGKGLEWVSVISES--GGTTYYADSVK-GRFTISRDNAKNSLFLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-100*02': 'DVQLVESGG-GLVKP-GGSLRLSCVAS-GFTF----SSYEMHWVRQAPGKGLEWVSVISES--GGTTYYADSVK-GRFTISRDNAKNSLFLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-100*03': 'EVQLVESGG-GLVKP-GGSLRLSCVAS-GFTF----SSYEMHWVRQAPGKGLEWVSVISES--GGTTYYADSVK-GRFTISRDNAKNSLSLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-100*04': 'EVQLVESGG-GLVKP-GGSLRLSCVAS-GFTF----SSYVMHWVRQAPGKGLEWVSVISES--GGTTYYADSVK-GRFTISRDNAKNSLFLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-100*06': 'DVQLVESGG-GLVKP-GGSLRLSCVAS-GFTF----SSYVMHWVRQAPGKGLEWVSVISES--GGTIYYADSVK-GRFTISRDNAKNSLFLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-103*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SSYAMHWVRQAPGKGLEWVSAINSG---GSTYYADSVK-GRFTISRDNSKNTLSLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-103*05': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SSYAMHWVRQAPGKGLEWVSAISSG---GSTYYADSVK-GRFTISRDNSKNTLSLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-103*06': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SSYWMNWVRQTPGKGLEWISAINSG--GGSTYYADSVK-GRFTISRDNSKNTLSLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-108*01': 'EVQLVESGR-GLVQP-GGSLRLSCAVS-GFTF----SDHYMSWVRQAPGKGPEWVGFMRNKANGGRTEYAASGK-GRFTISRDDSKSIASLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-108*02': 'EVQLVDSGR-GLVQP-GGSLRLSCAVS-GFTF----SDHYMSWVRQAPGKGPEWVGFMRNKANGGRTEYAASGK-GRFTISRDDSKSIASLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-110*01': 'EVQLVESGG-GLVQP-GGSLRLSCVAS-GFSF----SDHYMDWVRQAPGKGLEWVSSISSGS-GSTTLYPDSVK-GRFTISRDNAKNTVYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-110*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDHYMDWVRQAPGKGLEWVSSISSGS-GSTTLYPDSVK-GRFTISRDNAKNTVYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-110*03': 'EVQLVESGG-GLVQP-GGSLRLSCVAS-GFTF----SDHYMDWVRQAPGKGLEWVSSISSGS-GSTTLYPDSVK-GRFIISRDNAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-110*04': 'EVQLVESGG-GLVQP-GGSLRLSCVAS-GFSF----SDHYMDWVRQAPGKGLEWVSSISTGS-GSTTLYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-115*01': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTF----SGYEMHWVRQAPGKGLESVSVIGGD--SSYTHYADSVK-GRFTISRDNAKNSLSLQMNSLRAADTAVYYCAR--------------------',
                'IGHV3-115*02': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTF----SGYEMHWVRQAPGKGLESVSVIGGD--SSYTHYADSVK-GRFTISRDNAKNSLSLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-115*03': 'EVQLAESGG-GVVQP-GGSLRLSCAAS-GFTF----SGYEMHWVRQAPGKWLESVSVIGGD--SSYTHYADSVK-GRFTISRDNAKNSLSLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-115*04': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTF----SGYEMHWVRQAPGKGLESVSVIGGD--SSYTHYADSVK-GRFTISRDNAKNSLSLQMNSLRAEDTAMYYC----------------------',
                'IGHV3-116*01': 'EVQLVESGG-GLVQP-GGSLRVSCAAS-GFTF----SDYYMQWVRQAPGKGPEWVGFIRNKANGGTAEYAASVK-GRFTISRDDSKSIASLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3-116*02': 'EVRLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYYMSWVRQAPGKGPEWVGFIRNKANGGTAEYAASVK-GRFTISRDDSKSIASLQMNSLKTEDTAVYYCAR--------------------',
                'IGHV3-116*03': 'EVQLVESGG-GLVQP-GGSLRVSCAAS-GFTF----SDHYMQWVRQAPGKGPEWVGFIRNKANGGTAEYAASVK-GRFTISRDDSKSIAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-116*04': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDHYMSWVRQAPGKGPEWVGFIRNKANGGTAEYAASVK-GRFTISRDDSKSIASLQMNSLKTEDTAVYYCAR--------------------',
                'IGHV3-117*04': 'EVQLVESGG-GLVQP-GGSLRLSCVAS-GFTF----SDYCMDWVRQASGKGLEWVSSISGS--SSNTYYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-117*05': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDHYMDWVRQAPGKGLEWVSSISGS--SSSTYYPDSVK-GRFTISRDNAKNTLYLQMNSPRAEDTAVYYCAR--------------------',
                'IGHV3-117-1*01': 'EAQLVESGG-ALAQP-GGSLRPSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVGRIRNKANSYTTEYAASVK-GRFTISRDDSKNTLYLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-118*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSSAMHWVRQASGKGLEWVGRIRSKSNNYETGYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV3-118*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSSAMHWVRQASGKGLEWVGRIRSKSNNYETEYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCAR--------------------',
                'IGHV3-118*04': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYAMSWVRQASGKGLEWVGYIRSKYNNYATEYAASVK-GRFTISRDDSKNTLYLQMSSLKTEDTAVYYCTT--------------------',
                'IGHV3-119*01': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYWMYWVRQAPGKGLEWVSRISSD--GSSTSYADSVK-GRFTISRENAKNSLYLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-119*03': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYWMYWVRQAPGKGLEWVSRISSD--GSSTSYADSVK-GRFTISRENAKNSLYLQMNSLRAEDRAVYYCTT--------------------',
                'IGHV3-12*01': 'EVQLVESGG-GLVQP-GRSLRPSCAAS-GFTF----SSYGMHWVRQAPEEGLVWVSYIGS----STMYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCVR--------------------',
                'IGHV3-12*02': 'EVQLVESGG-GLVQP-GRSLRPSCAAS-GFTF----SSYGMHWVRQAPEEGLVWVSYIGS----STMYYADSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCVR--------------------',
                'IGHV3-124*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVSRIKWW---GSTYYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-124*02': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFSF----SDYYMYWVRQAPGKGLEWVSGISYT--GGSTYYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-124*03': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFSF----SDYYMYWVRQAPGKGLEWVSGISYT--GGSTYYADSVK-GRFTIFRENAKNTLYLQMDSLRAEDTAVYYCSR--------------------',
                'IGHV3-13*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYYMHWVRQAQGKGLEWVGLIRNKANSYTTEYAAAVK-GRFTISRDDSKNTLYLQMSSLKTEDTALYYCTK--------------------',
                'IGHV3-13*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYYMHWVRQAQGKGLEWVGLIRNKANSYTTEYAAAVK-GRFTISRDDSKNTLYLQMSSLKTEDTAVYYCTK--------------------',
                'IGHV3-13*03': 'EVQLVESGG-GLVQP-VGSLRLSCAAS-GFTF----SNYYMHWVRQAQGKGLEWVGLIRNKANSYTTEYAAAVK-GRFTISRDDSKNTLYLQMTSLKTEDTALYYCTK--------------------',
                'IGHV3-13*04': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYYMHWVRQAQGKGLEWVGLIKNKANSYTTEYAAAVK-GRFTISRDDSKNTLYLQMSSLKTEDTALYYCTK--------------------',
                'IGHV3-132*01': 'VEQLVESGG-GLVQP-GASLRLSCAAS-EFTF----SSYDMHWVRQAPGKGLEWVSGISIG---GGTYYPDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-132*02': 'VEQLVESGG-ALVQP-GASLRLSCAAS-EFTF----SSYDMHWVRQAPGKGLEWVSAISIG---GGTYYPDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-132*04': 'VEQLVESGG-GLVQP-GASLRLSCAAS-EFTF----SSYDMHWVRQAPGKGLEWVSAISIG---GGTYYPDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-134*01': 'EVQLVESGG-GLVKP-GGSLRLSCAAS-GFTF----DDYAMSWVRQAPGKGLEWVSRISWD--GGSTYYADSVK-GRFTISRDNAKNTLYLQMDRLRAEDTALYYCSR--------------------',
                'IGHV3-136*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYDMSWVRQAPGKGLEWVSYISYT--GKTIYYADSVK-GRFTISRDNAKNSLSLQMSSLRAEDTAVYYCTR--------------------',
                'IGHV3-153*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYAMDWVRQAPGKGPEWVGFIRSKAYGGTAEYAASVK-GRFTISRDDSKNTAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-153*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYAMDWVRQAPGKGLEWVGFIRSKAYGGTAEYAASVK-GRFTISRDDSKNTAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-153D*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYAMDWVRQAPGKGLEWVGFIRSKAYGGTAEYAASVK-GRFTISRDDSKNTAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-153D*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYAMDWVRQAPGKGLEWVGFIRSKAYGGTAEYAASVK-GRFTISRDDSKNTAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-16*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMNWVRQAPGKGLEWVGFIKNKADGGTAAYAESVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3-16*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMSWVRQAPGKGLDWVGFIKNKADGGTAAYAESVK-GRFTISRDDSKNTLYLQMSSLNTEDTAVYYCTR--------------------',
                'IGHV3-16*03': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMSWVRQAPGKGLEWVGFIKNKADGGTAAYAESVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3-16*05': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMSWVRQAPGKGLEWVGFIKNKADGGTAAYAESVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3-175*01': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTI----SSYWMSWVCQAPGKGLEWLSDIYG----STMYYGDSVK-GLFTVSRDNAKNSLYLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-175*02': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTI----SSYWMSWVCQAPGKGLEWLSDIYG----STMYYGDSVK-GLFTVSRDNAKNSLSLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-175*03': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTI----SSYWMSWVCQAPGKGLEWLSDIYG----STMYYGDSVK-GLFTVSRDNAKNSLYLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-175*04': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTV----SSYWMSWVRQAPGKGLEWLSDIYG----STMYYGDSVK-GRFTVSRDNAKNSLYLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-175*05': 'EVQLAESGG-GLVQP-GGSLRLSCAAS-GFTV----SSYWMSWVRQAPGKGLEWLSDIYG----STMYYGDSVK-GRFTVSRDNAKNSLYLQMNSLRAEDTAVYYCTR--------------------',
                'IGHV3-176*01': 'EVQLVESGG-GLVQPGGGSLRLSCAAS-GFTF----SDDYMEWVRQAPGKGLEWVGQINPN--GGTTFLMDSVK-GRFTISRDNAKNTLYLQINSLKIEDTAVYYCTR--------------------',
                'IGHV3-176*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDDYMEWVRQAPGKGLEWVGQINPN--GGTTFLMDSVK-GRFTISRDNAKNTLYLQINSLKIEDTAVYYCTR--------------------',
                'IGHV3-176*03': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDDYMEWVRQAPGKGLEWVGQINPN--GGTTFLMDSVK-GRFTISRDNAKNTLYLQINSLKIEDTAVYYCTR--------------------',
                'IGHV3-178*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-178*02': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-178*03': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMHWVRQASGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-178*04': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-178*05': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-182*05': 'EVQLVETGG-GLVQP-GGSLKLSCAAS-GFTF----SSYGMSWVRQAPGKGLEWVSAINSG--GGSTYYADSVK-GRFTISRDNSKNTLSLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-182*06': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYYMYWVRQAPGKGLEWISAINTG--GGSTYYADSVK-GRFTISRDNSKNTLSLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3-183*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GDYGMHWVRQAPGKGLEWVSSISNT--GKTVYYADSVK-GRFTISRDNAKNSLSLQMSSLRAEDTAVYYCTR--------------------',
                'IGHV3-183*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GDYGMHWVRQAPGKGLEWVSSISNT--GKTVYYADSVK-GRFTVSRDNAKNSLSLQMSSLRAEDTAVYYCTR--------------------',
                'IGHV3-183*03': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GDYGMHWVRQAPGKGLEWVSSISSA--SSYIYYADSVK-GRFTISRDNAKNSLSLQMSSLRAEDTAVYYCTR--------------------',
                'IGHV3-183*04': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GDYGMHWVRQAPGKGLEWVSFISYT--GKTIYYADSVK-GRFTVSRDNAKNSLSLQMSSLRAEDTAVYYCTR--------------------',
                'IGHV3-183*06': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GDYGMHWVRQAPGKGLEWVSSISNT--GKTIYYADSVK-GRFTVSRDNAKNSLSLQMSSLRAEDTAVYYCTR--------------------',
                'IGHV3-184*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYYMYWVRQAPGKGLEWVGFIRSKAYGGTAEYAASVK-GRFTISRDDSKSIAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-184*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYYMYWVRQAPGKGLEWVGFIRSKAYGGTAEYAASVK-GRFTISRDDSKSIAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3-186*02': 'EVQLVESGG-GLVQP-GGSLRPSCAAS-GFTF----SSSAMHWVRQASGKGLEWVGRIRSKSNNYATEYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCAR--------------------',
                'IGHV3-186*03': 'EVQLVESGG-GLVQP-GGSLRPSCAAS-GFTF----SSSAMHWVRQASGKGLEWVGRIRSKSNNYATEYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCAR--------------------',
                'IGHV3-186*04': 'EVQLVESGG-GLVQP-GGSLRPSCAAS-GFTF----SSSAMHWVRQASGKGLEWVGRIRSKSNNYATEYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCAR--------------------',
                'IGHV3-201*01': 'EVQLVESGG-GVVQP-GGSLRLSCAAS-GFTF----DDYAMHWVRQAPGKGLEWVSGISWS--GGSTYYADSVK-GRFTISRDNAKNSLYLQMGSLRAEDTALYYCAK--------------------',
                'IGHV3-28*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SSYWMHWVRQAPGKGLEWISAINSA--GSSTYYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTALYYCAG--------------------',
                'IGHV3-28*02': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SNYWMHWVRQAPGKGLEWISAINSA--GSSTYYADSVK-GRFTISRENAKNTLYLQMDGLRAEDTAVYYCAG--------------------',
                'IGHV3-28*05': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SSYWMHWVRQAPGKGLEWISAINSA--GSSTYYADSVK-GRFTISRENAKNTLYLQMDGLRAEDTAVYYCAG--------------------',
                'IGHV3-28*06': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SNYWMYWVRQAPGKGLEWISAINSA--GSSTYYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAK--------------------',
                'IGHV3-30*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNVWMNWVRQAPGKGLEWVARIKRKADGETADYAASVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV3-30*02': 'EVQLVESGA-GLVQP-GGSLRLSCAAS-GFTF----SNSWMSWVRQAPGKGLEWVARIKRKADGETADYAASVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV3-30*03': 'EVQLVESGA-GLVQP-GGSLRLSCAAS-GFTF----SNSWMSWVRQAPGKGLEWVARIKRKADGETADYAASVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV3-32*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAP-GFTS----GNSDLIWIRQAPGKGLEWVSYISSG---GSIYYSDSVK-GRFTISRDNAKNTLYLQMSSLRVEDTAVYYCAK--------------------',
                'IGHV3-32*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTS----GNSDLIWIRQAPGKGLEWVSYISSG---GSIYYSDSVK-GRFTISRDNAKNTLYLQMSSLRVEDTAVYYCAK--------------------',
                'IGHV3-32*03': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTS----GNSDLIWIRQAPGKGLEWVSYISSG---GSIYYSDSVK-GRFTISRDNAKNTLYLQMSSLRVEDTAVYYCAK--------------------',
                'IGHV3-32*05': 'EVQLVESGG-GLVQP-GGSLRLSCAAP-GFTS----GNSDLIWIRQAPGKGLEWVSYISSG---GSIYYSDSVK-GRFTISRDNAKNTLYLQMSSLRVEDTAVYYCAK--------------------',
                'IGHV3-32*06': 'EVQLVESGG-GLVQP-GGSLRLSCAAP-GFTS----GNSDLIWIRQAPGKGLEWVSYISSG---GSIYYSDSVK-GRFTISRDNAKNTLYLQMSSLRVEDTAVYYCAK--------------------',
                'IGHV3-34*01': 'EVKLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMFWVRQAPGKGLEWVSSISGS--SSSTYYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVHYCAR--------------------',
                'IGHV3-34*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMFWVRQAPGKGLEWVSSISGS--SSSTYYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTTVHYCAR--------------------',
                'IGHV3-34*04': 'EVKLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SNYWMFRFRQAPGKGLEWVSSISGS--SSSTYYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVHYCAR--------------------',
                'IGHV3-35*01': 'EVQLVEYGG-GLVQP-GGSLRLSC----GFTF----SVHFMSWVRQAPGKGPEWVGFMRNKANGGTAEYATSVK-GRFTISRDDSKSIAYLQMSSLNTEDTAVYYCAR--------------------',
                'IGHV3-35*02': 'EVQLVAYGG-GLEQP-GGSLRLSC----GFTF----SDHYMSWVRQAPGKGPEWVGFMRNKANGGTTEYATSVK-GRFTISRADSKSMASLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-35*03': 'EVQLVEYGG-GLEQP-GGSLRLSC----GFTF----SVHFMSWVRQAPGKGPEWVGFMRNKANGGTAEYATSVK-GRFTISRDDSKSIAYLQMSSLNTEDTAVYYCAR--------------------',
                'IGHV3-35*04': 'EVQLVAYGG-GLEQP-GGSLRLSC----GFTF----SDHYMSWVCQAPGKGPEWVGFMRNKANGGTTEYATSVK-GRFTISRADSKSMASLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-37*01': 'EVQLVESGG-GLVQP-GGSLRLSCVAS-GFTF----SDYCMDWVRQASGKGLEWVSSISGS--SSNTYYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-37*02': 'EVQLVESGG-GLVQP-GGSLRLSCATS-GFTF----SNYWMFWVRQAPGKGLEWVSSISGS--SSSTYYPDSVK-GRFTISRDNAKNTLYLQMNSPRAEDTAVYYCAR--------------------',
                'IGHV3-37*04': 'EVQLVESGG-GLVQP-GGSLRLSCVAS-GFTF----SDHYMDWVRQATGKGLEWVSSISQPS-GSNTYYLDPVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-37*05': 'EVQLVESGG-GLVQP-PGSLRLSCATS-GFTF----SNYWMFWVRQAPGKGLEWVSSISGS--SSSTYYPDSVK-GRFTISRDNAKNTLYLQMNSPRAEDTAVYYCAR--------------------',
                'IGHV3-38*01': 'EVQLVESGG-GLAQP-GGSLRLSCAAS-GFTF----SDHYMDWVRQAPGKGLEWVGRIRNKANSYTTEYAASVK-GRFTISRDDSKNTLYLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-38*02': 'EVQLVESGG-GLAQP-GGSLRLSCAAS-GFTF----SDHYMDWVRQAPGKGLEWVSRIRNKANSYTTEYAASVK-GRFTISRDDSKNTLYLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-38*03': 'EVQLVESGG-GLAQP-GGSLRLSCAAS-GFTF----SDHYMDWVRQAPGKGLEWVGRIRNKANSYPTEYAASVK-GRFTISRDDSKNTLYLQMSSLKTEDTAVYYCAR--------------------',
                'IGHV3-46*02': 'EVQLVESGG-GLVQP-GGSLRLSCTAS-GFTF----SSTRINWIRQSPGKRLEWVADIKYD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCVR--------------------',
                'IGHV3-46*05': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSTRMNWIRQAPGKRLEWVADIKYD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCVR--------------------',
                'IGHV3-46*06': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSTRMNWIRQAPGKRLEWVADIKYD--GSEKYYVDSVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCVR--------------------',
                'IGHV3-5-2*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GSYGMHWARQAPGKGLEWVSAINTG--GGSTWYTDSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-5-3*01': 'EMQLVESGG-GLVQP-GGSLRVSCAAS-GFTF----SDHYMYWFRQAPGKGPEWVGFIRNKAKGGTAEYAASVK-GRFTISRDDSKNVTYLQMSSLKTEDTAVYYCTT--------------------',
                'IGHV3-5-3*02': 'EMQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDHYMEWFRQAPGRGPEWVGFIRNKAKGGTAEYAASVK-GRFTISRDDSKNVTYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV3-54*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYGMSWVRQTPGKGLEWVAVIWYD--GSKKYYADSVK-DRFTISRDNSKNMLYLQMNNLKLEDTAVYYCGR--------------------',
                'IGHV3-54*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSKKYYADSVK-DRFTISRDNSKNMLYLQMNNLKLEDTAVYYCAR--------------------',
                'IGHV3-54*03': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYGMHWVRQAPGKGLEWVAVIWYD--GSKKYYADSVK-DRFTISRDNSKNMLYLQMNNLRVEDMAVYYCVK--------------------',
                'IGHV3-54*04': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYGMHWVRQAPGKGLEWVAVISYD--GSKKYYADSVK-DRFTISRDNSKNMLYLQMNNLKLEDTAVYYCAR--------------------',
                'IGHV3-58*01': 'EAQLMETGG-GLVQP-GGSLRLSCAAS-GFTF----SDHYMQWVRQAPGKGLEWVGLIRNKADGETTDYALSVK-GRFTISRDDSKSITYLQMNNLKTEDTAVYYCAR--------------------',
                'IGHV3-58*02': 'EAQLMETGG-GLVQP-GGSLRLSCADS-GFTF----SDHYMQWVRQAPGKGLEWVGLIRNKADGETTDYAASVK-GRFTISRDDSKSITYLQMNNLKTEDTAVYYCAR--------------------',
                'IGHV3-58*04': 'EAQLMESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYYMQWVRQAPGKGLEWVGLIRNKADGETTDYALSVK-GRVTISRDDSKSITYLQMNNLKTEDTAVYYCAR--------------------',
                'IGHV3-58*05': 'EAQLMETGG-GLVQP-GGSLRLSCAAS-GFTF----SDHYMQWVRQAPGKGLEWVGLIRNKADGETTDYAASVK-GRFTISRDDSKSITYLQMNNLKTEDTAVYYCAR--------------------',
                'IGHV3-59*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMHWVRQASGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-59*02': 'EVQLVESGG-GLVKP-GGSLRLSCAAS-GFTF----SDYYMHWVRQASGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYFQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-59*03': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYNMHWVRQASGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-59*04': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYYMHWVRQASGKGLEWVSRISNG--GGSTWYADSVK-GRFTISRENAKNTLYLQMDSLRAEDTAVYYCAR--------------------',
                'IGHV3-67*02': 'EVQLVESGG-GVVQP-GGSLRLSCAAS-GFTF----DDYAMHWVRQAPGKGLEWVSGISWS--GGSTYYADSVK-GRFTISRDNAKNSLYLQMGSLRAEDTALYYCAK--------------------',
                'IGHV3-67*04': 'EVQLVESGG-GVVQP-GGSLRLLCAAS-GFTF----DDYAMGWVRQAPGKGLEWVSAISWD--GGSTGYADSVK-GRFTISRDNAKNSLYLQMDRLRAEDTALYYCAR--------------------',
                'IGHV3-72*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYAMQWVHQAPGKGLEWVSAIGPG---GDTYYADAVK-GRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3-78*01': 'EVQLVESGG-GVVQP-GGSLRLSCAAS-GFTF----DDYAMGWVRQAPGKGLEWVSAISWN--GDSTYYADSVK-GRFTISRENAKNSLYLQINRLRAEDTALYYCAR--------------------',
                'IGHV3-78*02': 'EVQLVESGG-GVVQP-GGSLRLLCAAS-GFTF----DDYAMHWVRQAPGKGLEWVSDISWS--GGSTYYADSVK-GRFTISRDNAKNSLYLQMNRLRAEDTALYYCAR--------------------',
                'IGHV3-8*01': 'EVQLVESGG-GLVQP-GGSLRLSCTGS-GFTF----SSYYMYWVRQAPGKGLEWVSAINTG--GGSTWYTDSVK-GRFTISKENAKNTLYLQMDSLRAEDTAVYYCAK--------------------',
                'IGHV3S4*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYDMSWVRQALGKGLEWVSSISNT--GKTIYYADSVK-GRFTISRDNAKNSLSLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3S4*02': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYDMSWVRQALGKGLEWVSSISNT--GKTIYYADSVK-GRFTISRDNAKNSLSLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3S40*01': 'EVQRVESGG-GLVQP-GGSLRLSCAAS-GFTI----SSSWMNWDFQAPGKGLECVSHISSG---VSTDYPDSIK-GQFTISRDNTETMLYVQMNSLRAEDMAVNYCAR--------------------',
                'IGHV3S41*01': 'EVQLVESGG-GLVQP-GGSLRLSCATS-GFTF----SNYWMYWFRQAPGKGLEWVSSISGS--SSNTYYPDSVK-GRFTISRDNAKNTLYLQMNSLRAEDTAVYYCAR--------------------',
                'IGHV3S42*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SSYWMNWVRQTPGKGLEWISAINSG--GGSTYYADSVK-GRFTISRDNSKNTLSLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3S43*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAP-GFTS----GNSDLIWIRQAPGKGLEWVSYISSG---GSIYYSDSVK-GRFTISRDNAKNTLYLQMSSLRVEDTAVYYCAK--------------------',
                'IGHV3S46*01': 'EVQLVESGG-GLAKP-GGSLRLSCAAS-GFTF----SDYAMDWVRQAPGKGLEWVGFIRSKAYGGTAEYAAFVK-GRFTISRDDSKNTAYLQMSSLKTEDTAVYYCTR--------------------',
                'IGHV3S51*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYDMSWVRQAPGKGLEWVSSISSA--SSYIYYADSVK-GRFTISRDNAKNSLSLQMNSLKTEDTAVYYCTR--------------------',
                'IGHV3S56*01': 'EVQLVESGG-GVVQP-GGSLRLLCAAS-GFTF----DDYAMGWVRQAPGKGLEWVSAISWD--GDSTGYADSVK-GRFTISTDNAKNSLYLQMGSLRAEDTALYYCAR--------------------',
                'IGHV3S60*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----GDYGMSWVRQAPGKGLEWVSGISWS--GGSTYYADSVK-GRFTISRDNAKNSLYLQMGSLRAEDTALYYCAK--------------------',
                'IGHV3S61*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----DDYAMGWVRQAPGKGLEWVSGISWS--GGSTGYADSVK-GRFTISRDNAKNSLYLQMDRLRAEDTALYYCAR--------------------',
                'IGHV3S62*01': 'EVQLVESGG-GLVQP-GGSLRPSCAAS-GFTF----SSSAMHWVRQASGKGLEWVGRIRSKSNNYATEYAASVK-GRFTISRDDSKNTAYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV3S64*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SSYGMHWVRQAPGKGLEWVAIIYYD--GSQKYYADSVK-GRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK--------------------',
                'IGHV3S66*01': 'EVQLVESGG-GLVQP-GGSLRLSCAAS-GFTF----SDYYMDWVRQAPGKGLEWVGFIKNKADGGTADYAASVK-GRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTT--------------------',
                'IGHV4-106*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---DDYYWSWIRQPPGKGLEWIGYIYGS--GGGTNYNPSLK-NRVTISIDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-106*03': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSF----SGYYWGWIRQPPGKGLEWIGYISGS--SGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-106*05': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSF----SGYYWGWIRQPPGKGLEWIGYISGS--SGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-122*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS--SSYYYWSWIRQAPGKGLEWIGYIYGG--SGSTSYNPSLK-SRVTISRDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-122*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS--SGYYYWSWIRQPPGKGLEWIGYITYS---GSTSYNPSLK-SRVTISRDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-127*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYGWSWIRQPPGKGLEWIGYIGGS--SGSTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-127*03': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYGWSWIRQPPGKGLEWIGYIGGS--SGSTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-127*04': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYGWSWIRQPPGKGLEWIGYISGS--SGSTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-143*01': 'QVQLQESGP-GLVKP-SETLSLTCTVS-GGSIS---GYYYWSWIRQPPGKGLEWIGGIYGN--SASTYYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-147*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRIYGS--SGSTSYNPSLT-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-147*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRIYGS--SGSTSYNPSLT-SRVTISKDTSKNQFFLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-147*03': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRIYGS--SGSTSYNPSLT-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-160*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRIYGS--GGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-160*02': 'QVQLQESGP-GLVKP-SETLPLTCAVS-GASI----SSNYWSWIRQSPGKGLEWIGRIYGS--GGSTDYNPSLK-SRVTISIDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-160*03': 'QLQLQESGP-GLVKP-SETLPLTCAVS-GASI----SSNYWSWIRQAPGKGLEWIGRIYGS--GGSTDYNPSLK-SRVTISIDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-160*04': 'QVQLQESGP-GLVKP-SETLPLTCAVS-GASI----SSNYWSWIRQSPGKGLEWIGRIYGS--GGSTDYNPSLK-SRVTISRDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-160*05': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRIYGS--GGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-165*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWSWIRQPPGKGLEWIGYIGGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-165*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWNWIRQPPGKGLEWIGYIGGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-165*03': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWNWIRQPPGKGLEWIGYIGGS--SGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-165*05': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWNWIRQTPGKGLEWIGYIGGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-169*01': 'QLQLQESGP-GLVKP-SETLSVTCAVS-GGSI----SSSYWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-169*02': 'QLQLQESGP-GLVKP-SETLSVTCAVS-GGSI----SSSYWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAS--------------------',
                'IGHV4-169*03': 'QLQLQESGP-GLVKP-SETLSVTCAVS-GGSI----SSSYWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-169*06': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSSYWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-169*07': 'QLQLQESGP-GLVKP-SETLSVTCAVS-GGSI----SSSYWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-169*08': 'QVQLQESGP-GLVKP-SETLSVTCAVS-GGSI----SSSYWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAS--------------------',
                'IGHV4-173*01': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRISGS--GGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-173*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQPPGKGLEWIGRISGS--GGSTDYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-5-1*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSF----SSDWWGWIRQPPGKGLEWIGSIYGS--GGSNYLNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-5-1*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSF----SSDWWGWIRQPPGKGLEWIGSIYGS--GGSNYLNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-57*02': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGRISGS--GGSTSDNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-57*04': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGRISGS--GGSTSYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-57*05': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNWWSWIRQPPGKGLEWIGRISGS--GGSTSYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-65*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGYISGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-65*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGNIGGS--SGSTYYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-65*03': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGYISGN--SASTYYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-65*05': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGYISGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-73*01': 'QVKLQQWGE-GLVKP-SETLSLTCAVY-GGSIS---GYYYWSWIRQPPGKGLEWIGYIYGN--SASTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-73*02': 'QVKLQQWGE-GLVKP-SETLSLTCAVY-GGSIS---DYYYWSWIRQAPGKGLEWIGYIYGN--SASTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-73*03': 'QVKLQQWGE-GLVKP-SETLSLTCAVY-GGSIS---GYYYWSWIRQPPGKGLEWIGYIYGN--SASTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-73*04': 'QVKLQQWGE-GLVKP-SETLSLTCAVY-GGSIS---G-YYWSWIRQPPGKGLEWIGNIDGN--SASTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-76*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---GGYDWSWIRQPPGKGLEWIGYIYGS--SGSTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-76*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SGYDWSWIRQPPGKGLEWIGYIYGS--SGSTNYNPSLK-NRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-76*04': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---SGYDWSWIRQPPGKGLEWIGYIYGS--SGSTNYNPSLK-NRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-80*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GASI----SSYWWSWIRQPPGKGLEWIGEINGN--SGSTYYNPSLK-SRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-80*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSF----SSYWWSWIRQPPGKGLEWIGEINGN--SGSTNYNPSLK-SRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-80*03': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GAPI----SSYWWSWIRQPPGKGLEWIGEINGN--SGSTYYNPSLK-SRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-80*04': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GASI----SSYWWSWIRQPPGKGLEWIGEINGN--SGSTYYNPSLK-SRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-80*05': 'QVQLQESGP-GLVKP-SETLSLTCTVS-GASI----SSYWWSWIRQPPGKGLEWIGEINGN--SGSTNYNPSLK-SRVTISKDASKNQFSLKLSSVTTADTAVYYCAR--------------------',
                'IGHV4-81*01': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWSWIRQPPGKGLEWIGNIDGN--IAGTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-86*02': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---GGYGWSWIRQPPGKGLEWIEYIGGS--SGSTNYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-92*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGRISGS--GGSTSDNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-92*03': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSSYWSWIRQSPGKGLEWIGYIYGG--SGSTSYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-92*05': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQSPGKGLEWIGYIYGG--SGSTSYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-93*01': 'QVQLQESGP-AVVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQSPGKGLEWIGGIYGS--GGSTEYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-93*02': 'QVQLQESGP-AVVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQSPGKGLEWIGGIYGS--GGSTEYNPSLK-SRVTISIDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-93*04': 'QVQLQESGP-AVVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQSPGKGLEWIGGIYGS--GGSTEYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-99*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYYWGWIRQPPGKGLEYIGYISGS--SGSTYYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-99*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYYWGWIRQPPGKGLEYIGYISGS--SGSTYYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-99*03': 'QVQLQESGP-GLVKP-SETLSLTCTVS-GGSF----SSYWWGWIRQPPGKGLEWIGHI-SS--GGSNYLNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4-99*04': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYYWGWIRQPPGKGLEYIGYISGS--SGSTYYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S10*01': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---DSYWWSWIRQPPGKGLEWIGYIYGS--STSTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S10*02': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---DSYWWSWIRQPPGKGLEWIGYIYGS--STSTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S10*04': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---DSYRWSWIRQPPGKGLEWIGYIYGS--STSTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S13*01': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS--SGYYYWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S13*02': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS--SGYYYWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTTADTAVYYCAR--------------------',
                'IGHV4S13*03': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---GGYYWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S13*04': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---GYYYWSWIRQPPGKGLEWIGSIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTATDTAVYYCAR--------------------',
                'IGHV4S14*01': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---GYYLWSWIRQPPGKGLEWIGYIGGS--SGSTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S14*02': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---GYYLWSWIRQPPGKGLEWIGYIGGS--SGSTNYNPSLK-NRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S15*01': 'QVQLQESGP-GLLKP-SDTLSLTCAVS-GGSIS---GGYGWGWIRQPPGKGLEWIGSIYSS--NGNTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S15*02': 'QVQLQESGP-GLLKP-SETLSLTCAVS-GGSIS---GGYGWGWIRQPPGKGLEWIGSIYSS--SGNTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S15*03': 'QVQLQESGP-GLLKP-SETLSLTCAVS-GGSIS---GGYGWGWIRQPPGKGLEWIGSIYSS--SGNTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S16*01': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGTIS--SGYYYWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTATDTAVYYCAR--------------------',
                'IGHV4S16*02': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGTIS--SGYYYWSWIRQPRGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTATDTAVYYCAR--------------------',
                'IGHV4S16*03': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGTIS--SGYCYWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTATDTAVYYCAR--------------------',
                'IGHV4S17*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S17*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SSNWWSWIRQPPGKGLEWIGGIYSN--SESTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S18*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS--GGYYYWSWIRQPPGKGLEWIGGIYSS--SGNTYYNPSLK-SRVTISRDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S19*01': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---GGYGWSWIRQPPGKGLEWIGHIFGS--IGSTYYNPSLK-SRVTISIDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S19*02': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---SGYGWSWIRQPPGKGLEWIGNIFGS--IGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S2*01': 'QVQLQESGP-GLVKP-SETLPLTCAVS-GASI----SSNYWSWIRQAPGKGLEWIGRIYGS--GGSTDYNPSLK-SRVTISIDTCKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S21*01': 'QLQLQESGP-GLVKP-SETLSLTCTVS-GASI----SSYWWSWIRQAPGKGLEWIGYIYGS--GSSTNYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S22*01': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWSWIRQPPGKGLEWIGRI-DS--SGSTYYNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S22*02': 'QLQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYWWSWIRQPPGKGLEWIGRI-DS--SGSTYYNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S23*01': 'QVQLKESGP-GLVKP-SETLSLTCAVS-GGSIS---SGYGWGWIRQPPGKGLEWIVTIYSS--TGNTYYDPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S25*01': 'QVQLQESGP-GLVKP-SETLSLTCTVS-GGSI----SGYYWSWIRQPPGKGLEWIGGIGGG--SGSTEYNPSLK-SRVTISRDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S25*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWSWIRQPPGKGLEWIGYIGGG--SGSTEYNPSLK-SRVTISRDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S26*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---GGYYWSWIRQPPGKGLEWIGNIYGS--SASTNYNPSLK-SRVTISKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S27*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYYWSWIRQPPGKGLEWIGYIGGG--SGSTEYNPSLK-SRVTISRDTSKNHFSLKLSSVTAADTAVYYCAS--------------------',
                'IGHV4S28*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYWWGWIRQPPGKGLEWIGYIGGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S28*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SGYWWGWIRQPPGKGLEWIGYIGGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAS--------------------',
                'IGHV4S29*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSYWWGWIRQPPGKGLEWIGHI-SS--GGSNYLNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S30*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSIS---SGYYWGWIRQPPGKGLEWIGHIS-S--GGSNYLNPSLK-SRVTLSVDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S31*01': 'QVQLQESGP-GVVKP-SETLSLTCAVS-GGSIS---SGYGWSWIRQPPGKGLEWIGYIYGS--SGSTNYNPSLK-NRVTISKDASKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S32*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---GGYYWSWIRQPPGKGLEWIGQIYGG--SGSTYYNPSLK-SRVTVSKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S32*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---GGYGWSWIRQPPGKGLEWIGQIYGG--SGSTYYNPSLK-SRVTVSKDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S34*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GYSI----SSNYWSWIRQPPGKGLEWIGYIYGS--SGSTYYNPSLK-SRVTISTDTSKNQFSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S35*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSI----SSNYWSWIRQAPGKGLEWIGYIYGS--G-STYYNPSLK-SRVTLSVDTSKNQLSLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S9*01': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSIS---DYYYWNWIRQPPGKGLEWIGNIYGN--SASTYYNPSLK-SRVTISKDTSKNQFFLKLSSVTAADTAVYYCAR--------------------',
                'IGHV4S9*02': 'QVQLQESGP-GLVKP-SETLSLTCAVS-GGSSG---DYYYWNWIRQPPGKGLEWIGNIGGN--SASTYYNPSLK-SRVTISKDTSNNQFFLKLSSVTAADTAVYYCAR--------------------',
                'IGHV5-20*01': 'EVQLVQSGA-EVKRP-GESLKISCKTS-GYSF----TSYWISWVRQMPGKGLEWMGAIDPS--DSDTRYNPSFQ-GQVTISADKSISTAYLQWSRLKASDTATYYCAK--------------------',
                'IGHV5-20*02': 'EVQLVQSGA-EVKRP-GESLKISCKTS-GYSF----TSYWISWVRQMPGKGLEWMGAIDPS--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTATYYCAK--------------------',
                'IGHV5-20*03': 'EVQLVQSGA-EVKRP-GESLKISCKTS-GYSF----TSYWISWVRQMPGKGLEWMGAIDPS--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKDSDSATYYCAK--------------------',
                'IGHV5-20*04': 'EVQLVQSGA-EVKRP-GESLKISCKTS-GYSF----TSYWISWVRQMPGKGLEWMGAIDPS--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTATYYCAK--------------------',
                'IGHV5-43*01': 'EVQLVQSGA-EVKRP-GESLRISCKTS-GYSF----TSSWISWVRQMPGKGLEWMGSIYPG--DSDTRYNPSFQ-GHVTISADKSISTTYLQWSSLKASDTATYYCAK--------------------',
                'IGHV5-43*02': 'EVQLVQSGA-EVKRP-GESLKISCKTS-GYSF----TSSWISWVRQMPGKGLEWMGSIYPG--DSDTKYNPSFQ-GHVTISADKSISTTYLQWSSLKASDTATYYCAK--------------------',
                'IGHV5-43*03': 'EVQLVQSGA-EVKRP-GESLRISCKTS-GYSF----TSYWISWVRQMPGKGLEWMGMIYPG--DSDTRYSPSFQ-GQVTISADKSISTTYLQWSSLKASDTATYYCAK--------------------',
                'IGHV5-43*04': 'EVQLVQSGA-EVKRP-GESLRISCKTS-GYSF----TSSWISWVRQMPGKGLEWMGSIYPG--DSDTRYNPSFQ-GHVTISADKSISTTYLQWSSLKASDTATYYCAK--------------------',
                'IGHV5-43*05': 'EVQLVQSGA-EVKRP-GESLRISCKTS-GYSF----TSYWISWVRQMPGKGLEWMGMIYPG--DSDTRYSPSFQ-GQVTISADKSISTAYLQWSSLKASDTATYYCAK--------------------',
                'IGHV6-1*01': 'QVQLQESGP-GLVKP-SQTLSLTCAIS-GDSVS--SNSATWNWIRQSPSRGLEWLGRTYYRS-KWYNDYAQSVQ-NRISINPDTSKNQFSLQLNSVTPEDMAVYYCAR--------------------',
                'IGHV6-1*02': 'QVQLQESGP-GLVKP-SQTLSLTCAIS-GDSVS--SNSATWNWIRQSPSRGLEWLGRTYYRS-KWYNDYAQSVQ-NRITINPDTSKNQFSLQLNSVTPEDMAVYYCAR--------------------',
                'IGHV7-114*01': 'QVQLVQSGA-EVKQP-GASVKVSCKAS-GYTF----TSYGMNWVRQAHGQRLEWMGWINTD--TGNPTYAQGFK-ERFTFSMDTSISTAYLQISSLKAEDTAVYYCAR--------------------',
                'IGHV7-114*02': 'QVQLVQSGA-EVKQP-GASVKVSCKAS-GYTF----TSYSMHWVRQAHGQRLEWMGWINTD--TGNPTYAQGFK-ERFTFSMDTSISTAYLQISSLKAEDTAVYYCAR--------------------',
                'IGHV7-114*03': 'QVQLVQSGA-EVKQP-GASVKVSCKAS-GYTF----TSYGMNWVRQAHGQRLEWMGWINTD--TGNPTYAQGFK-ERFTFSMDTSISTAYLQISSLKAEDTAVYYCAR--------------------',
                'IGHV7-114*04': 'QVQLVQSGA-EVKQP-GASVKVSCKAS-GYTF----TSYGMNWVRQAHGQRLEWMGWINTD--TGNPTYAQGFK-ERFTFSMDTSISMAYLQISSLKAEDTAVYYCAR--------------------',
                'IGHV7-193*01': 'QVQLVQSGP-EVKQP-GASVKVSCKAS-GYSS----TTYGMNWVRQAPGQGLEWMGWMNTY--TGNPTYAQGFT-ERFVFSMDTSVSTVYLQISSLKAEDTAVYYCAR--------------------',
                'IGHV7-193*02': 'QVQLVQSGP-EVKQP-GASVKVSCKAS-GYSF----TTYGMNWVRQAPGQGLEWMGWMNTY--TGNPTYAQGFT-ERFVFSMDTSVSTVYLQISSLKGEDTAVYYCAR--------------------',
                'IGHV7-193*03': 'QVQLVQSGP-EVKQP-GASVKVSCKAS-GYSF----TTYGMNWVRQAPGQGLEWMGWMNTY--TGNPTYAQGFT-ERFVFSMDTSISTVYLQISSLKAEDTAVYYCAK--------------------',
                'IGHV7-193*04': 'QVQLVQSGP-EVKQP-GASVKVSCKAS-GYSS----TTYGMNWVQQAPGQGLEWMGWMNTY--TGNPTYAQGFT-ERFVFSMDTSVSTVYLQISSLKAEDTAVYYCAR--------------------'},
                'pig': {'IGHV1-10*01': 'EVKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSTYINWVRQAPGKGLEWLAAISTS--GGSTYYADSVK-GRFTISRDDSQNTAYLQMNSLRTEDTARYYCAT----------------------',
                'IGHV1-11*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFDF----SSYGVGWVRQAPGKGLESLASIGSGSYIGSTYYADSVK-GRFTISSDDSQNTVYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1-12*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFDF----SDYAFSWVRQAPGKGLEWVAAIASSDYDGSTYYADSVK-GRFTISSDDSQNMVYLQMNSLRTEDTARYYCAI----------------------',
                'IGHV1-14*01': 'EVKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSYEISWVRQAPGKGLEWLAAISTS--GGSTYYADSVK-GRFTISKDDSQNTAYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1-15*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSYSMSWVRQAPGKGLEWLAGIYSS--GSSTYYADSVK-GRFTISSDNSQNTAYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1-2*01': 'EVKLVECGG-GLVQPGGSLRLSCVGSGYTF----SSYGMSWVRQAPGKGLEWLAGIDSGSYSGSSYYADSVK-GRFTISRDDSQNTAYLQMNSLRTEDTARYYCAT----------------------',
                'IGHV1-4*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSTYINWVRQAPGKGLEWLAAISTS--GGSTYYADSVK-GRFTISRDNSQNTAYLQMNSLRTEDTARYYCAT----------------------',
                'IGHV1-4*02': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSTYINWVRQAPGKGLEWLAAISTS--GGSTYYADSVK-GRFTISRDNSQNTAYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1-5*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGITF----SSYAVSWVRQAPGKGLESLASIGSGSYIGSTDYADSVK-GRFTISSDDSQNTVYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1-6*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFDF----SDNAFSWVRQAPGKGLEWVAAIASSDYDGSTYYADSVK-GRFTISSDNSQNTVYLQMNSLRTEDTARYYCAI----------------------',
                'IGHV1-6*02': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFDF----SDNAFSWVRQAPGKGLEWVAAIASSDYDGSTYYADSVK-GRFTISRDNSQNTVYLQMNSLRTEDTARYYCAI----------------------',
                'IGHV1-8*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGITF----SSYAVSWVRQAPGKGLEWLAGIDSGSYSGSTYYADSVK-GRFTISRDDSQNTAYLQMTSLRTEDTARYYCAG----------------------',
                'IGHV1S2*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSYEISWVRQAPGKGLEWLAGIYSS--GGSTYYADSVK-GRFTISRDNSQNTAYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1S5*01': 'EEKLVESGG-GLVQPGGSLRLSCVGSGFTF----SSYAVSWVRQAPGKGLEWLAGIDSGSYSGSTYYADSVK-GRFTISRDNSQNTAYLQMNSLRTEDTARYYCAR----------------------',
                'IGHV1S6*01': 'QEKLVESGG-GLVQPGGSLRLSCVGSGFDF----SSYGVGWVRQAPGKGLESLASIGSGSYIGSTDDADSVK-GRFTISSDNSQNTAYLQMNSLRTEDTARYYCAR----------------------'},
                'alpaca': {'IGHV3-1*01': 'EVQLVESGG-GLVQPGGSLRLSCAASGFTF----DDYAMSWVRQAPGKGLEWVSAISWN--GGSTYYAESMK-GRFTISRDNAKNTLYLQMNSLKSEDTAVYYCAK----------------------',
                'IGHV3-3*01': 'QVQLVESGG-GLVQAGGSLRLSCAASGRTF----SSYAMGWFRQAPGKEREFVAAISWS--GGSTYYADSVK-GRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAA----------------------',
                'IGHV3S1*01': 'QVQLVESGG-GLVQPGGSLRLSCAASGFTF----SSYWMYWVRQAPGKGLEWVSAINTG--GGSTYYADSVK-GRFTISRDNAKNTLYLQMNSLKSEDTAVYYCAK----------------------',
                'IGHV3S53*01': 'QVQLVESGG-GLVQPGGSLRLSCAASGSIF----SINAMGWYRQAPGKQRELVAAITSG---GSTNYADSVK-GRFTISRDNAKNTVYLQMNSLKPEDTAVYYCNA----------------------',
                'IGHV4S1*01': 'QVQLQESGP-GLVKPSQTLSLTCTVSGGSIT--TSYYAWSWIRQPPGKGLEWMGVIAYD---GSTYYSPSLK-SRTSISRDTSKNQFSLQLSSVTPEDTAVYYCAR----------------------'},
                'cow': {'IGHV1-10*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SSYGVGWVRQAPGKALECLGGISSG---GSTGYNPALK-YRLSITKDNSKSQVSLSLSSVTTEDTATYYCAK----------------------',
                'IGHV1-14*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SDNSVGWVRQAPGKALEWLGVIYSG---GSTGYNPALK-SRLSITKDNSKSQVSLSLSSVTTEDTATYYCAR----------------------',
                'IGHV1-14*02': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SDNSVGWVRQAPGKALEWLGVIYSG---GSTGYNPALK-SRLSITKDNSKSQVSLSLSSVTTEDTATYYCAR----------------------',
                'IGHV1-17*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SSYAVSWVRQAPGKALEWLGDISSG---GSTGYNPALK-SRLSITKDNSKSQVSLSVSSVTPEDTATYYCAK----------------------',
                'IGHV1-20*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SSYAVGWVRQAPGKALEWLGGISSG---GSTYYNPALK-SRLSITKDNSKSQVSLSVSSVTPEDTATYYCAK----------------------',
                'IGHV1-21*01': 'QVQLRESGP-SLVKPSQTLSLTCTISGFSL----SSYAVGWVRQAPGKALEWVGGISSG---GSTCLNPALK-SRLSITKDNSKSQVSLSVSSVTTEDTATYYCAK----------------------',
                'IGHV1-25*01': 'QVQLQESGP-SLVKTSQTLSLTCTASGLSL----TRYGIHWVRQAPGKALEWLGDISSG---GSTGYNPGLK-SRLSITKDNSKSQVSLSLSSLTPEDSATYYCAR----------------------',
                'IGHV1-27*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SSNGVGWVRQAPGKALEWVGGIDND---GDTYYNPALK-SRLSITKDNSKSQVSLSVSSVTPEDTATYYCAK----------------------',
                'IGHV1-30*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SSNGVVWVRQAPGKALEWLGGICSG---GSTSFNPALK-SRLSITKDNSKSQVSLSVSSVTPEDTATYYCAR----------------------',
                'IGHV1-30*02': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SSNGVVWVRQAPGKALEWLGGICSG---GSTSFNPALK-SRLSITKDNSKSQVSLSVSSVTPEDTATYYCAKR---------------------',
                'IGHV1-33*01': 'QVQLRESGP-SLVKPSQTLSLTCTISGFSL----SSYAVGWVRQAPGKALEWVGGISSG---GSTCLNPALK-SRLSITKDNSKSQVSLSVSSVTTEDTATYYCAK----------------------',
                'IGHV1-37*01': 'QVQLQESGP-SLVKTSQTLSLTCTASGLSL----TRYGIHWVRQAPGKALEWLGDISSG---GSTGYNPGLK-SRLSITKDNSKSQVSLSLSSLTPEDSATYYCAR----------------------',
                'IGHV1-39*01': 'KVQLQESGP-SLVKPSQTLSLTCTTSGFSL----TSYGVSWVRQAPGKALEWLGGIDSG---GSTGYNPGLK-SRLSITRDNSKSQVSLSVSSVTPEDTATYYCAK----------------------',
                'IGHV1-7*01': 'QVQLRESGP-SLVKPSQTLSLTCTVSGFSL----SDKAVGWVRQAPGKALEWLGGIDTG---GSTGYNPGLK-SRLSITKDNSKSQVSLSVSSVTTEDSATYYCTTVH--------------------'}},
                'K': {'human': {'IGKV1-12*01': 'DIQMTQSPSSVSASVGDRVTITCRASQGI------SSWLAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQAN--------------------',
                'IGKV1-12*02': 'DIQMTQSPSSVSASVGDRVTITCRASQGI------SSWLAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQAN--------------------',
                'IGKV1-13*02': 'AIQLTQSPSSLSASVGDRVTITCRASQGI------SSALAWYQQKPGKAPKLLIYDA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQFN--------------------',
                'IGKV1-16*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNYLAWFQQKPGKAPKSLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQYN--------------------',
                'IGKV1-16*02': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNYLAWFQQKPGKAPKSLIYAA-------SSLQSGVP-SKFSGSG--SGTDFTLTISSLQPEDFATYYCQQYN--------------------',
                'IGKV1-17*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------RNDLGWYQQKPGKAPKRLIYAA-------SSLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCLQHN--------------------',
                'IGKV1-17*02': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------RNDLGWYQQKPGKAPKRLIYAA-------SSLQSGVP-SRFSGSG--SGTEFTLTISNLQPEDFATYYCLQHN--------------------',
                'IGKV1-17*03': 'DIQMTQSPSAMSASVGDRVTITCRASQGI------SNYLAWFQQKPGKVPKRLIYAA-------SSLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCLQHN--------------------',
                'IGKV1-27*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNYLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDVATYYCQKYN--------------------',
                'IGKV1-27*02': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNYLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDVATYYCQKYN--------------------',
                'IGKV1-27*03': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNYLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDVATYYCQKYN--------------------',
                'IGKV1-33*01': 'DIQMTQSPSSLSASVGDRVTITCQASQDI------SNYLNWYQQKPGKAPKLLIYDA-------SNLETGVP-SRFSGSG--SGTDFTFTISSLQPEDIATYYCQQYD--------------------',
                'IGKV1-39*01': 'DIQMTQSPSSLSASVGDRVTITCRASQSI------SSYLNWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQSY--------------------',
                'IGKV1-40-1*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNNLNWYQQKPGKTPKLLIYAA-------PSLQSGIP-SRFSDSG--SGADYTLTIRSLQPEDFATYYCQQSD--------------------',
                'IGKV1-5*01': 'DIQMTQSPSTLSASVGDRVTITCRASQSI------SSWLAWYQQKPGKAPKLLIYDA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPDDFATYYCQQYN--------------------',
                'IGKV1-5*02': 'DIQMTQSPSTLSASVGDRVTIICRASQSI------SSWLAWYQQKPGKAPKLLIYDA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPDDFATYYCQQYN--------------------',
                'IGKV1-5*03': 'DIQMTQSPSTLSASVGDRVTITCRASQSI------SSWLAWYQQKPGKAPKLLIYKA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPDDFATYYCQQYN--------------------',
                'IGKV1-5*04': 'DIQMTQSPSTLSASVGDRVTITCRASQSI------SSWLAWYQQKPGKAPKLLIYKA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPDDFATYYCQQYN--------------------',
                'IGKV1-5*05': 'DIQMTQSPSTLSASVGDRVTITCRASQSI------SSWLAWYQQKPGKAPKLLIYKA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPDDFATYYCQQYN--------------------',
                'IGKV1-6*01': 'AIQMTQSPSSLSASVGDRVTITCRASQGI------RNDLGWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCLQDY--------------------',
                'IGKV1-6*02': 'AIQMTQSPSSLSASVGDRVTITCRASQGI------RNDLGWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCLQDY--------------------',
                'IGKV1-8*01': 'AIRMTQSPSSFSASTGDRVTITCRASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV1-8*02': 'AIRITQSPSSLSASTGDRVTITCRASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV1-8*03': 'AIRMTQSPSSLSASTGDRVTITCRASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV1-8*04': 'AIRMTQSPSSLSASTGDRVTITCRASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV1-9*01': 'DIQLTQSPSFLSASVGDRVTITCRASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQLN--------------------',
                'IGKV1-9*02': 'DIQLTQSPSFLSASVGDRVTITCWASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQLN--------------------',
                'IGKV1-9*03': 'AIQLTQSPSSLSASVGDRVTITCRASQGI------SSYLAWYQQKPGKAPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQLN--------------------',
                'IGKV1D-12*01': 'DIQMTQSPSSVSASVGDRVTITCRASQGI------SSWLAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQAN--------------------',
                'IGKV1D-12*02': 'DIQMTQSPSSVSASVGDRVTITCRASQGI------SSWLAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQAN--------------------',
                'IGKV1D-13*01': 'AIQLTQSPSSLSASVGDRVTITCRASQGI------SSALAWYQQKPGKAPKLLIYDA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQFN--------------------',
                'IGKV1D-13*02': 'AIQLTQSPSSLSASVGDRVTITCRASQGI------SSALAWYQQKPGKAPKLLIYDA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQFN--------------------',
                'IGKV1D-16*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SSWLAWYQQKPEKAPKSLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQYN--------------------',
                'IGKV1D-16*02': 'DIQMTQSPSSLSASVGDRVTITCRARQGI------SSWLAWYQQKPEKAPKSLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQYN--------------------',
                'IGKV1D-17*01': 'NIQMTQSPSAMSASVGDRVTITCRARQGI------SNYLAWFQQKPGKVPKHLIYAA-------SSLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCLQHN--------------------',
                'IGKV1D-33*01': 'DIQMTQSPSSLSASVGDRVTITCQASQDI------SNYLNWYQQKPGKAPKLLIYDA-------SNLETGVP-SRFSGSG--SGTDFTFTISSLQPEDIATYYCQQYD--------------------',
                'IGKV1D-39*01': 'DIQMTQSPSSLSASVGDRVTITCRASQSI------SSYLNWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQSY--------------------',
                'IGKV1D-40-1*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNNLNWYQQKPGKTPKLLIYAA-------PSLQSGIP-SRFSDSG--SGADYTLTIRSLQPEDFATYYCQQSD--------------------',
                'IGKV1D-43*01': 'AIRMTQSPFSLSASVGDRVTITCWASQGI------SSYLAWYQQKPAKAPKLFIYYA-------SSLQSGVP-SRFSGSG--SGTDYTLTISSLQPEDFATYYCQQYY--------------------',
                'IGKV1D-7-1*01': 'DIQMTQSPSSLSASVGDRVTITCRASQGI------SNSLAWYQQKPGKAPKLLLYAA-------SRLESGVP-SRFSGSG--SGTDYTLTISSLQPEDFATYYCQQYY--------------------',
                'IGKV1D-8*01': 'VIWMTQSPSLLSASTGDRVTISCRMSQGI------SSYLAWYQQKPGKAPELLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV1D-8*02': 'AIWMTQSPSLLSASTGDRVTISCRMSQGI------SSYLAWYQQKPGKAPELLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV1D-8*03': 'VIWMTQSPSLLSASTGDRVTISCRMSQGI------SSYLAWYQQKPGKAPELLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISCLQSEDFATYYCQQYY--------------------',
                'IGKV2-24*01': 'DIVMTQTPLSSPVTLGQPASISCRSSQSLVHS-DGNTYLSWLQQRPGQPPRLLIYKI-------SNRFSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCMQAT--------------------',
                'IGKV2-24*02': 'DIVMTQTPLSSPVTLGQPASISCRSSQSLVHS-DGNTYLSWLQQRPGQPPRLLIYKI-------SNRFSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCTQAT--------------------',
                'IGKV2-28*01': 'DIVMTQSPLSLPVTPGEPASISCRSSQSLLHS-NGYNYLDWYLQKPGQSPQLLIYLG-------SNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQAL--------------------',
                'IGKV2-29*02': 'DIVMTQTPLSLSVTPGQPASISCKSSQSLLHS-DGKTYLYWYLQKPGQSPQLLIYEV-------SSRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQGI--------------------',
                'IGKV2-29*03': 'DIVMTQTPLSLSVTPGQPASISCKSSQSLLHS-DGKTYLYWYLQKPGQSPQLLIYEV-------SSRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQGI--------------------',
                'IGKV2-30*01': 'DVVMTQSPLSLPVTLGQPASISCRSSQSLVYS-DGNTYLNWFQQRPGQSPRRLIYKV-------SNRDSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQGT--------------------',
                'IGKV2-40*01': 'DIVMTQTPLSLPVTPGEPASISCRSSQSLLDSDDGNTYLDWYLQKPGQSPQLLIYTL-------SYRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQRI--------------------',
                'IGKV2D-26*01': 'EIVMTQTPLSLSITPGEQASISCRSSQSLLHS-DGYTYLYWFLQKARPVSTLLIYEV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDFGVYYCMQDA--------------------',
                'IGKV2D-26*02': 'EIVMTQTPLSLSITPGEQASMSCRSSQSLLHS-DGYTYLYWFLQKARPVSTLLICEV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDFGVYYCMQDA--------------------',
                'IGKV2D-26*03': 'EIVMTQTPLSLSITPGEQASMSCRSSQSLLHS-DGYTYLYWFLQKARPVSTLLIYEV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDFGVYYCMQDA--------------------',
                'IGKV2D-26*04': 'EIVMTQTPLSLSITPGEQASISCRSSQSLLHS-DGYTYLYWFLQKARPVSTLLIYEV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDFGVYYCMQDA--------------------',
                'IGKV2D-28*01': 'DIVMTQSPLSLPVTPGEPASISCRSSQSLLHS-NGYNYLDWYLQKPGQSPQLLIYLG-------SNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQAL--------------------',
                'IGKV2D-28*02': 'DIVMTQPPLSLPVTPGEPASISCRSSQSLLHS-NGYNYLDWYLQKPGQSPQLLIYLG-------SNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQAL--------------------',
                'IGKV2D-29*01': 'DIVMTQTPLSLSVTPGQPASISCKSSQSLLHS-DGKTYLYWYLQKPGQPPQLLIYEV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQSI--------------------',
                'IGKV2D-29*02': 'DIVMTQTPLSLSVTPGQPASISCKSSQSLLHS-DGKTYLYWYLQKPGQSPQLLIYEV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQSI--------------------',
                'IGKV2D-30*01': 'DVVMTQSPLSLPVTLGQPASISCRSSQSLVYS-DGNTYLNWFQQRPGQSPRRLIYKV-------SNWDSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQGT--------------------',
                'IGKV2D-40*01': 'DIVMTQTPLSLPVTPGEPASISCRSSQSLLDSDDGNTYLDWYLQKPGQSPQLLIYTL-------SYRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQRI--------------------',
                'IGKV2D-40*02': 'DIVMTQTPLSLPVTPGEPASISCRSSQSLLDSDDGNTYLDWYLQKPGQSPQLLIYTL-------SYRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQRI--------------------',
                'IGKV3-11*01': 'EIVLTQSPATLSLSPGERATLSCRASQSV------SSYLAWYQQKPGQAPRLLIYDA-------SNRATGIP-ARFSGSG--SGTDFTLTISSLEPEDFAVYYCQQRS--------------------',
                'IGKV3-11*02': 'EIVLTQSPATLSLSPGERATLSCRASQSV------SSYLAWYQQKPGQAPRLLIYDA-------SNRATGIP-ARFSGSG--SGRDFTLTISSLEPEDFAVYYCQQRS--------------------',
                'IGKV3-15*01': 'EIVMTQSPATLSVSPGERATLSCRASQSV------SSNLAWYQQKPGQAPRLLIYGA-------STRATGIP-ARFSGSG--SGTEFTLTISSLQSEDFAVYYCQQYN--------------------',
                'IGKV3-15*02': 'EIVMTQSPATLSVSPGERATLSCRASQSV------SSNLAWYQQKPGQAPRLLIYGA-------STRATGIP-ARFSGSG--SGTEFTLTISSMQSEDFAVYYCQQYN--------------------',
                'IGKV3-20*01': 'EIVLTQSPGTLSLSPGERATLSCRASQSVS-----SSYLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTDFTLTISRLEPEDFAVYYCQQYG--------------------',
                'IGKV3D-11*01': 'EIVLTQSPATLSLSPGERATLSCRASQGV------SSYLAWYQQKPGQAPRLLIYDA-------SNRATGIP-ARFSGSG--PGTDFTLTISSLEPEDFAVYYCQQRS--------------------',
                'IGKV3D-11*02': 'EIVLTQSPATLSLSPGERATLSCRASQSV------SSYLAWYQQKPGQAPRLLIYDA-------SNRATGIP-ARFSGSG--PGTDFTLTISSLEPEDFAVYYCQQRS--------------------',
                'IGKV3D-11*03': 'EIVLTQSPATLSLSPGERATLSCRASQGV------SSNLAWYQQKPGQAPRLLIYDA-------SNRATGIP-ARFSGSG--PGTDFTLTISSLEPEDFAVYYCQQRS--------------------',
                'IGKV3D-15*01': 'EIVMTQSPATLSVSPGERATLSCRASQSV------SSNLAWYQQKPGQAPRLLIYGA-------STRATGIP-ARFSGSG--SGTEFTLTISSLQSEDFAVYYCQQYN--------------------',
                'IGKV3D-15*03': 'EIVMTQSPATLSVSPGERATLSCRASQSV------SSNLAWYQQKPGQAPRLLIYGA-------SIRATGIP-ARFSGSG--SGTEFTLTISILQSEDFAVYYCQQYN--------------------',
                'IGKV3D-20*01': 'EIVLTQSPATLSLSPGERATLSCGASQSVS-----SSYLAWYQQKPGLAPRLLIYDA-------SSRATGIP-DRFSGSG--SGTDFTLTISRLEPEDFAVYYCQQYG--------------------',
                'IGKV3D-7*01': 'EIVMTQSPATLSLSPGERATLSCRASQSVS-----SSYLSWYQQKPGQAPRLLIYGA-------STRATGIP-ARFSGSG--SGTDFTLTISSLQPEDFAVYYCQQDY--------------------',
                'IGKV4-1*01': 'DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKPGQPPKLLIYWA-------STRESGVP-DRFSGSG--SGTDFTLTISSLQAEDVAVYYCQQYY--------------------',
                'IGKV4-1*02': 'DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKPGQPPKLLIYWA-------STRESGVP-DRFSGSG--SGTDFTLTISSLQAEDVAVYYCQQYY--------------------',
                'IGKV4-1*03': 'DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKPGQPPKLLIYWA-------STRESGVP-DRFSGSG--SGTDFTLTISSLQAEDVAVYYCQQYY--------------------',
                'IGKV5-2*01': 'ETTLTQSPAFMSATPGDKVNISCKASQDI------DDDMNWYQQKPGEAAIFIIQEA-------TTLVPGIP-PRFSGSG--YGTDFTLTINNIESEDAAYYFCLQHD--------------------',
                'IGKV5-2*02': 'ETTLTQSPAFMSATPGDKVNISCKASQDI------DDDMNWYQQKPGEAAIFIIQEA-------TTLVPGIS-PRFSGSG--YGTDFTLTINNIESEDAAYYFCLQHD--------------------',
                'IGKV6-21*01': 'EIVLTQSPDFQSVTPKEKVTITCRASQSI------GSSLHWYQQKPDQSPKLLIKYA-------SQSFSGVP-SRFSGSG--SGTDFTLTINSLEAEDAATYYCHQSS--------------------',
                'IGKV6-21*02': 'EIVLTQSPDFQSVTPKEKVTITCRASQSI------GSSLHWYQQKPDQSPKLLIKYA-------SQSISGVP-SRFSGSG--SGTDFTLTINSLEAEDAATYYCHQSS--------------------',
                'IGKV6D-21*01': 'EIVLTQSPDFQSVTPKEKVTITCRASQSI------GSSLHWYQQKPDQSPKLLIKYA-------SQSFSGVP-SRFSGSG--SGTDFTLTINSLEAEDAATYYCHQSS--------------------',
                'IGKV6D-21*02': 'EIVLTQSPDFQSVTPKEKVTITCRASQSI------GSSLHWYQQKPDQSPKLLIKYA-------SQSISGVP-SRFSGSG--SGTDFTLTINSLEAEDAAAYYCHQSS--------------------'},
                'mouse': {'IGKV1-110*01': 'DVVMTQTPLSLPVSLGDQASISCRSSQSLVHS-NGNTYLHWYLQKPGQSPKLLIYKV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDLGVYFCSQST--------------------',
                'IGKV1-110*02': 'DVVMTQTPLSLPVSLGDQASISCRSSQSLVHS-NGNTYLYWYLQKPGQSPKLLIYRV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDLGVYFCFQGT--------------------',
                'IGKV1-117*01': 'DVLMTQTPLSLPVSLGDQASISCRSSQSIVHS-NGNTYLEWYLQKPGQSPKLLIYKV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDLGVYYCFQGS--------------------',
                'IGKV1-117*02': 'DVVMTQTPLSLPVSLGDQASISCRSSQSIVHS-NGNTYLEWYLQKPGQSPKLLIYKV-------SNRLSGVP-DRFSGSG--SGTDFTLKISRVEAEDLGVYYCFQGS--------------------',
                'IGKV1-122*01': 'DAVMTQTPLSLPVSLGDQASISCRSSQSLENS-NGNTYLNWYLQKPGQSPQLLIYRV-------SNRFSGVL-DRFSGSG--SGTDFTLKISRVEAEDLGVYFCLQVT--------------------',
                'IGKV1-132*01': 'DVVMTQTPLSLSVTIGQPASISCKSSQSLLYS-NGKTYLNWLQQRPGQAPKHLMYQV-------SKLDPGIP-DRFSGSG--SETDFTLKISRVEAEDLGVYYCLQGT--------------------',
                'IGKV1-133*01': 'DVVMTQTPLTLSVTIGQPASISCKSSQSLLYS-NGKTYLNWLLQRPGQSPKRLIYLV-------SKLDSGVP-DRFTGSG--SGTDFTLKISRVEAEDLGVYYCVQGT--------------------',
                'IGKV1-135*01': 'DVVMTQTPLTLSVTIGQPASISCKSSQSLLDS-DGKTYLNWLLQRPGQSPKRLIYLV-------SKLDSGVP-DRFTGSG--SGTDFTLKISRVEAEDLGVYYCWQGT--------------------',
                'IGKV1-88*01': 'DVVVTQTPLSLPVSFGDQVSISCRSSQSLANS-YGNTYLSWYLHKPGQSPQLLIYGI-------SNRFSGVP-DRFSGSG--SGTDFTLKISTIKPEDLGMYYCLQGT--------------------',
                'IGKV1-99*01': 'DVVLTQTPLSLPVNIGDQASISCKSTKSLLNS-DGFTYLDWYLQKPGQSPQLLIYLV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDLGVYYCFQSN--------------------',
                'IGKV10-94*01': 'DIQMTQTTSSLSASLGDRVTISCRASEDI------SNYLNWYQQKPDGTVKLLIYYA-------SSLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-94*02': 'DIQMTQTTSSLSASLGDRVTISCSASQGI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-94*03': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-94*04': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-94*05': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-94*06': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQDS--------------------',
                'IGKV10-94*07': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQRPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-94*08': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-95*01': 'DIQMTQTTSSLSASLGDRVTISCRASEDI------STYLNWYQQKPDGTVKLLIYYT-------SGLHSGVP-SRFSGSG--SGADYSLTISNLEPEDIATYYCQQYS--------------------',
                'IGKV10-96*01': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEQEDIATYFCQQGS--------------------',
                'IGKV10-96*02': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEQEDIATYFCQQGS--------------------',
                'IGKV10-96*03': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEQEDIATYFCQQDS--------------------',
                'IGKV10-96*04': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEQEDIATYFCQQDS--------------------',
                'IGKV10-96*07': 'DIQMTQTTSSLSASLGDRVTISCRASQDI------SNYLNWYQQKPDGTVKLLIYYT-------SRLHSGVP-SRFSGSG--SGTDYSLTISNLEQEDIATYFCQQGN--------------------',
                'IGKV11-106*02': 'DVLMTQSPSSLSASLGERVSLTCQASQGI------SNNLNWYQQTPGKAPRLLIYDA-------SKLEDGVP-SRFSGTG--YRTDFNFTISSLEEEDVATYFCLQHR--------------------',
                'IGKV11-125*01': 'DVQMIQSPSSLSASLGDIVTMTCQASQGT------SINLNWFQQKPGKAPKLLIYGA-------SNLEDGVP-SRFSGSR--YGTDFTLTISSLEDEDMATYFCLQHS--------------------',
                'IGKV12-104-2*01': 'DIRMIQTPASLSGSLGESVTITCQASQDI------GKSLLWYQQKTGNPPKILIYTT-------SNLADGIS-SRVSGSG--SGTQFFLKFSSLKPEDTATYYCCQGY--------------------',
                'IGKV12-38*01': 'DIQMTQSPASLAASVGETVTITCRASENI------YYSLAWYQQKQGKSPQLLIYNA-------NSLEDGVP-SRFSGSG--SGTQYSMKINSMQPEDTATYFCKQAY--------------------',
                'IGKV12-41*01': 'DIQMTQSPASLSASVGETVTITCRASGNI------HNYLAWYQQKQGKSPQLLVYNA-------KTLADGVP-SRFSGSG--SGTQYSLKINSLQPEDFGSYYCQHFW--------------------',
                'IGKV12-41*02': 'DIQMTQSPASLSASVGETVTITCRASGNI------HNYLAWYQQKQGKSPQLLVYNA-------KTLADGVP-SRFSGSG--SGTQYSLKINSLQPEDFGSYYCQHFW--------------------',
                'IGKV12-44*01': 'DIQMTQSPASLSASVGETVTITCRASENI------YSYLAWYQQKQGKSPQLLVYNA-------KTLAEGVP-SRFSGSG--SGTQFSLKINSLQPEDFGSYYCQHHY--------------------',
                'IGKV12-46*01': 'DIQMTQSPASLSVSVGETVTITCRASENI------YSNLAWYQQKQGKSPQLLVYAA-------TNLADGVP-SRFSGSG--SGTQYSLKINSLQSEDFGSYYCQHFW--------------------',
                'IGKV12-89*01': 'DIQMTQSPASLSASVGETVTITCGASENI------YGALNWYQRKQGKSPQLLIYGA-------TNLADGMS-SRFSGSG--SGRQYSLKISSLHPDDVATYYCQNVL--------------------',
                'IGKV12-98*01': 'DIQMTQSPASQSASLGESVTITCLASQTI------GTWLAWYQQKPGKSPQLLIYAA-------TSLADGVP-SRFSGSG--SGTKFSFKISSLQAEDFVSYYCQQLY--------------------',
                'IGKV12-e*01': 'DIQMTQSPASLSVSVGETVTITCRASENI------YSNLAWLFSRNRENPPSLVYAA-------TNLADGVP-SRFSGSG--SGTQYSLKINSQQPEDFGSYYCQHFW--------------------',
                'IGKV13-84*01': 'DIQMTQSSSSFSVSLGDRVTITCKASEDI------YNRLAWYQQKPGNAPRLLISGA-------TSLETGVP-SRFSGSG--SGKDYTLSITSLQTEDVATYYCQQYW--------------------',
                'IGKV13-85*01': 'DIQMTQSSSYLSVSLGGRVTITCKASDHI------NNWLAWYQQKPGNAPRLLISGA-------TSLETGVP-SRFSGSG--SGKDYTLSITSLQTEDVATYYCQQYW--------------------',
                'IGKV14-100*01': 'DILMTQSPSSMSVSLGDTVSITCHASQGI------SSNIGWLQQKPGKSFKGLIYHG-------TNLEDGVP-SRFSGSG--SGADYSLTISSLESEDFADYYCVQYA--------------------',
                'IGKV14-111*01': 'DIKMTQSPSSMYASLGERVTITCKASQDI------NSYLSWFQQKPGKSPKTLIYRA-------NRLVDGVP-SRFSGSG--SGQDYSLTISSLEYEDMGIYYCLQYD--------------------',
                'IGKV14-126*01': 'DIKMTQSPSSMYASLGERVTITCKASQDI------KSYLSWYQQKPWKSPKTLIYYA-------TSLADGVP-SRFSGSG--SGQDYSLTISSLESDDTATYYCLQHG--------------------',
                'IGKV14-130*01': 'EIQMTQSPSSMSASLGDRITITCQATQDI------VKNLNWYQQKPGKPPSFLIYYA-------TELAEGVP-SRFSGSG--SGSDYSLTISNLESEDFADYYCLQFY--------------------',
                'IGKV16-104*01': 'DVQITQSPSYLAASPGETITINCRASKSI------SKYLAWYQEKPGKTNKLLIYSG-------STLQSGIP-SRFSGSG--SGTDFTLTISSLEPEDFAMYYCQQHN--------------------',
                'IGKV17-121*01': 'ETTVTQSPASLSMAIGEKVTIRCITSTDI------DDDMNWYQQKPGEPPKLLISEG-------NTLRPGVP-SRFSSSG--YGTDFVFTIENMLSEDVADYYCLQSD--------------------',
                'IGKV17-127*01': 'ETTVTQSPASLSVATGEKVTIRCITSTDI------DDDMNWYQQKPGEPPKLLISEG-------NTLRPGVP-SRFSSSG--YGTDFVFTIENTLSEDVADYYCLQSD--------------------',
                'IGKV18-36*01': 'TGETTQAPASLSFSLGETATLSCRSSESV------GSYLAWYQQKAEQVPRLLIHSA-------STRAGGVP-VRFSGTG--SGTDFTLTISSLEPEDAAVYYCQPFK--------------------',
                'IGKV19-93*01': 'DIQMTQSPSSLSASLGGKVTITCKASQDI------NKYIAWYQHKPGKGPRLLIHYT-------STLQPGIP-SRFSGSG--SGRDYSFSISNLEPEDIATYYCLQYD--------------------',
                'IGKV19-93*02': 'DIQMTQSPSSLSASLGGKVTITCKASQDI------NKYIAWYQHKPGKGPRLLIHYT-------STLQPGIP-SRFSGSG--SGRDYSFSISNLEPEDIATYYCLQYD--------------------',
                'IGKV2-109*01': 'DIVMTQAAFSNPVTLGTSASISCRSSKSLLHS-NGITYLYWYLQKPGQSPQLLIYQM-------SNLASGVP-DRFSSSG--SGTDFTLRISRVEAEDVGVYYCAQNL--------------------',
                'IGKV2-109*02': 'DIVMTQAAFSNPVTLGTSASISCRSSKSLLHS-NGITYLYWYLQKPGQSPQLLIYQM-------SNLASGVP-DRFSSSG--SGTDFTLRISRVEAEDVGVYYCAQNL--------------------',
                'IGKV2-109*03': 'DIVMTQAAFSNPVTLGTSASISCSSSKSLLHS-NGITYLYWYLQRPGQSPQLLIYRM-------SNLASGVP-DRFSGSG--SGTDFTLRISRVEAEDVGVYYCAQML--------------------',
                'IGKV2-109*04': 'DIVMTQAAFSNPVTLGTSASISCRSSKSLLHS-DGITYLYWYLQRPGQSPQLLIYRM-------SNLASGVP-DRFSGSG--SGTDFTLRISRVEAEDVGVYYCAQML--------------------',
                'IGKV2-112*01': 'DIVITQDELSNPVTSGESVSISCRSSKSLLYK-DGKTYLNWFLQRPGQSPQLLIYLM-------STRASGVS-DRFSGSG--SGTDFTLEISRVKAEDVGVYYCQQLV--------------------',
                'IGKV2-112*02': 'DIVITQDELSNPVTSGESVSISCRSSKSLLYK-DGKTYLNWFLQRPGQSPQLLVYWM-------STRASGVS-DRFSGSG--SGTDFTLEISRVKAEDVGVYYCQQVV--------------------',
                'IGKV2-116*01': 'DIVMTQAAFSNPVTLGTSASISCRSSKNLLHS-NGITYLYWYLQRPGQSPQLLIYRV-------SNLASGVP-NRFSGSE--SGTDFTLRISRVEAEDVGVYYCAQLL--------------------',
                'IGKV2-137*01': 'DIVMTQAAPSVPVTPGESVSISCRSSKSLLHS-NGNTYLYWFLQRPGQSPQLLIYRM-------SNLASGVP-DRFSGSG--SGTAFTLRISRVEAEDVGVYYCMQHL--------------------',
                'IGKV2-a*01': 'DIVMTQAAFSNPVTLGTSASISCRSSKSLLHS-SGNTYLYWFLQKPGQSPQLLIYYI-------SNLASGVP-DRFSGSG--SGTDFTLRISRVEAEDVGVYYCMQGL--------------------',
                'IGKV20-101-2*01': 'NIQVIQSPF-LSASVGERVTISCKTHQHI------NSSIAWYQQKVGKAPILLIRDA-------SFSLTDTP-SRFTGNG--FGTDFTLSISSMQSQDGATYFCQQHF--------------------',
                'IGKV3-1*01': 'DIVLTQSPASLAVSLGQRATISCRASESVEY--YGTSLMQWYQQKPGQPPKLLIYAA-------SNVESGVP-ARFSGSG--SGTDFSLNIHPVEEDDIAMYFCQQSR--------------------',
                'IGKV3-10*01': 'NIVLTQSPASLAVSLGQRATISCRASESVDS--YGNSFMHWYQQKPGQPPKLLIYLA-------SNLESGVP-ARFSGSG--SRTDFTLTIDPVEADDAATYYCQQNN--------------------',
                'IGKV3-12*01': 'DIVLTQSPASLAVSLGQRATISCRASKSVST--SGYSYMHWYQQKPGQPPKLLIYLA-------SNLESGVP-ARFSGSG--SGTDFTLNIHPVEEEDAATYYCQHSR--------------------',
                'IGKV3-2*01': 'DIVLTQSPASLAVSLGQRATISCRASESVDN--YGISFMNWFQQKPGQPPKLLIYAA-------SNQGSGVP-ARFSGSG--SGTDFSLNIHPMEEDDTAMYFCQQSK--------------------',
                'IGKV3-3*01': 'DIVLTQSPASLAVSLGQRATIFCRASQSVDY--NGISYMHWFQQKPGQPPKLLIYAA-------SNLESGIP-ARFSGSG--SGTDFTLNIHPVEEEDAATYYCQQSI--------------------',
                'IGKV3-4*01': 'DIVLTQSPASLAVSLGQRATISCKASQSVDY--DGDSYMNWYQQKPGQPPKLLIYAA-------SNLESGIP-ARFSGSG--SGTDFTLNIHPVEEEDAATYYCQQSN--------------------',
                'IGKV3-5*01': 'DIVLTQSPASLAVSLGQRATISCRASESVDS--YGNSFMHWYQQKPGQPPKLLIYRA-------SNLESGIP-ARFSGSG--SRTDFTLTINPVEADDVATYYCQQSN--------------------',
                'IGKV3-7*01': 'DIVLTQSPASLAVSLGQRATISCRASQSVST--SSYSYMHWYQQKPGQPPKLLIKYA-------SNLESGVP-ARFSGSG--SGTDFTLNIHPVEEEDTATYYCQHSW--------------------',
                'IGKV3-7*02': 'DIVLTQSPASLAVSLGQRATISCRASQSVST--SSYSYMHWYQQKPGQPPKLLIKYA-------SNLESGVP-ARFSGSG--SGTDFTLNIHPVEEEDTATYYCQHSW--------------------',
                'IGKV3-9*01': 'DIVLTQSPASLAVSLGQRATISCQASESVSF--AGTSLMHWYQQKPGQPPKLLIYRA-------SNLESGVP-ARFSGSG--SESDFTLTIDPVEEDDAAMYYCMQSM--------------------',
                'IGKV4-50*01': 'ENVLTQSPAIMSASLGEKVTMSCRASSSV-------NYMYWYQQKSDASPKLWIYYT-------SNLAPGVP-ARFSGSG--SGNSYSLTISSMEGEDAATYYCQQFT--------------------',
                'IGKV4-51*01': 'ENVLTQSPAIMAASLGEKVTMTCSASSSVS-----SSYLHWYQQKSGTSPKLWIYGT-------SNLASGVP-ARFSGSG--AGISYSLTISSMEAENDATYYCQQWS--------------------',
                'IGKV4-53*01': 'EIVLTQSPALMAASPGEKVTITCSVSSSIS-----SSNLHWYQQKSETSPKPWIYGT-------SNLASGVP-VRFSGSG--SGTSYSLTISSMEAEDAATYYCQQWS--------------------',
                'IGKV4-55*01': 'QIVLTQSPAIMSASPGEKVTMTCSASSSV-------SYMYWYQQKPGSSPRLLIYDT-------SNLASGVP-VRFSGSG--SGTSYSLTISRMEAEDAATYYCQQWS--------------------',
                'IGKV4-57*01': 'QIVLTQSPAIMSASPGEKVTITCSASSSV-------SYMHWFQQKPGTSPKLWIYST-------SNLASGVP-ARFSGSG--SGTSYSLTISRMEAEDAATYYCQQRS--------------------',
                'IGKV4-57-1*01': 'ENVLTQSPAIMSASPGEKVTMTCRASSSVS-----SSYLHWYQQKSGASPKLWIYST-------SNLASGVP-ARFSGSG--SGTSYSLTISSVEAEDAATYYCQQYS--------------------',
                'IGKV4-58*01': 'ENVLTQSPAIMAASLGQKVTMTCSASSSVS-----SSYLHWYQQKSGASPKPLIHRT-------SNLASGVP-ARFSGSG--SGTSYSLTISSVEAEDDATYYCQQWS--------------------',
                'IGKV4-59*01': 'QIVLTQSPAIMSASPGEKVTMTCSASSSV-------SYMHWYQQKSGTSPKRWIYDT-------SKLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAATYYCQQWS--------------------',
                'IGKV4-61*01': 'QIVLTQSPAIMSASPGEKVTISCSASSSV-------SYMYWYQQKPGSSPKPWIYRT-------SNLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAATYYCQQYH--------------------',
                'IGKV4-63*01': 'ENVLTQSPAIMSASPGEKVTMTCSASSSV-------SYMHWYQQKSSTSPKLWIYDT-------SKLASGVP-GRFSGSG--SGNSYSLTISSMEAEDVATYYCFQGS--------------------',
                'IGKV4-68*01': 'QIVLTQSPALMSASPGEKVTMTCSASSSV-------SYMYWYQQKPRSSPKPWIYLT-------SNLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAATYYCQQWS--------------------',
                'IGKV4-69*01': 'QILLTQSPAIMSASPGEKVTMTCSASSSV-------SYMHWYQQKPGSSPKPWIYDT-------SNLASGFP-ARFSGSG--SGTSYSLIISSMEAEDAATYYCHQRS--------------------',
                'IGKV4-70*01': 'QIVLTQSPAIMSASPGEKVTMTCSASSSI-------SYMHWYQQKPGTSPKRWIYDT-------SKLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAATYYCHQRS--------------------',
                'IGKV4-71*01': 'QIVLTQSPAIMSASPGEKVTMTCSASSSV-------SYMHWYQQKPGSSPRLWIYLT-------FNLASGVP-ARFSGSG--SGTSYSLSISSMEAEDAATYYCQQWS--------------------',
                'IGKV4-72*01': 'QIVLSQSPAILSASPGEKVTMTCRASSSV-------SYMHWYQQKPGSSPKPWIYAT-------SNLASGVP-ARFSGSG--SGTSYSLTISRVEAEDAATYYCQQWS--------------------',
                'IGKV4-74*01': 'QIVLTQSPAIMSASLGERVTMTCTASSSVS-----SSYLHWYQQKPGSSPKLWIYST-------SNLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAATYYCHQYH--------------------',
                'IGKV4-78*01': 'QIVLTQSPAIMSASPGEKVTMTCSARSSVS-----SSYLYWYQQKPGSSPKLWIYST-------SNLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAATFYCQQYS--------------------',
                'IGKV4-79*01': 'QIVLTQSPAIMSASPGEKVTLTCSASSSVS-----SSYLYWYQQKPGSSPKLWIYST-------SNLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAASYFCHQWS--------------------',
                'IGKV4-80*01': 'QIVLTQSPAIMSASLGEEITLTCSASSSV-------SYMHWYQQKSGTSPKLLIYST-------SNLASGVP-SRFSGSG--SGTFYSLTISSVEAEDAADYYCHQWS--------------------',
                'IGKV4-81*01': 'ENVLTQSPAIMAASPGEKVTMTCSASSSVS-----SSNLHWYQQKSGTSTKFWIYRT-------SNLASEVP-APFSGSG--SGTSYSLTISSVEAEDAATYYCQQWS--------------------',
                'IGKV4-86*01': 'EIVLTQSPAITAASLGQKVTITCSASSSV-------SYMHWYQQKSGTSPKPWIYEI-------SKLASGVP-ARFSGSG--SGTSYSLTISSMEAEDAAIYYCQQWN--------------------',
                'IGKV4-90*01': 'EILLTQSPAIIAASPGEKVTITCSASSSV-------SYMNWYQQKPGSSPKIWIYGI-------SNLASGVP-ARFSGSG--SGTSFSFTINSMEAEDVATYYCQQRS--------------------',
                'IGKV4-91*01': 'EIVLTQSPTTMAASPGEKITITCSASSSIS-----SNYLHWYQQKPGFSPKLLIYRT-------SNLASGVP-ARFSGSG--SGTSYSLTIGTMEAEDVATYYCQQGS--------------------',
                'IGKV4-92*01': 'EMVLTQSPVSITASRGEKVTITCRASSSIS-----SNYLHWYQQKPGSSPKLLIYRT-------SILASGVL-DSFSGSG--SESSYTLTISCMQDEVAATYYCQQGS--------------------',
                'IGKV5-37*01': 'DILLTQSPATLSVTPGETVSLSCRASQSI------YKNLHWYQQKSHRSPRLLIKYA-------SDSISGIP-SRFTGSG--SGTDYTLSINSVKPEDEGIYYCLQGY--------------------',
                'IGKV5-39*01': 'DIVMTQSPATLSVTPGDRVSLSCRASQSI------SDYLHWYQQKSHESPRLLIKYA-------SQSISGIP-SRFSGSG--SGSDFTLSINSVEPEDVGVYYCQNGH--------------------',
                'IGKV5-43*01': 'DIVLTQSPATLSVTPGDSVSLSCRASQSI------SNNLHWYQQKSHESPRLLIKYA-------SQSISGIP-SRFSGSG--SGTDFTLSINSVETEDFGMYFCQQSN--------------------',
                'IGKV5-45*01': 'DIVLTQSPATLSVTPGDRVSLSCRASQSI------SNYLHWYQQKSHESPRLLIKYA-------SQSISGIP-SRFSGSG--SGTDFTLSINSVETEDFGMYFCQQSN--------------------',
                'IGKV5-48*01': 'DILLTQSPAILSVSPGERVSFSCRASQSI------GTSIHWYQQRTNGSPRLLIKYA-------SESISGIP-SRFSGSG--SGTDFTLSINSVESEDIADYYCQQSN--------------------',
                'IGKV6-13*01': 'DIVMTQSQKFMSTSVGDRVSITCKASQNV------GTAVAWYQQKPGQSPKLLIYSA-------SNRYTGVP-DRFTGSG--SGTDFTLTISNMQSEDLADYFCQQYS--------------------',
                'IGKV6-14*01': 'DIVMTQSQKFMSTSVGDRVSITCKASQNV------RTAVAWYQQKPGQSPKALIYLA-------SNRHTGVP-DRFTGSG--SGTDFTLTISNVQSEDLADYFCLQHW--------------------',
                'IGKV6-15*01': 'DIVMTQSQKFMSTSVGDRVSVTCKASQNV------GTNVAWYQQKPGQSPKALIYSA-------SYRYSGVP-DRFTGSG--SGTDFTLTISNVQSEDLAEYFCQQYN--------------------',
                'IGKV6-17*01': 'DIVMTQSHKFMSTSVGDRVSITCKASQDV------STAVAWYQQKPGQSPKLLIYSA-------SYRYTGVP-DRFTGSG--SGTDFTFTISSVQAEDLAVYYCQQHY--------------------',
                'IGKV6-20*01': 'NIVMTQSPKSMSMSVGERVTLSCKASENV------GTYVSWYQQKPEQSPKLLIYGA-------SNRYTGVP-DRFTGSG--SATDFTLTISSVQAEDLADYHCGQSY--------------------',
                'IGKV6-23*01': 'DIVMTQSHKFMSTSVGDRVSITCKASQDV------GTAVAWYQQKPGQSPKLLIYWA-------STRHTGVP-DRFTGSG--SGTDFTLTISNVQSEDLADYFCQQYS--------------------',
                'IGKV6-25*01': 'DIVMTQSHKFMSTSVGDRVSITCKASQDV------STAVAWYQQKPGQSPKLLIYWA-------STRHTGVP-DRFTGSG--SGTDYTLTISSVQAEDLALYYCQQHY--------------------',
                'IGKV6-29*01': 'NIVMTQSPKSMSMSVGERVTLSCKASENV------GTYVSWYQQKPEQSPKLLIYGA-------SNRYPGVP-DRFTGSG--SATDFTLTISSLQAEDLADYHCGQGY--------------------',
                'IGKV6-32*01': 'SIVMTQTPKFLLVSAGDRVTITCKASQSV------SNDVAWYQQKPGQSPKLLIYYA-------SNRYTGVP-DRFTGSG--YGTDFTFTISTVQAEDLAVYFCQQDY--------------------',
                'IGKV6-32*02': 'SIVMTQTPKFLLVSAGERVTITCKASQSV------SNDVAWYQQKPGQSPKLLIYYA-------SNRYTGVP-DRFTGSG--YGTDFTFTISTVQAEDLAVYFCQQDY--------------------',
                'IGKV6-b*01': 'SIVMTQTPKFLPVSAGDRVTMTCKASQSV------GNNVAWYQQKPGQSPKLLIYYA-------SNRYTGVP-DRFTGSG--SGTDFTFTISSVQVEDLAVYFCQQHY--------------------',
                'IGKV6-c*01': 'SIVMTQTPKFLPVTAEDRVTITCKASQSV------SNEVAWYQQKPGQSPKLLIYYA-------SNRYTGVP-DRFTGSG--SGTDFTFTISSVQVEDLAVYFCQQHY--------------------',
                'IGKV6-d*01': 'SIVMTQSPKSLPVSAGDRVTMTCKASQSV------SNDVAWYQQKPGQSPKLLIYYA-------SNRYTGVP-ERFTGSG--SGTDFTFTISGVQAEDLAVYFCQQHY--------------------',
                'IGKV7-33*01': 'DIVMTQSPTFLAVTASKKVTISCTASESLYSSKHKVHYLAWYQKKPEQSPKLLIYGA-------SNRYIGVP-DRFTGSG--SGTDFTLTISSVQVEDLTHYYCAQFY--------------------',
                'IGKV8-16*01': 'EIVLTQSIPSLTVSAGERVTISCKSNQNLLWSGNQRYCLVWHQWKPGQTPTPLITWT-------SDRYSGVP-DRFIGSG--SVTDFTLTISSVQAEDVAVYFCQQHL--------------------',
                'IGKV8-19*01': 'DIVMTQSPSSLTVTAGEKVTMSCKSSQSLLNSGNQKNYLTWYQQKPGQPPKLLIYWA-------STRESGVP-DRFTGSG--SGTDFTLTISSVQAEDLAVYYCQNDY--------------------',
                'IGKV8-21*01': 'DIVMSQSPSSLAVSAGEKVTMSCKSSQSLLNSRTRKNYLAWYQQKPGQSPKLLIYWA-------STRESGVP-DRFTGSG--SGTDFTLTISSVQAEDLAVYYCKQSY--------------------',
                'IGKV8-24*01': 'DIVMTQSPSSLAMSVGQKVTMSCKSSQSLLNSSNQKNYLAWYQQKPGQSPKLLVYFA-------STRESGVP-DRFIGSG--SGTDFTLTISSVQAEDLADYFCQQHY--------------------',
                'IGKV8-27*01': 'NIMMTQSPSSLAVSAGEKVTMSCKSSQSVLYSSNQKNYLAWYQQKPGQSPKLLIYWA-------STRESGVP-DRFTGSG--SGTDFTLTISSVQAEDLAVYYCHQYL--------------------',
                'IGKV8-28*01': 'DIVMTQSPSSLSVSAGEKVTMSCKSSQSLLNSGNQKNYLAWYQQKPGQPPKLLIYGA-------STRESGVP-DRFTGSG--SGTDFTLTISSVQAEDLAVYYCQNDH--------------------',
                'IGKV8-28*02': 'DIVMTQSPSSLSVSAGDKVTMSCKSSQSLLNSRNQKNYLAWYQQKPWQPPKLLIYGA-------STRESGVP-DRFTGSG--SGTDFTLTISSVQAEDLAVYYCQNDY--------------------',
                'IGKV8-28*03': 'DIVMTQSPSSLSVSAGEKVTMSCKSSQSLLNSGNQKNYLAWYQQKPGQPPKLLIYGA-------STRESGVP-DRFTGSG--SGTDFTLTISSVQAEDLAVYYCLNDH--------------------',
                'IGKV8-30*01': 'DIVMSQSPSSLAVSVGEKVTMSCKSSQSLLYSSNQKNYLAWYQQKPGQSPKLLIYWA-------STRESGVP-DRFTGSG--SGTDFTLTISSVKAEDLAVYYCQQYY--------------------',
                'IGKV8-34*01': 'DILMTQSPSSLTVSAGEKVTMSCKSSQSLLASGNQNNYLAWHQQKPGRSPKMLIIWA-------STRVSGVP-DRFIGSG--SGTDFTLTINSVQAEDLAVYYCQQSY--------------------',
                'IGKV9-120*01': 'DIQMTQSPSSLSASLGERVSLTCRASQDI------GSSLNWLQQEPDGTIKRLIYAT-------SSLDSGVP-KRFSGSR--SGSDYSLTISSLESEDFVDYYCLQYA--------------------',
                'IGKV9-120*02': 'DIQMTQSPSSLSASLGERVSLTCRASQDI------GSSLNWLQQEPDGTIKRLIYAT-------SSLDSGVP-KRFSGSR--SGSDYSLTISSLESEDFVDYYCLQYA--------------------',
                'IGKV9-123*01': 'DIQMIQSPSSMFASLGDRVSLSCRASQGI------RGNLDWYQQKPGGTIKLLIYST-------SNLNSGVP-SRFSGSG--SGSDYSLTISSLESEDFADYYCLQRN--------------------',
                'IGKV9-124*01': 'DIQMTQSPSSLSASLGERVSLTCRASQEI------SGYLSWLQQKPDGTIKRLIYAA-------STLDSGVP-KRFSGSR--SGSDYSLTISSLESEDFADYYCLQYA--------------------'},
                'rat': {'IGKV10S6*01': 'AIQVTQSPTSLSASLGDRVTLTCRASQDI------NNKMAWYQQKPGEVPQLLIYYA-------STLQSGTP-SRFSGSG--AGTDFSFTISHLQSEDFATYYCLQGY--------------------',
                'IGKV12S1*01': 'DIQVTQSPASLSASPEEIVTITCQASQDI------GSSLLWYQQKPGKSPQLLIYSA-------TILADGVP-SRFSGSR--SGTQYSLKISRLQVEDIGTYYCLQVS--------------------',
                'IGKV12S14*01': 'DIQMTQSPASLSASLEEIVTITCQASQDI------GNWLAWYQQKPGKSPQLLIYGA-------TSLADGVP-SRFSGSR--SGTQYSLKISRLQVEDIGIYYCQQAS--------------------',
                'IGKV12S16*01': 'DIQMTQSPASLSASLEEIVTITCQASQDI------GNWLSWYQQKPGKSPQLLIYGA-------TSLADGVP-SRFSGSR--SGTQYSLKISRLQVEDIGIYYCLQAY--------------------',
                'IGKV12S17*01': 'DIQMTQSPASLSASLEEIVTITCQASQDI------GNWLAWYQQKPGKSPQLLIYGA-------TSLADGVP-SRFSGSR--SGTQYSLKISRLQVEDPGIYYCLQGY--------------------',
                'IGKV12S20*01': 'DIQMTQSPASLSASPEEIVTITCQASQDI------GNWLAWYQQKPGKSPQLLIYSA-------TSLADGIP-SRFSGSR--SGTQYSLKISRLQVEDTGIYYCLQRY--------------------',
                'IGKV12S22*01': 'DIQMTQSPASLSASLEEIVTITCQPSQGI------GNYLSWYQQKLGKSPQLLIHSA-------TSLEDGVP-SRFSGSR--SGTQYSLKINRLQVEDTGIYYCLQIS--------------------',
                'IGKV12S24*01': 'DIQMTQSPASLSASLEEIVTITCQASQDI------GNYLSWYQQKPGKSPQLLIHSA-------TSLADGVP-SRFSGSR--SGTQYSLKINRLQVEDTGIYYCLQHY--------------------',
                'IGKV12S25*01': 'DIQMTQSPASLSASLDEIVTITCQASLDI------GNWLAWYQQKPGKSPQLLIYGA-------TSLADGVP-SRFSGSR--SGTQYSLKICKLQVEDTGIYYCLQHY--------------------',
                'IGKV12S7*01': 'DIHVTQSPASLSASPEEIVTITCQASQDI------GSSLLWYQQKPGKSPQLLIYSA-------TILADGVP-SRFSGSR--SGTQYSLKISRLQVEDIGTYYCLQFS--------------------',
                'IGKV14S1*01': 'DIQMTQSPSSMSASLGDRVTITCQASQDI------GNNLIWFQQKPGKSPRPMIYYA-------TNLANGVP-SRFSGSR--SGSDYSLTISSLESEDMADYHCLQYK--------------------',
                'IGKV14S13*01': 'DIQMTQSPSSLPASLGDRVTITCRASQDI------GNYLRWFQQKPGKSPRLMIYGA-------TNLAAGVP-SRFSGSR--SGSDYSLTISSLESEDMADYYCLQSK--------------------',
                'IGKV14S14*01': 'DIQMTQSPSSLSASLGDRVTITCRASQDI------GNYLTWFQQKPGKSPRRMIYGA-------TNLAAGVP-SRFSGSR--SGSDYSLTISSLESEDVADYHCLQSI--------------------',
                'IGKV14S15*01': 'DIQMTQSPSSMSASLGDRVTITCRASQDI------GNYLSWFQQKPGKSPRRMIYGA-------TNLAAGVP-SRFSGSR--SGSDYSLTISSLESEDMAIYYCLQSI--------------------',
                'IGKV14S16*01': 'NIQMTQSSSSMPASLIDREILACRASQDI------RNYLSWYQQKPGKSPKLMISGG-------TNLAARIP-SRFSGSR--SGSDYSLTISSLESEDEADYHCLQYD--------------------',
                'IGKV14S18*01': 'DIQMTQSPSSLPASLGDRVTITCRASQDI------GNYLRWFQQKPGKSPRLMIYGA-------TNLANGVP-SRFSGSR--SGSDYSLTINSLESEDMAIYYCLQHN--------------------',
                'IGKV14S19*01': 'DIQMTQSPSSMSVSLGDRVTITCRASQDI------GNYLSWYQQKPEKSPKLMIYGA-------TNLEDGVP-SRFSGSR--SGSDYSLTINSLESEDTGIYFCLQHK--------------------',
                'IGKV14S2*01': 'DIQMTQAPSSLPASLGDRVTITCRASQDI------GNYLRWFQQKPGKSPRRMIYGA-------TNLAAGVP-SRFSGSR--SGSDYSLTISSLESEDMADYYCVQSK--------------------',
                'IGKV14S8*01': 'DIQMTQSPSSMSASLGDTVTINCLASQDI------GNYLSWYQQKPGKSPKLMIYGA-------TNLEDGVP-SRFSGSR--SGSDYSLTINSLGYDDEGIYHCHQYY--------------------',
                'IGKV14S9*01': 'DIQMTQSPSSMSVSLGDTVTITCRASQDV------GIYVNWFQQKPGKSPRHMIYRA-------TNLADGVP-SRFSGSR--SGSDYSLTISSLESEDVADYHCLQYD--------------------',
                'IGKV15S4*01': 'DIQMTQSPSFLSASLGNSITITCHASQNI------KGWLAWYQQKSGNAPELLIYKA-------SSLQSGVP-SRFSGSG--SGTDYIFTISNLQPEDIATYYCQHYQ--------------------',
                'IGKV17S1*01': 'ETTVTQSPASLSMAVGEKVSISCKTSTDI------DDDMNWYQQKSGEAPKLLISEG-------NTLRPGVP-SRFSSSG--YGTDFVFTINNVLLGDEGIYYCQQSD--------------------',
                'IGKV1S1*01': 'DVVMTQTPVSLSLAIGQPASISCKSSQSLLGT-SGKTFLNWILQRPGQSPKRLIYQV-------SKLYSEVP-DRFSGSG--SETEFTLKISRVEAEDLGVYYCWQGT--------------------',
                'IGKV1S12*01': 'DVVMTQTPPSLSVAIGQSVSISCKSSQSLVHS-DGKTYLNWLLQNPGQSPKRLIYQV-------SNLGSGVP-DRFSGTG--SEKDFTLKISRVEAEDLGVYHCVQAT--------------------',
                'IGKV1S14*01': 'DVVMTQTPPSLSVAIGQSVSISCKSSQSLVYS-DGKTYLHWLLQSPGRSPKRLIYQV-------SNLGSGVP-DRFSGTG--SQKDFTLKISRVEAEDLGVYYCAQTT--------------------',
                'IGKV1S18*01': 'DVVMTQTPVSLSVAIGQPASISCKSSQSLVHS-DGKTYLNWLLQRPGQSPKRLIYLV-------SKLDSGIP-DRFSGSG--SETDFTLKISRVEADDLGVYYCLQGT--------------------',
                'IGKV1S5*01': 'DIVMTQTPLSLSVAIGQSAFICCKSSQSLLYS-NGKKYLNWFLQRPGQSPKCLIYLV-------SKLDFGVP-DRFTGSG--SETNFTLEISRVEAENLGVYYCMQGS--------------------',
                'IGKV1S7*01': 'DIVMTQTPLSLSVAIGQSASISCKSSQSLKYS-DGKTYLNWVFQSPGQSPKRLIYQV-------SKLDSGVP-DRFSGTG--SETDFTLKISRVEAEDLGVYYCCQVH--------------------',
                'IGKV1S8*01': 'DVVMTQTPVSLSLAIGQPASISCKSSQSLIHS-DGKTYLSWILQRPGQSPKRLIYLV-------SKLDSGVP-DRFSGSG--SETEFTLKISRVEAEDLGVYYCWQAT--------------------',
                'IGKV20S1*01': 'DIRMTQTPASLSASLGESVTITCRASQDI------GKSLLWFQQKTGKPPKILIYTA-------SNLVSGIS-PRFSGSG--SGTQFSLKISSLKPEDTANYYCCQGY--------------------',
                'IGKV22S1*01': 'DIQMTQSPSVLSASVGDRVTLNCKASQNI------NKYLNWYQQKLGEAPKLLIYNT-------NNLQTGIP-SRFSGSG--SGTDFTLTISSLQPEDFATYFCFQHN--------------------',
                'IGKV22S2*01': 'DIQMTQSPSFLSASVGDRVTLSCKASQNI------NKYLAWYQQKLGEAPKLLIYNA-------NSLQTGIP-SRFSGSG--SGTDFTLTISSLQPEDVATYFCLQHN--------------------',
                'IGKV22S4*01': 'DIQMTQSPSFLSASVGDRVTINCKASQNI------NRYLNWYQQKLGEAPKLLIYNA-------NSLQTGIP-SRFSGSG--SGTDFTLTISSLQPEDVATYFCLQHN--------------------',
                'IGKV22S7*01': 'DIQMTQSPSFLSASVGDRVTINCKASQNI------NKYLNWYQQKLGEAPKLLIYNT-------NNLQTGIP-SRFSGSG--SGTDYTLTISSLQPEDVATYFCLQHS--------------------',
                'IGKV22S9*01': 'DIQMTQSPSLLSASVGDRVTLSCKASQSI------YNSLAWYQQKLGEAPKLLIYKT-------NSLQTGIP-SSFSGSG--SGTDYTLTISSLQPEDVATYFCQKYN--------------------',
                'IGKV2S3*01': 'DIMMTQSPLSVAVTPGESASISCRSSKSLLHS-NGITYLSWYLQRPEKSPQLLIYQI-------SNLASGVS-GRFSGSG--SGTDFTLKISRVETEDVGIYYCVQFL--------------------',
                'IGKV3S1*01': 'DIVLTQSPA-LAVSLEQRATISCKTSQNVDN--YGISYMHWYQQKPGQQPKLLIYEG-------SNLASGIP-ARFSGSG--SGTDFTLTIDPVEADDIATYYCQQSK--------------------',
                'IGKV3S18*01': 'DIVLTQSPV-LAVSLGQRATISCRASQSVSI--SSINLMHWYQQKPGQQPKLLIYRA-------SNLASGIP-ARFSGSG--SGTDFTLTIDPVQADDIAAYYCQQSR--------------------',
                'IGKV3S19*01': 'DIVLTQSPA-LAVSLGQRATISCRASQSVSI--SRYNLMHWYQQKPGQQPKLLIYRA-------SNLASGIP-ARFSGSG--SGTDFTLTINPVQADDIATYYCQQSR--------------------',
                'IGKV3S8*01': 'DTVLTQSPA-LAVSPGERVTISCRASESV------STLMHWYQQKPGQQPTLLIYLA-------SNLESGVP-AMFSGSG--SGTDFTLTIDPVEADDTATYFCQQSW--------------------',
                'IGKV3S9*01': 'DTVLTQSPA-LAVSPGERVTISCRASESV------STLMHWYQQKPGQQPTLLIYLA-------SNLESGVP-ARFSGSG--SGTDFTLTIDPVEADDTATYYCQQSW--------------------',
                'IGKV6S11*01': 'NTVMTQSPTSMFISVGDRVTMNCKASQNV------GTNVDWYQQKTGQSPKLLIYGA-------SNRYTGVP-DRFTGSG--SGTDFTLTISNMQAEDLAVYYCLQYN--------------------',
                'IGKV6S8*01': 'ETVMTQSPTSMSTSIGERVTLNCKASQSV------GINVDWYQQTPGQSPKLLIYGA-------SNRHTGVP-DRFTGSG--FGRDFTLTISNMEAEDLAVYYCLQYG--------------------',
                'IGKV8S5*01': 'DIVMTQSPSSLAVSAGETVTINCKSSQSLFGSVRQKNYLAWYQQKPGQSPKLLIYLA-------STRESGVP-DRFIGSG--SGTDFTLTISSVQAEDLANYYCQQYY--------------------',
                'IGKV8S6*01': 'DIVMTQSPSSLAVSAGETVTINCKSSQSLLSSGNQKNYLAWYQQKPGQSPKLLIYLA-------STRESGVP-DRFIGSG--SGTDFTLTISSVQAEDLADYYCQQHY--------------------',
                'IGKV9S2*01': 'QITLTQQAESLWVSPGEKVSITCRASQSLLYT-DGKHYLSWYQQRPGQTTKALIYHA-------SIRTDGVP-TRFIGSG--SGTEFTLSIEDVQPEDIALYYCLQTL--------------------'},
                'rabbit': {'IGKV1S2*01': 'AQVLTQTESPVSAPVGGTVTINCQASQSVY----DNNWLSWYQQKPGQPPKLLIYDA-------SKLASGVP-SRFSGSG--SGTQFTLTISGVQCDDAATYYCQGSY--------------------',
                'IGKV1S2*02': 'AQVLTQTESPVSAPVGGTVTINCQASQSVY----DNNYLSWYQQKPGQPPKLLIYDA-------SKLASGVP-SRFSGSG--SGTQFTLTISGVQCDDAATYYCQGSY--------------------',
                'IGKV1S3*01': 'AQVLTQTPASVSAAVGGTVTINCQASESI------SSYLNWYQQKLGQPPKLLIYYA-------STLASGVP-SRFKGSG--SGTEYTLTISGVQCDDAATYYCQHGY--------------------',
                'IGKV1S3*02': 'AQVMTQTPASVSAAVGGTVTIICQASESI------SSYLNWYQQKLGQPPKLLIYYA-------STLASGVP-SRFKGSG--SGTEYTLTISGVQCDDAATYYCQHGY--------------------',
                'IGKV1S5*01': 'DPVMTQTPSSTSAAVGGTVTINCQSSQNVY----SNNYLSWFQQKPGQPPKLLIYGA-------SKLASGVP-SRFSGSG--SGKQFTLTISGVQCDDAATYYCAGYY--------------------',
                'IGKV1S6*01': 'DGVMTQTPAPVSAAVGGTVTINCQASQSI------GSDLSWYQQKPGQPPKLLIYSA-------SKLATGVP-SRFNGSG--SGTQFTLTISGVQCDDAATYYCQCTY--------------------'},
                'rhesus': {'IGKV1-16*01': 'DIQMTQSPSSLSASVGDKVT-ITCQASQSI------SSWLAWYQQKPGKAPKPLIYKA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQY--------------------',
                'IGKV1-18*01': 'DIQMTQSPSSLSASVGDKVT-ITCRASQGI------SSWLAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDYTLTISSLQPEDFATYYCQQG--------------------',
                'IGKV1-19*01': 'DIQMTQSPSSLSASVGDKVT-ITCHASQGI------SSWLAWYQQKPGKAPKPLIYAA-------SSLQSGVP-SRFSGSG--SGTDYTLTISSLQPEDFATYYCQQY--------------------',
                'IGKV1-21*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SSWLAWYQQKPGKAPKLLIYKA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQY--------------------',
                'IGKV1-22*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQSI------SSWLAWYQQKPGKAPKLLIYKA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCLQY--------------------',
                'IGKV1-25*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SSYLAWYQQKPGKAPKLLIYKA-------STLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1-27*01': 'DIQMTQSPSSLSASVGDRVT-ITCQASQGI------SSWLAWYQQKPGKAPKLLLYKA-------PGLQSGVP-SMFSGSG--SGTDFTLTISSLQPEYFATYYCQQF--------------------',
                'IGKV1-28*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SSYLNWFQQKPGKAPKLLIYAA-------TTLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFAAYYCLQH--------------------',
                'IGKV1-28*03': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SSYLNWFQQKPGKAPKLLIYAA-------TTLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFAAYYCLQH--------------------',
                'IGKV1-32*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SSYLNWYQQKPGKAPKLLIYYA-------NRLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQY--------------------',
                'IGKV1-32*03': 'DIQMSQSPSSLSASVGDRVT-ITCRASQGI------SSYLNWYQQKPGKAPKLLIYYA-------NSLASGVP-SRFSGSG--SGTEFTLTISSLQPEDFAAYYCLQG--------------------',
                'IGKV1-32*04': 'DIQMSQSPSSLSASVGDRVT-ITCRASQGI------SSYLNWYQQKPGKAPKLLIYYA-------NSLASGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQG--------------------',
                'IGKV1-32*05': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SSYLNWYQQKPGKAPKLLIYYA-------NRLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQY--------------------',
                'IGKV1-33*01': 'DIQMTQSPSSLSASVGDKVT-ITCRASQGI------SNALAWYQQKPGKAPKLLIYAA-------SNLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFAVYYCQQR--------------------',
                'IGKV1-33*02': 'DIQMTQSPSSLSASVGDKVT-ITCRASQGI------SNALAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1-36*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SNYLSWYQQKPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFAAYYCLQY--------------------',
                'IGKV1-36*03': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SDYLSWYQQKPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFAAYYCLQG--------------------',
                'IGKV1-37*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SSYLAWYQQKPGKAPKPLIYYA-------SNLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFAIYYCQQY--------------------',
                'IGKV1-38*01': 'DIQLTQSPSSLSASVGDRVT-ITCRASQGI------SSYLAWYQQKPGKAPKLLIYDA-------SNLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFAVYYCQQR--------------------',
                'IGKV1-41*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SNYLNWYQQEPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQPF--------------------',
                'IGKV1-43*01': 'DIQMTQSPSSLSASAGDRVT-ITCRASQGI------STYLNWYQQKPGKAPKRLIYAA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCLQY--------------------',
                'IGKV1-43*03': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------STYLNWYQQKPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCLQY--------------------',
                'IGKV1-44*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQTI------SSYLAWYQQKPGKVPKLLIYAA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1-44*03': 'DIQMTQSPSSLSASVGDRVT-ITCRASQTI------SSYLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1-46*01': 'DIQMTQSPSSLSASVGDSVT-ITCRASQSF------SSSLAWYQQKPGKAPKLLIYSA-------SSLQSGVP-SRFSGSK--SGTDFTLTISSLQPEDIASYYCQQY--------------------',
                'IGKV1-59*01': 'AIQMTQSPSSLSASVGDKVT-ITCRASQSI------GSNLAWYQQKPGKVPKLLIYAA-------STLQSEVP-SRFSGSG--SGTDFTLTISSLQPEEVATYYCQKC--------------------',
                'IGKV1-66*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------NNYLSWYQQKPGKAPKPLIYYA-------SSLERGVP-SRFSGSR--SGTDYTLTISSLQPEDIATYYCQQY--------------------',
                'IGKV1-69*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SNWLAWYQQKPGKAPKLLIYRA-------SNLETGVP-SRFSGSG--SGTDFTLTISSLQPEDIATYYCQQH--------------------',
                'IGKV1-69*02': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SNWLAWYQQKPGKAPKLLIYAA-------SNLETGVP-SRFSGSG--SGTDFTLTISSLQPEDIATYYCQQH--------------------',
                'IGKV1-74*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASENV------NNYLNWYQQKPGKAPKLLIYKA-------STLQSGVP-SRFSGSG--SGTDYTFTISSLQPEDVATYYCQHG--------------------',
                'IGKV1-80*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------NNELAWYQQKPGKAPTLLLYSG-------SSLHTGVP-SQFSGSG--SGTDFTLTISSLQPEDVATYYCRQD--------------------',
                'IGKV1-84*01': 'DIQMTQPPSSLSASVGDRVN-ITCQASQSI------SNYLNWYPQKTWKAPKFLTYRA-------SGLQRGVP-SQFSGSG--YGRDFTLTISSLRPEDFAIYYCQQE--------------------',
                'IGKV1-94*01': 'DIQMTQSPSSLSASVGDRVT-VTCRASQGI------NKELSWYQQKPGKAPTLLIYAA-------SSLQTGVS-SRFSGSG--SGTDFTLTISSLQPEDVATYYCQQD--------------------',
                'IGKV1S11*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SSWLAWYQQKPGKAPKLLIYAA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFAVYYCQQR--------------------',
                'IGKV1S12*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQTI------SSYLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCLQY--------------------',
                'IGKV1S13*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SNYLAWYQQKPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFAAYYCLQH--------------------',
                'IGKV1S14*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SNYLAWYQQKPGKAPKPLIYYA-------SNLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S15*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SNYLAWYQQKPGKAPKPLIYYA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQG--------------------',
                'IGKV1S16*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SNYLAWYQQKPGKAPKPLIYYA-------SSLESGVP-SRFSGSG--YGTDFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S17*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SNNLAWYQQKPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCLQY--------------------',
                'IGKV1S19*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SNYLNWYQQKPGKAPKLLIYAA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFAAYYCLQH--------------------',
                'IGKV1S21*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SNYLNWYQQKPGKAPKRLIYDA-------SSLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFAAYYCLQY--------------------',
                'IGKV1S25*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQGI------SSWLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S3*01': 'DIQMTQSPSSLSASVGDTVT-ITCRASQGI------SNYLAWYQQKPGKAPKPLIYYA-------SSLESGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S4*01': 'DIQMTQSPSSLSASVGDRVT-ITCQASQGI------SSWLAWYQQKPGKAPKLLLYKA-------PGLQSGVP-SMFSGSG--SGTEFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S5*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQTI------SSYLAWYQQKPGKAPKRLIYAA-------SSLESGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S6*01': 'DIQMTQSPSSLSASVGDKVT-ITCRASQGI------SSWLAWYQQKPGKAPKLLIYKA-------SSLASGVP-SRFSGSG--SGTEFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV1S8*01': 'DIQMTQSPSSLSASVGDRVT-ITCRASQTI------SSYLAWYQQKPGKVPKLLIYAA-------STLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQY--------------------',
                'IGKV1S9*01': 'DIQMSQSPSSLSASVGDTVT-ITCRASQGI------SNYLNWFQQKPGKAPKLLIYAA-------TTLQSGVP-SRFSGSG--SGTDFTLTISSLQPEDFATYYCQQH--------------------',
                'IGKV2-104*02': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLDSEDGNTYLDWYLQKPGQSPQLLIYEV-------SNRASGVP-DRFSGSG--SDTDFTLKISRVEAEDVGVYYCMQA--------------------',
                'IGKV2-58*01': 'DVAMTQSPLSLPVTLGQPAS-ISCRSSQSLLHS-NGNTYLSWFQQKPGQSPRRLIYKV-------SNRDSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCMEG--------------------',
                'IGKV2-58*03': 'DVAMTQSPLSLPVTPGQPAS-ISCRSSQSLLHS-NGNTYLSWFQQKPGQSPRRLIYKV-------SNRDSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCMEG--------------------',
                'IGKV2-60*01': 'DIVMTQTPLSLPVTLGEPAS-ISCRSSQSLLSS-NGYNYLNWYLQKPGQSPQLLIYYG-------SNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQA--------------------',
                'IGKV2-61*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLHT-DGYTYLDWYLQKPGQSPQLLIYGG-------SNRASGVP-DRFSGSG--SGTDFTLKISKVEAEDVGVYYCMQH--------------------',
                'IGKV2-61*03': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLHT-DGYTYLDWYLQKPGQSPQLLIYGG-------SNRASGVP-DRFSGSG--SGTDFTLKISKVEAEDVGVYYCMQH--------------------',
                'IGKV2-64*01': 'DVVMTQSPLSLPITPGQPAS-ISCRSSQSLVHS-DGNTYLSWYQQKPGQPPRLLIYKV-------SNRDSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCGQG--------------------',
                'IGKV2-64*02': 'DVVMTQSPLSLPITPGQPAS-ISCRSSQSLVHS-DGNTYLSWYQQKPGQPPRLLIYKV-------SNRYSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCGQG--------------------',
                'IGKV2-65*01': 'DVVMTQSPLSLPITPGQPAS-ISCRSSQSLVHS-NGNTYLSWYQQKPGQPPRRLIYEV-------SNRDSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCGQG--------------------',
                'IGKV2-68*01': 'DIVMTQTLLSLPVTPGEPAS-ISCRSSQSLLHS-NGNTYLDWYLQKPGQSPRFLIYKV-------TNREPGVP-DRFSGSG--SGTDFTLKISRVEPEDVGVCYCMQS--------------------',
                'IGKV2-7*02': 'DTVMTQTPLSLPVT-G-PAS-ISCRSSQSLPYG-NGVNYLNWYLQKPDQPPQLLIYLG-------SSRFPGVP-DRFTVSR--SDTDFTLQISRVKAEDVGVYYCVQC--------------------',
                'IGKV2-70*01': 'DIVMTQTPLSLSVTPREPAS-ISCRSSQSLLHT-DGRTYLYWYLQKPGQPPRLLIYRV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQA--------------------',
                'IGKV2-72*01': 'DIVMTQTPLSLPITPGEPAS-ISCRSSQSLLHS-NGNTYLHWYLQKPGQSPQLLIYGG-------SNRASGVP-DRFSGSG--SGTDFTLKISKVEAEDVGVYYCVQA--------------------',
                'IGKV2-72*02': 'DIVMTQTPLSLPITPGEPAS-ISCRSSQSLLHS-NGNTYLHWYLQKPGQSPQLLIYGG-------SNRASGVP-DRFSGSG--SGTDFTLKISKVEAEDVGVYYCVQA--------------------',
                'IGKV2-72*03': 'DIVMTQTPLSLPITPGEPAS-ISCRSSQSLLHS-NGNTYLHWYLQKPGQSPQLLIYGG-------SNRASGVP-DRFSGSG--SGTDFTLKISKVEAEDVGVYYCVHA--------------------',
                'IGKV2-73*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLHS-DGNTYLYWYLQKPGQPPRLLIYRV-------SNRFSGVP-DRFSGSG--SGTDFTLKISRVKAEDVGVYYCMQA--------------------',
                'IGKV2-76*01': 'DIVMTQTPLSLPITPGEPAS-ISCRSSQSFLDSDDGYTYLDWYLQKPGQPPQPLIYFV-------SSRASGVP-DRFNGSG--SGSDFTLKISGVEADDVGVYYCMQC--------------------',
                'IGKV2-78*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLDS-DGYTHLHWYLQKPGQSPQLLIYLG-------SNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQT--------------------',
                'IGKV2-82*01': 'DIVMTQTPLSLPVTLGEPAS-ISCRSSQSLVYS-DGKTYLDWYLQKPGQSPQLLMYLV-------SKRASGVP-DKFSGSG--SGTDFTLKISRVEAEDVGVYYCMQA--------------------',
                'IGKV2-82*02': 'DIVMIQTPLSLPVTLGEPAS-ISCRSSQSLVYS-DGKTYLYWYLQKPGQSPQLLMYLV-------SKRASGVP-DKFSGSG--SGTDFTLKISRVEAEDVGVYYCMQA--------------------',
                'IGKV2-82*03': 'DIVMIQTPLSLPVTLGEPAS-ISCRSSQSLVYS-DGKTYLYWYLQKPGQSPQLLMYLV-------SKRASGVP-DKFSGSG--SGTDFTLKISRVEAEDVGVYYCMQA--------------------',
                'IGKV2-86*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLDSEDGNTYLDWYLQKPGQSPQPLIYEV-------SNRASGVP-DRFSGSG--SDTDFTLKISRVEAEDVGVYYCMQY--------------------',
                'IGKV2-90*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLDS-DGYTCLDWYLQKPGQSPQLLIYEV-------SNRVSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQS--------------------',
                'IGKV2-91*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLHS-NGYTYLYWYLQKPGQSPQLLMYFA-------SYRASGVP-DRFSGSG--SGTDFTLGISRVEAEDIGVYYCMQG--------------------',
                'IGKV2-99*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLFDSDYANTYLDWCLQKPGQSPQLLIYML-------FNRVSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQS--------------------',
                'IGKV2S15*01': 'DIVMTQTPLSLSVTPGQPAS-ISCKSSQSLLHS-DGKTYLYWYLQKPGQSPQLLIYEV-------SSRFSGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCMQG--------------------',
                'IGKV2S2*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLHS-NGNTYLDWYLQKPGQSPRLLIYKV-------TNRESGVP-DRFSGSG--SGTDFTLKISRVEPEDVGVYYCMQS--------------------',
                'IGKV2S20*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLDSEDGNTYLEWYLQKPGQSPQPLIYEV-------SNRASGVP-DRFSGSG--SDTDFTLKISRVEAEDVGVYYCMQG--------------------',
                'IGKV2S3*01': 'DIVMTQTPLSLPVTPGEPAS-ISCRSSQSLLHS-NGNTYLHWYLQKPGQSPRLLIYKV-------TNRESGVP-DRFSGSG--SGTDFTLKISRVEPEDVGVYYCMQS--------------------',
                'IGKV2S8*01': 'DVVMTQSPLSLPVTPGQPAS-ISCRSSQSLVHS-DGKTYLNWLQQKPGQPPRRLIYQV-------SNRDSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCGQG--------------------',
                'IGKV2S9*01': 'DVVMTQSPLSLPVTPGQPAS-ISCRSSQSLVHS-DGKTYLNWLQQKPGQPPRRLIYQV-------SNRDSGVP-DRFSGSG--AGTDFTLKISRVEAEDVGVYYCVQG--------------------',
                'IGKV3-10*02': 'QVILTQSPATLSLSPGERAT-LSCRASQSV------SSYLAWYQQKPGQAPRLLIHSA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDVGVYHCYQY--------------------',
                'IGKV3-17*01': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSSLAWYQQKPGQAPRLLIYDA-------SSRVTGIP-DRFSGSG--SGTDFTLTISSLEPEDVGVYFCQQE--------------------',
                'IGKV3-17*02': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSRLAWYQQKPGQAPRLLIYDA-------SSRVTGIP-DRFSGSG--SGTDFTLTISSLEPEDVAVYFCQQE--------------------',
                'IGKV3-17*03': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSRLAWYQQKPGQAPRLLIYDA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDVAVYFCQQE--------------------',
                'IGKV3-24*01': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSSLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTDFTLTISSLEPEDVAVYYCLQR--------------------',
                'IGKV3-24*03': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSSLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDVAVYYCLQR--------------------',
                'IGKV3-24*04': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------GSSLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTDFTLTISSLEPEDVAVYYCLQR--------------------',
                'IGKV3-31*01': 'EIVMTQSPATLSLSPGETAT-ISCRTSQSV------SSYLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTDFTLTISSLEPEYFAVYYCQET--------------------',
                'IGKV3-35*01': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSNLAWYQQKPGQAPRLLIYDA-------SNRATGIP-DRFSGSG--SGTDFTLTISSLEPEDVGVYYCQQE--------------------',
                'IGKV3-35*02': 'EIVMTQSPATLSLSPRERAT-LSCRASQSV------SSNLAWYQQKPGQAPRLLIYYA-------SNRATGIP-DRFSGSG--SGTDFTLTISSLEPEDVGVYYCQQE--------------------',
                'IGKV3-35*03': 'EIVMTQSPATLSLSPRERAT-LSCRASQSV------SSNLAWYQQKPGQAPRLLIYYA-------SNRATGIP-DRFSGSG--SGTDFTLTISSLEPEDVGVYYCQQE--------------------',
                'IGKV3-40*01': 'EIVMTQSPATLSLSPGETAT-LSCRASESV------GSYLAWYQQKPGQAPKLLVRSA-------YFRATGIP-DRFSGSG--SRTDFTLTISSLEPEDVGVYHCQQY--------------------',
                'IGKV3-40*03': 'EIVMTQSPATLSLSPGETAT-LSCRASESV------GSYLAWYQQKPGQAPKLLVHSA-------YFRATGIP-DRFSGSG--SRTEFTLTISSLEPEDVGVYHCQQY--------------------',
                'IGKV3-42*01': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSSLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDVAVYYCQQN--------------------',
                'IGKV3-42*03': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSSLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDFAVYYCQQY--------------------',
                'IGKV3-53*01': 'QVILTQSPATLSLSPGERAT-LSCRASQSV------SSSLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDFAVYYCQKY--------------------',
                'IGKV3S11*01': 'QVILTQSPATLSLSPGERAT-LSCRASQSV------GSNLAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTDFTLTISSLEPEDVAVYYCLQR--------------------',
                'IGKV3S5*01': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSRLAWYKQKPGQAPRLLIYDA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDVAVYLCQQE--------------------',
                'IGKV3S9*01': 'EIVMTQSPATLSLSPGERAT-LSCRASQSV------SSYVAWYQQKPGQAPRLLIYGA-------SSRATGIP-DRFSGSG--SGTEFTLTISSLEPEDFAVYYCQQY--------------------',
                'IGKV4-1*02': 'DIVMTQSPDSLAVSLGERVT-INCKSSQSLLYSSNNKNYLAWYQQKPGQAPKLLIYWA-------STRESGVP-NRFSGSG--SGSDFTLTISGLQAEDVAVYYCQQY--------------------',
                'IGKV5-11*02': 'ETILTQSAAFVSATPGDKVT-ISCRAGQDI------DDDMNWYQQEPGEAPKLIIKDA-------TTLVSGIP-PRFSGSG--YGTDFTLTINNVESEDAAYYFCLQH--------------------',
                'IGKV6-47*01': 'DIVMTQSPAFVSVTPGEKVT-ITCQVSEGI------SNYLHWYQQKPDQAPKLFIQYA-------SQSISGVP-SRFTGSG--SGTDFTFTISSLEVEDAATYYCQQG--------------------',
                'IGKV6-55*01': 'EIVLTQSPAFRSVTLKEKVT-ITCQASQSI------GSSLHWYQQKPDQSPKLLIKYA-------SQSISGVP-SRFSGSG--SGTDFTLTINSLEAEDAATYYCQQS--------------------'},
                'pig': {'IGKV1-11*01': 'AIQLTQSPASLAASLGDTVSITCRASQSI------NKWLAWYQQQAGKAPKLLIYSA-------STLQSGVP-SRFKGSG--SGTDFTLTISGLQAEDVATYYCQQHH--------------------',
                'IGKV1-7*01': 'AIQLTQSPASLAASLGDTVSITCRAHQTI------SSYLAWYQQQPGKPPKLLLCDA-------CTLQSGVP-CGFKGSG--SGTHFTLTISGLQAEDVATYYCQQLN--------------------',
                'IGKV1-9*01': 'AIQLTQSPASLAASLGDTVSITCRASQSV------SNNLAWYQQQAGKPPKLLIYWA-------SALQSGVP-SRFKGSV--SGTDFTLTISGLQAEDVATYYCQQLN--------------------',
                'IGKV2-10*01': 'AIVLTQTPLSLSVSPGEPASISCRSSQSLVDS-DGDSLLHWYLQKPGQSPQLLIYEA-------TNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDVGVYYCFQAL--------------------',
                'IGKV2-12*01': 'AIVLTQTPLSLSVSPGEPASISCRSTQSLRGS-YGKNYLNWYQQKPGQSPKLLIYWA-------TNRASGVP-DRFSGSR--SGTDFTLKIIRLEAEDAGVYSCLQDI--------------------',
                'IGKV2-13*02': 'AIVLTQTPLSLSVSPGEPASISCRSSQSLEE--YGSNLLSWYQQKPGQSPQLLIYEA-------TNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDAGVYYCQQFK--------------------',
                'IGKV2-6*01': 'AIVLTQSPLSLSVSPGAPASISCRSSQSLES--YSYNFLSWYQQKPGQSPRLLIYFA-------TNKASGVP-DRFSGSG--SGTDFTLKISRVEAEDAGVYYCQQNK--------------------',
                'IGKV2-8*01': 'AIVLTQTPLSLSVSPGEPASISCRSSQSLEI--YGSNFLSWYQQKPGQSPQLLIYEA-------TNRASGVP-DRFSGSG--SGTDFTLKISRVEAEDAGVYYCQQHK--------------------'},
                'cow': {'IGKV1-4*01': 'DIQVTQSPSYLSASLGDRVSITCQANQSV------SHYLNWYQQKPGEAPKLLIYYA-------TSRYTRVP-SRFSGSG--SGTDFTLTISSLEADDAANYYCQQDY--------------------',
                'IGKV2-15*01': 'DVVLTQTPLSLSVIPGETVSISCKSTQSLKYS-DGKTYLRWVQHKPGQSPQGVIYQV-------SNRNTGVP-DRFTGSG--SETDFTLTISSVQAEDAGVYYCFQGT--------------------',
                'IGKV2-18*01': 'DVVLTQTPLSLSVIPGETVSISCKSTQSLKY--SGKTYLRWLQHKPGQSPQSLIYQV-------SNRYTGVP-DRFTGSG--SETDFTLTISSVQAEDAGVYYCVQET--------------------',
                'IGKV2-6*01': 'DVVLTQTPLSLSIIPGEMASISCKSSQSLVHS-DGKTYLNWIQYKPGQSPQGLIYQV-------SNRYSGVS-DRFTGSG--SGTDFTLTISRVQAEDAGVYYCYQGT--------------------',
                'IGKV2-9*01': 'DVVLTQTPLSLSVIPGETVTISCKSTQSLKYS-DGKTYLQWFQHKPGQSPRLLIYQI-------SNRYTGVP-DRFTGSG--SETDFTLTISSVQAEDAGVYYCLQRS--------------------',
                'IGKV8-3*01': 'EAVLYQTPAYIAASLGESISITCRANQSI------SDYLSWYKQKPGQAPMILIYDA-------DNRYNGVP-ERFTATQ--SETEFVFTISQVEADDAAMYYCQQDY--------------------'}},
                'L': {'human': {'IGLV1-36*01': 'QSVLTQPPS-VSEAPRQRVTISCSGSSSNI----GNNAVNWYQQLPGKAPKLLIYYD-------DLLPSGVS-DRFSGSK--SGTSASLAISGLQSEDEADYYCAAWD--------------------',
                'IGLV1-40*01': 'QSVLTQPPS-VSGAPGQRVTISCTGSSSNIG---AGYDVHWYQQLPGTAPKLLIYGN-------SNRPSGVP-DRFSGSK--SGTSASLAITGLQAEDEADYYCQSYD--------------------',
                'IGLV1-40*02': 'QSVVTQPPS-VSGAPGQRVTISCTGSSSNIG---AGYDVHWYQQLPGTAPKLLIYGN-------SNRPSGVP-DRFSGSK--SGTSASLAITGLQAEDEADYYCQSYD--------------------',
                'IGLV1-40*03': 'QSVVTQPPS-VSGAPGQRVTISCTGSSSNIG---AGYDVHWYQQLPGTAPKLLIYGN-------SNRPSGVP-DRFSGSK--SGASASLAITGLQAEDEADYYCQSYD--------------------',
                'IGLV1-44*01': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNTVNWYQQLPGTAPKLLIYSN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLQSEDEADYYCAAWD--------------------',
                'IGLV1-44*02': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNTVNWYQQLPGTGPKLLIYSN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLQSEDEADYYCAAWD--------------------',
                'IGLV1-44*03': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNTVNWYQQLPGTAPKLLIYSN-------NQRPSGVL-DRFSGSK--SGTSASLAISGLQSEDEADYYCAAWD--------------------',
                'IGLV1-44*04': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNTVNWYQQLPGTAPKLLIYRN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLQAEDEADYYCAAWD--------------------',
                'IGLV1-47*01': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNYVYWYQQLPGTAPKLLIYRN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLRSEDEADYYCAAWD--------------------',
                'IGLV1-47*02': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNYVYWYQQLPGTAPKLLIYSN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLRSEDEADYYCAAWD--------------------',
                'IGLV1-47*03': 'QSVLTQPPS-ASGTPGQRVTISCSGSSSNI----GSNYVYWYQQLPGTAPKLLIYRN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLWSEDEADYYCAAWD--------------------',
                'IGLV1-51*01': 'QSVLTQPPS-VSAAPGQKVTISCSGSSSNI----GNNYVSWYQQLPGTAPKLLIYDN-------NKRPSGIP-DRFSGSK--SGTSATLGITGLQTGDEADYYCGTWD--------------------',
                'IGLV1-51*02': 'QSVLTQPPS-VSAAPGQKVTISCSGSSSNI----GNNYVSWYQQLPGTAPKLLIYEN-------NKRPSGIP-DRFSGSK--SGTSATLGITGLQTGDEADYYCGTWD--------------------',
                'IGLV10-54*01': 'QAGLTQPPS-VSKGLRQTATLTCTGNSNNV----GNQGAAWLQQHQGHPPKLLSYRN-------NNRPSGIS-ERLSASR--SGNTASLTITGLQPEDEADYYCSAWD--------------------',
                'IGLV10-54*02': 'QAGLTQPPS-VSKGLRQTATLTCTGNSNIV----GNQGAAWLQQHQGHPPKLLSYRN-------NNRPSGIS-ERFSASR--SGNTASLTITGLQPEDEADYYCSALD--------------------',
                'IGLV10-54*04': 'QAGLTQPPS-VSKGLRQTATLTCTGNSNNV----GNQGAAWLQQHQGHPPKLLSYRN-------NNRPSGIS-ERFSASR--SGNTASLTITGLQPEDEADYYCSAWD--------------------',
                'IGLV10-54*05': 'QAGLTQPPS-VSKGLRQTATLTCTGNSNIV----GNQGAAWLQQHQGHPPKLLSYRN-------NNRPSGIS-ERFSASR--SGNTASLTITGLQPEDEADYYCSAWD--------------------',
                'IGLV10-54*06': 'QAGLTQPPS-VSKGLRQTATLTCTGNSNNA----GNQGAAWLQQHQGHPPKLLSYRN-------NNRPSGIS-ERFSASR--SGNTASLTITGLQPEDEADYYCSAWD--------------------',
                'IGLV2-11*01': 'QSALTQPRS-VSGSPGQSVTISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYDV-------SKRPSGVP-DRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-11*02': 'QSALTQPRS-VSGSPGQSVTISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYDV-------SKRPSGVP-DRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-14*01': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYEV-------SNRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-14*02': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---SYNLVSWYQQHPGKAPKLMIYEG-------SKRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-14*03': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYDV-------SNRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-14*04': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYDV-------SNRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-14*05': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYEV-------SNRPSGVP-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-18*01': 'QSALTQPPS-VSGSPGQSVTISCTGTSSDVG---SYNRVSWYQQPPGTAPKLMIYEV-------SNRPSGVP-DRFSGSK--SGNTASLTISGLQAEDEADYYCSLYT--------------------',
                'IGLV2-18*02': 'QSALTQPPS-VSGSPGQSVTISCTGTSSDVG---SYNRVSWYQQPPGTAPKLMIYEV-------SNRPSGVP-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-18*03': 'QSALTQPPS-VSGSPGQSVTISCTGTSSDVG---SYNRVSWYQQPPGTAPKLMIYEV-------SNRPSGVP-DRFSGSK--SGNTASLTTSGLQAEDEADYYCSSYT--------------------',
                'IGLV2-18*04': 'QSALTQPPS-VSGSPGQSVTISCTGTSSDVG---SYNRVSWYQQPPGTAPKLMIYEV-------SNRPSGVP-DRSSGSK--SGNTASLTISGLQAEDEADYYCSSYT--------------------',
                'IGLV2-23*01': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---SYNLVSWYQQHPGKAPKLMIYEG-------SKRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-23*02': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---SYNLVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-23*03': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---SYNLVSWYQQHPGKAPKLMIYEG-------SKRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-23*04': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYDV-------SKRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-23*05': 'QSALTQPAS-VSGSPGQSITISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYDV-------SKRPSGVS-NRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-8*01': 'QSALTQPPS-ASGSPGQSVTISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYEV-------SKRPSGVP-DRFSGSK--SGNTASLTVSGLQAEDEADYYCSSYA--------------------',
                'IGLV2-8*02': 'QSALTQPPS-ASRSPGQSVTISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYEV-------SKRPSGVP-DRFSGSK--SGNTASLTVSGLQAEDEADYYCSSYA--------------------',
                'IGLV2-8*04': 'QSALTQPPS-ASGSSGQSVTISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYEV-------SKRPSGVP-DRFSGSK--SGNTASLTVSGLQAEDEADYYCSSYA--------------------',
                'IGLV3-1*01': 'SYELTQPPS-VSVSPGQTASITCSGDKLG------DKYACWYQQKPGQSPVLVIYQD-------SKRPSGIP-ERFSGSN--SGNTATLTISGTQAMDEADYYCQAWD--------------------',
                'IGLV3-1*02': 'SYELTQPPS-VSVSPGQTASITCSGDKLG------DKYACWYQQKPGQSPVLVIYQD-------SERPSGIP-ERFSGSN--SGNTATLTISGTQAMDEADYYCQAWD--------------------',
                'IGLV3-10*01': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KKYAYWYQQKSGQAPVLVIYED-------SKRPSGIP-ERFSGSS--SGTMATLTISGAQVEDEADYYCYSTD--------------------',
                'IGLV3-10*03': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KKYAYWYQQKSGQAPVLVIYED-------SKRPSGIP-ERFSGSS--SGTMATLTISGAQVEDEDDYYCYSTD--------------------',
                'IGLV3-10*04': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KKYPYWYQQKSGQAPVLVIYED-------SKRPSGIP-ERFSGSS--SGTMATLTISGAQVEDEADYYCYSTD--------------------',
                'IGLV3-12*02': 'SYELTQPHS-VSVATAQMARITCGGNNIG------SKAVHWYQQKPGQDPVLVIYSD-------SNRPSGIP-ERFSGSN--PGNTATLTISRIEAGDEADYYCQVWD--------------------',
                'IGLV3-16*01': 'SYELTQPPS-VSVSLGQMARITCSGEALP------KKYAYWYQQKPGQFPVLVIYKD-------SERPSGIP-ERFSGSS--SGTIVTLTISGVQAEDEADYYCLSAD--------------------',
                'IGLV3-19*01': 'SSELTQDPA-VSVALGQTVRITCQGDSLR------SYYASWYQQKPGQAPVLVIYGK-------NNRPSGIP-DRFSGSS--SGNTASLTITGAQAEDEADYYCNSRD--------------------',
                'IGLV3-19*02': 'SSELTQDPA-VSVALGQTVRITCQGDSLR------SYYASWYQQKPGQAPVRVIYGK-------NNRPSGIP-DRFSGSS--SGNTASLTITGAQAEDEADYYCNSWD--------------------',
                'IGLV3-21*01': 'SYVLTQPPS-VSVAPGKTARITCGGNNIG------SKSVHWYQQKPGQAPVLVIYYD-------SDRPSGIP-ERFSGSN--SGNTATLTISRVEAGDEADYYCQVWD--------------------',
                'IGLV3-21*02': 'SYVLTQPPS-VSVAPGQTARITCGGNNIG------SKSVHWYQQKPGQAPVLVVYDD-------SDRPSGIP-ERFSGSN--SGNTATLTISRVEAGDEADYYCQVWD--------------------',
                'IGLV3-21*03': 'SYVLTQPPS-VSVAPGKTARITCGGNNIG------SKSVHWYQQKPGQAPVLVVYDD-------SDRPSGIP-ERFSGSN--SGNTATLTISRVEAGDEADYYCQVWD--------------------',
                'IGLV3-21*04': 'SYVLTQPPS-VSVAPGKTARITCGGNNIG------SKSVHWYQQKPGQAPVLVIYYD-------SDRPSGIP-ERFSGSN--SGNTATLTISRVEAGDEADYYCQVWD--------------------',
                'IGLV3-22*01': 'SYELTQLPS-VSVSPGQTARITCSGDVLG------ENYADWYQQKPGQAPELVIYED-------SERYPGIP-ERFSGST--SGNTTTLTISRVLTEDEADYYCLSGD--------------------',
                'IGLV3-22*03': 'SYELTQLPS-VSLSPGQKARITCSGDVLG------KNYADWYQQKPGQAPELVIYED-------SERYPGIP-ERFSGST--SGNTTTLTISRVLTEDEADYYCLSGN--------------------',
                'IGLV3-25*01': 'SYELMQPPS-VSVSPGQTARITCSGDALP------KQYAYWYQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTISGVQAEDEADYYCQSAD--------------------',
                'IGLV3-25*02': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KQYAYWYQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTISGVQAEDEADYYCQSAD--------------------',
                'IGLV3-25*03': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KQYAYWYQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTISGVQAEDEADYYCQSAD--------------------',
                'IGLV3-27*01': 'SYELTQPSS-VSVSPGQTARITCSGDVLA------KKYARWFQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTISGAQVEDEADYYCYSAA--------------------',
                'IGLV3-9*01': 'SYELTQPLS-VSVALGQTARITCGGNNIG------SKNVHWYQQKPGQAPVLVIYRD-------SNRPSGIP-ERFSGSN--SGNTATLTISRAQAGDEADYYCQVWD--------------------',
                'IGLV3-9*02': 'SYELTQPLS-VSVALGQAARITCGGNNLG------YKSVHWYQQKPGQAPVLVIYRD-------NNRPSGIP-ERFSGSN--SGNTATLTISRAQAGDEADYYCQVWD--------------------',
                'IGLV4-60*01': 'QPVLTQSSS-ASASLGSSVKLTCTLSSGHS-----SYIIAWHQQQPGKAPRYLMKLEGS---GSYNKGSGVP-DRFSGSS--SGADRYLTISNLQLEDEADYYCETWD--------------------',
                'IGLV4-60*02': 'QPVLTQSSS-ASASLGSSVKLTCTLSSGHS-----SYIIAWHQQQPGKAPRYLMKLEGS---GSYNKGSGVP-DRFSGSS--SGADRYLTISNLQFEDEADYYCETWD--------------------',
                'IGLV4-60*03': 'QPVLTQSSS-ASASLGSSVKLTCTLSSGHS-----SYIIAWHQQQPGKAPRYLMKLEGS---GSYNKGSGVP-DRFSGSS--SGADRYLTISNLQSEDEADYYCETWD--------------------',
                'IGLV4-69*01': 'QLVLTQSPS-ASASLGASVKLTCTLSSGHS-----SYAIAWHQQQPEKGPRYLMKLNSD---GSHSKGDGIP-DRFSGSS--SGAERYLTISSLQSEDEADYYCQTWG--------------------',
                'IGLV4-69*02': 'QLVLTQSPS-ASASLGASVKLTCTLSSGHS-----SYAIAWHQQQPEKGPRYLMKLNSD---GSHSKGDGIP-DRFSGSS--SGAERYLTISSLQSEDEADYYCQTWG--------------------',
                'IGLV5-37*01': 'QPVLTQPPS-SSASPGESARLTCTLPSDINV---GSYNIYWYQQKPGSPPRYLLYYYSD---SDKGQGSGVP-SRFSGSKDASANTGILLISGLQSEDEADYYCMIWP--------------------',
                'IGLV5-37*02': 'QPVLTQPPS-SSASPGESARLTCTLPSDINV---GSYNIYWYQQKPGSPPRYLLYYYSD---SDKGQGSGVP-SRFSGSKDASANTGILLISGLQSEDEADYYCMIWP--------------------',
                'IGLV5-37*03': 'QPVLTQPPS-SSASPGESARLTCTLPSDINV---SSYNIYWYQQKPGSPPRYLLYYYSD---SDKGQGSGVP-SRFSGSKDASANTGILLISGLQSEDEADYYCMIWP--------------------',
                'IGLV5-39*01': 'QPVLTQPTS-LSASPGASARFTCTLRSGINV---GTYRIYWYQQKPGSLPRYLLRYKSD---SDKQQGSGVP-SRFSGSKDASTNAGLLLISGLQSEDEADYYCAIWY--------------------',
                'IGLV5-39*02': 'QPVLTQPTS-LSASPGASARFTCTLRSGINV---GTYRIYWYQQNPGSLPRYLLRYKSD---SDKQQGSGVP-SRFSGSKDASTNAGLLLISGLQSEDEADYYCAIWY--------------------',
                'IGLV5-45*01': 'QAVLTQPAS-LSASPGASASLTCTLRSGINV---GTYRIYWYQQKPGSPPQYLLRYKSD---SDKQQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-45*02': 'QAVLTQPSS-LSASPGASASLTCTLRSGINV---GTYRIYWYQQKPGSPPQYLLRYKSD---SDKQQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-45*03': 'QAVLTQPSS-LSASPGASASLTCTLRSGINV---GTYRIYWYQQKPGSPPQYLLRYKSD---SDKQQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-45*04': 'QAVLTQPSS-LSASPGASASLTCTLCSGINV---GTYRIYWYQQKPGSPPQYLLRYKSD---SDKQQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-45*05': 'QAVLTQLAS-LSASPGASASLTCTLRSGINV---GTYRIYWYQQKPGSPPQYLLRYKSD---SDKQQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-48*01': 'QPVLTQPTS-LSASPGASARLTCTLRSGINL---GSYRIFWYQQKPESPPRYLLSYYSD---SSKHQGSGVP-SRFSGSKDASSNAGILVISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-52*01': 'QPVLTQPSS-HSASSGASVRLTCMLSSGFSV---GDFWIRWYQQKPGNPPRYLLYYHSD---SNKGQGSGVP-SRFSGSNDASANAGILRISGLQPEDEADYYCGTWH--------------------',
                'IGLV6-57*01': 'NFMLTQPHS-VSESPGKTVTISCTRSSGSI----ASNYVQWYQQRPGSSPTTVIYED-------NQRPSGVP-DRFSGSIDSSSNSASLTISGLKTEDEADYYCQSYD--------------------',
                'IGLV6-57*02': 'NFMLTQPHS-VSESPGKTVTISCTGSSGSI----ASNYVQWYQQRPGSAPTTVIYED-------NQRPSGVP-DRFSGSIDSSSNSASLTISGLKTEDEADYYCQSYD--------------------',
                'IGLV6-57*03': 'NFMLTQPHS-VSESPGKTVTISCTRSSGSI----ASNYVQWYQQRPGSAPTTVIYED-------NQRPSGVP-DRFSGSIDSSSNSASLTISGLKTEDEADYYCQSYD--------------------',
                'IGLV6-57*04': 'NFMLTQPHS-VSESPGKTVTISCTRSSGSI----ASNYVQWYQQRPGSAPTTVIYED-------NQRPSGVP-DRFSGSIDSSSNSASLTISGLKTEDEADYYCQSYD--------------------',
                'IGLV7-43*01': 'QTVVTQEPS-LTVSPGGTVTLTCASSTGAVT---SGYYPNWFQQKPGQAPRALIYST-------SNKHSWTP-ARFSGSL--LGGKAALTLSGVQPEDEAEYYCLLYY--------------------',
                'IGLV7-46*01': 'QAVVTQEPS-LTVSPGGTVTLTCGSSTGAVT---SGHYPYWFQQKPGQAPRTLIYDT-------SNKHSWTP-ARFSGSL--LGGKAALTLSGAQPEDEAEYYCLLSY--------------------',
                'IGLV7-46*02': 'QAVVTQEPS-LTVSPGGTVTLTCGSSTGAVT---SGHYPYWFQQKPGQAPRTLIYDT-------SNKHSWTP-ARFSGSL--LGGKAALTLLGAQPEDEAEYYCLLSY--------------------',
                'IGLV7-46*04': 'QAVVTQEPS-LTVSPGGTVTLTCGSSTGAVT---SGHYPYWFQQKPGQAPRTLIYDT-------SNKHSWTP-ARFSGSL--LGGKAALTLSGAQPEDEAEYYCLLSY--------------------',
                'IGLV7-46*05': 'QAVVTQEPS-LTVSPGGTVTLTCGSSTGAVT---SGHYPYWFQQKPGQAPRTLIYDT-------SNKHSWTP-ARFSGSL--LGGKAALTLSGAQPEDEAEYYCLLSY--------------------',
                'IGLV8-61*01': 'QTVVTQEPS-FSVSPGGTVTLTCGLSSGSVS---TSYYPSWYQQTPGQAPRTLIYST-------NTRSSGVP-DRFSGSI--LGNKAALTITGAQADDESDYYCVLYM--------------------',
                'IGLV8-61*02': 'QTVVTQEPS-FSVSPGGTVTLTCGLSSGSVS---TSYYPSWYQQTPGQAPRTLIYST-------NTRSSGVP-DCFSGSI--LGNKAALTITGAQADDESDYYCVLYM--------------------'},
                'mouse': {'IGLV1*01': 'QAVVTQESA-LTTSPGETVTLTCRSSTGAVT---TSNYANWVQEKPDHLFTGLIGGT-------NNRAPGVP-ARFSGSL--IGDKAALTITGAQTEDEAIYFCALWY--------------------',
                'IGLV1*02': 'QAVVTQESA-LTTSPGETVTLTCRSSTGAAT---TSNYANWVQEKPDHLFTGLIGGT-------NNRAPGVP-ARFSGSL--IGDKAALTITGAQTEDEAIYFCALWY--------------------',
                'IGLV2*01': 'QAVVTQESA-LTTSPGGTVILTCRSSTGAVT---TSNYAIWVQEKTDHLFAGVIGDT-------SNRAPGVP-ARFSGSL--IGDKAALTITGAQTEDDAMYFCALWY--------------------',
                'IGLV2*02': 'QAVVTQESA-LTTSPGGTVILTCRSSTGAVT---TSNYANWVQEKPDHLFTGLIGGT-------SNRAPGVP-VRFSGSL--IGDKAALTITGAQTEDDAMYFCALWY--------------------',
                'IGLV3*01': 'QPVLTQSSS-ASFSLGASAKLTCTLSSEHS-----TYIIEWYQQQPLKPPKYVMQLKKD---GSHSKGDGIP-DRFSGSS--SGADRYLSISNIQPEDEAIYICGVDD--------------------'},
                'rat': {'IGLV1-14*01': 'QAVVTQESA-LTTLPGGTVTLTCHSSTGAVT---TSNYANWIQEKADHLFTGIVGDT-------SNRAPGAP-ARFSGSL--LEGKAALTITGAQIEDEATYFCSLWY--------------------',
                'IGLV2-13*01': 'QFTLTQPKS-VSGSLRSTITIPCERSSGDI----GDSYVSWYQQHLGRPPINVIYAD-------DQRPSEVS-DRFSGSIDSSSNSASLTITNLQMDDEADYFCQSYD--------------------',
                'IGLV3-10*01': 'QFVLTQSNS-MSTSLGSTVKLSCKRSTGNI----GSSYVYWYQQHEGRSPTTMIYDD-------DKRPDGVP-DRFSGSIDSSSNSAFLTINNVQIEDEAIYFCQSYS--------------------',
                'IGLV3-11*01': 'QFVLTQPNS-VSTNLGSTVKLSCKRSTGNI----GSNYVNWYQQHEGRSPTTMIYRD-------DKRPDGVP-DRFSGSIDRSSNSALLTINNVQTEDEADYFCQSYS--------------------',
                'IGLV3-12*01': 'QAVLTQPNS-VSTSLGSTVKLSCTLSSGNI----ENNYVHWYQQYEGRSPTTMIYND-------DKRPDGVP-DRFSGSIDSSSNSAFLTINNVEIEDEAIYFCHSYV--------------------',
                'IGLV3-6*01': 'QVVLTQPNT-VSTSLGITVKLSCKCSSGNI----GSYSVHWYQQHEGRSPTTMIYKD-------DKRPDGVP-DRFSGSIDSSSNSAFLTINNVQTEDEAVYFCHSYD--------------------',
                'IGLV3-6*02': 'QVVLTQPNT-VSTSLGITVKLSCKRSTGNI----GSYSVHWYQQHERRSPTTMIYKY-------DKRPDGVP-DRFSGSTDSSSNSAFLTINNVQTEDEADYFCHSYD--------------------',
                'IGLV3-7*01': 'QVVLTQPKS-VSTSLESTVKLSCKLNSGNI----GSYYIHWYQQHEGRSPTTMIYRD-------DKRPDGVP-DRFSGSIDSSSNSAFLTINNVQTEDEAIYFCHSYD--------------------',
                'IGLV3-7*02': 'QVVLTQPKS-VSTSLESTVKLSCKLNSGNI----GSYYMHWYQQHEGRSPTNMIYRD-------DKRPDGVP-DRFSGSIDSSSNSAFLTINNVQTEDEAIYFCHSYD--------------------',
                'IGLV4-1*01': 'SYELIQPPS-ASVTLENTVSITCSGDELS------NKYAHWYQQKPDKTILEVMYKD-------SERPSGIS-DRFSGSS--SGTTAILTIRDAQAEDEADYYCLSTY--------------------',
                'IGLV4-1*02': 'SYELIQPPS-ASVTLENTVSITCSGDELS------NKYAHWYQQKPDKTILEVMYKD-------SERPSGIS-DRFSGSS--SGTTAILTIRDAQAEDEADYYCLSTY--------------------',
                'IGLV4-1-1*01': 'SYELIQPPS-ASVTLGNTVSITCSGDELP------KKYASWYQQKPDQSIVRVIYKD-------SERPSGIS-DWFSGSS--SGTTATLTIRDAQAEDEADYYCRSAY--------------------',
                'IGLV4-2*01': 'SYELIQPPS-ASVTLGNTVSITCSGDELP------KRYAYWYQQKPDKSIVRVIYKD-------SERPSGIS-DRFSGSS--SGTTATLTIRDTQAEDEADYYCHSTY--------------------',
                'IGLV4-3*01': 'SYELIQPPS-ASVTLGNTVSLTCVGDELP------KRYAYWYQQKPDQSIVRVIYED-------SKRPSGIS-DRFSGSS--SGTTATLTIRDAQAEDEADYYCHSTY--------------------',
                'IGLV4-3*02': 'SYELIQPPS-ASVTLGNTVSLTCVGDELP------KRYAYWYQQKPDQSIVRVIYED-------SKRPSGIS-DRFSGSS--SGTTATLTIRDAQNEDEADYYCHSTY--------------------',
                'IGLV4-8*01': 'SYELIQPPS-ASVTLGNTVSLTCVGDELS------KRYAQWYQQKPDKTIVSVIYKD-------SERPSGIS-DRFSGSS--SGTTATLTIHGTLAEDEADYYCLSTY--------------------',
                'IGLV4-9*01': 'SYELIQPPS-ASVTLGNTVSITCSGDELP------KKYAQWYQQKPDQSIVTVIYED-------SKRPSGIS-DRFSGSS--SGTTATLTIRDAQAEDEADYYCQSAY--------------------'},
                'rabbit': {'IGLV2S1*01': 'QPALTQPSS-AFGALGGSVTISCTGTSDDVG---YTNAVYWYRQLPGMSPTLLIYYD-------SKRPSGIP-ERFSGSK--SGNTASLTISWLQPEDEAAYYCSSYR--------------------',
                'IGLV2S2*01': 'QPALTQPSS-VSGALGGSVTITCAGSNSDIG---YNSLISWYQQLPGSVPKLLMFRV-------DRLASGIP-ERFSGSK--SGTTASLTISGLQPEDEADYYCVSYT--------------------',
                'IGLV3S2*01': 'SYELTQLPS-VSVSLGQTARITCGGNSIG------SKAVHWYQQKPGLAPGLLIYND-------DERPSGVP-DRFSGSN--SGDTATLTISGAQAGDEADYYCQLWD--------------------',
                'IGLV3S6*01': 'SHELTKLPS-VSVSLGQTARITCGGDSIE------EYSVHWYQKRPGQAPVLLIYRD-------SNRLSGIP-DHFSGSN--SGNTATLTISGAQAGDEADYYCQVWD--------------------',
                'IGLV3S9*01': 'SYELTQLPS-VSVSLGQTARITCGGDSIE------SYAVSWYQQKPGLAPVLLIYRD-------SKWPSGIP-DRFSGSN--SGNTATLTISRAQAGDEADYYCQVFN--------------------',
                'IGLV4S3*01': 'QPVLTQSPS-ASAALGSSAKLTCTLSSAHK-----TYYIEWYQQQQGEAPRYLMQLKSD---GSYTKGTGVP-DRFSGSS--SGADRYLIISSVQAEDEADYICGVTG--------------------',
                'IGLV4S4*01': 'QPVLTQSPS-VSAALGASAKLTCTLSSAHK-----TYTIDWYQQQQGEAPRYLMQLKSD---GSYTKGTGVP-DRFSGSS--SGADRYLIIPSVQADDEADYYCGADY--------------------',
                'IGLV5S1*01': 'QPVLTQPPS-LSASLGTTARLTCTLSTGYSV---GSLGVLWLQQVPGRPPRYLLTYHTE---EFKHQGSGVP-TRFSGSKDTSENSFVLSISGLQPEDEADYYCFTAH--------------------',
                'IGLV5S10*01': 'QPVLTQPPS-LSASLDTTARLTCTLSTGYSV---GEYPLVWLQQVPGRPPRYLLGYHTD---DIKHQGSGVP-SRFSGSKDDSANAGVLSISGLQPEDEADYYCAVG---------------------',
                'IGLV5S2*01': 'QPVLTQPPS-LSASLGTTARLTCTLSTGYSV---GEYPLVWLQQVPGRPPRYLLGYHTD---DIKHQGSGVH-SRFSGSKDTSENAGVLSISGLQPEDEADYYCATAH--------------------',
                'IGLV5S3*01': 'QPVLTQPPS-LSASLGTTARLTCTLSTGYSV---GKYPLVWLQQVPGRPPRYLLTYHTE---EFKHQGSGVH-SRFSGSKDTSENAGVLSISGLQPEDEADYYCVTAH--------------------',
                'IGLV5S5*01': 'QPVLTQPPS-LSASLDTTARLTCTLSTGYSV---GSLGVLWLQQVPGRPPRYLLAYHTD---DMKHQGSGVP-SRFSGSKDTSENSFVLSISGLQPEDEADYYCATAC--------------------',
                'IGLV5S6*01': 'QPVLTQPPS-LSASLGTTARLTCTLSTGYSV---GSLGVLWLQQVPGRAPRYLLSYNTD---EEKHQGSGVP-TRFSGSKDTSENSFVLSISGLQPEDEADYYCATAH--------------------',
                'IGLV5S9*01': 'QPVLTQPPS-LSASLDTTARLTCTLSTGYSV---GSYVIGWYQQVPGRPPRYLLTYHTE---EIKHQGSGVH-SRFSGSKDDSANAGVLSISGLQPEDEADYYCATAH--------------------',
                'IGLV6S1*01': 'SVVFTQPQA-VSGSLGETVSISCTRSSGNI----GANYVYWYQQHQGHAPSQLIYQY-------DKRPSGVP-DWFSDSKDSASNSASLTIAGLQPEDEADYYCLSGY--------------------',
                'IGLV6S3*01': 'QFVLTQPQS-VSGSLGQTVSISCNRDSGNI----EDYYVHWYQQHPGKAPTTVIYND-------DQRPSGVP-DRFSGSIDSTSNSASLTITGLLAEDEADYYCLSSD--------------------',
                'IGLV6S5*01': 'QFVLTQPQS-VSGSLGQTVSISCNRDSGNI----EDYYVHWYQQHPGKAPTTVIYND-------DQRPSGVP-DRFSGSIDSTSNSASLTIAGLQAEDEADYHCQSYD--------------------',
                'IGLV6S6*01': 'QFVLNQPQS-VSGSLGQTVSISCNRDSGNI----EEKYVHWYQQHPGKAPTTVIYSD-------DQRPSGVP-DRFSGSINSASNSASLTITGLLAEDEADYHCQSYD--------------------',
                'IGLV6S7*01': 'QFVLTQPQS-VSGSLGQTVSISCNRDSGNI----EDYYVHWYQQHPGKAPTTVIYND-------DQRPSGVP-DRFSGSIDSTSNSASLTITGLLAEDEADYYCLSSD--------------------'},
                'rhesus': {'IGLV1-60*01': 'QSVLTQPPS-ASEAARKSVTISCSGSSSNI----GSNSVSWYQQLPGTAPKLLIYYN-------DQRASGVS-DRFSGSK--SGTSASLAISGLQTEDEADYYCAAWD--------------------',
                'IGLV1-64*01': 'QSVLTQPPS-VSGAPGQRVTISCTGSSSNI----GGYYVSWYQQLPGTTPKLLIYQD-------NKRPSGVS-DRFSGSK--SGTSASLTITGLQTEDEADYYCLSYD--------------------',
                'IGLV1-64*02': 'QSVLTQPPS-VSGAPGQRVTISCTGSSSNI----GGYYVSWYQQLPGTTPKLLIYQD-------NKRPSGVS-DRFSGSK--SGTSASLTITGLQTEDEADYYCLSYD--------------------',
                'IGLV1-65*01': 'QSVLTQPPS-VSGDPGQRVTISCTGSSSNI----GGYYVYWYQQFPGTAPKLLIYDN-------NKRPSGVS-DRFSGSK--SGTSASLTITGLQPGDEADYYCGAWD--------------------',
                'IGLV1-66*01': 'QSVLTQPPS-VSGDPGQRVTISCTGSSSNI----GGYDVYWYQQLPGTAPKLLIYEN-------NKRPSGVS-DRFSGSK--SGTSASLTITGLQSEDEAEYYCETWD--------------------',
                'IGLV1-67*01': 'QSVLTQPPS-VSAAPGQRVTISCSGSSSNI----GRSYVSWYQQVPGTAPKLLIYQD-------NKRPSGVS-DRFSGSK--SGTSASLAITGLQTGDEADYYCSAWD--------------------',
                'IGLV1-67*02': 'QSVLTQPPS-VSAAPGQKVTISCSGSSSNI----GRSYVSWYQQVPGTAPKLLIYQD-------NKRPSGVS-DRFSGSK--SGTSASLAITGLQTGDEADYYCSAWD--------------------',
                'IGLV1-72*01': 'QSVLTQPPS-ASGAPGQSVTISCSGSSSNI----GSNYVYWYQQLSGKAPKLLIYNN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLQSKDEADYYCSAWD--------------------',
                'IGLV1-72*02': 'QSVLTQPPS-ASGAPGQSVTISCSGSSSNI----GSNYVYWYQQLSGKAPKLLIYNN-------NQRPSGVP-DRFSGSK--SGTSASLAISGLQSEDEADYYCAAWD--------------------',
                'IGLV1-77*01': 'QSVLTQPPS-ASGAPGQSVTISCSGSSSNI----RGNGVHWYQQLSGTAPKLLIYNN-------NQRPSGVP-DRFSGSK--SGTSASLAITGLRSEDEVDYYCEAWD--------------------',
                'IGLV1-77*02': 'QSVLTQPPS-ASGAPGQSVTISCSGSSSNI----RGNGVHWYQQLSGMAPKLLIYNN-------NQRPSGVP-DRFSGSK--SGTSASLAITGLQSEDEADYYCEAWD--------------------',
                'IGLV1-81*01': 'QSVLTQPPS-ASGAPGQSVTISCSGSSSNI----GSNYVYWYQQLPGTAPKLLIYYS-------NQRPSGVP-DRFSGSK--SGTSASLAITGLRSEDEADYYCAAWD--------------------',
                'IGLV1-85*01': 'QSVLTQPPS-VSGAPGQRVTISCTGSSSNI----GGYYVQWYQQLPGTAPKLLIYEN-------NKRPSGVS-DRFSGSQ--SGTSASLTITGLQSEDEADYYCQSYD--------------------',
                'IGLV1-86*01': 'QSVLTQPPS-VSAAPGQRVTISCSGSSFNF----RRYYVSWYQQLPGAAPKLLIYDV-------NKRPSGVS-DRFSGSQ--SGTSATLGISGLRPEDEADYYCSAWD--------------------',
                'IGLV1-86*02': 'QSVLTQPPS-VSAAPGQRVTISCSGSSFNF----RRYYVSWYQQLPGTAPKLLIYDV-------NKQPSGVS-DRFSGSQ--SGTSATLGISGLRPEDEADYYCSAWD--------------------',
                'IGLV10-114*01': 'QAGLTQPPS-VSKGLRQTATLTCTGNSNNV----GNQGAAWLQQHQGHPPKLLSYRN-------NNRPSGIS-ERFSASR--SGNTASLTITGLQPEDEADYYCSAWD--------------------',
                'IGLV11-117*01': 'QPVLTQPPS-LSASPGASARLPCTLSSDLSV---GSKNMYWYQQKPGSAPRLFLYYYSD---SDKQLGPGVP-NRVSGSKETSSNTAFLLISGLQPEDEADYYCQVYD--------------------',
                'IGLV1S1*01': 'QSVLTQPPS-VSGAPGQRVTISCTGSSSNIG---AGYYVQWYQQLPGTAPKLLIYEN-------NKRPSGVS-DRFSGSK--SGTSASLTITGLQSEDEADYYCQSYD--------------------',
                'IGLV1S2*01': 'QSVLTQPPS-VSGAPGQRVTISCTGSSSNIG---AGYGVQWYQQLPGTAPKLLIYEN-------NKRPSGVS-DRFSGSQ--SGTSASLTITGLQSEDEADYYCLSYD--------------------',
                'IGLV1S4*01': 'QSVLTQPPS-ASGAPGQSVTISCSGSSSNI----GGNNVYWYQQLPGTAPKLLIYYS-------NQRPSGVP-DRFSGSK--SGTSASLAITGLRSEDEADYYCAAWD--------------------',
                'IGLV1S6*01': 'QSVLTQPPS-VSAAPGQKVTISCSGSSSNI----GRSYVSWYQQVPGTAPKLLIYDN-------NKRPSGVS-DRFSGSK--SGTSASLAITGLQTGDEADYYCGAWD--------------------',
                'IGLV2-11*01': 'QSAPIQSPS-VSGSLGQSVTISCTGTSSDIG---RYNYVSWYRQQPGTTTKLMMYKV-------NMRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYE--------------------',
                'IGLV2-11*03': 'QSAPIQSPS-VSGSLGQSVTISCTGTSSDIG---RYNYVSWYRQQPGTTTKLMMYKV-------NMRPSGVS-DRFSGSK--SGNTASLTISGLQAEDKADYYCSSYE--------------------',
                'IGLV2-13*01': 'QAALTQSPS-VSGSPGQSVTISCTGTSSDIG---GYNRVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-13*02': 'QAALTQSPS-VSGSPGQSVTISCTGTSSDIG---GYNRVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-13*03': 'QAAPTQSPS-VSGSPGQSVTISCTGTSSDIG---GYNRVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-19*01': 'QAAPTQPPS-VSGSPGQSVTISCTGTSSDIG---YYNAVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-19*02': 'QAAPTQSPS-VSGSAGQSVTISCTGTSSDIG---YYNAVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-19*03': 'QAAPTQPPS-VSGSPGQSVTISCTGTSSDIG---YYNAVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-23*01': 'QAALTQPPS-VSGSPGQSVTISCTGTSSDIG---GYNYVSWYQQHPGKAPKLMIYDV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-23*02': 'QAALTQPPS-MSGSPGQSVTISCTGTSSDIG---GYNRVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-23*03': 'QAAPTQPPS-VSGSPGQSVTISCTGTSSDIG---GYNYVSWYQQHPGKAPKLMIYDV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-26*01': 'QAALTQPPS-VSKSLGQSVTISCAGTSSGIA---SYSDISWYQQHPGTDPRLLIYRV-------SNRPSGVS-DRFSGFK--SGSTTSLTISGLQAEDEAIYYCCSYR--------------------',
                'IGLV2-26*02': 'QAALTQPPS-VSKSLGQSVTISCAGTSSGIA---SYSDVSWYQQHPGTAPRLLIYRV-------SNRPSGVS-DRFSGFK--SGSTASLTISGLQAEDEAIYYCCSYR--------------------',
                'IGLV2-32*01': 'QAALTQPRS-VSGSPGQSVTISCTGTSSDIG---GYNYVSWYQQHPGTAPKLMIYAV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCCSYA--------------------',
                'IGLV2-32*02': 'QAALTQPRS-VSGSPGQSVTISCTGTSSDIG---GYNYVSWYQQHPGTAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2-38*01': 'QSALTQPPS-VSKSLGQSVTISCTGTSSDIG---GYNGVSWYQQHSGTAPRLLIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCGSYR--------------------',
                'IGLV2-46*01': 'QSAPTQPPS-VSGSPGQSVTISCTGTSSDIG---YYNAVSWYQQHPGTAPKLMIYGV-------SNRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCCSYT--------------------',
                'IGLV2S4*01': 'QAAPTQSPS-VSGSPGQSVTISCTGTSSDIG---GYNRVSWYQQHPGKAPKLMIYEV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCSSYA--------------------',
                'IGLV2S7*01': 'QSAPTQPPS-VSGSPGQSVTISCTGTSSDVG---GYNYVSWYQQHPGKAPKLMIYGV-------SNRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCCSYT--------------------',
                'IGLV2S9*01': 'QAALTQPPS-VSKSLGQSVTISCTGTSNDVG---GYNDVSWYQQHPGTAPRLLIYDV-------SKRPSGVS-DRFSGSK--SGNTASLTISGLQAEDEADYYCCSYR--------------------',
                'IGLV3-14*01': 'SYELTQPLS-VSVALGQMARITCGGNNIG------RKYVYWYQQKPDQAPVLVIYED-------SKRPSGIP-ERFAGSN--SGNTATLTIDGAQARDEADYYCQVWD--------------------',
                'IGLV3-16*02': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KKYAYWFQQKPGQSPVLIIYED-------SKRPSGIP-ERFSGSS--SGTVATLTISGAQVEDEADYYCYSTD--------------------',
                'IGLV3-22*01': 'SYELTQPPS-VSVSPGQTARITCSGEILA------KKYAQWFQQKPGQAPVLVIYKD-------SERPSGIP-ERFSSSS--SGTTVTLTISGAQAEDEADYYCQSAD--------------------',
                'IGLV3-25*01': 'SYELTQPPS-VSAASGQTARITCGGDNIG------SKYVHWYQQKPAQAPVQVIYAD-------SKRPSGIP-ERFSGSN--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-25*02': 'SYELTQPPS-VSAASGQTARITCGGDNIG------SKNVHWYQQKPAQAPVLVIYAD-------SKRPSGIP-ERFSGSN--SGNTATLTISRVEAGDEADYYCQVWD--------------------',
                'IGLV3-27*01': 'SSELTQDPA-VSVALGQTVRITCQGDSLR------SYYASWYQQKPGQAPVLVVYGN-------NNRPSGIP-ERFSGSS--SGNTASLTITGAQVEDEADYYCDSWD--------------------',
                'IGLV3-27*02': 'SSELTQDPA-VSVALGQTVRITCQGDSLR------SYYASWYQQKPGQAPVLVVYGN-------NNRPSGIP-ERFSGSS--SGNTASLTITGAQVEDEADYYCDSWD--------------------',
                'IGLV3-29*01': 'SYDVTQPRS-VSVSPGQTARITCGGDNIG------SKVVHWYQQKPAQAPVLVIYRD-------SKRPSGIP-ERFSGSN--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-30*01': 'SYELTQPPL-VSVSPGQTARITCSGDVLK------ENYADWYQQKPGQAPVLLIHED-------SKRPSGIP-ERFSGST--SGDTTTLTISSTLSEDEADYSCFSGN--------------------',
                'IGLV3-33*01': 'SSELTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*02': 'SSELTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*03': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*04': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*05': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNRGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*06': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*07': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-33*08': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-34*01': 'SYELTQPRS-VSVSPGQTARITCGGDNIG------SKSVQWYQQKPPQAPVLVIYAD-------SERPSGIP-ERFSGSN--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-36*01': 'SYELTQPPS-VSVSPGQMARITCGGDNLG------SKYVHWYQQKPAQAPVLVIYYD-------SDRPSGIP-ERFSGSK--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-36*02': 'SYELTQPPS-VSVSPGQTARITCGGDNLG------SKYVHWYQQKPAQAPVLVIYYD-------SDRPSGIP-ERFSGSK--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-39*01': 'SSELTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVVYGN-------NYRLSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-40*01': 'SYELTQPRS-VSVSPGQTARITCGGDNIG------SKSVQWYQQKPPQAPVLVIYAD-------SERPSGIP-ERFSGSN--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-43*01': 'SSGLTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3-44*01': 'SYDLTQPPS-VSVSPGQTARITCGGDNIG------SEAVHWYQQKPPQAPVQVIYSD-------SERPSGIP-ERFSGSK--SGNTATLTISGVEAGDEADYYCQVWD--------------------',
                'IGLV3-48*01': 'SYELTQPPS-VSVSPGQTARITCSGDALP------KNYAYWYQQKPGQVPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTVSGAQAEDEADYYCYSGD--------------------',
                'IGLV3S10*01': 'SSGLTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------TNRPSGIP-GRFSGSW--SGNTGSLTITGAQVEDEADYYCGSWD--------------------',
                'IGLV3S11*01': 'SSGLTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------TNRPSGIP-GRFSGSW--SGNTGSLTITGAQVEDEADYYCGSWD--------------------',
                'IGLV3S12*01': 'SSELTQKPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------TNRPSGIP-GRFSVSW--SGNTASLTITGAQVEDEADYYCGSWD--------------------',
                'IGLV3S13*01': 'SSELTQDPA-VSVALGQTVRITCQGDSLR------SYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNTGSLTITGAQVEDEADYYCGSWD--------------------',
                'IGLV3S14*01': 'SYELTQPPS-VSVSPGQTAKITCSGEILA------KKYARWFQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTVSGAQAEDEADYYCYSGD--------------------',
                'IGLV3S15*01': 'SYELTQPPS-VSVSPGQTAKITCSGEILA------KKYARWFQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTVSGAQAEDEADYYCYSGD--------------------',
                'IGLV3S16*01': 'SYELTQPPS-VSVSLGQTAKITCSGDVLA------KYYAHWFQQKPGQAPVLVIYKD-------SERPSGIP-ERFSGSS--SGTTVTLTISGAQAEDEADYYCYSGD--------------------',
                'IGLV3S3*01': 'SSELTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIS-ERFSGSS--SGNTASLTITGAQVEDEADYYCDSWD--------------------',
                'IGLV3S7*01': 'SSGLTQEPA-LSVALGHTVRMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------TNRPSGIP-GRFSGSW--SGNRGSLTITAAQVEDEADYYCNSWD--------------------',
                'IGLV3S9*01': 'SSGLTQEPA-LSVALGHTVSMTCQGDSLK------TYYASWYQQKPGQVPVLVIYGN-------NYRPSGIP-GRFSVSW--SGNRGSLTITAAQVEDEADYYCGSWD--------------------',
                'IGLV4-97*01': 'QPVLTQSPS-ASASLGASVKLTCTLSSGHS-----SYAIAWHQQQQGKAPRYLMRLNSV---GSHSKGDGIP-DRFSGSS--SGAERYLTISNLQSEDEADYYCQTWT--------------------',
                'IGLV4-97*02': 'QPVLTQSPS-ASASLGASVKLTCTLSSGHS-----SYAIAWHQQQQGKAPRYLMRLNSD---GSHSKGDGIP-DRFSGSS--SGAERYLTISNLQSEDEADYYCQTWD--------------------',
                'IGLV4-97*03': 'QPVLTQSPS-ASASLGASVKLTCTLSSGHS-----SYTIAWHQQQQGKAPRYLMWLKSD---GSHSKGDGIP-DRFSGSS--SGAERYLTISNLQSEDEADYYCQTWD--------------------',
                'IGLV4S1*01': 'QPVLTQSPS-ASASLGASIKLTCTLSSGHS-----SYTIAWHQQQQGKAPRYLMWLKSD---GSHSKGDGIP-DRFSGSS--SGAERYLTISNLQSEDEADYYCQTWD--------------------',
                'IGLV5-103*01': 'QPVLTQPPS-LSASQGASARLSCTLSSGFSA---DLYWIYWYQHKPGSPPRYLLSLYQN---SLHDLGSGVP-RRISGLMEDWSNKGLLLISDLQPEDEADYYCMIEH--------------------',
                'IGLV5-62*01': 'KPMLTQPAS-LSASPGASASLTCTFSGGINV---AGYHIFWYQQKPGSPPRYLLRYKSD---SDKGQGSGVP-SRFSGSKDASANTGILRISGLQSEDEADYYCAIGH--------------------',
                'IGLV5-62*02': 'KPMLTQPAS-LSASPGASASLTCTFSGGINV---AGYHIFWYQQKPGSPPRYLLRYKSD---SDKGQGSGVP-SRFSGSKDASANAGILRISGLQSEDEADYYCAIGR--------------------',
                'IGLV5-69*01': 'QPVLTQPTS-LSASPGASARLSCTLSSGINV---GSYSIFWYQQKPGSPPRYLLYYYSD---SSKHQGSGVP-SRFSGSKDASANAGLLLISGLQSEDEADYYCAIWH--------------------',
                'IGLV5-74*01': 'QPVLTQPTS-LSASPGASVRLTCTLRSGISV---GGYNIHWYQQKPRSPPRYLLYYYSD---SNKGQGSGVP-SRFSGSKDASANAGILLISGFQSEDEADYYCTTWH--------------------',
                'IGLV5-74*02': 'QPVLTQPTS-LSASPGASVRLTCTLRSGISV---GGYNIHWYQQKPGSPPRYLLYYYSD---SNKGQGSGVP-SRFSGSKDASANAGILLISGFQSEDEADYYCTTWH--------------------',
                'IGLV5-83*01': 'QPVLTQPTS-LSASPGASARLTCTLRSGISV---GSYRIFWYQQKPGSPPRYLLNYHTD---SDKHQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-83*02': 'QPVLTQPAS-LSASPGASARLTCTLSSGISV---GSYRIFWYQQKPGSPPRYLLNYHTD---SDKHQGSGVP-SRFSGSKDASANAGILLISGLQSEDEADYYCMIWH--------------------',
                'IGLV5-87*01': 'QPVLTQPAS-LSASPGASASLTCTFSGGTNV---GDYTIHWYQQKPGSPPRYLLKYKSD---SDKHQGSGVP-SRFSGSKDASANTGILRISGLQSEDEADYYCAIGR--------------------',
                'IGLV5-87*02': 'QPVLTQPAS-VSASPGASASLTCTFSGGTNV---GDYTIHWYQQKPGSPPRYLLKYKSD---SDKHQGSGVP-SRFSGSKDASANTGILRISGLQSEDEADYYCAIGR--------------------',
                'IGLV5-95*01': 'QPVLTQPSS-HSASPGAPARLTCTLSSGFSV---GDFWIRWYQQKPGSPPRYLLYYHSD---SDKHQGSGVP-SRFSGSNDASANAGILHISGLQPEDEADYYCCTWH--------------------',
                'IGLV5-99*01': 'QSVLTQPPS-LSASLEALARLTCTLSSGISV---GGKIVYWYQQKPGSNPRYLLSYYSE---SSKHQGSGVP-GRFSGSKDASTNSGILHVSGLQPEDEADYYCKIWH--------------------',
                'IGLV6-110*01': 'EVVFTQPHS-VSGSPGQTVTISCTRSSGSI----DSEYVQWYQQRPGNAPTTVIYKD-------NQRPSGVP-DRFSGSIDSSSNSASLAISGLKSEDEADYYCQSAD--------------------',
                'IGLV6-110*02': 'EVVFTQPHS-VSGSPGQTVTISCTRSSGSI----DSEYVQWYQQRPGNAPTTVIYKD-------NQRPSGVP-DRFSGSIDSSSNSASLAISGLKSEDEADYYCQSAD--------------------',
                'IGLV6-110*03': 'EVVFTQPHS-VSGSPGQTVTISCTRSSGSI----DSEYVQWYQQRPGNAPTTVIYKD-------NQRPSGVP-DRFSGSIDSSSNSASLAISGLKSEDEADYYCQSAD--------------------',
                'IGLV6-112*01': 'EVVFTQPHS-VSGSPGQTVTISCTRSSGSI----DSKYVQWYQQRPGSAPTTVIYKD-------NQRPSGVP-DRFSGSIDSSSNSASLTISGLKSEDEADYYCQSAD--------------------',
                'IGLV6-112*02': 'EVVFTQPHS-VSGSPGQTVTISCTRSSGSI----DSEYVQWYQQRPGSAPTTVIYKD-------NQRPSGVP-DRFSGSIDSSSNSASLAISGLKSEDEADYYCQSYD--------------------',
                'IGLV6-92*01': 'EVVFTQPHS-VSGSPGQTVTISCTHSSGSI----DNSYVYWYQQRPGSAPTTVIYND-------DQRPSGVP-DRSSGSIDSSSNSASLTISGLKSEDEADYYCQSYD--------------------',
                'IGLV7-71*01': 'QAVVTQEPS-LTVSPGGTVTLTCASSTGAVT---SGHSPHWCQQKPGQAPRTLIYNT-------SFKHSWTP-ARFSGSL--LGGKAALILSGAQPEDEAEYYCLLHY--------------------',
                'IGLV7-76*01': 'QAVVTQEPS-MTVSPGGTVTLTCASSTGAVT---SGHSPHWFQQKPGQAPKTLIYNT-------NYKHSWTP-ARFSGSL--LGGKAALTLSGAQPEDEAEYYCWLYY--------------------',
                'IGLV7-76*02': 'QAVVTQEPS-MTVSPGGTVTLTCASSTGAVT---SGHSPHWFQQKPGQAPKTLIYNT-------NYKHSWTP-ARFSGSL--LGGKAALTLSGAQPEDKAEYYCWLYY--------------------',
                'IGLV7-80*01': 'QAVVTQEPS-LTVSPGGTVTLTCASSTGAVT---SGHYPHWFQQKPGQAPKTLIYDT-------SNKLSWTP-ARFSGSL--AGGKAALTLSGAQPEDEAEYYCWLYY--------------------',
                'IGLV7-88*01': 'QAVVTQEPS-LTVSPGGTVTLTCGSSAGAVT---GSHYPYWFQQKPGQAPRTLIYDT-------SNKLSWTP-ARFSGSL--LGGKAALTLSGAQPEDEAEYYCWLHY--------------------',
                'IGLV8-125*01': 'ETVVTQEPS-LSVSPGGTVTLTCGLSSGSVS---TSNYPSWYQQTPGQAPRMLIYST-------NTRPSGVP-DRFSGSI--LGNKAALTITGAQADDESDYYCMLYM--------------------',
                'IGLV8-125*02': 'ETVVTQEPS-LSVYPGGTVTLTCGLSSGSVS---TSNYPSWYQQTPGQAPRTLIYST-------NTRPSGVP-DRFSGSI--LGNKAALTITGAQADDESDYYCTLYM--------------------'},
                'pig': {'IGLV2-6*01': 'QSALTQPPS-VSRNLKEMETISCAGTSSDI-----GGYVSWYQQHPGLAPKFLIYYV-------NTRASGIP-DGFCGSK--SGNTASLTISGLQAEDEADYYCSSPR--------------------',
                'IGLV5-14*01': 'QAVLTQPPS-LSASPGPSARLPCTLSSGSSV---GSYHISWYQRKPGRPPWYLLRFHFAS-SKDQGSGVPSC-FSGDKDA--SAHAGLLLISGLQPEDKADCDCLNWQ--------------------',
                'IGLV8-18*01': 'QTV-IQEPA-MSVSLGGTVTLTCAFSSGSVT---SSNYPGWFQQTPGQPPRTVIYST-------NSRPTGVP-SRFSGAI--SGNKATLTITGAQAEDEADYFCALYK--------------------',
                'IGLV8-19*01': 'QTV-IQEPA-MSVSLGGTVTLTCAFSSGSVT---SSNNPGWFQQTPGQPPRTVIYQT-------NNRPTGVP-SRFSGAI--SGNKATLTITGAQAEDEADYFCALGK--------------------',
                'IGLV8-19*02': 'QTV-IQEPA-MSVSLGGTVTLTCAFSSGSVT---SSNYPSWYQQTPGQPPRQLIYST-------NSRPTGVP-SRFSGAI--SGNKATLTITGAQAEDEADYFCALYK--------------------'},
                'cow': {'IGLV1-12*01': 'QAVLTQPPS-VSGSLGQTVTISCTGSSNNI----GILGVSWYQQIPGSAPRTLIYNS-------NKRPSGVP-DRFSGTK--SGNTGTLTIASLQAEDEADYYCASAD--------------------',
                'IGLV1-16*02': 'QDVLTQPSS-VSGSLGQNVSITCSGSSSNVG---YANYVSWHQQKQGSAPRTLIYGA-------TSRASGVP-DQFSGSK--SGNTATLTISSLQPEDEADYYCSSYD--------------------',
                'IGLV1-21*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSNNI----GSYGVGWYQQVPGSGLRTIIYGS-------SSRPSGVP-DRFSGSK--SGNTATLTISSLQAEDEADYFCATVD--------------------',
                'IGLV1-26*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSSNVG---YGNYVSWFQDIPGSAPRTLIYGD-------TSRASGVP-DRFSGSR--SGNTATLTISSLQAEDEADYFCASYQ--------------------',
                'IGLV1-31*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSSNVG---TGNYVSWFQQIPGSAPRTLIYGA-------TSRASGVP-DRFSGSR--SGNTATLTISSLQAEDEADYFCASYQ--------------------',
                'IGLV1-40*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSSNVG---LGNYVSWFQQIPGSAPRTLIYGA-------TSRASGVP-DRFSGSR--SGNTATLTISSLQAEDEADYFCASPD--------------------',
                'IGLV1-43*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSSNVG---YGNYVSWFQEIPGSAPRTLIYGD-------TSRASGVP-DRFSGSR--SGNTATLTISSLQAEDEADYFCASYQ--------------------',
                'IGLV1-47*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSSNV----GNGYVSWYQLIPGSAPRTLIYGD-------TSRASGVP-DRFSGSR--SGNTATLTISSLQAEDEADYFCASAE--------------------',
                'IGLV1-52*01': 'QAVLTQPSS-VSGSLGQRVSITCSGSSSNV----GNGYVSWYQLIPGSAPRTLIYGD-------TSRASGVP-DRFSGSR--SGNTATLTISSLQAEDEADYFCASAE--------------------',
                'IGLV1-67*02': 'QAVLTQPSS-VSGSLGQRVSITCSGSSNNI----GRYGVGWYQQVPGSGLRTIIYGS-------SSRPSGVP-DRFSGSK--SGNTATLTISSLQAEDEADYFCVAYD--------------------',
                'IGLV1-70*01': 'QAVLTQPPS-VSGSLGQRVTISCTGSSSNIG---GGNYVGWYQQIPGSAPKTLIYRS-------TSRPSGVP-DRFSGSR--SGNTATLTISSLQAEDEADYYCATYE--------------------',
                'IGLV1-71*01': 'QAVLTQLPS-VFRTLGQRVTISCTGSSNNI----GGYYVSWYQQLPGKAPRLLTYEI-------SKRPPGVP-DRVSGSK--SGNSASLT-SSVHAEDDTDYYCFSWA--------------------',
                'IGLV1-73*01': 'QAVLTQPPS-VSGSLGQRVTITCTGSSSYVS---RGNHVSWYQLIPGLAPKTLIYNS-------NKRPSGVP-DRFSGTK--SGNTGTLTIASLQAEDEADYYCASAD--------------------',
                'IGLV2-6*01': 'QSGLTQPSS-VSGNLGQTVTISCAGTSSDVG---AYNGVGWYQQLPGSAPKTLIYNL-------NKRSSGIP-ARFSGSK--SGNTATLTISGLQAEDEADYYCSSYK--------------------',
                'IGLV2-9*01': 'QSGLTQPSS-VSGNLGQTVITSCAGTSSYVG---SYNGVGWYQQLPGSAPKTLIYNV-------SKRPSGIP-DRFSGSK--SGNTATLTVSGLQAEDEADYYCSSYK--------------------',
                'IGLV3-2*01': 'SSQLTQPPA-VSVSLGQTASITCQGDDLE------LLSAHWYQQKPGQAPVLVIYAD-------DNLASGIP-DRFSGSK--SDTTATLTIRGAQAEDEADYYCQSAD--------------------',
                'IGLV3-3*01': 'SYELTQLTS-VSVALGQTAKITCSGELLD------EQYTQWYQQKPGQAPKLVIYKD-------SKRRSGIP-DQFSGSS--SGKTAILTISGVRAEDEADYYCLSWD--------------------',
                'IGLV3-4*01': 'SYELTQPTS-VSVALGQTAKITCSGDLLD------EQYTQWYQQKPGQGPVRVIYKD-------SERPSGIS-DRFSGSS--SGKTATLTISGAQTEDEADYYCQSAD--------------------',
                'IGLV3-5*01': 'SSQLTQPPA-VSVSLGQTASITCQGDDLE------SYYAHWYQQKPSQAPVLVIYES-------SERPSGIP-DRFSGSS--SGNTATLTISGAQTEDEADYYCQSYD--------------------',
                'IGLV5-72*01': 'QPVLTQPVT-VSASLGASARLSCTLSSGYNV---SNYSIYWYQQKAGNPLRYLLRFKSD---SDKHQGSGVP-SRFSGSKDASTNAGLLLISGLQPEDEADYYCAVWH--------------------',
                'IGLV8-38*01': 'QIV-IQEPS-LSVSPGGTVTLTCGLSSGSVT---TYNEPSWYQQTPGQAPRNVIYNT-------NTRTSGVP-DRFSASI--SGNKATLTITGAQPKDEADYHCLLYQ--------------------'}},
                'A': {'human': {'TRAV1-1*01': 'GQSLEQ-PSEVTAVEGAIVQINCTYQTSG------FYGLSWYQQHDGGAPTFLSYNAL----DGLEET-----GRFSSFLSRSDSYGYLLLQELQMKDSASYFCAVR---------------------',
                'TRAV1-2*01': 'GQNIDQ-PTEMTATEGAIVQINCTYQTSG------FNGLFWYQQHAGEAPTFLSYNVL----DGLEEK-----GRFSSFLSRSKGYSYLLLKELQMKDSASYLCAVR---------------------',
                'TRAV1-2*03': 'GQNIDQ-PTEMTATEGAIVQINCTYQTSG------FNGLFWYQQHAGEAPTFLSYNVL----DGLEEK-----GRFSSFLSRSKGYSYLLLKELQMKDSASYLCAVR---------------------',
                'TRAV10*01': 'KNQVEQSPQSLIILEGKNCTLQCNYTVSP------FSNLRWYKQDTGRGPVSLTIMTFS---ENTKSN-----GRYTATLDADTKQSSLHITASQLSDSASYICVVS---------------------',
                'TRAV10*02': 'KNQVEQSPQSLIILEGKNCTLQCNYTVSP------FSNLRWYKQDTGRGPVSLTIMTFS---ENTKSN-----GRYTATLDADTKQSSLHITASQLSDSASYICVVS---------------------',
                'TRAV12-1*01': 'RKEVEQDPGPFNVPEGATVAFNCTYSNSA------SQSFFWYRQDCRKEPKLLMSVYS----SGN-ED-----GRFTAQLNRASQYISLLIRDSKLSDSATYLCVVN---------------------',
                'TRAV12-1*03': 'RKEVEQDPGPFNVPEGATVAFNCTYSNSA------SQSFFWYRQDCRKEPKLLMSVYS----SGN-ED-----GRFTAQLNRASQYISLLIRDSKLSDSATYLCAVN---------------------',
                'TRAV12-2*01': 'QKEVEQNSGPLSVPEGAIASLNCTYSDRG------SQSFFWYRQYSGKSPELIMFIYS----NGDKED-----GRFTAQLNKASQYVSLLIRDSQPSDSATYLCAVN---------------------',
                'TRAV12-2*03': 'QKEVEQNSGPLSVPEGAIASLNCTYSDRV------SQSFFWYRQYSGKSPELIMSIYS----NGDKED-----GRFTAQLNKASQYVSLLIRDSQPSDSATYLCAVN---------------------',
                'TRAV12-2*04': 'QKEVEQNSGPLSVPEGAIASLNCTYSDQG------SQSFFWYRQYSGKSPELIMFIYS----NGDKED-----GRFTAQLNKASQYVSLLIRDSQPSDSATYLCAVN---------------------',
                'TRAV12-3*01': 'QKEVEQDPGPLSVPEGAIVSLNCTYSNSA------FQYFMWYRQYSRKGPELLMYTYS----SGNKED-----GRFTAQVDKSSKYISLFIRDSQPSDSATYLCAMS---------------------',
                'TRAV12-3*03': 'QKEVEKDPGPLSVPEGAIVSLNCTYSNSA------FQYFMWYRQYSRKGPELLMYTYS----SGNKED-----GRFTAQVDKSSKYISLFIRDSQPSDSATYLCAMS---------------------',
                'TRAV13-1*01': 'GENVEQHPSTLSVQEGDSAVIKCTYSDSA------SNYFPWYKQELGKGPQLIIDIRSN---VGEKKD-----QRIAVTLNKTAKHFSLHITETQPEDSAVYFCAAS---------------------',
                'TRAV13-1*02': 'GENVEQHPSTLSVQEGDSAVIKCTYSDSA------SNYFPWYKQELGKRPQLIIDIRSN---VGEKKD-----QRIAVTLNKTAKHFSLHITETQPEDSAVYFCAAS---------------------',
                'TRAV13-2*01': 'GESVGLHLPTLSVQEGDNSIINCAYSNSA------SDYFIWYKQESGKGPQFIIDIRSN---MDKRQG-----QRVTVLLNKTVKHLSLQIAATQPGDSAVYFCAEN---------------------',
                'TRAV14/DV4*01': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDP-----SYGLFWYKQPSSGEMIFLIYQGSY--DQQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV14/DV4*02': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDQ-----SYGLFWYKQPSSGEMIFLIYQGSY--DEQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV14/DV4*03': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDP-----SYGLFWYKQPSSGEMIFLIYQGSY--DQQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV14/DV4*05': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDQ-----SYGLFWYKQPSSGEMIFLIYQGSY--DEQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV16*01': 'AQRVTQPEKLLSVFKGAPVELKCNYSYSG------SPELFWYVQYSRQRLQLLLRHI------SRESI-----KGFTADLNKGETSFHLKKPFAQEEDSAMYYCALS---------------------',
                'TRAV17*01': 'SQQGEEDPQALSIQEGENATMNCSYKTSI-------NNLQWYRQNSGRGLVHLILIRSN---EREKHS-----GRLRVTLDTSKKSSSLLITASRAADTASYFCATD---------------------',
                'TRAV18*01': 'GDSVTQTEGPVTLPERAALTLNCTYQSSY------STFLFWYVQYLNKEPELLLKSSE----NQETDS-----RGFQASPIKSDSSFHLEKPSVQLSDSAVYYCALR---------------------',
                'TRAV19*01': 'AQKVTQAQTEISVVEKEDVTLDCVYETRDT-----TYYLFWYKQPPSGELVFLIRRNSF--DEQNEIS-----GRYSWNFQKSTSSFNFTITASQVVDSAVYFCALSE--------------------',
                'TRAV2*01': 'KDQVFQ-PSTVASSEGAVVEIFCNHSVSN------AYNFFWYLHFPGCAPRLLVKGS------KPSQQ-----GRYNMTYER--FSSSLLILQVREADAAVYYCAVE---------------------',
                'TRAV20*01': 'EDQVTQSPEALRLQEGESSSLNCSYTVSG------LRGLFWYRQDPGKGPEFLFTLYSA---GEEKEK-----ERLKATLTK--KESFLHITAPKPEDSATYLCAVQ---------------------',
                'TRAV20*02': 'EDQVTQSPEALRLQEGESSSLNCSYTVSG------LRGLFWYRQDPGKGPEFLFTLYSA---GEEKEK-----ERLKATLTK--KESFLHITAPKPEDSATYLCAVQ---------------------',
                'TRAV21*01': 'KQEVTQIPAALSVPEGENLVLNCSFTDSA------IYNLQWFRQDPGKGLTSLLLIQSS---QREQTS-----GRLNASLDKSSGRSTLYIAASQPGDSATYLCAVR---------------------',
                'TRAV21*03': 'KQEVTQIPAALSVPEGENLVLSCSFTDSA------IYNLQWFRQDPGKGLTSLLLIQSS---QREQTS-----GRLNASLDKSSGRSTLYIAASQPGDSATYLCAVR---------------------',
                'TRAV22*01': 'GIQVEQSPPDLILQEGANSTLRCNFSDSV-------NNLQWFHQNPWGQLINLFYIPS-----GTKQN-----GRLSATTVATERYSLLYISSSQTTDSGVYFCAVE---------------------',
                'TRAV23/DV6*01': 'QQQVKQSPQSLIVQKGGISIINCAYENTA------FDYFPWYQQFPGKGPALLIAIRPD---VSEKKE-----GRFTISFNKSAKQFSLHIMDSQPGDSATYFCAAS---------------------',
                'TRAV23/DV6*02': 'QQQVKQSPQSLIVQKGGIPIINCAYENTA------FDYFPWYQQFPGKGPALLIAIRPD---VSEKKE-----GRFTISFNKSAKQFSLHIMDSQPGDSATYFCAAS---------------------',
                'TRAV23/DV6*05': 'QQQVKQSPQSLIVQKGGISIINCAYENTA------FDYFPWYQQFPGKGPALLIAIRPD---VSEKKE-----GRFTISFNKSAKQFSSHIMDSQPGDSATYFCAAS---------------------',
                'TRAV24*01': 'ILNVEQSPQSLHVQEGDSTNFTCSFPSSN------FYALHWYRWETAKSPEALFVMTLN---GDEKKK-----GRISATLNTKEGYSYLYIKGSQPEDSATYLCAF----------------------',
                'TRAV25*01': 'GQQVMQIPQYQHVQEGEDFTTYCNSSTTL-------SNIQWYKQRPGGHPVFLIQLVKS---GEVKKQ-----KRLTFQFGEAKKNSSLHITATQTTDVGTYFCAG----------------------',
                'TRAV25*02': 'GQQVMQIPQYQHVQEGEDFTTYCNSSTTL-------SNIQWYKQRPGGHPVFLIQLVKS---GEVKKQ-----KRLTFQFGEAKKNSSLHITATQTTDVGTYFCAG----------------------',
                'TRAV26-1*01': 'DAKTTQ-PPSMDCAEGRAANLPCNHSTISG-----NEYVYWYRQIHSQGPQYIIHGLK-----NNETN-----EMASLIITEDRKSSTLILPHATLRDTAVYYCIVRV--------------------',
                'TRAV26-1*02': 'DAKTTQ-PTSMDCAEGRAANLPCNHSTISG-----NEYVYWYRQIHSQGPQYIIHGLK-----NNETN-----EMASLIITEDRKSSTLILPHATLRDTAVYYCIVRV--------------------',
                'TRAV26-2*01': 'DAKTTQ-PNSMESNEEEPVHLPCNHSTISG-----TDYIHWYRQLPSQGPEYVIHGLT-----SNVNN-----RMASLAIAEDRKSSTLILHRATLRDAAVYYCILRD--------------------',
                'TRAV27*01': 'TQLLEQSPQFLSIQEGENLTVYCNSSSVF-------SSLQWYRQEPGEGPVLLVTVVTG---GEVKKL-----KRLTFQFGDARKDSSLHITAAQPGDTGLYLCAG----------------------',
                'TRAV27*03': 'TQLLEQSPQFLSIQEGENLTVYCNSSSVF-------SSLQWYRQEPGEGPVLLVTVVTG---GEVKKL-----KRLTFQFGDARKDSSLHITAAQTGDTGLYLCAG----------------------',
                'TRAV27*04': 'TQLLEQSPQFLSIQEGENLTVYCNSSSVF-------SSLQWYRQEPGEGPVLLVTVVTG---GEVKKL-----KRLTFQFGDARKDSSLHITAAQPGDTGLYLCAG----------------------',
                'TRAV29/DV5*01': 'DQQVKQNSPSLSVQEGRISILNCDYTNSM------FDYFLWYKKYPAEGPTFLISISSI---KDKNED-----GRFTVFLNKSAKHLSLHIVPSQPGDSAVYFCAAS---------------------',
                'TRAV29/DV5*02': 'DQQVKQNSPSLSVQEGRISILNCDYTNSM------FDYFLWYKKYPAEGPTFLISISSI---KDKNED-----GRFTVFLNKSAKHLSLDIVPSQPGDSAVYFCAAS---------------------',
                'TRAV29/DV5*04': 'DQQVKQNSPSLSVQEGRISILNCDYTNSM------FDYFLWYKKYPAEGPTFLISISSI---KDKNED-----GRFTVFLNKSAKHLSLHIVPSQPGDSAVYFCAAS---------------------',
                'TRAV3*01': 'AQSVAQPEDQVNVAEGNPLTVKCTYSVSG------NPYLFWYVQYPNRGLQFLLKYITG--DNLVKGS-----YGFEAEFNKSQTSFHLKKPSALVSDSALYFCAVRD--------------------',
                'TRAV30*01': 'QQPV-QSPQAVILREGEDAVINCSSSKAL-------YSVHWYRQKHGEAPVFLMILLKG---GEQKGH-----EKISASFNEKKQQSSLYLTASQLSYSGTYFCGTE---------------------',
                'TRAV30*05': 'QQPV-QSPQAVILREGEDAVINCSSSKAL-------YSVHWYRQKHGEAPVFLMILLKG---GEQKGH-----DKISASFNEKKQQSSLYLTASQLSYSGTYFCGTE---------------------',
                'TRAV30*06': 'QQPV-QSPQAVILREGEDAVINCSSSKAL-------YSVHWYRQKHGEAPIFLMILLKG---GEQKGH-----DKISASFNEKKQQSSLYLTASQLSYSGTYFCGTE---------------------',
                'TRAV30*07': 'QQPV-QSPQAVILREGEDAVINCSSSKAL-------YSVHWYRQKHGEAPVFLMILLKG---GEQKGH-----DKISASFNEKKQQSSLYLTASQLSYSGTYFCGTE---------------------',
                'TRAV30*08': 'QQPV-QSPQAVILREGEDAVINCSSSKAL-------YSVHWYRQKHGEAPVFLMILLKG---GEQMRH-----EKISASFNEKKQQSSLYLTASQLSYSGTYFCGTE---------------------',
                'TRAV34*01': 'SQELEQSPQSLIVQEGKNLTINCTSSKTL-------YGLYWYKQKYGEGLIFLMMLQKG---GEEKSH-----EKITAKLDEKKQQSSLHITASQPSHAGIYLCGAD---------------------',
                'TRAV35*01': 'GQQLNQSPQSMFIQEGEDVSMNCTSSSIF-------NTWLWYKQEPGEGPVLLIALYKA---GELTSN-----GRLTAQFGITRKDSFLNISASIPSDVGIYFCAGQ---------------------',
                'TRAV35*02': 'GQQLNQSPQSMFIQEGEDVSMNCTSSSIF-------NTWLWYKQDPGEGPVLLIALYKA---GELTSN-----GRLTAQFGITRKDSFLNISASIPSDVGIYFCAGQ---------------------',
                'TRAV36/DV7*01': 'EDKVVQSPLSLVVHEGDTVTLNCSYEVTN------FRSLLWYKQEKKAP-TFLFMLTSS---GIEKKS-----GRLSSILDKKELSSILNITATQTGDSAIYLCAVE---------------------',
                'TRAV36/DV7*05': 'EDKVVQSPLSLVVHEGDTVTLNCSYEVTN------FRSLLWYKQEKKAP-TFLFMLTSS---GIEKKS-----GRLSSILDKKELFSILNITATQTGDSAIYLCAVE---------------------',
                'TRAV38-1*01': 'AQTVTQSQPEMSVQEAETVTLSCTYDTSEN-----NYYLFWYKQPPSRQMILVIRQEAY--KQQNATE-----NRFSVNFQKAAKSFSLKISDSQLGDTAMYFCAFMK--------------------',
                'TRAV38-1*03': 'AQTVTQSQPEMSVQEAETVTLSCTYDTSES-----NYYLFWYKQPPSRQMILVIRQEAY--KQQNATE-----NRFSVNFQKAAKSFSLKISDSQLGDTAMYFCAFMK--------------------',
                'TRAV38-2/DV8*01': 'AQTVTQSQPEMSVQEAETVTLSCTYDTSES-----DYYLFWYKQPPSRQMILVIRQEAY--KQQNATE-----NRFSVNFQKAAKSFSLKISDSQLGDAAMYFCAYRS--------------------',
                'TRAV39*01': 'ELKVEQNPLFLSMQEGKNYTIYCNYSTTS-------DRLYWYRQDPGKSLESLFVLLSN---GAVKQE-----GRLMASLDTKARLSTLHITAAVHDLSATYFCAVD---------------------',
                'TRAV4*01': 'LAKTTQ-PISMDSYEGQEVNITCSHNNIAT-----NDYITWYQQFPSQGPRFIIQGYK-----TKVTN-----EVASLFIPADRKSSTLSLPRVSLSDTAVYYCLVGD--------------------',
                'TRAV40*01': 'SNSVKQT-GQITVSEGASVTMNCTYTSTG------YPTLFWYVEYPSKPLQLLQRET------MENSK-----NFGGGNIKD--KNSPIVKYSVQVSDSAVYYCLLG---------------------',
                'TRAV41*01': 'KNEVEQSPQNLTAQEGEFITINCSYSVGI-------SALHWLQQHPGGGIVSLFMLSS-----GKKKH-----GRLIATINIQEKHSSLHITASHPRDSAVYICAVR---------------------',
                'TRAV5*01': 'GEDVEQS-LFLSVREGDSSVINCTYTDSS------STYLYWYKQEPGAGLQLLTYIFSN---MDMKQD-----QRLTVLLNKKDKHLSLRIADTQTGDSAIYFCAES---------------------',
                'TRAV6*01': 'SQKIEQNSEALNIQEGKTATLTCNYTNYS------PAYLQWYRQDPGRGPVFLLLIREN---EKEKRK-----ERLKVTFDTTLKQSLFHITASQPADSATYLCALD---------------------',
                'TRAV6*02': 'SQKIEQNSEALNIQEGKTATLTCNYTNYS------PAYLQWYRQDPGRGPVFLLLIREN---EKEKRK-----ERLKVTFDTTLKQSLFHITASQPADSATYLCALD---------------------',
                'TRAV6*07': 'SQKIEQNSEALNIQEGKTATLTCNYTNYS------PAYLQWYRQDPGRGPVFLLLIREN---EKEKRK-----ERLKVTFDTTLKQSLFHITASQPADSATYLCALD---------------------',
                'TRAV7*01': 'ENQVEHSPHFLGPQQGDVASMSCTYSVSR------FNNLQWYRQNTGMGPKHLLSMYSA---GYEKQK-----GRLNATLLK--NGSSLYITAVQPEDSATYFCAVD---------------------',
                'TRAV8-1*01': 'AQSVSQHNHHVILSEAASLELGCNYSYGG------TVNLFWYVQYPGQHLQLLLKYFSG--DPLVKGI-----KGFEAEFIKSKFSFNLRKPSVQWSDTAEYFCAVN---------------------',
                'TRAV8-2*01': 'AQSVTQLDSHVSVSEGTPVLLRCNYSSSY------SPSLFWYVQHPNKGLQLLLKYTSA--ATLVKGI-----NGFEAEFKKSETSFHLTKPSAHMSDAAEYFCVVS---------------------',
                'TRAV8-2*03': 'AQSVTQLDSHVSVSEGTPVLLRCNYSSSY------SPSLFWYVQHPNKGLQLLLKYTSA--ATLVKGI-----NGFEAEFKKSETSFHLTKPSAHMSDAAEYFCVVS---------------------',
                'TRAV8-3*01': 'AQSVTQPDIHITVSEGASLELRCNYSYGA------TPYLFWYVQSPGQGLQLLLKYFSG--DTLVQGI-----KGFEAEFKRSQSSFNLRKPSVHWSDAAEYFCAVG---------------------',
                'TRAV8-3*02': 'AQSVTQPDIHITVSEGASLELRCNYSYGA------TPYLFWYVQSPGQGLQLLLKYFSG--DTLVQGI-----KGFEAEFKRSQSSFNLRKPSVHWSDAAEYFCAVG---------------------',
                'TRAV8-4*01': 'AQSVTQLGSHVSVSEGALVLLRCNYSSSV------PPYLFWYVQYPNQGLQLLLKYTSA--ATLVKGI-----NGFEAEFKKSETSFHLTKPSAHMSDAAEYFCAVS---------------------',
                'TRAV8-4*08': 'AQSVTQLGSHVSVSEGALVLLRCNYSSSV------PPYLFWYVQYPNQGLQLLLKYTSA--ATLVKGI-----NGFEAEFKKSETSFHLTKPSAHMSDAAEYFCAVS---------------------',
                'TRAV8-4*09': 'AQSVTQLGSHVSVSEGALVLLRCNYSSSV------PPYLFWYVQYPNQGLQLLLKYTSA--ATLVKGI-----NGFEAEFKKSETSFHLTKPSAHMSDAAEYFCAVS---------------------',
                'TRAV8-6*01': 'AQSVTQLDSQVPVFEEAPVELRCNYSSSV------SVYLFWYVQYPNQGLQLLLKYLSG--STLVESI-----NGFEAEFNKSQTSFHLRKPSVHISDTAEYFCAVS---------------------',
                'TRAV8-6*02': 'AQSVTQLDSQVPVFEEAPVELRCNYSSSV------SVYLFWYVQYPNQGLQLLLKYLSG--STLVKGI-----NGFEAEFNKSQTSFHLRKPSVHISDTAEYFCAVS---------------------',
                'TRAV9-1*01': 'GDSVVQTEGQVLPSEGDSLIVNCSYETTQ------YPSLFWYVQYPGEGPQLHLKAMKA---NDKGRN-----KGFEAMYRKETTSFHLEKDSVQESDSAVYFCALS---------------------',
                'TRAV9-1*02': 'GDSVVQTEGQVLPSEGDSLIVNCSYETTQ------YPSLFWFVQYPGEGPQLHLKAMKA---NDKGRN-----KGFEAMYRKETTSFHLEKDSVQESDSAVYFCALS---------------------',
                'TRAV9-2*01': 'GNSVTQMEGPVTLSEEAFLTINCTYTATG------YPSLFWYVQYPGEGLQLLLKATKA---DDKGSN-----KGFEATYRKETTSFHLEKGSVQVSDSAVYFCALS---------------------',
                'TRAV9-2*02': 'GDSVTQMEGPVTLSEEAFLTINCTYTATG------YPSLFWYVQYPGEGLQLLLKATKA---DDKGSN-----KGFEATYRKETTSFHLEKGSVQVSDSAVYFCALS---------------------',
                'TRAV9-2*03': 'GDSVTQMEGPVTLSEEAFLTINCTYTATG------YPSLFWYVQYPGEGLQLLLKATKA---DDKGSN-----KGFEATYRKETTSFHLEKGSVQVSDSAVYFCALS---------------------'},
                'mouse': {'TRAV1*01': 'GQGVEQ-PDNLMSVEGTFARVNCTYSTSG------FNGLSWYQQREGHAPVFLSYVVL----DGLKDS-----GHFSTFLSRSNGYSYLLLTELQIKDSASYLCAVR---------------------',
                'TRAV1*02': 'GQGVEQ-PAKLMSVEGTFARVNCTYSTSG------FNGLSWYQQREGQAPVFLSYVVL----DGLKDS-----GHFSTFLSRSNGYSYLLLTELQIKDSASYLCAVR---------------------',
                'TRAV10*01': 'GEKVEQHESTLSVREGDSAVINCTYTDTA------SSYFPWYKQEAGKGLHFVIDIRSN---VDRKQS-----QRLIVLLDKKAKRFSLHITATQPEDSAIYFCAAS---------------------',
                'TRAV10*02': 'GEKVEQHESTLSVREGDSAVINCTYTDTA------SSYFPWYKQEAGKSLHFVIDIRSN---VDRKQS-----QRLIVLLDKKAKRFSLHITATQPEDSAIYFCAAS---------------------',
                'TRAV10D*01': 'GEKVEQHQSTLSVREGDSAVINCTYTDTA------SSYFPWYKQEAGKSLHFVIDIRSN---VDRKQS-----QRLTVLLDKKAKRFSLHITATQPEDSAIYFCAAS---------------------',
                'TRAV10D*03': 'GEKVEQHESTLSVREGDSAVINCTYTDTA------SSYFPWYKQEAGKSLHFVIDIRSN---VDRKQS-----QRLIVLLDKKAKRFSLHITATQPEDSAIYFCAAS---------------------',
                'TRAV10N*01': 'GEKVEQHESTLSVREGDSAVINCTYTDTA------SSYFPWYKQEAGKSLHFVIDIRSN---VDRKQS-----QRLTVLLDKKAKRFSLHITATQPEDSAIYFCAAS---------------------',
                'TRAV11*01': 'KTQVEQSPQSLVVRQGENCVLQCNYSVTP------DNHLRWFKQDTGKGLVSLTVLVHE---NDKTSN-----GRYSATLDKDAKHSTLHITATLLDDTATYICVVG---------------------',
                'TRAV11*02': 'KTQVEQSPQSLVVRQGENCVLQCNYSVTP------DNHLRWFKQDTGKGLVSLTVLVDQ---KDKTSN-----GRYSATLDKDAKHSTLHITATLLDDTATYICVVG---------------------',
                'TRAV11D*01': 'KTQVEQSPQSLVVRQGENCVLQCNYSVTP------DNHLRWFKQDTGKGLVSLTVLVHE---NDKTSN-----GRYSATLDKDAKHSTLHITATLLDDTATYICVVG---------------------',
                'TRAV12-1*01': 'GDSVTQTEGLVTVTEGLPVMLNCTYQTAYS-----DVAFFWYVQYLNEAPKLLLRSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSVQLSDSALYYCALS---------------------',
                'TRAV12-1*02': 'GDSVTQTEGLVTVTEGLPVMLNCTYQTTYS-----DVAFFWYVQYLNEAPKLLLRSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSYAQLSDSGLYYCALS---------------------',
                'TRAV12-1*06': 'GDSVTQTEGLVTVTEGLPVKLNCTYQTTYL-----TIAFFWYVQYLNEAPQVLLKSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12-2*01': 'GDSVTQTEGLVTLTEGLPVMLNCTYQTAY------STFLFWYVQHLNEAPKLLLKSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12-2*02': 'GDSVTQTEGLVTLTEGLPVMLNCTYQSTY------SPFLFWYVQHLNEAPKLLLKSFTD---NKRPEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12-3*01': 'GDSVTQTEGLVTLTEGLPVMLNCTYQTIYS-----NPFLFWYVHYLNESPRLLLKSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12-3*05': 'GDSVTQKEGLVTLTEGLPVMLNCTYQTIYS-----NAFLFWYVHYLNESPRLLLKSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12D-1*01': 'GDSVTQTEGLVTVTEGLPVKLNCTYQTTYL-----TIAFFWYVQYLNEAPQVLLKSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12D-1*02': 'GDSVTQTEGLVTVTEGLPVKLNCTYQTTYL-----TIAFFWYVQYLNEAPQVLLRSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12D-1*06': 'GDSVTQTEGLVTVTEGLPVMLNCTYQTAYS-----DVAFFWYVQYLNEAPKLLLRSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSVQLSDSALYYCALS---------------------',
                'TRAV12D-2*01': 'GDSVTQTEGLVTLTEGLPVMLNCTYQSTY------SPFLFWYVQHLNEAPKLLLKSFTD---NKRPEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12D-2*02': 'GDSVTQTEGLVTLTKGLPVMLNCTYQTTY------SPFLFWYVQHLNEAPKLLLKSSTD---NKRTEH-----QGFYATLHKSSSSFHLQKSSVQLSDSALYFCALS---------------------',
                'TRAV12D-3*02': 'GDSVTQTEGLVTLTEGLPVMLNCTYQTIYS-----NAFLFWYVHYLNESPWLLLRSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12D-3*04': 'GDSVTQTEGLVTLTEGLPVMLNCTYQTIYS-----NPFLFWYVQHLNESPRLLLKSFTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV12N-1*01': 'GDSVTQTEGLVTVTEGLPVMLNCTYQTAYS-----DVAFFWYVQYLNEAPKLLLRSSTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSVQLSDSALYYCALS---------------------',
                'TRAV12N-2*01': 'GDSVTQTEGLVTLTKGLPVMLNCTYQTTY------SPFLFWYVQHLNEAPKLLLKSSTD---NKRTEH-----QGFYATLHKSSSSFHLQKSSVQLSDSALYFCALS---------------------',
                'TRAV12N-3*01': 'GDSVTQTEGLVTLTEGLPVMLNCTYQTIYS-----NPFLFWYVQHLNESPRLLLKSFTD---NKRTEH-----QGFHATLHKSSSSFHLQKSSAQLSDSALYYCALS---------------------',
                'TRAV13-1*01': 'GQQVQQSPASLVLQEGENAELQCNFSTSL-------NSMQWFYQRPEGSLVSLFYNPS-----GTKQS-----GRLTSTTVIKERRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13-1*02': 'GQQVQQSPASLVLQEGENAELQCNFSTSL-------NSMQWFYQRPGGSLVSLFYNPS-----GTKHS-----GRLTSTTVIKERRSSLHISSSQTTDSGTYLCALE---------------------',
                'TRAV13-2*01': 'GQQVQQSPSSLVLQEGENAELQCNFSSTA-------TQLQWFYQSPGGSLVSLLSNPS-----GTKHT-----GRLTSTTVTKERRSSLHISSSQTTDSGTYLCAID---------------------',
                'TRAV13-2*03': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TRLQWFYQHPGGRLVSLFYNPS-----GTKHT-----GRLTSTTVTNERRSSLHISSSQTTDSGTYFCAID---------------------',
                'TRAV13-3*01': 'GQQVQQSPASLVLQEGENAELQCTYSTTL-------NSMQWFYQRPGGRLVSLLYSPS----WAEQRG-----GRLTSSAASNESRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13-4/DV7*01': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TQLQWFYQRPGGSLVSLLYNPS-----GTKHT-----GRLTSTTVTKERRSSLHISSSQITDSGTYFCAME---------------------',
                'TRAV13-4/DV7*02': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TRLQWFYQRPGGSLVSLLSNPS-----GTKHT-----GRLTSTTVTKERRGSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13-5*01': 'GQQVQQSPASLVLQEGENAELQCSFSIST-------NQVQWFYQRPGGRLIGLSYIP------GMKPT-----GKQTSSTVTKGRHSSLTISSSQTTDSGTYFCVLS---------------------',
                'TRAV13D-1*01': 'GQQVQQSPASLVLQEGENAELQCNFSTSL-------NSMQWFYQRPGGSLVSLFYNPS-----GTKQS-----GRLTSTTVIKERRSSLHISSSQTTDSGTYLCAME---------------------',
                'TRAV13D-1*04': 'GQQVQQSPTSLVLQEGENAELQCNFSTSL-------NSMQWFYQRPGGSLVSVFYNPS-----GTKQS-----GRLTSTTVIKERRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13D-2*01': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TRLQWFYQHPGGRLVSLFYNPS-----GTKHT-----GRLTSTTVTNERRGSLHISSSQTTDSGTYFCAID---------------------',
                'TRAV13D-2*03': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TQLQWFYQSPGGSLVSLLSNPS-----GTKHT-----GRLTSTTVTKERRSSLHISSSQTTDSGTYLCAID---------------------',
                'TRAV13D-3*01': 'GQQVEQSPASLVLQEGENAELQCTYSTTL-------NSMQWFYQRPGGRLVSLLYSPS----WAEQRG-----GRLTSSAASNESRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13D-3*02': 'GQQVQQSPASLVLQEGENAELQCTYSTTL-------NSMQWFYQRPGGRLVSLLYSPS----WAEQRG-----GRLTSSAASNESRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13D-4*01': 'EQQVQQSPASLVLQEGENAELQCSFSIFT-------NQVQWFYQRPGGRLVSLLYNPS-----GTKQS-----GRLTSTTVIKERRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13D-4*03': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TRLQWFYQRPGGSLVSLLYNPS-----GTKHT-----GRLTSTTVTKERRSSLHISSSQTTDSGTYFCAME---------------------',
                'TRAV13N-1*01': 'GQQVQQSPTSLVLQEGENAELQCNFSTSL-------NSMQWFYQRPGGSLISVFYNPS-----GTKQS-----GRLTSTTVIKERRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13N-2*01': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TQLQWFYQSPGGSLVSLLSNPS-----GTKHT-----GRLTSTTVTKERRSSLHISSSQTTDSGTYLCAID---------------------',
                'TRAV13N-3*01': 'GQQVQQSPASLVLQEGENAELQCTYSTTL-------NSMQWFYQRPGGRLVSLLYSPS----WAEQRG-----GRLTSSAASNESRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV13N-4*01': 'EQQVQQSPASLVLQEAENAELQCSFSIFT-------NQVQWFYQRPGGRLVSLLYNPS-----GTKQS-----GRLTSTTVIKERRSSLHISSSQITDSGTYLCAME---------------------',
                'TRAV14-1*01': 'QQQVRQSPQSLTVWEGGTTVLTCSYEDST------FNYFPWYQQFPGEGPALLISILSV---SDKKED-----GRFTTFFNKREKKLSLHIIDSQPGDSATYFCAAS---------------------',
                'TRAV14-1*02': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FNYFPWYQQFPGEGPALLISISSV---SDKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14-2*01': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FDYFPWYWQFPRESPALLIAIRPV---SNKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14-2*02': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FDYFPWYHQFPGESPALLIAIRPV---SNKKED-----GRFTIFFNKREKKFSLHIADSQPGDSATYFCAAS---------------------',
                'TRAV14-2*03': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FDYFPWYRLFPGESPALLIAIRPV---SNKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14-3*01': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLISILSV---SDKKED-----GRFTIFFNKREKKLSLHIADSQPGDSATYFCAAS---------------------',
                'TRAV14-3*02': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAILSV---SNKKED-----GRFTIFFNKREKKLSLHIADSQPGDSATYFCAAS---------------------',
                'TRAV14-3*04': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14-3*05': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLISILSV---SNKKED-----GRFTIFFNKREKKLSLHIADSQPGDSATYFCAAS---------------------',
                'TRAV14D-1*01': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FNYFPWYQQFPGEGPALLISIRSV---SDKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14D-2*01': 'QQQVRQSPQSLTVWEGETTILNCSYEDST------FDYFPWYRQFPGKSPALLIAISLV---SNKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14D-2*03': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FDYFPWYWQFPRESPALLIAIRPV---SNKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*01': 'QQQVRQSSQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GRFTIFFNKREKNLSLHIKDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*02': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*03': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GGFTIFFNKREKNLSLHIKDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*08': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLISILSV---SDKKED-----GRFTIFFNKREKKLSLHIADSQPGDSATYFCAAS---------------------',
                'TRAV14N-1*01': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FNYFPWYQQFPGEGPALLISIRSV---SDKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14N-2*01': 'QQQVRQSPQSLTVWEGETAILNCSYEDST------FDYFPWYWQFPRESPALLIAIRPV---SNKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14N-3*01': 'QQQVRQSSQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GRFTIFFNKREKNLSLHIKDSQPGDSATYFCAAS---------------------',
                'TRAV15-1/DV6-1*01': 'AQKVIQVWSTTSRQEGEKLTLDCSYKTSQV-----LYHLFWYKHLLSGEMVLLIRQMPS--TIAIERS-----GRYSVVFQKSRKSISLVISTLQPDDSGKYFCALWE--------------------',
                'TRAV15-2/DV6-2*01': 'AQKVTQVQSTGSSQWG-EVTLHCSYETSEY-----FYVILWYKQLFSGEMVFLIYQTSF--DTQNQRN-----SRYSVVFQKSLKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15-2/DV6-2*02': 'AQRVTQVQSTGSSQWG-EVTLDCSYETSEY-----SYLILWYRQLFSGEMVFLIYQPSF--DTQNQRS-----GHYSVVFQKSFKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15D-1/DV6D-1*01': 'AQKVIQVWSTPSRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQMSS--STAKERS-----GRYSVVFQKSLKSISLVISALQPDDSGKYFCALWE--------------------',
                'TRAV15D-1/DV6D-1*02': 'AQKVIQVWSTASRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQMSS--STAKERS-----GRYSVVFQKSLKSISLVISALQPDDSGKYFCALWE--------------------',
                'TRAV15D-1/DV6D-1*07': 'AEKVIQVWSTASRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQTSS--STAKERS-----GRYSVVFQKSLKSISLIISALQPDDSGKYFCALWE--------------------',
                'TRAV15D-2/DV6D-2*01': 'AQRVTQVQPTGSSQWGEEVTLDCSYETSEY-----FYCIIWYRQLFSGEMVFLIYQTSF--DTQNQRN-----GRYSVVFQKSLKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15D-2/DV6D-2*03': 'AQRVTQVQPTGSSQWGEEVTLDCSYETSEY-----FYRIFWYRQLFSGEMVFLIYQPSF--DTQNQRS-----GRYSVVFQKSFKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15N-1*01': 'AEKVIQVWSTASRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQTSS--STAKERS-----GRYSVVFQKSLKSISLIISALQPDDSGKYFCALWE--------------------',
                'TRAV15N-2*01': 'AQRVTQVQPTGSSQWGEEVTLDCSYETSEY-----FYRIFWYRQLFSGEMVFLIYQPSF--DTQNQRS-----GRYSVVFQKSFKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV16*01': 'AQKVTQTQTSISVMEKTTVTMDCVYETQDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATV-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV16*06': 'AQKVTQTQTSISVMEKTTVTMDCVYETRDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATE-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV16D/DV11*01': 'AQKVTQTQTSISVMEKTTVTMDCVYETQDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATV-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV16D/DV11*02': 'AQKVTQTQTSISVMEKTTVTMDCVYETQDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATV-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV16N*01': 'AQKVTQTQTSISVVEKTTVTMDCVYETRDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATV-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV17*01': 'AQSVDQPDAHVTLYEGASLELRCSYSYSA------APYLFWYVQYPGQSLQFLLKYITG--DAVVKGT-----KGFEAEFRKSNSSFNLKKSPAHWSDSAKYFCALE---------------------',
                'TRAV17*02': 'AQSVDQPDAHVTLSEGASLELRCSYSYSA------APYLFWYVQYPGQSLQFLLKYITG--DTVVKGT-----KGFEAEFRKSNSSFNLKKSPAHWSDSAKYFCALE---------------------',
                'TRAV19*01': 'GQQVKQSSPSLTVQEGGILILNCDYENDM------FDYFAWYKKYPDNSPTLLISVRSN---VDKRED-----GRFTVFLNKSGKHFSLHITASQPEDTAVYLCAAG---------------------',
                'TRAV19*03': 'GQQVKQSSPSLTVQEGGISILNCDYENDM------FDYFAWYKKYPDNSPTLLISVRSN---VDKRED-----GRLTVFLNKSGKHFSLHITASQPEDTAVYLCAAG---------------------',
                'TRAV2*01': 'LAKTTQ-PPSMEAYEGQEVNVSCSHTNIAT-----SEYIYWYRQVPHQGPQFIIQGYK-----DYVVN-----EVASLFISADRKLSTLSLPWVSLRDAAVYYCIVTD--------------------',
                'TRAV21/DV12*01': 'DAKTTQ-PDSMESTEGETVHLPCSHATISG-----NEYIYWYRQVPLQGPEYVTHGLQ-----QNTTN-----SMAFLAIASDRKSSTLILTHVSLRDAAVYHCILRV--------------------',
                'TRAV3-1*01': 'GEQVEQRPPHLSVREGDSAIIICTYTDSA------TAYFSWYKQEAGAGLQLLMSVLSN---VDRKEE-----QGLTVLLNKKDKRLSLNLTAAHPGDSAVYFCAVS---------------------',
                'TRAV3-1*03': 'GEQVEQRPPHLSVREGDSAIIICTYTDSA------TAYFSWYKQEAGAGLQLLMSVLSN---VDRKEE-----QGLTVLLNKKDKRLSLNLTAAHPGDSAVYFCAVS---------------------',
                'TRAV3-3*01': 'GEQVEQRPPHLSVREGDSAVITCTYTDPN------SYYFFWYKQEPGASLQLLMKVFSS---TEINEG-----QGFTVLLNKKDKRLSLNLTAAHPGDSAAYFCAVS---------------------',
                'TRAV3-4*01': 'GEQVEQRPPHLSVPEGDSAVIICTYTDSA------TAYFYWYKQEPGAGLQLLMSVFSN---VDRKEE-----QGLTVLLNKKDKQLSLNLTAAHPGDSAVYFCAVS---------------------',
                'TRAV3-4*02': 'GEQVEQRPPHLSVREGDSAFIICTYTDSA------TAYFYWYKQEPGAGLQLLMSVFSN---VDRKEE-----QGLTVLLNKKDKRLSLNLTAAHPGDSAVYFCAVS---------------------',
                'TRAV3D-3*01': 'GEQVEQRPPHLSVREGDSAFITCTYTDPN------SYYFFWYKQEPGASLQLLMKVFSS---TEINEG-----QGFTVLLNKKDKRLSLNLTAAHPGDSAAYFCAVS---------------------',
                'TRAV3D-3*02': 'GEQVEQRPPHLSVREGDSAVIICTYTDPN------SYYFFWYKQEPGAGLQLLMKVFSS---TEINEG-----QGFTVLLNKKDKQLSLNLTAAHPGDSAVYFCAVS---------------------',
                'TRAV3N-3*01': 'GEQVEQRPPHLSVREGDSAVIICTYTDPN------SYYFFWYKQEPGAGLQLLMKVFSS---TEINEG-----QGFTVLLNKKDKQLSLNLTAAHPGDSAVYFCAVS---------------------',
                'TRAV4-2*01': 'GMPVEQNPPALSLYEGADSGLRCNFSTTM-------KSVQWFQQNHRGRLITLFYLAQ-----GTKEN-----GRLKSTFNSKERYSTLHIKDAQLEDSGTYFCAAE---------------------',
                'TRAV4-2*02': 'GMPVEQNPPALSLYEGAESGLRCNFSTTM-------KGVQWFQQNHRGRLITLFYLAQ-----GTKEN-----GRLKSTFNSKERYSTLHIKDAQLEDSGTYFCAVE---------------------',
                'TRAV4-3*01': 'GDQVKQSPSALSLQEGTNSALRCNFSIAT-------TTVQWFLQNPRGSLMNLFYLVP-----GTKEN-----GRLKSTFNSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4-3*03': 'GDKVKQSPSALSLQEGTNSALRCNFSIAA-------TTVQWFLQNPRGSLINLFYLVP-----GTKEN-----GRLKSAFDSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4-4/DV10*01': 'GDQVEQSPSALSLHEGTDSALRCNFTTTM-------RSVQWFRQNSRGSLISLFYLAS-----GTKEN-----GRLKSAFDSKERYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4-4/DV10*02': 'GDQVEQSPSALSLHEGTGSALRCNFTTTM-------RAVQWFQQNSRGSLINLFYLAS-----GTKEN-----GRLKSTFNSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4D-3*01': 'GDKVKQSPSALSLQEGTNSALRCNFSIAA-------TTVQWFLQNPRGSLINLFYLVP-----GTKEN-----GRLKSTFNSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4D-3*06': 'GDKVKQSPSALSLQEGTNSALRCNFSIAA-------TTVQWFLQNPRGSLMNLFYLVP-----GTKEN-----GRLKSAFDSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4D-4*01': 'GDQVEQSPSALSLHEGTSSALRCNFTTTT-------RSVQWFRQNSRGSLINLFYLAS-----GTKEN-----GRLKSAFDSKELYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4D-4*03': 'GDQVEQSPSALSLHEGTGSALRCNFTTTM-------RAVQWFRKNSRGSLINLFYLAS-----GTKEN-----GRLKSAFDSKERYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4N-3*01': 'GDKVKQSPSALSLQEGTNSALRCNFSIAA-------TTVQWFLQNPRGSLMNLFYLVP-----GTKEN-----GRLKSAFDSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4N-4*01': 'GDQVEQSPSALSLHEGTGSALRCNFTTTM-------RAVQWFRKNSRGSLINLFYLAS-----GTKEN-----GRLKSAFDSKERYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV5-1*01': 'GEQVEQLPSSLIVQEGASVLINCSYTDSA------SVYFPWYKQEPGKRLQFIIDIRSN---MERKQN-----QRLTLLFDKKTKHLSLHITATQPGDSAIYFCSAS---------------------',
                'TRAV5-1*02': 'GEQVEQLPSSLIVQEGASVLINCSYTDSA------SVYFPWYKQEPGKRLQFIIDIRSN---MERKQT-----QRLTLLFDKKTKHLSLHITATQPGDSAIYFCSAS---------------------',
                'TRAV5-4*01': 'GEQVEQLPSILRVQERSSASINCTYENSA------SNYFPWYKQEPGENPKLIIDIRSN---MERKQI-----QGLIVLLDKKAKRFSLHITDTQPADSAMYFCAAS---------------------',
                'TRAV5-4*02': 'GEQVEQLPSILRVQEGSSASINCSYEDSA------SNYFPWYKQEPGENPKLIIDIRSN---MERKQI-----QELIVLLDKKAKRFSLHITDTQPGDSAMYFCAAS---------------------',
                'TRAV5D-4*05': 'GEQVEQLPSILRVQEGSSASINCTYENSA------SNYFPWYKQEPGENPKLIIDIRSN---MERKQT-----QGLIVLLDKKAKRFSLHITDTQPGDSAMYFCAAS---------------------',
                'TRAV5N-4*01': 'GEQVEQLPSILRVQEGSSASINCTYENSA------SNYFPWYKQEPGENPKLIIDIRSN---MERKQT-----QGLIVLLDKKAKRFSLHITDTQPGDSAMYFCAAS---------------------',
                'TRAV6-1*01': 'GDSVTQMQGQVTLSEDDFLFINCTYSTTW------YPTLFWYVQYPGEGPQLLLKVTTA---NNKGIS-----RGFEATYDKRTTSFHLQKASVQESDSAVYYCVLG---------------------',
                'TRAV6-1*03': 'GDSVTQMQGQVTLSEDDFLFINCTYSTTW------YPTLFWYVQYPGEGPQLLLKVTTA---NNKGIS-----RGFEATYDKGTTSFHLQKASVQESDSAVYYCVLG---------------------',
                'TRAV6-2*01': 'GNSVTQMQGQVTLSEEEFLFINCTYSTTG------YPTLFWYVQYPGEGPQLLLKVTTA---NNKGSS-----RGFEATYDKGTTSFHLQKASVQESDSAVYYCVLG---------------------',
                'TRAV6-2*02': 'GNSVTQMQGQVTLSEEEFLFINCTYSTTG------YPTLFWYVQYPGEGPQLLLKVTTA---NNKGSS-----RGFEATYDKGTTSFHLQKASVQESDSAVYYCVLG---------------------',
                'TRAV6-3*01': 'GDSVIQMQGQVTLSENDFLFINCTYSTTG------YPTLFWYVQYSGEGPQLLLQVTTA---NNKGSS-----RGFEATYDKGTTSFHLQKTSVQEIDSAVYYCAMR---------------------',
                'TRAV6-3*02': 'GDSVIQMQGQVTLSENDFLFINCTYSTTG------YPTLFWYVQYSGEGPQLLLQVTTA---NNKGSS-----RGFEATYDKGTTSFHLQKTSVQEIDSAVYYCAMR---------------------',
                'TRAV6-4*01': 'GDSVTQKQGQVTLSEDDFLFINCTYSTTT------YPTLLWYVQYPGQGPQLLLKVTTA---NNKGIS-----RGFEATYDKGTTSFHLQKASVQESDSAVYFCALV---------------------',
                'TRAV6-4*03': 'GDSVTQKQGQVTLSEDDFLFINCTYSTTT------YPTLLWYVQYLGQGPQLLLKVTTA---NNKGIS-----RGFEATYDKGTTSFHLQKASVQESDSAVYFCALV---------------------',
                'TRAV6-5*01': 'GDSVTQTEGPVTLSEGTSLTVNCSYETKQ------YPTLFWYVQYPGEGPQLLFKVPKA---NEKGSS-----RGFEATYNKEATSFHLQKASVQESDSAVYYCALS---------------------',
                'TRAV6-5*04': 'GDSVTQTEGPVTLSEGTSLTVNCSYETKQ------YPTLFWYVQYPGEGPQLLFKVPKA---NEKGSN-----RGFEATYNKEATSFHLQKASVQESDSAVYYCALS---------------------',
                'TRAV6-6*01': 'GDSVTQTEGQVTVSESKSLIINCTYSTTSI----AYPNLFWYVRYPGEGLQLLLKVITA---GQKGSS-----RGFEATYNKETTSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6-6*04': 'GDSVTQTEGPVTVSESESLIINCTYSATSI----AYPNLFWYVRYPGEGLQLLLKVITA---GQKGSS-----RGFEATYNKETTSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6-7/DV9*01': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPALFWYVQYPGEGPQFLFRASRD---KEKGSS-----RGFEATYNKETTSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6-7/DV9*04': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPTLFWYVQYPGEGPQLLFRASRD---KEKGSS-----RGFEATYDKGTTSFHLRKASVQESDSAVYYCALS---------------------',
                'TRAV6D-3*01': 'GDSVIQMQGQVTLSENDFLFINCTYSTTG------YPTLFWYVQYSGEGPQLLLQVTTA---NNKGSS-----RGFEATYDKGTTSFHLQKTSVQEIDSAVYYCAMR---------------------',
                'TRAV6D-3*03': 'GDSVIQMQGQVTLSENDFLFINCTYSTTG------YPTLFWYVQYSGEGPQLLLQVTTA---NNKGSS-----RGFEATYDKGTTSFHLQKTSVQEIDSAVYYCAMS---------------------',
                'TRAV6D-4*01': 'GDSVTQKQGQVTLSEDDFLFINCTYSTTT------YPTLFWYVQYPGQGPQLLLKVTTA---NNKGIS-----RGFEATYDKGTTSFHLQKASVQESDSAVYFCALV---------------------',
                'TRAV6D-5*01': 'GDSVTQTEGPVTLSEGTSLTVNCSYETKQ------YPTLFWYVQYPGEGPQLLFKVPKA---NEKGSN-----RGFEATYNKEATSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6D-5*02': 'GDSVTQTKGPVTLSEGTSLTVNCSYETKQ------YPTLFWYVQYPGEGPQLLFKVPKA---NEKGSN-----RGFEATYDKGTTSFHLQKASVQESDSAVYYCVLG---------------------',
                'TRAV6D-6*01': 'GDSVTQTEGQVTVSESKSLIINCTYSATSI----AYPNLFWYVRYPGEGLQLLLKVITA---GQKGSS-----RGFEATYNKETTSFHLQKASVQESDSAVYYCALS---------------------',
                'TRAV6D-6*02': 'GDSVTQTEGPVTVSESESLIINCTYSATSI----AYPNLFWYVRYPGEGLQLLLKVITA---GQKGSS-----RGFEATYNKETTSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6D-7*01': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPALFWYVQYPGEGPQFLFRASRD---KEKGSS-----RGFEATYNKEATSFHLQKASVQESDSAVYYCALS---------------------',
                'TRAV6D-7*04': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPALFWYVQYPGEGPQFLFRASRD---KEKGSS-----RGFEATYDKGTTSFHLRKASVQESDSAVYYCALG---------------------',
                'TRAV6N-5*01': 'GDSVTQTEGPVTLSEGTSLTVNCSYETKQ------YPTLFWYVQYPGEGPQLLFKVPKA---NEKGSN-----RGFEATYNKEATSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6N-6*01': 'GDSVTQTEGQVTVSESKSLIINCTYSATSI----GYPNLFWYVRYPGEGLQLLLKVITA---GQKGSS-----RGFEATYNKEATSFHLQKASVQESDSAVYYCALS---------------------',
                'TRAV6N-7*01': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPALFWYVQYPGEGPQFLFRASRD---KEKGSS-----RGFEATYDKGTTSFHLRKASVQESDSAVYYCALG---------------------',
                'TRAV7-1*01': 'QQKVQQSPESLIVPEGGMASLNCTFSDRN------SQYFWWYRQHSGEGPKALMSIFS----NGDKKE-----GRFTAHLNKASLYVSLHIKDSQPSDSALYFCAVS---------------------',
                'TRAV7-1*02': 'QQKVQQSPESLIVPEGGMASLNCTFSDRN------SQYFWWYRQHSGEGPKALMSIFS----NGDKKE-----GRFTAHLNKASLHVSLHIKDSQPSDSALYFCAVS---------------------',
                'TRAV7-2*01': 'QQKVQQSPESLIVPEGGMASLNCTSSDRN------VDYFWWYRQRSGKSPKMLMAIFS----NGEKEE-----GRFTVHLNKASLHTSLHIRDSQPSDSALYFCAVS---------------------',
                'TRAV7-2*02': 'QQKVQQSPESLIVPEGGMASLNCTSSDRN------VDYFWWYRQHSGKSPKMLMSIFS----NGEKEE-----GRFTVHLNKASLHTSLHIRDSQPSDSALYLCAAS---------------------',
                'TRAV7-3*01': 'QQNVQQSPESLIVPEGARTSLNCTFSDSA------SQYFWWYRQHSGKAPKALMSIFS----NGEKEE-----GRFTIHLNKASLHFSLHIRDSQPSDSALYLCAVS---------------------',
                'TRAV7-3*04': 'QQKVQQSPESLIVPEGAMTSLNCTFSDSA------SQYFAWYRQHSGKAPKALMSIFS----NGEKEE-----GRFTIHLNKASLHFSLHIRDSQPSDSALYLCAVS---------------------',
                'TRAV7-4*01': 'QQKVQQSPESLSVPEGGMASFNCTSSDRN------FQYFWWYRQHSGEGPKALMSIFS----DGDKKE-----GRFTAHLNKASLHVSLHIRDSQPSDSALYFCAASE--------------------',
                'TRAV7-4*02': 'QQKVQQSPESLSVPEGGMASLNCTSSDRN------FQYFWWYRQHSGEGPKALMSIFS----DGDKKE-----GRFTAHLNKASLHVSLHIRDSQPSDSALYFCAASE--------------------',
                'TRAV7-5*01': 'QQKVQQSPESLTVSEGAMASLNCTFSDGT------SDNFRWYRQHSGKGLEVLVSIFS----DGEKEE-----GRFTAHLNRASLHVSLHIREPQPSDSAVYLCAMS---------------------',
                'TRAV7-5*03': 'QQKVQQSPESLTVSEGAMASLNCTFSDGT------SNNFRWYRQHSAKGLEVLVSIFS----DGEKEE-----GRFTAHLNRANLHVSLHIREPQPSDSAVYLCAVS---------------------',
                'TRAV7-6*01': 'QEKVQQSPESLIVPEGAMASLNCTFSNSA------SQSIWWYQQHPGKGPEALISIFS----NGNKKE-----GRLTVYLNRASLHVSLHIRDSQPTDSAIYLCAVS---------------------',
                'TRAV7-6*02': 'QEKVQQSPESLIVPEGAMVSLNCSFSDSA------SQSIWWYQQHPGKGPKALISIFS----NGNKKE-----GRLTVYLNRASLHVSLHIKDSQPSDSAVYLCAVS---------------------',
                'TRAV7D-2*01': 'QQKVQQSPESLIVPEGGMASLNCTSSDRN------VDYFWWYRQHSGKSPKMLMSIFS----NGEKEE-----GRFTVHLNKASLHTSLHIRDSQPSDSALYLCAAS---------------------',
                'TRAV7D-2*02': 'QQKVQQSPESLIVPEGGMASLNCTSSDRN------VDYFWWYRQHSGKSPKMLMSIFS----NGEKEE-----GRFTVHLNKASLHTSLHIRDSQPSDSALYLCAAS---------------------',
                'TRAV7D-3*01': 'QQKVQQSPESLIVPEGAMTSLNCTFSDSA------SQYFAWYRQHSGKAPKALMSIFS----NGEKEE-----GRFTIHLNKASLHFSLHIRDSQPSDSALYLCAVS---------------------',
                'TRAV7D-3*03': 'QQKVQQSPESLIVPEGAMTSLNCTFSDSA------SQYFAWYRQHSGKAPKALMSIFS----NGEKEE-----GRFTIHLNKASLHFSLHIRDSQPSDSALYLCAVS---------------------',
                'TRAV7D-4*01': 'QQKVQQSPESLSVPEGGMASLNCTSSDRN------FQYFWWYRQHSGEGPKALMSIFS----DGDKKE-----GRFTAHLNKASLHVSLHIRDSQPSDSALYFCAASE--------------------',
                'TRAV7D-4*02': 'QQKVQQSPESLSVPEGGMASLNCTSSDRN------FQYFWWYRQHSGEGPKALMSIFS----DGDKKE-----GRFTAHLNKASLHVSLHIRDSQPSDSALYFCAASE--------------------',
                'TRAV7D-5*01': 'QQKVQQSPESLTVSEGAMASLNCTFSDGT------SDNFRWYRQHSGKGLEMLVSIFS----DGEKEE-----GRFTAHLNRASLHVSLHIREPQPSDSAVYLCAVS---------------------',
                'TRAV7D-5*02': 'QQKVQQSPESLTVSEGAMASLNCTFSDRS------SDNFRWYRQHSGKGLEVLVSIFS----DGEKEE-----GSFTAHLNRASLHVFLHIREPQPSDSALYLCAVS---------------------',
                'TRAV7D-6*01': 'QEKVQQSPESLTVPEGAMASLNCTISDSA------SQSIWWYQQNPGKGPKALISIFS----NGNKKE-----GRLTVYLNRASLHVSLHIRDSHPSDSAVYLCAAS---------------------',
                'TRAV7D-6*02': 'QEKVQQSPESLIVPEGAMSSLNCTFSNSA------SQSIWWYQQHPGKGPEALISIFS----NGNKKE-----GRLTVYLNRASLHVSLHIRDSQPSDSAVYLCAVS---------------------',
                'TRAV7N-4*01': 'QQKVQQSPESLSVPEGGMASLNCTSSDRN------FQYFWWYRQHSGEGPKALMSIFS----DGDKKE-----GRFTAHLNKASLHVSLHIRDSQPSDSALYFCAVSE--------------------',
                'TRAV7N-5*01': 'QQKVQQSPESLTVSEGAMASLNCTFSDRS------SDNFRWYRQHSGKGLEVLVSIFS----DGEKEE-----GSFTAHLNRASLHVFLHIREPQPSDSALYLCAVS---------------------',
                'TRAV7N-6*01': 'QEKVQQSPESLIVPEGAMSSLNCTFSNSA------SQSIWWYQQHPGKGPEALISIFS----NGNKKE-----GRLTVYLNRASLHVSLHIRDSQPSDSAVYLCAVS---------------------',
                'TRAV8-1*01': 'SQLAEENPWALSVHEGESVTVNCSYKTSI-------TALQWYRQKSGEGPAQLILIRSN---EREKRN-----GRLRATLDTSSQSSSLSITATRCEDTAVYFCATD---------------------',
                'TRAV8-1*03': 'SQLAEENSWALSVHEGESVTVNCSYKTSI-------TALQWYRQKSGKGPAQLILIRSN---EREKRN-----GRLRATLDTSSQSSSLSITATRCEDTAVYFCATD---------------------',
                'TRAV8-2*01': 'SQWGEENLQALSIQEGEDVTMNCSYKTYT-------TVVHWYRQDSGRGPALIILIRSN---EREKRS-----GRLRATLDTSSQSSSLSITAAQCEDTAVYFCATD---------------------',
                'TRAV8D-1*01': 'SQLAEENLWALSVHEGESVTVNCSYKTSI-------TALQWYRQKSGEGPAQLILIRSN---EREKRN-----GRLRATLDTSSQSSSLSITATRCEDTAVYFCATD---------------------',
                'TRAV8D-2*01': 'SQWGEENLQALSIQEGEDVTMNCSYKTYT-------TVVQWYRQKSGKGPALIILIRSN---EREKRS-----GRLRATLDTSSQSSSLSITGTLATDTAVYFCATD---------------------',
                'TRAV8D-2*03': 'SQWGEENLQALSIQEGEDVTMNCSYKTYT-------TVVHWYRQDSGRGPALIILIRSN---EREKRS-----GRLRATLDTSSQSSSLSITAAQCEDTAVYFCATD---------------------',
                'TRAV8N-2*01': 'SQWGEENLQALSIQEGEDVTMNCSYKTYT-------TVVQWYRQKSGKGPAQLILIRSN---EREKRS-----GRLRATLDTSSQSSSLSITGTLATDTAVYFCATD---------------------',
                'TRAV9-1*01': 'TQTVSQSDAHVTVFEGDSVELRCNYSYGG------SIYLSWYIQHHGRGLQFLLKYYSG--NPVVQGV-----NGFEAEFSKSDSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9-1*02': 'TQTVSQSDAHVTVFEGDSVELRCNYSYGG------SIYLSWYIQHHGHGLQFLLKYYSG--NPVVQGV-----NGFEAEFSKSDSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9-2*01': 'AQSVTQPDARVTVSEGASLQLRCKYSYSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSDSSFHLRKASVHWSDSAVYFCAAS---------------------',
                'TRAV9-2*02': 'AQSVTQPDARVTVSEGASLQLRCKYSYSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDSAVYFCVLS---------------------',
                'TRAV9-3*01': 'AQSVTQPDARVTVSEGASLQLRCKYSSSV------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9-4*01': 'AQSVTQPDARVTVSEGASLQLRCKYSYSA------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9-4*02': 'AQSVTQPDARVTVSEGASLQLRCKYSYSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFIKSNSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9D-1*01': 'TQTVSQSDAHVTVFEGDSVELRCNYSYGG------SIYLSWYIQHHGRGLQFLLKYYSG--NPVVQGV-----NGFEAEFSKSDSSFHLRKASVHWSDSAVYFCAAS---------------------',
                'TRAV9D-1*03': 'TQTVSQSDAHVTVFEGDSVELRCNYSYGG------SIYLSWYIQHHGRGLQFLLKYYSG--NPVVQGV-----NGFKAEFSKSDSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9D-2*01': 'AQSVTQPDARVTVSQGASLQLRCKYSYSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHPRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9D-2*02': 'AQSVTQPDARVTVSQGASLQLRCKYSYSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHPRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9D-3*01': 'AQSVTQPDARVTVSEGASLQLRCKYSYSA------TPYLFWYVQYPRQGLQMLLKYYSG--DPVVQGV-----NGFEAEFSKSDSSFHLRKASVHWSDSAVYFCAVS---------------------',
                'TRAV9D-3*03': 'AQSVTQPDARVTVSEGASLQLRCKYSYFG------TPYLFWYVQYPRQGLQLLLKYYPG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDWAVYFCAVS---------------------',
                'TRAV9D-4*01': 'AQSVTQPDARVTVSEGASLQLRCKYSYSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDSAVYFCALS---------------------',
                'TRAV9N-2*01': 'AQSVTQPDARVTVSEGASLQLRCKYSSSG------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDSAVYFCVLS---------------------',
                'TRAV9N-3*01': 'AQSVTQPDARVTVSEGASLQLRCKYSYFG------TPYLFWYVQYPRQGLQLLLKYYPG--DPVVQGV-----NGFEAEFSKSNSSFHLRKASVHWSDWAVYFCAVS---------------------',
                'TRAV9N-4*01': 'AQSVTQPDARVTVSEGASLQLRCKYSYSA------TPYLFWYVQYPRQGLQLLLKYYSG--DPVVQGV-----NSFEAEFSKSNSSFHLQKASVHWSDSAVYFCALS---------------------'}},
                'B': {'human': {'TRBV10-1*01': 'DAEITQSPRHKITETGRQVTLACHQTWNH-------NNMFWYRQDLGHGLRLIHYSYG----VQDTNKGEVS-DGYSVSRS-NTEDLPLTLESAASSQTSVYFCASSE--------------------',
                'TRBV10-1*02': 'DAEITQSPRHKITETGRQVTLACHQTWNH-------NNMFWYRQDLGHGLRLIHYSYG----VHDTNKGEVS-DGYSVSRS-NTEDLPLTLESAASSQTSVYFCASSE--------------------',
                'TRBV10-2*01': 'DAGITQSPRYKITETGRQVTLMCHQTWSH-------SYMFWYRQDLGHGLRLIYYSAA----ADITDKGEVP-DGYVVSRS-KTENFPLTLESATRSQTSVYFCASSE--------------------',
                'TRBV10-2*02': 'DAGITQSPRYKITETGRQVTLMCHQTWSH-------SYMFWYRQDLGHGLRLIYYSAA----ADITDKGEVP-DGYVVSRS-KTENFPLTLESATRSQTSVYFCASSE--------------------',
                'TRBV10-3*01': 'DAGITQSPRHKVTETGTPVTLRCHQTENH-------RYMYWYRQDPGHGLRLIHYSYG----VKDTDKGEVS-DGYSVSRS-KTEDFLLTLESATSSQTSVYFCAISE--------------------',
                'TRBV10-3*02': 'DAGITQSPRHKVTETGTPVTLRCHQTENH-------RYMYWYRQDPGHGLRLIHYSYG----VKDTDKGEVS-DGYSVSRS-KTEDFLLTLESATSSQTSVYFCAISE--------------------',
                'TRBV10-3*03': 'DAGITQSPRHKVTETGTPVTLRCHQTENH-------RYMYWYRQDPGHGLRLIHYSYG----VKDTDKGEVS-DGYSVSRS-KTEDFLLTLESATSSQTSVYFCAISE--------------------',
                'TRBV11-1*01': 'EAEVAQSPRYKITEKSQAVAFWCDPISGH-------ATLYWYRQILGQGPELLVQFQD----ESVVDDSQLPKDRFSAERL-KGVDSTLKIQPAELGDSAMYLCASSL--------------------',
                'TRBV11-2*01': 'EAGVAQSPRYKIIEKRQSVAFWCNPISGH-------ATLYWYQQILGQGPKLLIQFQN----NGVVDDSQLPKDRFSAERL-KGVDSTLKIQPAKLEDSAVYLCASSL--------------------',
                'TRBV11-2*03': 'EAGVAQSPRYKIIEKRQSVAFWCNPISGH-------ATLYWYQQILGQGPKLLIQFQN----NGVVDDSQLPKDRFSAERL-KGVDSTLKIQPAKLEDSAVYLCASSL--------------------',
                'TRBV11-3*01': 'EAGVVQSPRYKIIEKKQPVAFWCNPISGH-------NTLYWYLQNLGQGPELLIRYEN----EEAVDDSQLPKDRFSAERL-KGVDSTLKIQPAELGDSAVYLCASSL--------------------',
                'TRBV11-3*04': 'EAGVVQSPRYKIIEKKQPVAFWCNPISGH-------NTLYWYRQNLGQGPELLIRYEN----EEAVDDSQLPKDRFSAERL-KGVDSTLKIQPAELGDSAVYLCASSL--------------------',
                'TRBV12-2*02': 'DAGIIQSPKHEVTEMGQTVTLRCEPIFGH-------NFLFWYRDTFVQGLELLSYFRS----RSIIDNAGMPTERFSAERP-DGSFSTLKIQPAEQGDSAVYVCASRL--------------------',
                'TRBV12-3*01': 'DAGVIQSPRHEVTEMGQEVTLRCKPISGH-------NSLFWYRQTMMRGLELLIYFNN----NVPIDDSGMPEDRFSAKMP-NASFSTLKIQPSEPRDSAVYFCASSL--------------------',
                'TRBV12-4*01': 'DAGVIQSPRHEVTEMGQEVTLRCKPISGH-------DYLFWYRQTMMRGLELLIYFNN----NVPIDDSGMPEDRFSAKMP-NASFSTLKIQPSEPRDSAVYFCASSL--------------------',
                'TRBV12-4*03': 'DAGVIQSPRHEVTEMGQEVTLRCKPISGH-------DYLFWYRQTMMRGLELLIYFNN----NVPIDDSGMPEDRFSAKMP-NASFSTLKIQPSEPRDSAVYFCASSL--------------------',
                'TRBV12-5*01': 'DARVTQTPRHKVTEMGQEVTMRCQPILGH-------NTVFWYRQTMMQGLELLAYFRN----RAPLDDSGMPKDRFSAEMP-DATLATLKIQPSEPRDSAVYFCASGL--------------------',
                'TRBV12-5*02': 'DARVTQTPRDKVTEMGQEVTMRCQPILGH-------NTVFWYRQTMMQGLELLAYFRN----RAPLDDSGMPKDRFSAEMP-DATLATLKIQPSEPRDSAVYFCASGL--------------------',
                'TRBV13*01': 'AAGVIQSPRHLIKEKRETATLKCYPIPRH-------DTVYWYQQGPGQDPQFLISFYE----KMQSDKGSIP-DRFSAQQF-SDYHSELNMSSLELGDSALYFCASSL--------------------',
                'TRBV13*03': 'AAGVIQSPRHLIKEKRETATLKCYPISRH-------DTVYWYQQGPGQDPQFLISFYE----KMQSDKGSIP-DRFSAQQF-SDYHSELNMSSLELGDSALYFCASSL--------------------',
                'TRBV14*01': 'EAGVTQFPSHSVIEKGQTVTLRCDPISGH-------DNLYWYRRVMGKEIKFLLHFVK----ESKQDESGMPNNRFLAERT-GGTYSTLKVQPAELEDSGVYFCASSQ--------------------',
                'TRBV14*02': 'EAGVTQFPSHSVIEKGQTVTLRCDPISGH-------DNLYWYRRVMGKEIKFLLHFVK----ESKQDESGMPNNRFLAERT-GGTYSTLKVQPAELEDSGVYFCASSQ--------------------',
                'TRBV15*01': 'DAMVIQNPRYQVTQFGKPVTLSCSQTLNH-------NVMYWYQQKSSQAPKLLFHYYD----KDFNNEADTP-DNFQSRRP-NTSFCFLDIRSPGLGDTAMYLCATSR--------------------',
                'TRBV15*02': 'DAMVIQNPRYQVTQFGKPVTLSCSQTLNH-------NVMYWYQQKSSQAPKLLFHYYD----KDFNNEADTP-DNFQSRRP-NTSFCFLDIRSPGLGDAAMYLCATSR--------------------',
                'TRBV16*01': 'GEEVAQTPKHLVRGEGQKAKLYCAPIKGH-------SYVFWYQQVLKNEFKFLISFQN----ENVFDETGMPKERFSAKCL-PNSPCSLEIQATKLEDSAVYFCASSQ--------------------',
                'TRBV18*01': 'NAGVMQNPRHLVRRRGQEARLRCSPMKGH-------SHVYWYRQLPEEGLKFMVYLQK----ENIIDESGMPKERFSAEFP-KEGPSILRIQQVVRGDSAAYFCASSP--------------------',
                'TRBV19*01': 'DGGITQSPKYLFRKEGQNVTLSCEQNLNH-------DAMYWYRQDPGQGLRLIYYSQI----VNDFQKGDIA-EGYSVSRE-KKESFPLTVTSAQKNPTAFYLCASSI--------------------',
                'TRBV19*02': 'DGGITQSPKYLFRKEGQNVTLSCEQNLNH-------DAMYWYRQVPGQGLRLIYYSHI----VNDFQKGDIA-EGYSVSRE-KKESFPLTVTSAQKNPTAFYLCASSI--------------------',
                'TRBV19*03': 'DGGITQSPKYLFRKEGQNVTLSCEQNLNH-------DAMYWYRQDPGQGLRLIYYSHI----VNDFQKGDIA-EGYSVSRE-KKESFPLTVTSAQKNPTAFYLCASSI--------------------',
                'TRBV19*04': 'DGGITQSPKYLFRKEGQNVTLSCEQNLNH-------DAMYWYRQDPGQGLRLIYYSQI----VNDFQKGDIA-EGYSVSRE-KKESFPLTVTSAQKNPTAFYLCASSI--------------------',
                'TRBV2*01': 'EPEVTQTPSHQVTQMGQEVILRCVPISNH-------LYFYWYRQILGQKVEFLVSFYN----NEISEKSEIFDDQFSVERP-DGSNFTLKIRSTKLEDSAMYFCASSE--------------------',
                'TRBV20-1*01': 'GAVVSQHPSWVICKSGTSVKIECRSLDFQ------ATTMFWYRQFPKQSLMLMATSNEG---SKATYEQGVEKDKFLINHA-SLTLSTLTVTSAHPEDSSFYICSAR---------------------',
                'TRBV20-1*02': 'GAVVSQHPSRVICKSGTSVKIECRSLDFQ------ATTMFWYRQFPKQSLMLMATSNEG---SKATYEQGVEKDKFLINHA-SLTLSTLTVTSAHPEDSSFYICSAR---------------------',
                'TRBV24-1*01': 'DADVTQTPRNRITKTGKRIMLECSQTKGH-------DRMYWYRQDPGLGLRLIYYSFD----VKDINKGEIS-DGYSVSRQ-AQAKFSLSLESAIPNQTALYFCATSD--------------------',
                'TRBV24-1*02': 'DADVTQTPRNRITKTGKRIMLECSQTKGH-------DRMYWYRQDPGLGLQLIYYSFD----VKDINKGEIS-DGYSVSRQ-AQAKFSLSLESAIPNQTALYFCATSD--------------------',
                'TRBV25-1*01': 'EADIYQTPRYLVIGTGKKITLECSQTMGH-------DKMYWYQQDPGMELHLIHYSYG----VNSTEKGDLS-SESTVSRI-RTEHFPLTLESARPSHTSQYLCASSE--------------------',
                'TRBV25-1*02': 'EADIYQTPRYLVIGTGKKITLECSQTMGH-------DKMYWYQQDPGMELHLIHYSYG----VNSTEKGDLS-SESTVSRI-RTEHFPLTLESARPSHTSQYLCASSE--------------------',
                'TRBV27*01': 'EAQVTQNPRYLITVTGKKLTVTCSQNMNH-------EYMSWYRQDPGLGLRQIYYSMN----VEVTDKGDVP-EGYKVSRK-EKRNFPLILESPSPNQTSLYFCASSL--------------------',
                'TRBV27*02': 'EAQVTQNPRYLITVTGKKLTVTCSQNMNH-------EYMSWYRQDPGLGLRQIYYSMN----VEATDKGDVP-EGYKVSRK-EKRNFPLILESPSPNQTSLYFCASSL--------------------',
                'TRBV28*01': 'DVKVTQSSRYLVKRTGEKVFLECVQDMDH-------ENMFWYRQDPGLGLRLIYFSYD----VKMKEKGDIP-EGYSVSRE-KKERFSLILESASTNQTSMYLCASSL--------------------',
                'TRBV29-1*01': 'SAVISQKPSRDICQRGTSLTIQCQVDSQV-------TMMFWYRQQPGQSLTLIATANQG---SEATYESGFVIDKFPISRP-NLTFSTLTVSNMSPEDSSIYLCSVE---------------------',
                'TRBV3-1*01': 'DTAVSQTPKYLVTQMGNDKSIKCEQNLGH-------DTMYWYKQDSKKFLKIMFSYNN----KELIINETVP-NRFSPKSP-DKAHLNLHINSLELGDSAVYFCASSQ--------------------',
                'TRBV30*02': 'SQTIHQWPATLVQPVGSPLSLECTVEGTS------NPNLYWYRQAAGRGLQLLFYSVG-----IGQISSEVP-QNLSASRP-QDRQFILSSKKLLLSDSGFYLCAWS---------------------',
                'TRBV4-1*01': 'DTEVTQTPKHLVMGMTNKKSLKCEQHMGH-------RAMYWYKQKAKKPPELMFVYSY----EKLSINESVP-SRFSPECP-NSSLLNLHLHALQPEDSALYLCASSQ--------------------',
                'TRBV4-1*03': 'DTEVIQTPKHLVMGMTNKKSLKCEQHMGH-------RAMYWYKQKAKKPPELMFVYSY----EKLSINESVP-SRFSPECP-NSSLLNLHLHALQPEDSALYLCASSQ--------------------',
                'TRBV4-2*01': 'ETGVTQTPRHLVMGMTNKKSLKCEQHLGH-------NAMYWYKQSAKKPLELMFVYNF----KEQTENNSVP-SRFSPECP-NSSHLFLHLHTLQPEDSALYLCASSQ--------------------',
                'TRBV4-3*01': 'ETGVTQTPRHLVMGMTNKKSLKCEQHLGH-------NAMYWYKQSAKKPLELMFVYSL----EERVENNSVP-SRFSPECP-NSSHLFLHLHTLQPEDSALYLCASSQ--------------------',
                'TRBV5-1*01': 'KAGVTQTPRYLIKTRGQQVTLSCSPISGH-------RSVSWYQQTPGQGLQFLFEYFS----ETQRNKGNFP-GRFSGRQF-SNSRSEMNVSTLELGDSALYLCASSL--------------------',
                'TRBV5-4*01': 'ETGVTQSPTHLIKTRGQQVTLRCSSQSGH-------NTVSWYQQALGQGPQFIFQYYR----EEENGRGNFP-PRFSGLQF-PNYSSELNVNALELDDSALYLCASSL--------------------',
                'TRBV5-4*05': 'ETGVTQSPTHLIKTRGQQVTLRCSSQSGH-------NTVSWYQQALGQGPQFIFQYYR----EEENGRGNFP-PRFSGLQF-PNYSSELNVNALELDDSALYLCASSL--------------------',
                'TRBV5-5*01': 'DAGVTQSPTHLIKTRGQQVTLRCSPISGH-------KSVSWYQQVLGQGPQFIFQYYE----KEERGRGNFP-DRFSARQF-PNYSSELNVNALLLGDSALYLCASSL--------------------',
                'TRBV5-5*02': 'DAGVTQSPTHLIKTRGQHVTLRCSPISGH-------KSVSWYQQVLGQGPQFIFQYYE----KEERGRGNFP-DRFSARQF-PNYSSELNVNALLLGDSALYLCASSL--------------------',
                'TRBV5-6*01': 'DAGVTQSPTHLIKTRGQQVTLRCSPKSGH-------DTVSWYQQALGQGPQFIFQYYE----EEERQRGNFP-DRFSGHQF-PNYSSELNVNALLLGDSALYLCASSL--------------------',
                'TRBV5-6*02': 'DAGVTQSPTHLIKTRGQQVTLRCSPKSGH-------DTVSWYQQALGQGPQFIFQYYE----EEERQRGNFP-DRFSGHQF-PNYSSELNVNALWLGDSALYLCASSL--------------------',
                'TRBV5-8*01': 'EAGVTQSPTHLIKTRGQQATLRCSPISGH-------TSVYWYQQALGLGLQFLLWYDE----GEERNRGNFP-PRFSGRQF-PNYSSELNVNALELEDSALYLCASSL--------------------',
                'TRBV6-1*01': 'NAGVTQTPKFQVLKTGQSMTLQCAQDMNH-------NSMYWYRQDPGMGLRLIYYSAS----EGTTDKGEVP-NGYNVSRL-NKREFSLRLESAAPSQTSVYFCASSE--------------------',
                'TRBV6-2*01': 'NAGVTQTPKFRVLKTGQSMTLLCAQDMNH-------EYMYWYRQDPGMGLRLIHYSVG----EGTTAKGEVP-DGYNVSRL-KKQNFLLGLESAAPSQTSVYFCASSY--------------------',
                'TRBV6-3*01': 'NAGVTQTPKFRVLKTGQSMTLLCAQDMNH-------EYMYWYRQDPGMGLRLIHYSVG----EGTTAKGEVP-DGYNVSRL-KKQNFLLGLESAAPSQTSVYFCASSY--------------------',
                'TRBV6-4*01': 'IAGITQAPTSQILAAGRRMTLRCTQDMRH-------NAMYWYRQDLGLGLRLIHYSNT----AGTTGKGEVP-DGYSVSRA-NTDDFPLTLASAVPSQTSVYFCASSD--------------------',
                'TRBV6-4*02': 'TAGITQAPTSQILAAGRSMTLRCTQDMRH-------NAMYWYRQDLGLGLRLIHYSNT----AGTTGKGEVP-DGYSVSRA-NTDDFPLTLASAVPSQTSVYFCASSD--------------------',
                'TRBV6-5*01': 'NAGVTQTPKFQVLKTGQSMTLQCAQDMNH-------EYMSWYRQDPGMGLRLIHYSVG----AGITDQGEVP-NGYNVSRS-TTEDFPLRLLSAAPSQTSVYFCASSY--------------------',
                'TRBV6-6*01': 'NAGVTQTPKFRILKIGQSMTLQCTQDMNH-------NYMYWYRQDPGMGLKLIYYSVG----AGITDKGEVP-NGYNVSRS-TTEDFPLRLELAAPSQTSVYFCASSY--------------------',
                'TRBV6-6*02': 'NAGVTQTPKFRILKIGQSMTLQCAQDMNH-------NYMYWYRQDPGMGLKLIYYSVG----AGITDKGEVP-NGYNVSRS-TTEDFPLRLELAAPSQTSVYFCASSY--------------------',
                'TRBV6-8*01': 'NAGVTQTPKFHILKTGQSMTLQCAQDMNH-------GYMSWYRQDPGMGLRLIYYSAA----AGTTDK-EVP-NGYNVSRL-NTEDFPLRLVSAAPSQTSVYLCASSY--------------------',
                'TRBV6-8*02': 'NAGVTQTPKFHILKTGQSMTLQCAQDMNH-------GYMSWYRQDPGMGLRLIYYSAA----AGTTDK-EVP-NGYNVSRL-NTEDFPLRLVSAAPSRTSVYLCASSY--------------------',
                'TRBV6-9*01': 'NAGVTQTPKFHILKTGQSMTLQCAQDMNH-------GYLSWYRQDPGMGLRRIHYSVA----AGITDKGEVP-DGYNVSRS-NTEDFPLRLESAAPSQTSVYFCASSY--------------------',
                'TRBV6-9*02': 'NAGVTQTPKFHILKTGQSMTLQCAQDMNH-------GYLSWYRQDPGMGLRRIHYSVA----AGITDKGEVP-DGYNVSRS-NTEDFPLRLESAAPSQTSVYFCASSY--------------------',
                'TRBV7-2*01': 'GAGVSQSPSNKVTEKGKDVELRCDPISGH-------TALYWYRQSLGQGLEFLIYFQG----NSAPDKSGLPSDRFSAERT-GGSVSTLTIQRTQQEDSAVYLCASSL--------------------',
                'TRBV7-2*02': 'GAGVSQSPSNKVTEKGKDVELRCDPISGH-------TALYWYRQRLGQGLEFLIYFQG----NSAPDKSGLPSDRFSAERT-GESVSTLTIQRTQQEDSAVYLCASSL--------------------',
                'TRBV7-2*03': 'GAGVSQSPSNKVTEKGKDVELRCDPISGH-------TALYWYRQRLGQGLEFLIYFQG----NSAPDKSGLPSDRFSAERT-GESVSTLTIQRTQQEDSAVYLCTSSL--------------------',
                'TRBV7-3*01': 'GAGVSQTPSNKVTEKGKYVELRCDPISGH-------TALYWYRQSLGQGPEFLIYFQG----TGAADDSGLPNDRFFAVRP-EGSVSTLKIQRTERGDSAVYLCASSL--------------------',
                'TRBV7-4*01': 'GAGVSQSPRYKVAKRGRDVALRCDSISGH-------VTLYWYRQTLGQGSEVLTYSQS----DAQRDKSGRPSGRFSAERP-ERSVSTLKIQRTEQGDSAVYLCASSL--------------------',
                'TRBV7-6*01': 'GAGVSQSPRYKVTKRGQDVALRCDPISGH-------VSLYWYRQALGQGPEFLTYFNY----EAQQDKSGLPNDRFSAERP-EGSISTLTIQRTEQRDSAMYRCASSL--------------------',
                'TRBV7-7*01': 'GAGVSQSPRYKVTKRGQDVTLRCDPISSH-------ATLYWYQQALGQGPEFLTYFNY----EAQPDKSGLPSDRFSAERP-EGSISTLTIQRTEQRDSAMYRCASSL--------------------',
                'TRBV7-7*03': 'GAGVSQSPRYKVTKRGQDVTLRCDPISSH-------ATLYWYQQALGQGPEFLTYFNY----EAQPDKSGLPSDRFSAERP-EGSISTLTIQRTEQRDSAMYRCASSL--------------------',
                'TRBV7-8*01': 'GAGVSQSPRYKVAKRGQDVALRCDPISGH-------VSLFWYQQALGQGPEFLTYFQN----EAQLDKSGLPSDRFFAERP-EGSVSTLKIQRTQQEDSAVYLCASSL--------------------',
                'TRBV7-8*02': 'GAGVSQSPRYKVAKRGQDVALRCDPISGH-------VSLFWYQQALGQGPEFLTYFQN----EAQLDKSGLPSDRFFAERP-EGSVSTLKIQRTQKEDSAVYLCASSL--------------------',
                'TRBV7-9*01': 'DTGVSQNPRHKITKRGQNVTFRCDPISEH-------NRLYWYRQTLGQGPEFLTYFQN----EAQLEKSRLLSDRFSAERP-KGSFSTLEIQRTEQGDSAMYLCASSL--------------------',
                'TRBV7-9*03': 'DTGVSQDPRHKITKRGQNVTFRCDPISEH-------NRLYWYRQTLGQGPEFLTYFQN----EAQLEKSRLLSDRFSAERP-KGSFSTLEIQRTEQGDSAMYLCASSL--------------------',
                'TRBV9*01': 'DSGVTQTPKHLITATGQRVTLRCSPRSGD-------LSVYWYQQSLDQGLQFLIQYYN----GEERAKGNIL-ERFSAQQF-PDLHSELNLSSLELGDSALYFCASSV--------------------',
                'TRBV9*02': 'DSGVTQTPKHLITATGQRVTLRCSPRSGD-------LSVYWYQQSLDQGLQFLIHYYN----GEERAKGNIL-ERFSAQQF-PDLHSELNLSSLELGDSALYFCASSV--------------------'},
                'mouse': {'TRBV1*01': 'VTLLEQNPRWRLVPRGQAVNLRCILKNSQ------YPWMSWYQQDLQKQLQWLFTLRS----PGDKEVKSLPGADYLATRV-TDTELRLQVANMS--QGRTLYCTCSA--------------------',
                'TRBV12-1*01': 'DSGVVQSPRHIIKEKGGRSVLTCIPISGH-------SNVVWYQQTLGKELKFLIQHYE----KVERDKGFLP-SRFSVQQF-DDYHSEMNMSALELEDSAMYFCASSL--------------------',
                'TRBV12-2*01': 'NSGVVQSPRYIIKGKGERSILKCIPISGH-------LSVAWYQQTQGQELKFFIQHYD----KMERDKGNLP-SRFSVQQF-DDYHSEMNMSALELEDSAVYFCASSL--------------------',
                'TRBV13-1*01': 'EAAVTQSPRNKVTVTGGNVTLSCRQTNSH-------NYMYWYRQDTGHGLRLIHYSYG----AGNLRIGDVP-DGYKATRT-TQEDFFLLLELASPSQTSLYFCASSD--------------------',
                'TRBV13-1*02': 'EAAVTQSPRNKVTVTGGNVTLSCRQTNSH-------NYMYWYRQDTGHGLRLIHYSYG----AGNLQIGDVP-DGYKATRT-TQEDFFLLLELASPSQTSLYFCASSD--------------------',
                'TRBV13-2*01': 'EAAVTQSPRNKVAVTGGKVTLSCNQTNNH-------NNMYWYRQDTGHGLRLIHYSYG----AGSTEKGDIP-DGYKASRP-SQENFSLILELATPSQTSVYFCASGD--------------------',
                'TRBV13-3*01': 'EAAVTQSPRSKVAVTGGKVTLSCHQTNNH-------DYMYWYRQDTGHGLRLIHYSYV----ADSTEKGDIP-DGYKASRP-SQENFSLILELASLSQTAVYFCASSD--------------------',
                'TRBV14*01': 'EAGVTQSPRYAVLQEGQAVSFWCDPISGH-------DTLYWYQQPRDQGPQLLVYFRD----EAVIDNSQLPSDRFSAVRP-KGTNSTLKIQSAKQGDTATYLCASSF--------------------',
                'TRBV15*01': 'DAGVTQTPRHEVAEKGQTIILKCEPVSGH-------NDLFWYRQTKIQGLELLSYFRS----KSLMEDGGAFKDRFKAEML-NSSFSTLKIQPTEPKDSAVYLCASSL--------------------',
                'TRBV16*01': 'NAGVIQTPRHKVTGKGQEATLWCEPISGH-------SAVFWYRQTIVQGLEFLTYFRN----QAPIDDSGMPKERFSAQMP-NQSHSTLKIQSTQPQDSAVYLCASSL--------------------',
                'TRBV17*01': 'DTTVKQNPRYKLARVGKPVNLICSQTMNH-------DTMYWYQKKPNQAPKLLLFYYD----KILNREADTF-EKFQSSRP-NNSFCSLYIGSAGLEYSAMYLCASSR--------------------',
                'TRBV19*01': 'GGIITQTPKFLIGQEGQKLTLKCQQNFNH-------DTMYWYRQDSGKGLRLIYYSIT----ENDLQKGDLS-EGYDASRE-KKSSFSLTVTSAQKNEMAVFLCASSI--------------------',
                'TRBV2*01': 'DPKIIQKPKYLVAVTGSEKILICEQYLGH-------NAMYWYRQSAKKPLEFMFSYSY----QKLMDNQTAS-SRFQPQSS-KKNHLDLQITALKPDDSATYFCASSQ--------------------',
                'TRBV20*01': 'GALVYQYPRRTICKSGTSMRMECQAVGFQ------ATSVAWYRQSPQKTFELIALSTVN---SAIKYEQNFTQEKFPISHP-NLSFSSMTVLNAYLEDRGLYLCGAR---------------------',
                'TRBV23*01': 'DAAVTQKPRYLIKMKGQEAEMKCIPEKGH-------TAVFWYQQKQSKELKFLIYFQN----QQPLDQIDMVKERFSAVCP-SSSLCSLGIRTCEAEDSALYLCSSSQ--------------------',
                'TRBV24*01': 'VAGVTQTPRYLVKEKGQKAHMSCSPEKGH-------TAFYWYQQNQKQELTFLISFRN----EEIMEQTDLVKKRFSAKCS-SNSRCILEILSSEEDDSALYLCASSL--------------------',
                'TRBV26*01': 'NSKVIQTPRYLVKGQGQKAKMRCIPEKGH-------PVVFWYQQNKNNEFKFLINFQN----QEVLQQIDMTEKRFSAECP-SNSPCSLEIQSSEAGDSALYLCASSL--------------------',
                'TRBV29*01': 'DMKVTQMPRYLIKRMGENVLLECGQDMSH-------ETMYWYRQDPGLGLQLIYISYD----VDSNSEGDIP-KGYRVSRK-KREHFSLILDSAKTNQTSVYFCASSL--------------------',
                'TRBV3*01': 'GPKVLQIPSHQIIDMGQMVTLNCDPVSNH-------LYFYWYKQILGQQMEFLVNFYN----GKVMEKSKLFKDQFSVERP-DGSYFTLKIQPTALEDSAVYFCASSL--------------------',
                'TRBV30*01': 'SVLLYQKPNRDICQSGTSLKIQCVADSQV-------VSMFWYQQFQEQSLMLMATANEG---SEATYESGFTKDKFPISRP-NLTFSTLTVNNARPGDSSIYFCSSR---------------------',
                'TRBV31*01': 'AQTIHQWPVAEIKAVGSPLSLGCTIKGKS------SPNLYWYWQATGGTLQQLFYSIT-----VGQVESVVQ-LNLSASRP-KDDQFILSTEKLLLSHSGFYLCAWS---------------------',
                'TRBV5*01': 'NTKITQSPRYLIL-GRANKSLECEQHLGH-------NAMYWYKQSAEKPPELMFLYNL----KQLIRNETVP-SRFIPECP-DSSKLLLHISAVDPEDSAVYFCASSQ--------------------'}},
                'G': {'human': {'TRGV2*01': 'SSNLEGRTKSVIRQTGSSAEITCDLAEGS------NGYIHWYLHQEGKAPQRLQYYDSY--NSKVVLESGVSPGKYYTYAS-TRNNLRLILRNLIENDSGVYYCATWD--------------------',
                'TRGV2*03': 'SSNLEGRTKSVIRQTGSSAEITCDLAEGS------NGYIHWYLHQEGKAPQRLQYYDSY--NSKVVLESGVSPGKYYTYAS-TRNNLRLILRNLIENDFGVYYCATWD--------------------',
                'TRGV3*01': 'SSNLEGRTKSVTRQTGSSAEITCDLTVTN------TFYIHWYLHQEGKAPQRLLYYDVS--TARDVLESGLSPGKYYTHTP-RRWSWILRLQNLIENDSGVYYCATWD--------------------',
                'TRGV3*02': 'SSNLEGRTKSVTRQTGSSAEITCDLTVTN------TFYIHWYLHQEGKAPQRLLYYDVS--TARDVLESGLSPGKYYTHTP-RRWSWILRLQNLIENDSGVYYCATWD--------------------',
                'TRGV3*03': 'SSNLEGRTKSVTRQTGSSAEITCDLTVTN------TFYIHWYLHQEGKAPQRLLYYDVS--TARDVLESGLSPGKYYTHTP-RRWSWILRLQNLIENDSGVYYCATWD--------------------',
                'TRGV3*04': 'SSNLEGRTKSVTRQTGSSAEITCDLTVTN------TFYIHWYLHQEGKAPQRLLYYDVS--TTRDVLESGLSPGKYYTHTP-RRWSWILRLQNLIENDSGVYYCATWD--------------------',
                'TRGV4*01': 'SSNLEGRTKSVIRQTGSSAEITCDLAEGS------TGYIHWYLHQEGKAPQRLLYYDSY--TSSVVLESGISPGKYDTYGS-TRKNLRMILRNLIENDSGVYYCATWD--------------------',
                'TRGV4*02': 'SSNLEGRTKSVIRQTGSSAEITCDLAEGS------TGYIHWYLHQEGKAPQRLLYYDSY--TSSVVLESGISPGKYDTYGS-TRKNLRMILRNLIENDSGVYYCATWD--------------------',
                'TRGV4*03': 'SSNLEGRTKSVIRQTGSSAEITCDLAEGS------TGYIHWYLHQEGKAPQRLLYYDSY--TSSVVLESGISPGKYDTYGS-TRKNLRMILRNLIENDSGVYYCATWD--------------------',
                'TRGV5*01': 'SSNLEGGTKSVTRPTRSSAEITCDLTVIN------AFYIHWYLHQEGKAPQRLLYYDVS--NSKDVLESGLSPGKYYTHTP-RRWSWILILRNLIENDSGVYYCATWD--------------------',
                'TRGV8*01': 'SSNLEGRTKSVTRPTGSSAVITCDLPVEN------AVYTHWYLHQEGKAPQRLLYYDSY--NSRVVLESGISREKYHTYAS-TGKSLKFILENLIERDSGVYYCATWD--------------------',
                'TRGV9*01': 'AGHLEQPQISSTKTLSKTARLECVVSGITI----SATSVYWYRERPGEVIQFLVSISYD---GTVRKESGIPSGKFEVDRIPETSTSTLTIHNVEKQDIATYYCALWE--------------------',
                'TRGV9*02': 'AGHLEQPQISSTKTLSKTARLECVVSGIKI----SATSVYWYRERPGEVIQFLVSISYD---GTVRKESGIPSGKFEVDRIPETSTSTLTIHNVEKQDIATYYCALWE--------------------'},
                'mouse': {'TRGV1*01': 'LGQLEQTELSVTRETDESAQISCIVSLPYF----SNTAIHWYRQKAKK-FEYLIYVST----NYNQRPLGGKNKKIEASKDFQTSTSTLKINYLKKEDEATYYCAVWI--------------------',
                'TRGV1*02': 'LGQLEQTELSVTRETDESAQISCIVSLPYF----SNTAIHWYRQKAKK-FEYLIYVST----NYNQRPLGGKNKKIEASKDFQTSTSTLKINYLKKEDEATYYCAVWI--------------------',
                'TRGV1*04': 'LGQLEQTELSVTRATDESAQISCIVSLPYF----SNTAIHWYRQKAKK-FEYLIYVST----NYNQRPLGGKNKKIEASKDFQTSTSTLKINYLKKEDEATYYCAVWI--------------------',
                'TRGV2*01': 'LGQLEQTELSVTRETDENVQISCIVYLPYF----SNTAIHWYRQKTNQQFEYLIYVAT----NYNQRPLGGKHKKIEASKDFKSSTSTLEINYLKKEDEATYYCAVWM--------------------',
                'TRGV3*01': 'LGQLEQTELSVTRATDESAQISCIVSLPCF----SNTAIHWYRQKPNQQFEYLIYVET----NYNQQPLGGKNKKIEASKDFQTSTSTLKINYLKKEDEATYYCAVWI--------------------',
                'TRGV5*01': 'DSWISQDQLSFTRRPNKTVHISCKLSGVPL----HNTIVHWYQLKEGEPLRRIFYGS------VKTYKQDKSHSRLEIDEK-DDGTFYLIINNVVTSDEATYYCACWD--------------------',
                'TRGV6*01': 'TSLTSPLGSYVIKRKGNTAFLKCQIKTSVQK---PDAYIHWYQEKPGQRLQRMLCSSSK---ENIVYEKDFSDERYEARTWQSDLSSVLTIHQVREEDTGTYYCACWD--------------------',
                'TRGV6*02': 'SSLTSPLGSYVIKRKGNTAFLKCQIKTSVQK---PDAYIHWYQEKPGQRLQRMLCSSSK---ENIVYEKDFSDERYEARTWQSDLSSVLTIHQVTEEDTGTYYCACWD--------------------',
                'TRGV7*01': 'SSNLEERIMSITKLEGSSAIMTCDTHRTG-------TYIHWYRFQKGRAPEHLLYYNFV--SSTTVVDSRFNSEKYHVYEG-PDKRYKFVLRNVEESDSALYYCASWA--------------------',
                'TRGV7*02': 'SSNLEERIMSITKLEGSSAIMTCDTHRTG-------TYIHWYRFQKGRAPEHLLYYNFV--SSTTVVDSRFNLEKYHVYEG-PDKRYKFVLRNVEESDSALYYCASWA--------------------'}},
                'D': {'human': {'TRAV14/DV4*01': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDP-----SYGLFWYKQPSSGEMIFLIYQGSY--DQQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV14/DV4*02': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDQ-----SYGLFWYKQPSSGEMIFLIYQGSY--DEQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV14/DV4*03': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDP-----SYGLFWYKQPSSGEMIFLIYQGSY--DQQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV14/DV4*05': 'AQKITQTQPGMFVQEKEAVTLDCTYDTSDQ-----SYGLFWYKQPSSGEMIFLIYQGSY--DEQNATE-----GRYSLNFQKARKSANLVISASQLGDSAMYFCAMRE--------------------',
                'TRAV23/DV6*01': 'QQQVKQSPQSLIVQKGGISIINCAYENTA------FDYFPWYQQFPGKGPALLIAIRPD---VSEKKE-----GRFTISFNKSAKQFSLHIMDSQPGDSATYFCAAS---------------------',
                'TRAV23/DV6*02': 'QQQVKQSPQSLIVQKGGIPIINCAYENTA------FDYFPWYQQFPGKGPALLIAIRPD---VSEKKE-----GRFTISFNKSAKQFSLHIMDSQPGDSATYFCAAS---------------------',
                'TRAV23/DV6*05': 'QQQVKQSPQSLIVQKGGISIINCAYENTA------FDYFPWYQQFPGKGPALLIAIRPD---VSEKKE-----GRFTISFNKSAKQFSSHIMDSQPGDSATYFCAAS---------------------',
                'TRAV29/DV5*01': 'DQQVKQNSPSLSVQEGRISILNCDYTNSM------FDYFLWYKKYPAEGPTFLISISSI---KDKNED-----GRFTVFLNKSAKHLSLHIVPSQPGDSAVYFCAAS---------------------',
                'TRAV29/DV5*02': 'DQQVKQNSPSLSVQEGRISILNCDYTNSM------FDYFLWYKKYPAEGPTFLISISSI---KDKNED-----GRFTVFLNKSAKHLSLDIVPSQPGDSAVYFCAAS---------------------',
                'TRAV29/DV5*04': 'DQQVKQNSPSLSVQEGRISILNCDYTNSM------FDYFLWYKKYPAEGPTFLISISSI---KDKNED-----GRFTVFLNKSAKHLSLHIVPSQPGDSAVYFCAAS---------------------',
                'TRAV36/DV7*01': 'EDKVVQSPLSLVVHEGDTVTLNCSYEVTN------FRSLLWYKQEKKAP-TFLFMLTSS---GIEKKS-----GRLSSILDKKELSSILNITATQTGDSAIYLCAVE---------------------',
                'TRAV36/DV7*05': 'EDKVVQSPLSLVVHEGDTVTLNCSYEVTN------FRSLLWYKQEKKAP-TFLFMLTSS---GIEKKS-----GRLSSILDKKELFSILNITATQTGDSAIYLCAVE---------------------',
                'TRAV38-2/DV8*01': 'AQTVTQSQPEMSVQEAETVTLSCTYDTSES-----DYYLFWYKQPPSRQMILVIRQEAY--KQQNATE-----NRFSVNFQKAAKSFSLKISDSQLGDAAMYFCAYRS--------------------',
                'TRDV1*01': 'AQKVTQAQSSVSMPVRKAVTLNCLYETSWW-----SYYIFWYKQLPSKEMIFLIRQG-------SDEQNAKS-GRYSVNFKKAAKSVALTISALQLEDSAKYFCALGE--------------------',
                'TRDV2*01': 'AIELVPEHQTVPVSIGVPATLRCSMKGEAI----GNYYINWYRKTQGNTITFIYREK-------DIYGPGFK-DNFQGDIDIAKNLAVLKILAPSERDEGSYYCACDT--------------------',
                'TRDV2*03': 'AIELVPEHQTVPVSIGVPATLRCSMKGEAI----GNYYINWYRKTQGNTMTFIYREK-------DIYGPGFK-DNFQGDIDIAKNLAVLKILAPSERDEGSYYCACDT--------------------',
                'TRDV3*01': 'CDKVTQSSPDQTVASGSEVVLLCTYDTVYS-----NPDLFWYRIRPDYSFQFVFYGDN----SRSEGADFTQ-GRFSVKHILTQKAFHLVISPVRTEDSATYYCAF----------------------',
                'TRDV3*02': 'CDKVTQSSPDQTVASGSEVVLLCTYDTVYS-----NPDLFWYWIRPDYSFQFVFYGDN----SRSEGADFTQ-GRFSVKHILTQKAFHLVISPVRTEDSATYYCAF----------------------',
                'TRDV3*03': 'CDKVTQSSPDQTVASGSEVVLLCTYNTVYS-----NPDLFWYRIRPDYSFQFVFYGDN----SRSEGADFTQ-GRFSVKHILTQKAFHLVISPVRTEDSATYYCAF----------------------'},
                'mouse': {'TRAV13-4/DV7*01': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TQLQWFYQRPGGSLVSLLYNPS-----GTKHT-----GRLTSTTVTKERRSSLHISSSQITDSGTYFCAME---------------------',
                'TRAV13-4/DV7*02': 'GQQVQQSPASLVLQEGENAELQCNFSSTA-------TRLQWFYQRPGGSLVSLLSNPS-----GTKHT-----GRLTSTTVTKERRGSLHISSSQITDSGTYLCAME---------------------',
                'TRAV14D-3/DV8*01': 'QQQVRQSSQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GRFTIFFNKREKNLSLHIKDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*02': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GRFTIFFNKREKKLSLHITDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*03': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLIAIRSV---SDKKED-----GGFTIFFNKREKNLSLHIKDSQPGDSATYFCAAS---------------------',
                'TRAV14D-3/DV8*08': 'QQQVRQSPQSLTVWEGETAILNCSYENSA------FDYFPWYQQFPGEGPALLISILSV---SDKKED-----GRFTIFFNKREKKLSLHIADSQPGDSATYFCAAS---------------------',
                'TRAV15-1/DV6-1*01': 'AQKVIQVWSTTSRQEGEKLTLDCSYKTSQV-----LYHLFWYKHLLSGEMVLLIRQMPS--TIAIERS-----GRYSVVFQKSRKSISLVISTLQPDDSGKYFCALWE--------------------',
                'TRAV15-2/DV6-2*01': 'AQKVTQVQSTGSSQWG-EVTLHCSYETSEY-----FYVILWYKQLFSGEMVFLIYQTSF--DTQNQRN-----SRYSVVFQKSLKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15-2/DV6-2*02': 'AQRVTQVQSTGSSQWG-EVTLDCSYETSEY-----SYLILWYRQLFSGEMVFLIYQPSF--DTQNQRS-----GHYSVVFQKSFKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15D-1/DV6D-1*01': 'AQKVIQVWSTPSRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQMSS--STAKERS-----GRYSVVFQKSLKSISLVISALQPDDSGKYFCALWE--------------------',
                'TRAV15D-1/DV6D-1*02': 'AQKVIQVWSTASRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQMSS--STAKERS-----GRYSVVFQKSLKSISLVISALQPDDSGKYFCALWE--------------------',
                'TRAV15D-1/DV6D-1*07': 'AEKVIQVWSTASRQEGEELTLDCSYETSQV-----LYHLFWYKHLLSGEMVFLIRQTSS--STAKERS-----GRYSVVFQKSLKSISLIISALQPDDSGKYFCALWE--------------------',
                'TRAV15D-2/DV6D-2*01': 'AQRVTQVQPTGSSQWGEEVTLDCSYETSEY-----FYCIIWYRQLFSGEMVFLIYQTSF--DTQNQRN-----GRYSVVFQKSLKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV15D-2/DV6D-2*03': 'AQRVTQVQPTGSSQWGEEVTLDCSYETSEY-----FYRIFWYRQLFSGEMVFLIYQPSF--DTQNQRS-----GRYSVVFQKSFKSISLVISASQPEDSGTYFCALSE--------------------',
                'TRAV16D/DV11*01': 'AQKVTQTQTSISVMEKTTVTMDCVYETQDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATV-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV16D/DV11*02': 'AQKVTQTQTSISVMEKTTVTMDCVYETQDS-----SYFLFWYKQTASGEIVFLIRQDSY--KKENATV-----GHYSLNFQKPKSSIGLIITATQIEDSAVYFCAMRE--------------------',
                'TRAV21/DV12*01': 'DAKTTQ-PDSMESTEGETVHLPCSHATISG-----NEYIYWYRQVPLQGPEYVTHGLQ-----QNTTN-----SMAFLAIASDRKSSTLILTHVSLRDAAVYHCILRV--------------------',
                'TRAV4-4/DV10*01': 'GDQVEQSPSALSLHEGTDSALRCNFTTTM-------RSVQWFRQNSRGSLISLFYLAS-----GTKEN-----GRLKSAFDSKERYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV4-4/DV10*02': 'GDQVEQSPSALSLHEGTGSALRCNFTTTM-------RAVQWFQQNSRGSLINLFYLAS-----GTKEN-----GRLKSTFNSKESYSTLHIRDAQLEDSGTYFCAAE---------------------',
                'TRAV6-7/DV9*01': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPALFWYVQYPGEGPQFLFRASRD---KEKGSS-----RGFEATYNKETTSFHLQKASVQESDSAVYYCALG---------------------',
                'TRAV6-7/DV9*04': 'GDSVTQTEGQVALSEEDFLTIHCNYSASG------YPTLFWYVQYPGEGPQLLFRASRD---KEKGSS-----RGFEATYDKGTTSFHLRKASVQESDSAVYYCALS---------------------',
                'TRDV1*01': 'TQMLHQSPQSLTIQEGDEVTMSCNLSTSL-------YALLWYRQGDDGSLVSLVTLQ-------KGGDEKSK-DKITAKLDKKMQQSSLQIQASQPSHSGTYLCGGK---------------------',
                'TRDV1*02': 'TQMLHQSPQSLTIQEGDEVTMSCNLSTSL-------YALLWYRQGDDGSLVSLVTLQ-------KGGDEKSK-DKITANLDKKMQQSSLWIQASQPSHSGTYLCGGK---------------------',
                'TRDV2-1*01': 'AQTVSQHQQEKSVQVAESATLDCTYDTSDT-----NYLLFWYKQQGGQVTLVIRQEA-------YKQYNAME-NRFSVNFQKAAKSFSLEISDSQLGDAATYFCALRG--------------------',
                'TRDV2-1*02': 'AQTVSQPQKKKSVQVAESATLDCTYDTSDT-----NYLLFWYKQQGGQVTLVILQEA-------YKQYNATL-NRFSVNFQKAAKSFSLEISDSQLGDAATYFCALRG--------------------',
                'TRDV2-2*01': 'AQTVSQPQKKKSVQVAESATLDCTYDTSDT-----NYLLFWYKQQGGQVTLVILQEA-------YKQYNATL-NRFSVNFQKAAKSFSLEISDSQLGDAATYFCALME--------------------',
                'TRDV4*01': 'DVYLEPVAKTFTVVAGDPASFYCTVTGGDM----KNYHMSWYKKNGTNALFLVYKLN-------SNSTDGGK-SNLKGKINISKNQFILDIQKATMKDAGTYYCGSDI--------------------',
                'TRDV5*01': 'CITLTQSSTDQTVASGTEVTLLCTYNADSP-----NPDLFWYRKRPDRSFQFILYRDD----TSSHDADFVQ-GRFSVKHSKANRTFHLVISPVSLEDSATYYCASGY--------------------',
                'TRDV5*02': 'CITLTQSSTDQTVASGTEVTLLCTYNADSP-----NPDLFWYRKRPDRSFQFILYRDD----TSSHDADFVQ-GRFSVKHSKANRTFHLVISPVSLEDSATYYCASGY--------------------',
                'TRDV5*03': 'CITLTQSSTDQTVASGTEVTLLCTYNADSP-----NPDLFWYRKRPDRSFQFILYRDD----TSSHDADFVQ-GRFSVKHSKANRTFHLVISPVSLEDSATYYCASGY--------------------'}}}}


In [14]:
from pprint import pprint

In [22]:
from bioomics import ProcessJson
ProcessJson.save_json(all_germlines, '/home/yuan/bio/ANARCI/src/all_germlines.json')

True

In [24]:
data = ProcessJson.load_json('/home/yuan/bio/ANARCI/src/all_germlines.json')
data

{'J': {'A': {'human': {'TRAJ10*01': '------------------------------------------------------------------------------------------------------------------KLTFGTGTQLKVEL',
    'TRAJ11*01': '------------------------------------------------------------------------------------------------------------------TLTFGKGTMLLVSP',
    'TRAJ12*01': '------------------------------------------------------------------------------------------------------------------KLIFGSGTRLLVRP',
    'TRAJ13*01': '------------------------------------------------------------------------------------------------------------------KVTFGIGTKLQVIP',
    'TRAJ13*02': '------------------------------------------------------------------------------------------------------------------KVTFGTGTKLQVIP',
    'TRAJ14*01': '------------------------------------------------------------------------------------------------------------------TFIFGSGTRLSVKP',
    'TRAJ15*01': '---------------------------------------------------------------------